# Team Bahlil Notebook

1. Wisnu Daarmawan (Leader)
2. Clarosa Canara
3. Annisa Wafa Jayyida

## SECTION 1 - Environment Setup & Library Import

In [ ]:
import os
import sys
import glob
import math
import random
import warnings
import subprocess
from pathlib import Path
from typing import Optional

warnings.filterwarnings("ignore")

# Global Configuration
TARGET = "property_organic_content"
ID_COL = "sample_id"
N_SPLITS = 5
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 200
FAST_MODE = False

# Training size control
MAIN_N_ESTIMATORS = 500 if FAST_MODE else 3000
BASELINE_N_ESTIMATORS = 300 if FAST_MODE else 1000


def seed_everything(seed: int = RANDOM_STATE) -> None:
    """Set seeds"""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    import numpy as np
    np.random.seed(seed)

seed_everything(RANDOM_STATE)


def install_if_missing(package_name: str, import_name: Optional[str] = None) -> None:
    """Install a package if import fails"""
    import importlib
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

# Required stack
install_if_missing("catboost")
install_if_missing("lightgbm")
install_if_missing("xgboost")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
from lightgbm import LGBMRegressor
import xgboost as xgb
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


def rmse(y_true, y_pred) -> float:
    """RMSE"""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print("Environment ready")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"lightgbm: {lgb.__version__}")
print(f"xgboost: {xgb.__version__}")

## SECTION 2 - Load Data

Di bagian ini, notebook dibuat supaya bisa jalan di berbagai tempat.

Terdapat pengecekan di awal supaya kalau ada file yang salah, kurang, atau kolom pentingnya nggak ada, akan muncul error.

In [ ]:
def find_csv_file(filename: str) -> Optional[str]:
    """Find CSV file across common Kaggle, Colab, and local"""
    search_patterns = [
        filename,
        f"./{filename}",
        f"/content/{filename}",
        f"/kaggle/working/{filename}",
        f"/kaggle/input/**/{filename}",
    ]
    matches = []
    for pattern in search_patterns:
        matches.extend(glob.glob(pattern, recursive=True))
    matches = [m for m in matches if os.path.isfile(m)]
    if not matches:
        return None
    # Prefer local/current files, then /content, then /kaggle/working, then /mnt/data, then /kaggle/input.
    priority = {".": 0, "/content": 1, "/kaggle/working": 2, "/kaggle/input": 3}
    def score(path: str):
        p = os.path.abspath(path)
        for key, value in priority.items():
            if p == os.path.abspath(filename) or p.startswith(os.path.abspath(key)):
                return value
        return 99
    return sorted(set(matches), key=score)[0]


def maybe_colab_upload(required_files: list[str]) -> None:
    """Upload files for google collab"""
    missing = [f for f in required_files if find_csv_file(f) is None]
    if not missing:
        return
    try:
        from google.colab import files
        print("Missing files detected:", missing)
        print("Please upload train.csv, test.csv, and sample_submission.csv")
        uploaded = files.upload()
        print("Uploaded:", list(uploaded.keys()))
    except Exception as exc:
        print("Not running in Colab or upload is unavailable.")
        print("Missing files:", missing)
        raise FileNotFoundError(f"Required files not found: {missing}") from exc


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load train, test, and sample submission"""
    required_files = ["train.csv", "test.csv", "sample_submission.csv"]
    maybe_colab_upload(required_files)

    paths = {name: find_csv_file(name) for name in required_files}
    if any(v is None for v in paths.values()):
        raise FileNotFoundError(f"Could not locate all required files. Found paths: {paths}")

    print("Resolved paths:")
    for k, v in paths.items():
        print(f"- {k}: {v}")

    train = pd.read_csv(paths["train.csv"])
    test = pd.read_csv(paths["test.csv"])
    sample_submission = pd.read_csv(paths["sample_submission.csv"])

    # Strict validation
    assert TARGET in train.columns, f"Train file must contain target column: {TARGET}"
    assert ID_COL in train.columns, f"Train file must contain ID column: {ID_COL}"
    assert ID_COL in test.columns, f"Test file must contain ID column: {ID_COL}"
    assert ID_COL in sample_submission.columns, f"Sample submission must contain ID column: {ID_COL}"
    assert TARGET in sample_submission.columns, f"Sample submission must contain target column: {TARGET}"
    assert len(test) == len(sample_submission), "Test and sample_submission must have the same number of rows"
    assert train[ID_COL].is_unique, f"Train {ID_COL} must be unique"
    assert test[ID_COL].is_unique, f"Test {ID_COL} must be unique"
    assert sample_submission[ID_COL].is_unique, f"Sample submission {ID_COL} must be unique"
    
    if not test[ID_COL].equals(sample_submission[ID_COL]):
        print("Warning: test ID order differs from sample_submission. Final output will follow sample_submission order.")

    print("\nShapes:")
    print(f"train: {train.shape}")
    print(f"test: {test.shape}")
    print(f"sample_submission: {sample_submission.shape}")

    return train, test, sample_submission

train_raw, test_raw, sample_submission = load_data()

print("\nTrain head:")
display(train_raw.head())
print("\nTest head:")
display(test_raw.head())
print("\nSample submission head:")
display(sample_submission.head())

## SECTION 3 - Initial Data Audit

Bagian ini akan mengecek dataset shape, data types, duplicated rows, duplicated IDs, missing values, constant columns, high-missing columns, and categorical/numerical feature groups.

In [ ]:
def get_feature_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in [ID_COL, TARGET]]


def missing_summary(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    """Return side by side missing value summary for train and test"""
    all_cols = sorted(set(train.columns).union(test.columns))
    rows = []
    for col in all_cols:
        tr_miss = train[col].isna().sum() if col in train.columns else np.nan
        te_miss = test[col].isna().sum() if col in test.columns else np.nan
        rows.append({
            "column": col,
            "train_missing_count": tr_miss,
            "train_missing_pct": tr_miss / len(train) if col in train.columns else np.nan,
            "test_missing_count": te_miss,
            "test_missing_pct": te_miss / len(test) if col in test.columns else np.nan,
        })
    return pd.DataFrame(rows).sort_values("train_missing_pct", ascending=False)


def audit_dataframe(df: pd.DataFrame, name: str, has_target: bool = True) -> dict:
    """Print and return key audit findings"""
    feature_cols = get_feature_columns(df)
    categorical_cols = df[feature_cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
    duplicated_rows = int(df.duplicated().sum())
    duplicated_ids = int(df[ID_COL].duplicated().sum()) if ID_COL in df.columns else None

    print(f"===== {name} audit =====")
    print(f"Shape: {df.shape}")
    print(f"Duplicated rows: {duplicated_rows}")
    print(f"Duplicated {ID_COL}: {duplicated_ids}")
    print(f"Categorical features: {len(categorical_cols)}")
    print(f"Numerical features: {len(numerical_cols)}")
    print(f"Constant feature candidates: {constant_cols}")

    return {
        "feature_cols": feature_cols,
        "categorical_cols": categorical_cols,
        "numerical_cols": numerical_cols,
        "constant_cols": constant_cols,
        "duplicated_rows": duplicated_rows,
        "duplicated_ids": duplicated_ids,
    }

train_audit = audit_dataframe(train_raw, "train", has_target=True)
test_audit = audit_dataframe(test_raw, "test", has_target=False)

print("\nData types:")
display(train_raw.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))

missing_df = missing_summary(train_raw, test_raw)
print("\nMissing value summary:")
display(missing_df)

high_missing_cols = missing_df.loc[missing_df["train_missing_pct"].fillna(0) >= 0.40, "column"].tolist()
high_missing_cols = [c for c in high_missing_cols if c not in [TARGET]]
print("\nHigh-missing feature candidates >= 40% missing:")
print(high_missing_cols)

constant_cols = sorted(set(train_audit["constant_cols"]).intersection(set(test_audit["constant_cols"])))
print("\nConstant columns in both train and test, safe candidates to drop:")
print(constant_cols)

categorical_cols_raw = train_audit["categorical_cols"]
numerical_cols_raw = train_audit["numerical_cols"]
print("\nRaw categorical columns:", categorical_cols_raw)
print("Raw numerical columns:", numerical_cols_raw)

## SECTION 4 - Exploratory Data Analysis
Difokuskan pada target distribution, missingness, numeric correlations, categorical relationships, and train & test distribution checks.

In [ ]:
def plot_target_distribution(y: pd.Series) -> None:
    print("Target summary:")
    display(y.describe().to_frame().T)
    print(f"Skewness: {y.skew():.4f}")

    plt.figure(figsize=(8, 4))
    plt.hist(y.dropna(), bins=50)
    plt.title("Target Distribution")
    plt.xlabel(TARGET)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 2.5))
    plt.boxplot(y.dropna(), vert=False)
    plt.title("Target Boxplot")
    plt.xlabel(TARGET)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(np.log1p(y.dropna()), bins=50)
    plt.title("log1p(Target) Distribution")
    plt.xlabel(f"log1p({TARGET})")
    plt.ylabel("Frequency")
    plt.show()

plot_target_distribution(train_raw[TARGET])

In [ ]:
# Categorical cardinality and target summaries
categorical_cardinality = []
for c in categorical_cols_raw:
    categorical_cardinality.append({
        "column": c,
        "n_unique_train": train_raw[c].nunique(dropna=False),
        "n_unique_test": test_raw[c].nunique(dropna=False) if c in test_raw.columns else np.nan,
        "top_train": train_raw[c].value_counts(dropna=False).index[0] if c in train_raw.columns else np.nan,
    })
cat_cardinality_df = pd.DataFrame(categorical_cardinality).sort_values("n_unique_train", ascending=False)
print("Categorical cardinality:")
display(cat_cardinality_df)

important_cat_cols = [
    "source_id", "geo_zone_macro", "geo_zone_meso", "geo_zone_micro",
    "biome", "land_cover_type", "parent_rock_type", "sampling_strategy",
    "has_band_A_spectrum", "has_band_B_spectrum", "sampling_depth_cm",
]
important_cat_cols = [c for c in important_cat_cols if c in train_raw.columns]

for c in important_cat_cols:
    summary = train_raw.groupby(c, dropna=False)[TARGET].agg(["count", "mean", "median", "std"]).sort_values("median", ascending=False)
    print(f"\nTarget by {c}:")
    display(summary.head(20))

In [ ]:
# Numerical correlation with target
num_cols_for_corr = [c for c in numerical_cols_raw if c != TARGET and c in train_raw.columns]
corr_with_target = train_raw[num_cols_for_corr + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
print("Top absolute correlations with target:")
display(corr_with_target.head(30).to_frame("corr_with_target"))

# Compact correlation heatmap for top numeric features
heatmap_cols = corr_with_target.head(20).index.tolist() + [TARGET]
if len(heatmap_cols) > 1:
    corr_matrix = train_raw[heatmap_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    plt.imshow(corr_matrix, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
    plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
    plt.title("Correlation Heatmap - Top Numeric Features")
    plt.tight_layout()
    plt.show()

In [ ]:
# Missing value visualization and train & test comparison
missing_plot_df = missing_df[(missing_df["column"] != TARGET)].copy()
missing_plot_df = missing_plot_df.sort_values("train_missing_pct", ascending=False).head(30)

plt.figure(figsize=(10, 5))
plt.bar(missing_plot_df["column"], missing_plot_df["train_missing_pct"])
plt.xticks(rotation=90)
plt.ylabel("Train Missing Percentage")
plt.title("Top Missing Features in Train")
plt.tight_layout()
plt.show()

print("Train-test missing percentage comparison:")
display(missing_df.sort_values("train_missing_pct", ascending=False).head(40))

# Categorical train & test distribution comparison using total variation distance
cat_shift_rows = []
for c in important_cat_cols:
    train_dist = train_raw[c].astype(str).fillna("Missing").value_counts(normalize=True, dropna=False)
    test_dist = test_raw[c].astype(str).fillna("Missing").value_counts(normalize=True, dropna=False)
    cats = sorted(set(train_dist.index).union(set(test_dist.index)))
    tvd = 0.5 * sum(abs(train_dist.get(cat, 0) - test_dist.get(cat, 0)) for cat in cats)
    cat_shift_rows.append({"column": c, "train_unique": len(train_dist), "test_unique": len(test_dist), "total_variation_distance": tvd})
cat_shift_df = pd.DataFrame(cat_shift_rows).sort_values("total_variation_distance", ascending=False)
print("Categorical train-test distribution shift summary:")
display(cat_shift_df)

## SECTION 5 - Table and Graph

In [ ]:
research_assets = {}

def grouped_target_summary(df: pd.DataFrame, group_cols: list[str], min_count: int = 10) -> pd.DataFrame:
    available = [c for c in group_cols if c in df.columns]
    if not available:
        return pd.DataFrame()
    out = (
        df.groupby(available, dropna=False)[TARGET]
        .agg(count="count", mean="mean", median="median", std="std", min="min", max="max")
        .reset_index()
    )
    return out[out["count"] >= min_count].sort_values("median", ascending=False)

# Geographic distribution evidence
for c in ["geo_zone_macro", "geo_zone_meso", "geo_zone_micro"]:
    if c in train_raw.columns:
        research_assets[f"target_by_{c}"] = grouped_target_summary(train_raw, [c], min_count=5)
        print(f"Target distribution evidence by {c}:")
        display(research_assets[f"target_by_{c}"].head(20))

# Ecosystem and land cover evidence
for c in ["biome", "land_cover_type", "parent_rock_type"]:
    if c in train_raw.columns:
        research_assets[f"target_by_{c}"] = grouped_target_summary(train_raw, [c], min_count=5)
        print(f"Target distribution evidence by {c}:")
        display(research_assets[f"target_by_{c}"].head(20))

# Low acidity + low CEC evidence
if {"property_acidity_index", "cation_exchange_capacity"}.issubset(train_raw.columns):
    acidity_q25 = train_raw["property_acidity_index"].quantile(0.25)
    cec_mean = train_raw["cation_exchange_capacity"].mean()
    low_acidity_low_cec = train_raw[
        (train_raw["property_acidity_index"] < acidity_q25) &
        (train_raw["cation_exchange_capacity"] < cec_mean)
    ].copy()
    research_assets["low_acidity_low_cec"] = low_acidity_low_cec
    print(f"Low acidity threshold Q25: {acidity_q25:.4f}")
    print(f"Low CEC threshold mean: {cec_mean:.4f}")
    print(f"Filtered rows: {len(low_acidity_low_cec)}")
    for c in ["biome", "land_cover_type", "geo_zone_macro"]:
        if c in low_acidity_low_cec.columns and len(low_acidity_low_cec) > 0:
            print(f"Low acidity + low CEC target summary by {c}:")
            display(grouped_target_summary(low_acidity_low_cec, [c], min_count=3).head(20))

# Outlier combinations: land_cover_type + geo_zone_macro
if {"land_cover_type", "geo_zone_macro"}.issubset(train_raw.columns):
    combo = grouped_target_summary(train_raw, ["land_cover_type", "geo_zone_macro"], min_count=10)
    if not combo.empty:
        combo_mean = combo["mean"].mean()
        combo_std = combo["mean"].std(ddof=0)
        combo["z_score_mean"] = (combo["mean"] - combo_mean) / combo_std if combo_std > 0 else 0
        combo["is_outlier_abs_z_gt_2"] = combo["z_score_mean"].abs() > 2
        research_assets["land_cover_geo_outliers"] = combo.sort_values("z_score_mean", key=lambda s: s.abs(), ascending=False)
        print("Outlier evidence for land_cover_type + geo_zone_macro:")
        display(research_assets["land_cover_geo_outliers"].head(30))

# High correlation pairs evidence
numeric_for_corr = train_raw[get_feature_columns(train_raw) + [TARGET]].select_dtypes(include=[np.number])
corr = numeric_for_corr.corr(numeric_only=True).abs()
pairs = []
cols = corr.columns.tolist()
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        val = corr.loc[c1, c2]
        if pd.notna(val) and val >= 0.80:
            pairs.append({"feature_1": c1, "feature_2": c2, "abs_corr": val})
high_corr_pairs = pd.DataFrame(pairs).sort_values("abs_corr", ascending=False) if pairs else pd.DataFrame(columns=["feature_1", "feature_2", "abs_corr"])
research_assets["high_corr_pairs_abs_gt_0_8"] = high_corr_pairs
print("High-correlation pairs, abs(corr) >= 0.8:")
display(high_corr_pairs.head(50))

## SECTION 6 - Preprocessing Strategy

In [ ]:
# Safe constant columns from train only. A column constant in train cannot provide useful supervised splits.
DROP_CONSTANT_COLUMNS = True
constant_cols_train = train_audit["constant_cols"]
constant_cols_train = [c for c in constant_cols_train if c in test_raw.columns]

base_drop_cols = [ID_COL, TARGET]
if DROP_CONSTANT_COLUMNS:
    base_drop_cols += constant_cols_train

base_feature_cols = [c for c in train_raw.columns if c not in base_drop_cols]
print("Base dropped columns:", base_drop_cols)
print("Number of base features:", len(base_feature_cols))

X_base = train_raw[base_feature_cols].copy()
y = train_raw[TARGET].copy()
X_test_base = test_raw[base_feature_cols].copy()

base_categorical_cols = X_base.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
base_numerical_cols = X_base.select_dtypes(include=[np.number]).columns.tolist()

print("Base categorical columns:", base_categorical_cols)
print("Base numerical columns:", base_numerical_cols)

assert ID_COL not in X_base.columns, "ID column leaked into features"
assert TARGET not in X_base.columns, "Target column leaked into features"
assert TARGET not in X_test_base.columns, "Target column leaked into test features"

## SECTION 7 - Feature Engineering

Feature engineering is performed on combined train-test features without using the target, preventing target leakage while keeping train and test transformations consistent.

Included features:

- Missing indicators and row-level missing counts
- Spectral block statistics for Band A and Band B
- Soil particle and cation chemistry ratios
- Geography bins and geography-combination categorical features
- Frequency encoding for important categorical columns

In [ ]:
def safe_divide(a, b):
    """Vectorized safe division that preserves NaN when division is not meaningful."""
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    result = a / b.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)


def add_if_columns_exist(df: pd.DataFrame, new_col: str, cols: list[str], operation) -> None:
    """Add engineered feature only if required source columns exist."""
    if all(c in df.columns for c in cols):
        df[new_col] = operation(df)


def create_features(df: pd.DataFrame, categorical_frequency_cols: Optional[list[str]] = None) -> pd.DataFrame:
    """Create leakage-safe features. This function must not use the target."""
    df = df.copy()
    original_cols = [c for c in df.columns if c not in [ID_COL, TARGET]]
    original_numeric_cols = df[original_cols].select_dtypes(include=[np.number]).columns.tolist()

    # Missing indicators and counts
    original_missing_mask = df[original_cols].isna()
    df["original_total_missing_count"] = original_missing_mask.sum(axis=1)
    df["original_numerical_missing_count"] = df[original_numeric_cols].isna().sum(axis=1) if original_numeric_cols else 0

    missing_indicator_cols = [c for c in original_cols if df[c].isna().any()]
    priority_missing_cols = [
        "property_acidity_index", "cation_Na", "latitude", "longitude",
        "cation_Ca", "cation_Mg", "cation_exchange_capacity",
    ]
    missing_indicator_cols = sorted(set(missing_indicator_cols).union([c for c in priority_missing_cols if c in df.columns]))
    for c in missing_indicator_cols:
        df[f"missing_{c}"] = df[c].isna().astype(int)

    # Spectral block features
    spectral_a_cols = [c for c in df.columns if c.startswith("spectral_band_A_PC_")]
    spectral_b_cols = [c for c in df.columns if c.startswith("spectral_band_B_PC_")]

    def add_spectral_stats(prefix: str, cols: list[str]) -> None:
        if not cols:
            return
        block = df[cols].apply(pd.to_numeric, errors="coerce")
        df[f"{prefix}_mean"] = block.mean(axis=1)
        df[f"{prefix}_std"] = block.std(axis=1)
        df[f"{prefix}_min"] = block.min(axis=1)
        df[f"{prefix}_max"] = block.max(axis=1)
        df[f"{prefix}_median"] = block.median(axis=1)
        df[f"{prefix}_abs_mean"] = block.abs().mean(axis=1)
        df[f"{prefix}_l2_norm"] = np.sqrt((block.fillna(0) ** 2).sum(axis=1))
        df[f"{prefix}_missing_count"] = block.isna().sum(axis=1)
        df[f"{prefix}_available_count"] = block.notna().sum(axis=1)

    add_spectral_stats("spectral_band_A", spectral_a_cols)
    add_spectral_stats("spectral_band_B", spectral_b_cols)
    df["missing_band_B"] = df[spectral_b_cols].isna().all(axis=1).astype(int) if spectral_b_cols else 0
    df["missing_band_A"] = df[spectral_a_cols].isna().all(axis=1).astype(int) if spectral_a_cols else 0

    # Soil particle features
    add_if_columns_exist(df, "particle_fine_minus_coarse", ["property_particle_fine", "property_particle_coarse"],
                         lambda x: x["property_particle_fine"] - x["property_particle_coarse"])
    add_if_columns_exist(df, "particle_fine_ratio", ["property_particle_fine", "property_particle_coarse"],
                         lambda x: safe_divide(x["property_particle_fine"], x["property_particle_fine"] + x["property_particle_coarse"]))
    add_if_columns_exist(df, "particle_coarse_ratio", ["property_particle_fine", "property_particle_coarse"],
                         lambda x: safe_divide(x["property_particle_coarse"], x["property_particle_fine"] + x["property_particle_coarse"]))

    # Soil chemistry and cation features
    cation_cols = [c for c in ["cation_Ca", "cation_Mg", "cation_Na"] if c in df.columns]
    if cation_cols:
        df["cation_total"] = df[cation_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)

    add_if_columns_exist(df, "Ca_Mg_ratio", ["cation_Ca", "cation_Mg"], lambda x: safe_divide(x["cation_Ca"], x["cation_Mg"]))
    add_if_columns_exist(df, "Ca_CEC_ratio", ["cation_Ca", "cation_exchange_capacity"], lambda x: safe_divide(x["cation_Ca"], x["cation_exchange_capacity"]))
    add_if_columns_exist(df, "Mg_CEC_ratio", ["cation_Mg", "cation_exchange_capacity"], lambda x: safe_divide(x["cation_Mg"], x["cation_exchange_capacity"]))
    add_if_columns_exist(df, "Na_CEC_ratio", ["cation_Na", "cation_exchange_capacity"], lambda x: safe_divide(x["cation_Na"], x["cation_exchange_capacity"]))
    add_if_columns_exist(df, "fine_x_CEC", ["property_particle_fine", "cation_exchange_capacity"], lambda x: x["property_particle_fine"] * x["cation_exchange_capacity"])
    add_if_columns_exist(df, "acidity_x_CEC", ["property_acidity_index", "cation_exchange_capacity"], lambda x: x["property_acidity_index"] * x["cation_exchange_capacity"])

    # Geography features
    if {"latitude", "longitude"}.issubset(df.columns):
        df["lat_lon_missing"] = (df["latitude"].isna() | df["longitude"].isna()).astype(int)
        df["lat_lon_interaction"] = pd.to_numeric(df["latitude"], errors="coerce") * pd.to_numeric(df["longitude"], errors="coerce")
        df["lat_abs"] = pd.to_numeric(df["latitude"], errors="coerce").abs()
        df["lon_abs"] = pd.to_numeric(df["longitude"], errors="coerce").abs()
        df["lat_bin"] = pd.cut(pd.to_numeric(df["latitude"], errors="coerce"), bins=10, labels=False, duplicates="drop")
        df["lon_bin"] = pd.cut(pd.to_numeric(df["longitude"], errors="coerce"), bins=10, labels=False, duplicates="drop")

    def combine_cat(new_col: str, cols: list[str]) -> None:
        if all(c in df.columns for c in cols):
            values = [df[c].astype(str).fillna("Missing") for c in cols]
            combined = values[0]
            for v in values[1:]:
                combined = combined + "__" + v
            df[new_col] = combined

    combine_cat("geo_macro_meso", ["geo_zone_macro", "geo_zone_meso"])
    combine_cat("geo_meso_micro", ["geo_zone_meso", "geo_zone_micro"])
    combine_cat("geo_macro_biome", ["geo_zone_macro", "biome"])
    combine_cat("biome_land_cover", ["biome", "land_cover_type"])
    combine_cat("rock_biome", ["parent_rock_type", "biome"])
    combine_cat("land_cover_geo_macro", ["land_cover_type", "geo_zone_macro"])

    # Frequency encoding
    categorical_frequency_cols = categorical_frequency_cols or []
    for c in categorical_frequency_cols:
        if c in df.columns:
            key = df[c].astype(str).fillna("Missing")
            freq = key.value_counts(dropna=False, normalize=True)
            count = key.value_counts(dropna=False)
            df[f"{c}_freq"] = key.map(freq).astype(float)
            df[f"{c}_count"] = key.map(count).astype(float)

    # Clean infinities
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

# Feature engineering is fit on combined train+test features only, without target.
combined_features = pd.concat([X_base, X_test_base], axis=0, ignore_index=True)
frequency_cols = [
    "source_id", "geo_zone_macro", "geo_zone_meso", "geo_zone_micro",
    "land_cover_type", "biome", "parent_rock_type", "sampling_strategy",
    "has_band_A_spectrum", "has_band_B_spectrum",
]
frequency_cols = [c for c in frequency_cols if c in combined_features.columns]

combined_fe = create_features(combined_features, categorical_frequency_cols=frequency_cols)
X_fe = combined_fe.iloc[:len(train_raw)].reset_index(drop=True)
X_test_fe = combined_fe.iloc[len(train_raw):].reset_index(drop=True)

# Final feature columns
feature_cols = X_fe.columns.tolist()
cat_features = X_fe.select_dtypes(include=["object", "category"]).columns.tolist()
num_features = [c for c in feature_cols if c not in cat_features]

print("Engineered train shape:", X_fe.shape)
print("Engineered test shape:", X_test_fe.shape)
print("Categorical features after FE:", len(cat_features))
print("Numerical features after FE:", len(num_features))
print("New feature count:", X_fe.shape[1] - X_base.shape[1])

assert X_fe.shape[0] == len(train_raw)
assert X_test_fe.shape[0] == len(test_raw)
assert ID_COL not in X_fe.columns
assert TARGET not in X_fe.columns

In [ ]:
# ==================================================
# SECTION 7B - Domain Priority & Robust Feature Set
# ==================================================
# Goal:
# 1. Prioritize features based on soil-domain relevance.
# 2. Create a robust feature set to test whether source_id / micro-geography / high-cardinality features overfit.
# 3. No external data is used. All features are derived only from official train/test columns.

def classify_feature_domain(feature_name: str) -> str:
    f = feature_name.lower()

    if "spectral_band" in f:
        return "1_spectral_core"
    if any(k in f for k in ["cation", "cec", "acidity", "ca_", "mg_", "na_"]):
        return "2_soil_chemistry"
    if any(k in f for k in ["particle", "fine", "coarse"]):
        return "3_soil_texture"
    if any(k in f for k in ["geo", "biome", "land_cover", "rock", "latitude", "longitude", "lat_", "lon_"]):
        return "4_ecology_geography"
    if any(k in f for k in ["missing", "available_count", "total_missing"]):
        return "5_missing_pattern"
    if "source_id" in f:
        return "6_source_lab_risk"
    return "7_other"

feature_domain_df = pd.DataFrame({
    "feature": X_fe.columns,
    "domain_group": [classify_feature_domain(c) for c in X_fe.columns],
    "dtype": [str(X_fe[c].dtype) for c in X_fe.columns],
    "missing_pct_train": [X_fe[c].isna().mean() * 100 for c in X_fe.columns],
    "nunique_train": [X_fe[c].nunique(dropna=False) for c in X_fe.columns],
}).sort_values(["domain_group", "feature"])

print("Feature domain summary:")
display(
    feature_domain_df
    .groupby("domain_group")
    .agg(
        n_features=("feature", "count"),
        avg_missing_pct=("missing_pct_train", "mean"),
        max_nunique=("nunique_train", "max")
    )
    .reset_index()
)

display(feature_domain_df.head(30))


# --------------------------------------------------
# Robust feature set
# --------------------------------------------------
# Rationale:
# - source_id can represent lab/source bias.
# - geo_zone_micro and very detailed geo-combinations can overfit local CV.
# - We keep core features: spectral, soil chemistry, texture, macro/meso geography, biome, land cover.

protected_categorical_features = [
    "geo_zone_macro",
    "geo_zone_meso",
    "biome",
    "land_cover_type",
    "parent_rock_type",
    "sampling_strategy",
    "has_band_A_spectrum",
    "has_band_B_spectrum",
    "geo_macro_biome",
    "biome_land_cover",
    "rock_biome",
]

risky_exact_features = [
    "source_id",
    "geo_zone_micro",
    "geo_meso_micro",
]

risky_prefixes = [
    "source_id_",
    "geo_zone_micro_",
    "geo_meso_micro_",
]

risky_contains = [
    "geo_meso_micro",
]

# Remove high-cardinality categorical features that are not protected.
high_cardinality_cat_features = []
for c in cat_features:
    if c not in protected_categorical_features:
        nunique = X_fe[c].nunique(dropna=False)
        if nunique > 80:
            high_cardinality_cat_features.append(c)

risky_feature_cols = set()

for c in X_fe.columns:
    if c in risky_exact_features:
        risky_feature_cols.add(c)

    if any(c.startswith(prefix) for prefix in risky_prefixes):
        risky_feature_cols.add(c)

    if any(pattern in c for pattern in risky_contains):
        risky_feature_cols.add(c)

for c in high_cardinality_cat_features:
    risky_feature_cols.add(c)

risky_feature_cols = sorted([c for c in risky_feature_cols if c in X_fe.columns])

X_fe_robust = X_fe.drop(columns=risky_feature_cols, errors="ignore").copy()
X_test_fe_robust = X_test_fe.drop(columns=risky_feature_cols, errors="ignore").copy()

cat_features_robust = X_fe_robust.select_dtypes(include=["object", "category"]).columns.tolist()
num_features_robust = [c for c in X_fe_robust.columns if c not in cat_features_robust]

print("Original feature shape:", X_fe.shape)
print("Robust feature shape:", X_fe_robust.shape)
print("Removed risky features:", len(risky_feature_cols))
display(pd.DataFrame({"removed_feature": risky_feature_cols}).head(100))

print("Robust categorical features:", len(cat_features_robust))
print("Robust numerical features:", len(num_features_robust))

assert X_fe_robust.shape[0] == X_fe.shape[0]
assert X_test_fe_robust.shape[0] == X_test_fe.shape[0]
assert list(X_fe_robust.columns) == list(X_test_fe_robust.columns)

## SECTION 8 - Validation Strategy

The main validation uses `StratifiedKFold` adapted for regression by binning the target with quantiles. This helps keep the target distribution balanced across folds.

Each model returns:

- Out-of-fold predictions
- Averaged test predictions
- Fold RMSE scores
- Mean and standard deviation CV RMSE
- Train RMSE per fold for overfit/underfit analysis
- Trained fold models

In [ ]:
def make_regression_stratified_folds(y: pd.Series, n_splits: int = N_SPLITS, seed: int = RANDOM_STATE):
    """Create stratified folds for regression using quantile-binned targets."""
    y = pd.Series(y).reset_index(drop=True)
    n_bins = min(10, max(2, len(y) // n_splits))
    try:
        y_bins = pd.qcut(y, q=n_bins, labels=False, duplicates="drop")
        if pd.Series(y_bins).nunique() < 2:
            raise ValueError("Not enough unique target bins")
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = list(splitter.split(np.zeros(len(y)), y_bins))
        strategy = "StratifiedKFold with target quantile bins"
    except Exception as exc:
        print(f"Falling back to KFold because target binning failed: {exc}")
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = list(splitter.split(np.zeros(len(y))))
        strategy = "KFold fallback"
    return folds, strategy

folds, validation_strategy = make_regression_stratified_folds(y, N_SPLITS, RANDOM_STATE)
print("Validation strategy:", validation_strategy)

# Check target balance per fold
fold_balance = []
for fold, (_, val_idx) in enumerate(folds, 1):
    yy = y.iloc[val_idx]
    fold_balance.append({
        "fold": fold,
        "n_valid": len(val_idx),
        "target_mean": yy.mean(),
        "target_median": yy.median(),
        "target_min": yy.min(),
        "target_max": yy.max(),
        "target_std": yy.std(),
    })
fold_balance_df = pd.DataFrame(fold_balance)
display(fold_balance_df)


def prepare_catboost_frame(X: pd.DataFrame, cat_cols: list[str]) -> pd.DataFrame:
    """Prepare a DataFrame for CatBoost native categorical handling."""
    X_out = X.copy()
    for c in cat_cols:
        if c in X_out.columns:
            X_out[c] = X_out[c].astype(str).fillna("Missing")
    return X_out




def make_median_imputer():
    """Create a median imputer while preserving all-empty features when sklearn supports it."""
    try:
        return SimpleImputer(strategy="median", keep_empty_features=True)
    except TypeError:
        return SimpleImputer(strategy="median")

def prepare_encoded_fold(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
):
    """Fold-safe preprocessing for LightGBM/XGBoost/Ridge: median impute numeric and ordinal encode categorical."""
    Xtr_parts, Xva_parts, Xte_parts, feature_names = [], [], [], []

    if num_cols:
        imputer = make_median_imputer()
        Xtr_num = pd.DataFrame(imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
        Xva_num = pd.DataFrame(imputer.transform(X_valid[num_cols]), columns=num_cols, index=X_valid.index)
        Xte_num = pd.DataFrame(imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)
        Xtr_parts.append(Xtr_num)
        Xva_parts.append(Xva_num)
        Xte_parts.append(Xte_num)
        feature_names.extend(num_cols)

    if cat_cols:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        Xtr_cat_raw = X_train[cat_cols].astype(str).fillna("Missing")
        Xva_cat_raw = X_valid[cat_cols].astype(str).fillna("Missing")
        Xte_cat_raw = X_test[cat_cols].astype(str).fillna("Missing")
        encoded_cat_cols = [f"{c}__ord" for c in cat_cols]
        Xtr_cat = pd.DataFrame(encoder.fit_transform(Xtr_cat_raw), columns=encoded_cat_cols, index=X_train.index)
        Xva_cat = pd.DataFrame(encoder.transform(Xva_cat_raw), columns=encoded_cat_cols, index=X_valid.index)
        Xte_cat = pd.DataFrame(encoder.transform(Xte_cat_raw), columns=encoded_cat_cols, index=X_test.index)
        Xtr_parts.append(Xtr_cat)
        Xva_parts.append(Xva_cat)
        Xte_parts.append(Xte_cat)
        feature_names.extend(encoded_cat_cols)

    Xtr = pd.concat(Xtr_parts, axis=1)
    Xva = pd.concat(Xva_parts, axis=1)
    Xte = pd.concat(Xte_parts, axis=1)
    return Xtr, Xva, Xte, feature_names

experiment_log = []
model_outputs = {}

In [ ]:
# ==================================================
# SECTION 8B - Harder Validation Diagnostic
# ==================================================
# Goal:
# Compare normal stratified CV with harder group-based validation.
# This helps diagnose whether the model overfits source/lab or geographic groups.

from sklearn.model_selection import GroupKFold

RUN_VALIDATION_DIAGNOSTIC = True
DIAGNOSTIC_N_ESTIMATORS = 300 if FAST_MODE else 700


def make_group_folds(df_raw: pd.DataFrame, group_col: str, n_splits: int = N_SPLITS):
    """Create GroupKFold folds if the group column is usable."""
    if group_col not in df_raw.columns:
        print(f"Skip {group_col}: column not found")
        return None

    groups = df_raw[group_col].astype(str).fillna("Missing")
    n_groups = groups.nunique()

    if n_groups < 2:
        print(f"Skip {group_col}: not enough unique groups")
        return None

    actual_splits = min(n_splits, n_groups)
    splitter = GroupKFold(n_splits=actual_splits)
    return list(splitter.split(np.zeros(len(df_raw)), y, groups))


def fit_lgbm_diagnostic(model, X_tr, y_tr, X_val, y_val):
    """Lightweight LightGBM fit for validation diagnostics."""
    try:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(0)
            ],
        )
    except TypeError:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=100,
            verbose=False,
        )
    return model


def quick_lgbm_cv_from_folds(
    X: pd.DataFrame,
    X_test_unused: pd.DataFrame,
    y: pd.Series,
    folds_to_use,
    cat_cols: list[str],
    num_cols: list[str],
    label: str,
):
    """Quick LightGBM CV score for comparing validation difficulty."""
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(folds_to_use, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        X_tr, X_val, X_te_dummy, feature_names = prepare_encoded_fold(
            X_tr_raw,
            X_val_raw,
            X_test_unused,
            cat_cols,
            num_cols
        )

        model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=DIAGNOSTIC_N_ESTIMATORS,
            learning_rate=0.04,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.2,
            reg_lambda=2.0,
            min_child_samples=30,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        model = fit_lgbm_diagnostic(model, X_tr, y_tr, X_val, y_val)
        val_pred = model.predict(X_val)
        score = rmse(y_val, val_pred)
        fold_scores.append(score)

    return {
        "validation_label": label,
        "mean_rmse": np.mean(fold_scores),
        "std_rmse": np.std(fold_scores),
        "fold_scores": fold_scores,
    }


validation_diagnostic_rows = []

if RUN_VALIDATION_DIAGNOSTIC:
    # Normal stratified CV, full features
    validation_diagnostic_rows.append(
        quick_lgbm_cv_from_folds(
            X_fe, X_test_fe, y, folds,
            cat_features, num_features,
            "stratified_full_features"
        )
    )

    # Normal stratified CV, robust features
    validation_diagnostic_rows.append(
        quick_lgbm_cv_from_folds(
            X_fe_robust, X_test_fe_robust, y, folds,
            cat_features_robust, num_features_robust,
            "stratified_robust_features"
        )
    )

    # Harder group validation
    for group_col in ["source_id", "geo_zone_macro", "geo_zone_meso"]:
        group_folds = make_group_folds(train_raw, group_col, N_SPLITS)

        if group_folds is not None:
            validation_diagnostic_rows.append(
                quick_lgbm_cv_from_folds(
                    X_fe, X_test_fe, y, group_folds,
                    cat_features, num_features,
                    f"group_{group_col}_full_features"
                )
            )

            validation_diagnostic_rows.append(
                quick_lgbm_cv_from_folds(
                    X_fe_robust, X_test_fe_robust, y, group_folds,
                    cat_features_robust, num_features_robust,
                    f"group_{group_col}_robust_features"
                )
            )

validation_diagnostic_df = pd.DataFrame(validation_diagnostic_rows).sort_values("mean_rmse")
display(validation_diagnostic_df)

## SECTION 9 - Baseline Model

Baselines provide a reference point for the main models.

Included baselines:

1. Mean target baseline
2. Ridge regression baseline with fold-safe preprocessing
3. HistGradientBoosting baseline with fold-safe preprocessing

In [ ]:
def log_experiment(model_name: str, features: str, fold_scores: list[float], notes: str, train_scores: Optional[list[float]] = None):
    row = {
        "model_name": model_name,
        "features": features,
        "mean_cv_rmse": float(np.mean(fold_scores)),
        "std_cv_rmse": float(np.std(fold_scores)),
        "fold_scores": fold_scores,
        "notes": notes,
    }
    if train_scores is not None:
        row["mean_train_rmse"] = float(np.mean(train_scores))
        row["train_valid_gap"] = float(np.mean(fold_scores) - np.mean(train_scores))
    experiment_log.append(row)
    return row

# Baseline 1: mean prediction
mean_oof = np.zeros(len(y))
mean_test_pred = np.zeros(len(test_raw))
mean_fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    pred_value = y.iloc[tr_idx].mean()
    mean_oof[val_idx] = pred_value
    fold_rmse = rmse(y.iloc[val_idx], mean_oof[val_idx])
    mean_fold_scores.append(fold_rmse)
mean_test_pred[:] = y.mean()
log_experiment("Mean Baseline", "target mean only", mean_fold_scores, "Predicts each fold training target mean")
print("Mean baseline RMSE:", np.mean(mean_fold_scores), "+/-", np.std(mean_fold_scores))

model_outputs["mean_baseline"] = {
    "oof": mean_oof,
    "test_pred": mean_test_pred,
    "scores": mean_fold_scores,
}

In [ ]:
def run_ridge_baseline(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=10.0, random_state=RANDOM_STATE)),
        ])
        model.fit(X_tr, y_tr)
        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        print(f"Ridge fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    return {"oof": oof, "test_pred": test_pred, "scores": fold_scores, "train_scores": train_scores}

ridge_output = run_ridge_baseline(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("Ridge Baseline", "all engineered features, encoded", ridge_output["scores"], "Linear baseline with median imputation, ordinal encoding, scaling", ridge_output["train_scores"])
model_outputs["ridge"] = ridge_output
print("Ridge baseline RMSE:", np.mean(ridge_output["scores"]), "+/-", np.std(ridge_output["scores"]))

In [ ]:
def run_hgb_baseline(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        model = HistGradientBoostingRegressor(
            learning_rate=0.04,
            max_iter=300 if FAST_MODE else 800,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=RANDOM_STATE,
            early_stopping=True,
        )
        model.fit(X_tr, y_tr)
        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        print(f"HGB fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    return {"oof": oof, "test_pred": test_pred, "scores": fold_scores, "train_scores": train_scores}

hgb_output = run_hgb_baseline(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("HistGradientBoosting Baseline", "all engineered features, encoded", hgb_output["scores"], "Tree baseline with fold-safe preprocessing", hgb_output["train_scores"])
model_outputs["hgb"] = hgb_output
print("HGB baseline RMSE:", np.mean(hgb_output["scores"]), "+/-", np.std(hgb_output["scores"]))

## SECTION 10 - Main Models: CatBoost, LightGBM, XGBoost

Three main models are trained using the same fold split:

1. CatBoostRegressor: native categorical handling
2. LightGBMRegressor: fold-safe numeric median imputation and categorical ordinal encoding
3. XGBoostRegressor: same fold-safe encoded data as LightGBM

In [ ]:
def run_catboost_cv(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols: list[str]):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []
    models = []
    feature_importance_rows = []

    X_cb_full = prepare_catboost_frame(X, cat_cols)
    X_test_cb = prepare_catboost_frame(X_test, cat_cols)

    cat_indices = [X_cb_full.columns.get_loc(c) for c in cat_cols if c in X_cb_full.columns]

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr, X_val = X_cb_full.iloc[tr_idx], X_cb_full.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=MAIN_N_ESTIMATORS,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5,
            random_strength=1.0,
            bagging_temperature=0.5,
            random_seed=RANDOM_STATE + fold,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=200,
            allow_writing_files=False,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_indices,
            use_best_model=True,
        )

        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_test_cb)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        models.append(model)

        fi = pd.DataFrame({
            "feature": X_cb_full.columns,
            "importance": model.get_feature_importance(),
            "fold": fold,
        })
        feature_importance_rows.append(fi)
        print(f"CatBoost fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    feature_importance = pd.concat(feature_importance_rows, ignore_index=True)
    return {
        "oof": oof,
        "test_pred": test_pred,
        "scores": fold_scores,
        "train_scores": train_scores,
        "models": models,
        "feature_importance": feature_importance,
    }

catboost_output = run_catboost_cv(X_fe, X_test_fe, y, folds, cat_features)
log_experiment("CatBoostRegressor", "all engineered features, native categorical", catboost_output["scores"], "Main model with native categorical and missing handling", catboost_output["train_scores"])
model_outputs["catboost"] = catboost_output
print("CatBoost CV RMSE:", np.mean(catboost_output["scores"]), "+/-", np.std(catboost_output["scores"]))

In [ ]:
def fit_lgbm_compatible(model, X_tr, y_tr, X_val, y_val):
    """Fit LightGBM across multiple API versions."""
    try:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(200)],
        )
    except TypeError:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=200,
        )
    return model


def run_lgbm_cv(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []
    models = []
    feature_importance_rows = []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=MAIN_N_ESTIMATORS,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            min_child_samples=20,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            verbosity=-1,
        )
        model = fit_lgbm_compatible(model, X_tr, y_tr, X_val, y_val)

        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        models.append(model)

        fi = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_,
            "fold": fold,
        })
        feature_importance_rows.append(fi)
        print(f"LightGBM fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    feature_importance = pd.concat(feature_importance_rows, ignore_index=True)
    return {
        "oof": oof,
        "test_pred": test_pred,
        "scores": fold_scores,
        "train_scores": train_scores,
        "models": models,
        "feature_importance": feature_importance,
    }

lgbm_output = run_lgbm_cv(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("LightGBMRegressor", "all engineered features, encoded", lgbm_output["scores"], "Main model with fold-safe imputation and ordinal encoding", lgbm_output["train_scores"])
model_outputs["lightgbm"] = lgbm_output
print("LightGBM CV RMSE:", np.mean(lgbm_output["scores"]), "+/-", np.std(lgbm_output["scores"]))

In [ ]:
def fit_xgb_compatible(params: dict, X_tr, y_tr, X_val, y_val):
    """Fit XGBoost across old/new API differences for early stopping."""
    params = params.copy()
    try:
        model = XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)
    except TypeError:
        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=200)
    return model


def run_xgb_cv(X: pd.DataFrame, X_test: pd.DataFrame, y: pd.Series, folds, cat_cols, num_cols):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    fold_scores, train_scores = [], []
    models = []
    feature_importance_rows = []

    params = dict(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=MAIN_N_ESTIMATORS,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        tree_method="hist",
        n_jobs=-1,
    )

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(X_tr_raw, X_val_raw, X_test, cat_cols, num_cols)

        params["random_state"] = RANDOM_STATE + fold
        model = fit_xgb_compatible(params, X_tr, y_tr, X_val, y_val)

        tr_pred = model.predict(X_tr)
        val_pred = model.predict(X_val)
        te_pred = model.predict(X_te)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)
        fold_scores.append(rmse(y_val, val_pred))
        train_scores.append(rmse(y_tr, tr_pred))
        models.append(model)

        fi = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_,
            "fold": fold,
        })
        feature_importance_rows.append(fi)
        print(f"XGBoost fold {fold}: train RMSE={train_scores[-1]:.5f}, valid RMSE={fold_scores[-1]:.5f}")

    feature_importance = pd.concat(feature_importance_rows, ignore_index=True)
    return {
        "oof": oof,
        "test_pred": test_pred,
        "scores": fold_scores,
        "train_scores": train_scores,
        "models": models,
        "feature_importance": feature_importance,
    }

xgb_output = run_xgb_cv(X_fe, X_test_fe, y, folds, cat_features, num_features)
log_experiment("XGBoostRegressor", "all engineered features, encoded", xgb_output["scores"], "Main model with fold-safe imputation and ordinal encoding", xgb_output["train_scores"])
model_outputs["xgboost"] = xgb_output
print("XGBoost CV RMSE:", np.mean(xgb_output["scores"]), "+/-", np.std(xgb_output["scores"]))

## SECTION 11 - Ensemble, Model Selection, and Final Prediction

The final Kaggle submission uses out-of-fold validated ensemble predictions.

Ensemble strategies:

1. Equal weight: CatBoost + LightGBM + XGBoost
2. Fixed weighted ensemble: 0.50 CatBoost + 0.30 LightGBM + 0.20 XGBoost
3. Simple OOF weight search with step 0.05

The best OOF RMSE ensemble is used to generate `submission.csv`.

In [ ]:
experiment_log_df = pd.DataFrame(experiment_log).sort_values("mean_cv_rmse")
print("Experiment log:")
display(experiment_log_df)

main_model_names = ["catboost", "lightgbm", "xgboost"]
assert all(name in model_outputs for name in main_model_names), "All three main models must be trained before ensembling."

main_oofs = {name: model_outputs[name]["oof"] for name in main_model_names}
main_test_preds = {name: model_outputs[name]["test_pred"] for name in main_model_names}


def weighted_average(pred_dict: dict[str, np.ndarray], weights: dict[str, float]) -> np.ndarray:
    pred = None
    for name, w in weights.items():
        if pred is None:
            pred = w * pred_dict[name]
        else:
            pred += w * pred_dict[name]
    return pred

ensemble_rows = []

# 1. Equal weight
weights_equal = {name: 1 / len(main_model_names) for name in main_model_names}
oof_equal = weighted_average(main_oofs, weights_equal)
ensemble_rows.append({"ensemble_name": "equal_weight", "weights": weights_equal, "oof_rmse": rmse(y, oof_equal)})

# 2. Recommended fixed weighted ensemble
weights_fixed = {"catboost": 0.50, "lightgbm": 0.30, "xgboost": 0.20}
oof_fixed = weighted_average(main_oofs, weights_fixed)
ensemble_rows.append({"ensemble_name": "fixed_0.50_0.30_0.20", "weights": weights_fixed, "oof_rmse": rmse(y, oof_fixed)})

# 3. Simple weight search, step 0.05
step = 0.05
values = np.round(np.arange(0, 1 + step, step), 2)
best = {"ensemble_name": "search_weight", "weights": None, "oof_rmse": np.inf}
for w_cat in values:
    for w_lgb in values:
        w_xgb = round(1.0 - w_cat - w_lgb, 2)
        if w_xgb < 0 or w_xgb > 1:
            continue
        weights = {"catboost": float(w_cat), "lightgbm": float(w_lgb), "xgboost": float(w_xgb)}
        oof_pred = weighted_average(main_oofs, weights)
        score = rmse(y, oof_pred)
        if score < best["oof_rmse"]:
            best = {"ensemble_name": "search_weight", "weights": weights, "oof_rmse": score}
ensemble_rows.append(best)

ensemble_results = pd.DataFrame(ensemble_rows).sort_values("oof_rmse")
print("Ensemble results:")
display(ensemble_results)

best_ensemble = ensemble_results.iloc[0]
best_weights = best_ensemble["weights"]
print("Selected ensemble:", best_ensemble["ensemble_name"])
print("Selected weights:", best_weights)
print("Selected OOF RMSE:", best_ensemble["oof_rmse"])

final_oof_pred = weighted_average(main_oofs, best_weights)
final_test_pred = weighted_average(main_test_preds, best_weights)
final_test_pred = np.clip(final_test_pred, 0, None)
final_oof_pred_clipped = np.clip(final_oof_pred, 0, None)

print("Final OOF RMSE before clipping:", rmse(y, final_oof_pred))
print("Final OOF RMSE after clipping:", rmse(y, final_oof_pred_clipped))
print("Final test prediction summary:")
display(pd.Series(final_test_pred, name=TARGET).describe().to_frame().T)

# Create submission following sample_submission order exactly
prediction_frame = pd.DataFrame({ID_COL: test_raw[ID_COL].values, TARGET: final_test_pred})
submission = sample_submission[[ID_COL]].merge(prediction_frame, on=ID_COL, how="left")

# Strict final validation
assert list(submission.columns) == [ID_COL, TARGET], f"Submission columns must be {[ID_COL, TARGET]}"
assert len(submission) == len(sample_submission), "Submission row count must match sample_submission"
assert submission[ID_COL].equals(sample_submission[ID_COL]), "Submission ID order must follow sample_submission"
assert submission[TARGET].notna().all(), "Submission predictions must not contain NaN"
assert np.isfinite(submission[TARGET]).all(), "Submission predictions must be finite"
assert (submission[TARGET] >= 0).all(), "Submission predictions must not be negative"

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())
display(submission.shape)
display(submission[TARGET].describe().to_frame().T)

# Optional Colab download
try:
    from google.colab import files
    files.download("submission.csv")
except Exception:
    pass

In [ ]:
# ==================================================
# SECTION 11B - New Candidate Submissions
# ==================================================
# Goal:
# Create additional submission candidates based on public leaderboard diagnostics.
#
# Current findings:
# - Original ensemble public score is better than CatBoost single and XGBoost single.
# - CatBoost single is worse on public LB, so we test lower CatBoost weights.
# - No external data. No AutoML. Only official train/test-derived features.

def save_submission_from_pred(pred, filename: str):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)
    return sub


# Existing model predictions from current notebook
base_pred_dict = {
    "catboost": model_outputs["catboost"]["test_pred"],
    "lightgbm": model_outputs["lightgbm"]["test_pred"],
    "xgboost": model_outputs["xgboost"]["test_pred"],
}

base_oof_dict = {
    "catboost": model_outputs["catboost"]["oof"],
    "lightgbm": model_outputs["lightgbm"]["oof"],
    "xgboost": model_outputs["xgboost"]["oof"],
}


def weighted_pred(pred_dict, weights):
    out = np.zeros(len(next(iter(pred_dict.values()))))
    for name, weight in weights.items():
        out += weight * pred_dict[name]
    return out


candidate_weight_sets = {
    "equal_weight": {
        "catboost": 1/3,
        "lightgbm": 1/3,
        "xgboost": 1/3,
    },
    "catlow_025_035_040": {
        "catboost": 0.25,
        "lightgbm": 0.35,
        "xgboost": 0.40,
    },
    "catlow_020_040_040": {
        "catboost": 0.20,
        "lightgbm": 0.40,
        "xgboost": 0.40,
    },
    "xgb_lgb_heavy_010_045_045": {
        "catboost": 0.10,
        "lightgbm": 0.45,
        "xgboost": 0.45,
    },
}

candidate_rows = []
for name, weights in candidate_weight_sets.items():
    oof_pred = weighted_pred(base_oof_dict, weights)
    test_pred = weighted_pred(base_pred_dict, weights)
    oof_score = rmse(y, np.clip(oof_pred, 0, None))

    candidate_rows.append({
        "candidate": name,
        "weights": weights,
        "oof_rmse": oof_score,
    })

    save_submission_from_pred(test_pred, f"submission_{name}.csv")

candidate_weight_df = pd.DataFrame(candidate_rows).sort_values("oof_rmse")
display(candidate_weight_df)

In [ ]:
# ==================================================
# SECTION 11C - Robust Feature Ensemble
# ==================================================
# Goal:
# Train CatBoost, LightGBM, and XGBoost again using robust features.
# This tests whether removing risky features improves public/private generalization.

RUN_ROBUST_ENSEMBLE = True

if RUN_ROBUST_ENSEMBLE:
    print("Training robust CatBoost...")
    catboost_robust_output = run_catboost_cv(
        X_fe_robust,
        X_test_fe_robust,
        y,
        folds,
        cat_features_robust
    )
    model_outputs["catboost_robust"] = catboost_robust_output
    log_experiment(
        "CatBoostRegressor Robust",
        "robust features, native categorical",
        catboost_robust_output["scores"],
        "Removed source_id, geo_zone_micro, and high-cardinality risky features",
        catboost_robust_output["train_scores"]
    )

    print("Training robust LightGBM...")
    lgbm_robust_output = run_lgbm_cv(
        X_fe_robust,
        X_test_fe_robust,
        y,
        folds,
        cat_features_robust,
        num_features_robust
    )
    model_outputs["lightgbm_robust"] = lgbm_robust_output
    log_experiment(
        "LightGBMRegressor Robust",
        "robust features, encoded",
        lgbm_robust_output["scores"],
        "Removed source_id, geo_zone_micro, and high-cardinality risky features",
        lgbm_robust_output["train_scores"]
    )

    print("Training robust XGBoost...")
    xgb_robust_output = run_xgb_cv(
        X_fe_robust,
        X_test_fe_robust,
        y,
        folds,
        cat_features_robust,
        num_features_robust
    )
    model_outputs["xgboost_robust"] = xgb_robust_output
    log_experiment(
        "XGBoostRegressor Robust",
        "robust features, encoded",
        xgb_robust_output["scores"],
        "Removed source_id, geo_zone_micro, and high-cardinality risky features",
        xgb_robust_output["train_scores"]
    )

    robust_oof_dict = {
        "catboost": catboost_robust_output["oof"],
        "lightgbm": lgbm_robust_output["oof"],
        "xgboost": xgb_robust_output["oof"],
    }

    robust_test_pred_dict = {
        "catboost": catboost_robust_output["test_pred"],
        "lightgbm": lgbm_robust_output["test_pred"],
        "xgboost": xgb_robust_output["test_pred"],
    }

    robust_weight_sets = {
        "robust_equal": {
            "catboost": 1/3,
            "lightgbm": 1/3,
            "xgboost": 1/3,
        },
        "robust_catlow_025_035_040": {
            "catboost": 0.25,
            "lightgbm": 0.35,
            "xgboost": 0.40,
        },
        "robust_search_safe": None,
    }

    robust_candidate_rows = []

    # Fixed robust weights
    for name, weights in robust_weight_sets.items():
        if weights is None:
            continue

        robust_oof_pred = weighted_pred(robust_oof_dict, weights)
        robust_test_pred = weighted_pred(robust_test_pred_dict, weights)

        robust_candidate_rows.append({
            "candidate": name,
            "weights": weights,
            "oof_rmse": rmse(y, np.clip(robust_oof_pred, 0, None)),
        })

        save_submission_from_pred(robust_test_pred, f"submission_{name}.csv")

    # Search robust weights
    step = 0.05
    values = np.round(np.arange(0, 1 + step, step), 2)
    best_robust = {
        "candidate": "robust_search_safe",
        "weights": None,
        "oof_rmse": np.inf,
        "test_pred": None,
    }

    for w_cat in values:
        for w_lgb in values:
            w_xgb = round(1.0 - w_cat - w_lgb, 2)
            if w_xgb < 0 or w_xgb > 1:
                continue

            weights = {
                "catboost": float(w_cat),
                "lightgbm": float(w_lgb),
                "xgboost": float(w_xgb),
            }

            robust_oof_pred = weighted_pred(robust_oof_dict, weights)
            score = rmse(y, np.clip(robust_oof_pred, 0, None))

            if score < best_robust["oof_rmse"]:
                best_robust = {
                    "candidate": "robust_search_safe",
                    "weights": weights,
                    "oof_rmse": score,
                    "test_pred": weighted_pred(robust_test_pred_dict, weights),
                }

    robust_candidate_rows.append({
        "candidate": best_robust["candidate"],
        "weights": best_robust["weights"],
        "oof_rmse": best_robust["oof_rmse"],
    })

    save_submission_from_pred(best_robust["test_pred"], "submission_robust_search_safe.csv")

    robust_candidate_df = pd.DataFrame(robust_candidate_rows).sort_values("oof_rmse")
    display(robust_candidate_df)

    experiment_log_df = pd.DataFrame(experiment_log).sort_values("mean_cv_rmse")
    display(experiment_log_df)

In [ ]:
# ==================================================
# SECTION 11D - Optional Multi-Seed Ensemble
# ==================================================
# This is computationally expensive.
# Run only if you have enough Colab time.
#
# Goal:
# Reduce prediction variance by averaging several seed-based ensembles.

RUN_MULTI_SEED_ENSEMBLE = False

MULTI_SEEDS = [7, 123, 2024]

if RUN_MULTI_SEED_ENSEMBLE:
    multi_seed_test_preds = []
    multi_seed_oof_preds = []
    multi_seed_rows = []

    old_random_state = RANDOM_STATE

    for seed in MULTI_SEEDS:
        print("=" * 80)
        print(f"Running multi-seed ensemble with seed = {seed}")

        RANDOM_STATE = seed
        seed_everything(seed)

        seed_folds, seed_strategy = make_regression_stratified_folds(y, N_SPLITS, seed)

        seed_cb = run_catboost_cv(
            X_fe,
            X_test_fe,
            y,
            seed_folds,
            cat_features
        )

        seed_lgb = run_lgbm_cv(
            X_fe,
            X_test_fe,
            y,
            seed_folds,
            cat_features,
            num_features
        )

        seed_xgb = run_xgb_cv(
            X_fe,
            X_test_fe,
            y,
            seed_folds,
            cat_features,
            num_features
        )

        seed_oof_dict = {
            "catboost": seed_cb["oof"],
            "lightgbm": seed_lgb["oof"],
            "xgboost": seed_xgb["oof"],
        }

        seed_test_pred_dict = {
            "catboost": seed_cb["test_pred"],
            "lightgbm": seed_lgb["test_pred"],
            "xgboost": seed_xgb["test_pred"],
        }

        # Public diagnostics showed CatBoost single was weaker,
        # so use a more balanced / CatBoost-lower weighting.
        seed_weights = {
            "catboost": 0.25,
            "lightgbm": 0.35,
            "xgboost": 0.40,
        }

        seed_oof_pred = weighted_pred(seed_oof_dict, seed_weights)
        seed_test_pred = weighted_pred(seed_test_pred_dict, seed_weights)

        multi_seed_oof_preds.append(seed_oof_pred)
        multi_seed_test_preds.append(seed_test_pred)

        multi_seed_rows.append({
            "seed": seed,
            "weights": seed_weights,
            "oof_rmse": rmse(y, np.clip(seed_oof_pred, 0, None)),
        })

    RANDOM_STATE = old_random_state
    seed_everything(RANDOM_STATE)

    multi_seed_oof_final = np.mean(multi_seed_oof_preds, axis=0)
    multi_seed_test_final = np.mean(multi_seed_test_preds, axis=0)

    print("Multi-seed individual results:")
    display(pd.DataFrame(multi_seed_rows))

    print("Multi-seed final OOF RMSE:", rmse(y, np.clip(multi_seed_oof_final, 0, None)))

    save_submission_from_pred(
        multi_seed_test_final,
        "submission_multiseed_catlow_ensemble.csv"
    )

In [ ]:
from google.colab import files

files.download("submission_catlow_025_035_040.csv")
# files.download("submission_equal_weight.csv")
# files.download("submission_xgb_lgb_heavy_010_045_045.csv")

In [ ]:
def make_single_submission(model_key, filename):
    pred = model_outputs[model_key]["test_pred"]
    pred = np.clip(pred, 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub.shape)
    display(sub[TARGET].describe().to_frame().T)

make_single_submission("catboost", "submission_catboost.csv")
make_single_submission("xgboost", "submission_xgboost.csv")
make_single_submission("lightgbm", "submission_lightgbm.csv")

In [ ]:
from google.colab import files

files.download("submission_catboost.csv")

In [ ]:
# ==================================================
# SECTION 11E - Error Analysis by Target Range
# ==================================================
# Goal:
# Find whether the model performs poorly on low, medium, or high organic content.

# Use selected ensemble OOF prediction if available
if "final_oof_pred" in globals():
    oof_pred_current = final_oof_pred
else:
    # fallback: rebuild old selected ensemble from current best weights
    current_weights = {"catboost": 0.50, "lightgbm": 0.25, "xgboost": 0.25}
    oof_pred_current = (
        current_weights["catboost"] * model_outputs["catboost"]["oof"] +
        current_weights["lightgbm"] * model_outputs["lightgbm"]["oof"] +
        current_weights["xgboost"] * model_outputs["xgboost"]["oof"]
    )

error_df = train_raw[[ID_COL, TARGET]].copy()
error_df["oof_pred"] = np.clip(oof_pred_current, 0, None)
error_df["error"] = error_df["oof_pred"] - error_df[TARGET]
error_df["abs_error"] = error_df["error"].abs()
error_df["squared_error"] = error_df["error"] ** 2

# Target bins
error_df["target_bin_5"] = pd.qcut(
    error_df[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
)

error_df["target_bin_10"] = pd.qcut(
    error_df[TARGET],
    q=10,
    duplicates="drop"
)

def segment_metrics(df, group_col):
    out = (
        df.groupby(group_col)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("oof_pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
        .sort_values("rmse", ascending=False)
    )
    return out

target_bin_metrics = segment_metrics(error_df, "target_bin_5")
display(target_bin_metrics)

print("Overall OOF RMSE:", np.sqrt(np.mean(error_df["squared_error"])))
print("Worst target range:")
display(target_bin_metrics.head(3))

In [ ]:
# ==================================================
# SECTION 11F - Error Analysis by Soil, Source, and Geography Segment
# ==================================================
# Goal:
# Find which categorical segments produce the largest validation error.

segment_cols = [
    "source_id",
    "has_band_A_spectrum",
    "has_band_B_spectrum",
    "sampling_strategy",
    "geo_zone_macro",
    "geo_zone_meso",
    "geo_zone_micro",
    "land_cover_type",
    "biome",
    "parent_rock_type",
]

segment_cols = [c for c in segment_cols if c in train_raw.columns]

segment_reports = {}

for col in segment_cols:
    temp = pd.concat([train_raw[[col]].reset_index(drop=True), error_df.reset_index(drop=True)], axis=1)
    report = segment_metrics(temp, col)
    
    # filter small groups to avoid misleading conclusion
    report_filtered = report[report["count"] >= 50].copy()
    segment_reports[col] = report_filtered
    
    print("=" * 80)
    print(f"Segment error by: {col}")
    display(report_filtered.head(15))

# Compact summary: worst segment per column
worst_segments = []

for col, report in segment_reports.items():
    if len(report) > 0:
        top = report.iloc[0].to_dict()
        top["segment_column"] = col
        worst_segments.append(top)

worst_segment_df = pd.DataFrame(worst_segments)
cols_order = ["segment_column"] + [c for c in worst_segment_df.columns if c != "segment_column"]
worst_segment_df = worst_segment_df[cols_order].sort_values("rmse", ascending=False)

print("Worst segment summary:")
display(worst_segment_df)

In [ ]:
# ==================================================
# SECTION 11G - Train-Test Distribution Shift Check
# ==================================================
# Goal:
# Compare train and test distribution for important features.

important_cat_cols = [
    "source_id",
    "has_band_A_spectrum",
    "has_band_B_spectrum",
    "sampling_strategy",
    "geo_zone_macro",
    "geo_zone_meso",
    "geo_zone_micro",
    "land_cover_type",
    "biome",
    "parent_rock_type",
]

important_cat_cols = [c for c in important_cat_cols if c in train_raw.columns and c in test_raw.columns]

def category_shift_report(train_df, test_df, col):
    train_dist = train_df[col].astype(str).fillna("Missing").value_counts(normalize=True)
    test_dist = test_df[col].astype(str).fillna("Missing").value_counts(normalize=True)
    
    all_cats = sorted(set(train_dist.index).union(set(test_dist.index)))
    
    out = pd.DataFrame({
        "category": all_cats,
        "train_pct": [train_dist.get(cat, 0) * 100 for cat in all_cats],
        "test_pct": [test_dist.get(cat, 0) * 100 for cat in all_cats],
    })
    
    out["abs_diff_pct"] = (out["train_pct"] - out["test_pct"]).abs()
    out["feature"] = col
    return out.sort_values("abs_diff_pct", ascending=False)

shift_reports = []

for col in important_cat_cols:
    report = category_shift_report(train_raw, test_raw, col)
    shift_reports.append(report)
    
    print("=" * 80)
    print(f"Train-test category shift: {col}")
    display(report.head(15))

cat_shift_df = pd.concat(shift_reports, ignore_index=True)
print("Largest categorical distribution shifts:")
display(cat_shift_df.sort_values("abs_diff_pct", ascending=False).head(30))


# Missing rate shift
missing_shift_df = pd.DataFrame({
    "feature": train_raw.columns.intersection(test_raw.columns),
})

missing_shift_df["train_missing_pct"] = missing_shift_df["feature"].map(lambda c: train_raw[c].isna().mean() * 100)
missing_shift_df["test_missing_pct"] = missing_shift_df["feature"].map(lambda c: test_raw[c].isna().mean() * 100)
missing_shift_df["abs_diff_pct"] = (missing_shift_df["train_missing_pct"] - missing_shift_df["test_missing_pct"]).abs()

missing_shift_df = missing_shift_df.sort_values("abs_diff_pct", ascending=False)

print("Largest missing-rate shifts:")
display(missing_shift_df.head(30))

In [ ]:
# ==================================================
# SECTION 11H - High Target Calibration & Tail Correction
# ==================================================
# Goal:
# Diagnose and reduce underprediction on high organic-content samples.
# No external data. No AutoML. Only OOF-based post-processing.

from sklearn.isotonic import IsotonicRegression

OLD_BEST_OOF_RMSE = 11.335988679052521  # from previous selected ensemble
PUBLIC_BEST_SCORE = 12.13087             # current best public score

def get_weighted_oof_and_test(weights):
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(sample_submission.shape[0])

    for model_name, weight in weights.items():
        oof_pred += weight * model_outputs[model_name]["oof"]
        test_pred += weight * model_outputs[model_name]["test_pred"]

    return np.clip(oof_pred, 0, None), np.clip(test_pred, 0, None)


def make_error_report_from_pred(oof_pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["oof_pred"] = np.clip(oof_pred, 0, None)
    temp["error"] = temp["oof_pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5")
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("oof_pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    overall_rmse = rmse(temp[TARGET], temp["oof_pred"])

    print("=" * 90)
    print(label)
    print("Overall OOF RMSE:", overall_rmse)
    display(report.sort_values("rmse", ascending=False))

    return {
        "label": label,
        "overall_rmse": overall_rmse,
        "report": report,
    }


def save_submission_from_pred_safe(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


# Candidate base ensembles
candidate_base_weights = {
    "old_best_050_025_025": {
        "catboost": 0.50,
        "lightgbm": 0.25,
        "xgboost": 0.25,
    },
    "equal_weight": {
        "catboost": 1/3,
        "lightgbm": 1/3,
        "xgboost": 1/3,
    },
    "catlow_025_035_040": {
        "catboost": 0.25,
        "lightgbm": 0.35,
        "xgboost": 0.40,
    },
}

base_candidate_outputs = {}

for name, weights in candidate_base_weights.items():
    oof_pred, test_pred = get_weighted_oof_and_test(weights)
    report = make_error_report_from_pred(oof_pred, f"Base ensemble: {name}")

    base_candidate_outputs[name] = {
        "weights": weights,
        "oof_pred": oof_pred,
        "test_pred": test_pred,
        "oof_rmse": report["overall_rmse"],
    }


# --------------------------------------------------
# Approach 1: Isotonic calibration
# --------------------------------------------------
# Fits a monotonic mapping from OOF prediction -> actual target.
# It can lift high predictions if OOF shows systematic underprediction.

calibration_rows = []
calibrated_outputs = {}

for name, obj in base_candidate_outputs.items():
    base_oof = obj["oof_pred"]
    base_test = obj["test_pred"]

    iso = IsotonicRegression(
        y_min=0,
        y_max=None,
        out_of_bounds="clip"
    )

    iso.fit(base_oof, y)

    cal_oof = np.clip(iso.predict(base_oof), 0, None)
    cal_test = np.clip(iso.predict(base_test), 0, None)

    score = rmse(y, cal_oof)

    calibration_rows.append({
        "candidate": f"{name}_isotonic",
        "base_oof_rmse": obj["oof_rmse"],
        "calibrated_oof_rmse": score,
        "delta_vs_base": score - obj["oof_rmse"],
        "delta_vs_old_best": score - OLD_BEST_OOF_RMSE,
    })

    calibrated_outputs[f"{name}_isotonic"] = {
        "oof_pred": cal_oof,
        "test_pred": cal_test,
        "oof_rmse": score,
    }

    make_error_report_from_pred(cal_oof, f"Isotonic calibrated: {name}")


calibration_df = pd.DataFrame(calibration_rows).sort_values("calibrated_oof_rmse")
display(calibration_df)


# --------------------------------------------------
# Approach 2: Smooth high-tail boost
# --------------------------------------------------
# If high target is underpredicted, gently increase only high predictions.
# This is tested on OOF before creating a submission.

tail_rows = []
tail_outputs = {}

for name, obj in base_candidate_outputs.items():
    base_oof = obj["oof_pred"]
    base_test = obj["test_pred"]

    # Use prediction threshold, not true target threshold, because test target is unknown.
    thresholds = [
        np.percentile(base_oof, 70),
        np.percentile(base_oof, 75),
        np.percentile(base_oof, 80),
        np.percentile(base_oof, 85),
    ]

    alphas = np.round(np.arange(0.02, 0.31, 0.02), 2)

    for threshold in thresholds:
        for alpha in alphas:
            adj_oof = base_oof + alpha * np.maximum(base_oof - threshold, 0)
            adj_oof = np.clip(adj_oof, 0, None)

            score = rmse(y, adj_oof)

            candidate_name = f"{name}_tail_t{threshold:.2f}_a{alpha:.2f}"

            tail_rows.append({
                "candidate": candidate_name,
                "base": name,
                "threshold": threshold,
                "alpha": alpha,
                "base_oof_rmse": obj["oof_rmse"],
                "tail_oof_rmse": score,
                "delta_vs_base": score - obj["oof_rmse"],
                "delta_vs_old_best": score - OLD_BEST_OOF_RMSE,
            })

            adj_test = base_test + alpha * np.maximum(base_test - threshold, 0)
            adj_test = np.clip(adj_test, 0, None)

            tail_outputs[candidate_name] = {
                "oof_pred": adj_oof,
                "test_pred": adj_test,
                "oof_rmse": score,
                "base": name,
                "threshold": threshold,
                "alpha": alpha,
            }

tail_df = pd.DataFrame(tail_rows).sort_values("tail_oof_rmse")
display(tail_df.head(20))

best_tail_name = tail_df.iloc[0]["candidate"]
best_tail = tail_outputs[best_tail_name]

print("Best tail candidate:", best_tail_name)
print("Best tail OOF RMSE:", best_tail["oof_rmse"])
print("Old best OOF RMSE:", OLD_BEST_OOF_RMSE)

make_error_report_from_pred(best_tail["oof_pred"], f"Best tail correction: {best_tail_name}")


# --------------------------------------------------
# Save only candidates that are not obviously bad
# --------------------------------------------------

# Best isotonic candidate
best_iso_name = calibration_df.iloc[0]["candidate"]
best_iso = calibrated_outputs[best_iso_name]

print("Best isotonic candidate:", best_iso_name)
print("Best isotonic OOF RMSE:", best_iso["oof_rmse"])

save_submission_from_pred_safe(
    best_iso["test_pred"],
    f"submission_{best_iso_name}.csv"
)

save_submission_from_pred_safe(
    best_tail["test_pred"],
    "submission_best_tail_correction.csv"
)

print("Decision guide:")
print("- Submit calibrated/tail candidate only if OOF RMSE is close to or better than old best.")
print("- Strong submit condition: OOF RMSE < old best OOF RMSE.")
print("- Soft submit condition: OOF slightly worse, but very_high RMSE and bias improve clearly.")

In [ ]:
# ==================================================
# SECTION 11I - Cross-Validated Isotonic Calibration
# ==================================================
# Goal:
# Validate isotonic calibration safely using calibration folds.
# This checks whether isotonic improvement is real or just overfitting OOF predictions.
#
# Important:
# - Base model OOF predictions are already out-of-fold.
# - Calibration is also done out-of-fold.
# - Test prediction uses isotonic fitted on full OOF after CV validation.
# - No external data. No AutoML.

from sklearn.isotonic import IsotonicRegression

OLD_BEST_OOF_RMSE = 11.335988679052521
PUBLIC_BEST_SCORE = 12.13087

# --------------------------------------------------
# Utility fallback
# --------------------------------------------------

def weighted_pred_from_model_outputs(weights):
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(sample_submission.shape[0])

    for model_name, weight in weights.items():
        oof_pred += weight * model_outputs[model_name]["oof"]
        test_pred += weight * model_outputs[model_name]["test_pred"]

    return np.clip(oof_pred, 0, None), np.clip(test_pred, 0, None)


def save_submission_from_pred_safe_11i(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_error_report(oof_pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["oof_pred"] = np.clip(oof_pred, 0, None)
    temp["error"] = temp["oof_pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5")
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("oof_pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
        .sort_values("rmse", ascending=False)
    )

    print("=" * 90)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(oof_pred, 0, None)))
    display(report)

    return report


# --------------------------------------------------
# Base ensemble candidates to calibrate
# --------------------------------------------------

calibration_base_weight_sets = {
    "old_best_050_025_025": {
        "catboost": 0.50,
        "lightgbm": 0.25,
        "xgboost": 0.25,
    },
    "equal_weight": {
        "catboost": 1/3,
        "lightgbm": 1/3,
        "xgboost": 1/3,
    },
    "catlow_025_035_040": {
        "catboost": 0.25,
        "lightgbm": 0.35,
        "xgboost": 0.40,
    },
}

# Use the same folds for calibration validation
# If folds exists from Section 8, use it directly.
calibration_folds = folds

cv_iso_rows = []
cv_iso_outputs = {}

for base_name, weights in calibration_base_weight_sets.items():
    print("=" * 100)
    print(f"Cross-validated isotonic calibration for: {base_name}")

    base_oof, base_test = weighted_pred_from_model_outputs(weights)

    cv_calibrated_oof = np.zeros(len(y))

    for fold, (cal_tr_idx, cal_val_idx) in enumerate(calibration_folds, 1):
        x_cal_train = base_oof[cal_tr_idx]
        y_cal_train = y.iloc[cal_tr_idx].values

        x_cal_valid = base_oof[cal_val_idx]

        iso = IsotonicRegression(
            y_min=0,
            y_max=None,
            out_of_bounds="clip"
        )

        iso.fit(x_cal_train, y_cal_train)
        cv_calibrated_oof[cal_val_idx] = iso.predict(x_cal_valid)

    cv_calibrated_oof = np.clip(cv_calibrated_oof, 0, None)

    base_score = rmse(y, base_oof)
    cv_iso_score = rmse(y, cv_calibrated_oof)

    # Fit final calibrator on all OOF predictions for test-time mapping
    final_iso = IsotonicRegression(
        y_min=0,
        y_max=None,
        out_of_bounds="clip"
    )

    final_iso.fit(base_oof, y.values)
    calibrated_test = np.clip(final_iso.predict(base_test), 0, None)

    cv_iso_rows.append({
        "candidate": f"{base_name}_cv_isotonic",
        "base_oof_rmse": base_score,
        "cv_isotonic_oof_rmse": cv_iso_score,
        "delta_vs_base": cv_iso_score - base_score,
        "delta_vs_old_best": cv_iso_score - OLD_BEST_OOF_RMSE,
        "weights": weights,
    })

    cv_iso_outputs[f"{base_name}_cv_isotonic"] = {
        "oof_pred": cv_calibrated_oof,
        "test_pred": calibrated_test,
        "oof_rmse": cv_iso_score,
        "base_oof_rmse": base_score,
        "weights": weights,
    }

    target_bin_error_report(
        base_oof,
        f"Before CV isotonic: {base_name}"
    )

    target_bin_error_report(
        cv_calibrated_oof,
        f"After CV isotonic: {base_name}"
    )

cv_iso_df = pd.DataFrame(cv_iso_rows).sort_values("cv_isotonic_oof_rmse")
display(cv_iso_df)


# --------------------------------------------------
# Save best CV-isotonic submission
# --------------------------------------------------

best_cv_iso_name = cv_iso_df.iloc[0]["candidate"]
best_cv_iso = cv_iso_outputs[best_cv_iso_name]

print("Best CV isotonic candidate:", best_cv_iso_name)
print("Best CV isotonic OOF RMSE:", best_cv_iso["oof_rmse"])
print("Old best OOF RMSE:", OLD_BEST_OOF_RMSE)
print("Delta vs old best:", best_cv_iso["oof_rmse"] - OLD_BEST_OOF_RMSE)

save_submission_from_pred_safe_11i(
    best_cv_iso["test_pred"],
    f"submission_{best_cv_iso_name}.csv"
)

print("Decision guide:")
print("- Strong submit: CV isotonic OOF < 11.33599")
print("- Medium submit: CV isotonic OOF close to old best, but very_high bias/RMSE improves clearly")
print("- Do not submit: CV isotonic OOF worse by > 0.05 without clear high-target improvement")

In [ ]:
from google.colab import files

files.download(f"submission_{best_cv_iso_name}.csv")

In [ ]:
# ==================================================
# SECTION 11J - Fold-Safe Residual Correction
# ==================================================
# Goal:
# Train a second-stage residual model to correct systematic errors from the ensemble.
#
# Why:
# - Current model underpredicts very_high target values.
# - Tail correction helped slightly.
# - Isotonic global overfit, CV isotonic failed.
# - Residual correction is tested fold-safely before submission.
#
# No external data. No AutoML.

OLD_BEST_OOF_RMSE = 11.335988679052521

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def make_base_prediction(weights):
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(sample_submission.shape[0])

    for model_name, weight in weights.items():
        oof_pred += weight * model_outputs[model_name]["oof"]
        test_pred += weight * model_outputs[model_name]["test_pred"]

    return np.clip(oof_pred, 0, None), np.clip(test_pred, 0, None)


def save_submission_safe(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)
    return sub


def target_bin_report(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(pred, 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5")
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
        .sort_values("rmse", ascending=False)
    )

    print("=" * 90)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(pred, 0, None)))
    display(report)
    return report


def fit_lgbm_residual(model, X_tr, y_tr, X_val, y_val):
    try:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(150, verbose=False),
                lgb.log_evaluation(0)
            ],
        )
    except TypeError:
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=150,
            verbose=False,
        )
    return model


# --------------------------------------------------
# Base candidates
# --------------------------------------------------

base_weight_sets_11j = {
    "old_best_050_025_025": {
        "catboost": 0.50,
        "lightgbm": 0.25,
        "xgboost": 0.25,
    },
    "equal_weight": {
        "catboost": 1/3,
        "lightgbm": 1/3,
        "xgboost": 1/3,
    },
    "catlow_025_035_040": {
        "catboost": 0.25,
        "lightgbm": 0.35,
        "xgboost": 0.40,
    },
}

# If best tail correction exists from Section 11H, add it as a base.
extra_base_predictions_11j = {}

if "best_tail" in globals():
    extra_base_predictions_11j["best_tail_from_11H"] = {
        "oof_pred": best_tail["oof_pred"],
        "test_pred": best_tail["test_pred"],
    }


# --------------------------------------------------
# Residual correction function
# --------------------------------------------------

def run_residual_correction_for_base(base_name, base_oof, base_test):
    print("=" * 100)
    print(f"Residual correction for base: {base_name}")

    base_oof = np.clip(np.asarray(base_oof), 0, None)
    base_test = np.clip(np.asarray(base_test), 0, None)

    base_rmse = rmse(y, base_oof)
    print("Base OOF RMSE:", base_rmse)

    # Residual target: actual - prediction
    residual_target = y.values - base_oof

    # Create residual model features
    X_res = X_fe.copy()
    X_res_test = X_test_fe.copy()

    X_res["base_pred"] = base_oof
    X_res_test["base_pred"] = base_test

    X_res["base_pred_sq"] = base_oof ** 2
    X_res_test["base_pred_sq"] = base_test ** 2

    X_res["base_pred_log1p"] = np.log1p(np.clip(base_oof, 0, None))
    X_res_test["base_pred_log1p"] = np.log1p(np.clip(base_test, 0, None))

    # High prediction flags, because main issue is high-target underprediction
    for q in [60, 70, 75, 80, 85, 90]:
        threshold = np.percentile(base_oof, q)
        X_res[f"base_pred_gt_p{q}"] = (base_oof > threshold).astype(int)
        X_res_test[f"base_pred_gt_p{q}"] = (base_test > threshold).astype(int)

    cat_cols_res = X_res.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols_res = [c for c in X_res.columns if c not in cat_cols_res]

    residual_oof = np.zeros(len(y))
    residual_test = np.zeros(sample_submission.shape[0])
    residual_fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw = X_res.iloc[tr_idx]
        X_val_raw = X_res.iloc[val_idx]
        X_te_raw = X_res_test

        y_tr_res = residual_target[tr_idx]
        y_val_res = residual_target[val_idx]

        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(
            X_tr_raw,
            X_val_raw,
            X_te_raw,
            cat_cols_res,
            num_cols_res
        )

        model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=2000,
            learning_rate=0.025,
            num_leaves=15,
            max_depth=5,
            subsample=0.75,
            colsample_bytree=0.75,
            reg_alpha=1.0,
            reg_lambda=10.0,
            min_child_samples=120,
            random_state=RANDOM_STATE + 1000 + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        model = fit_lgbm_residual(model, X_tr, y_tr_res, X_val, y_val_res)

        val_res_pred = model.predict(X_val)
        test_res_pred = model.predict(X_te)

        residual_oof[val_idx] = val_res_pred
        residual_test += test_res_pred / len(folds)

        fold_corrected = np.clip(base_oof[val_idx] + val_res_pred, 0, None)
        fold_score = rmse(y.iloc[val_idx], fold_corrected)
        residual_fold_scores.append(fold_score)

        print(f"Fold {fold}: corrected RMSE = {fold_score:.5f}")

    # Search shrinkage alpha
    rows = []
    outputs = {}

    for alpha in np.round(np.arange(0.05, 1.01, 0.05), 2):
        corrected_oof = np.clip(base_oof + alpha * residual_oof, 0, None)
        corrected_test = np.clip(base_test + alpha * residual_test, 0, None)

        score = rmse(y, corrected_oof)

        rows.append({
            "base": base_name,
            "alpha": alpha,
            "base_rmse": base_rmse,
            "corrected_rmse": score,
            "delta_vs_base": score - base_rmse,
            "delta_vs_old_best": score - OLD_BEST_OOF_RMSE,
        })

        outputs[alpha] = {
            "oof_pred": corrected_oof,
            "test_pred": corrected_test,
            "score": score,
        }

    result_df = pd.DataFrame(rows).sort_values("corrected_rmse")
    display(result_df.head(10))

    best_alpha = float(result_df.iloc[0]["alpha"])
    best_score = float(result_df.iloc[0]["corrected_rmse"])
    best_output = outputs[best_alpha]

    print("Best alpha:", best_alpha)
    print("Best corrected OOF RMSE:", best_score)
    print("Delta vs old best:", best_score - OLD_BEST_OOF_RMSE)

    target_bin_report(
        best_output["oof_pred"],
        f"Residual corrected: {base_name}, alpha={best_alpha}"
    )

    return {
        "base": base_name,
        "base_rmse": base_rmse,
        "best_alpha": best_alpha,
        "best_score": best_score,
        "best_oof_pred": best_output["oof_pred"],
        "best_test_pred": best_output["test_pred"],
        "result_df": result_df,
    }


# --------------------------------------------------
# Run residual correction
# --------------------------------------------------

residual_results = []

# Weighted bases
for base_name, weights in base_weight_sets_11j.items():
    base_oof, base_test = make_base_prediction(weights)
    result = run_residual_correction_for_base(base_name, base_oof, base_test)
    residual_results.append(result)

# Extra base from tail correction
for base_name, preds in extra_base_predictions_11j.items():
    result = run_residual_correction_for_base(
        base_name,
        preds["oof_pred"],
        preds["test_pred"]
    )
    residual_results.append(result)


residual_summary_df = pd.DataFrame([
    {
        "base": r["base"],
        "base_rmse": r["base_rmse"],
        "best_alpha": r["best_alpha"],
        "best_score": r["best_score"],
        "delta_vs_base": r["best_score"] - r["base_rmse"],
        "delta_vs_old_best": r["best_score"] - OLD_BEST_OOF_RMSE,
    }
    for r in residual_results
]).sort_values("best_score")

display(residual_summary_df)


# --------------------------------------------------
# Save best residual correction candidate
# --------------------------------------------------

best_residual_result = sorted(residual_results, key=lambda x: x["best_score"])[0]

best_residual_filename = (
    "submission_residual_"
    + best_residual_result["base"]
    + f"_alpha_{best_residual_result['best_alpha']:.2f}"
    + ".csv"
)

save_submission_safe(
    best_residual_result["best_test_pred"],
    best_residual_filename
)

print("Best residual candidate:", best_residual_filename)
print("Best residual OOF RMSE:", best_residual_result["best_score"])
print("Old best OOF RMSE:", OLD_BEST_OOF_RMSE)

print("Decision guide:")
print("- Strong submit if best residual OOF < old best OOF.")
print("- Medium submit if it improves very_high RMSE/bias clearly and OOF is close.")
print("- Do not submit if residual correction improves train-looking segments but overall OOF is worse.")

In [ ]:
# ==================================================
# SECTION 11K - Tail Correction Explainability & CV-Safe Check
# ==================================================
# Goal:
# 1. Explain why tail correction improves OOF RMSE.
# 2. Check whether the improvement comes from high-target samples.
# 3. Check what segments improve or get worse.
# 4. Validate tail correction parameters in a fold-safe way.
#
# No external data. No AutoML. Only OOF-based post-processing.

OLD_BEST_OOF_RMSE = 11.335988679052521

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def get_old_best_pred():
    """Rebuild old best ensemble prediction."""
    weights = {
        "catboost": 0.50,
        "lightgbm": 0.25,
        "xgboost": 0.25,
    }

    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(sample_submission.shape[0])

    for model_name, weight in weights.items():
        oof_pred += weight * model_outputs[model_name]["oof"]
        test_pred += weight * model_outputs[model_name]["test_pred"]

    return np.clip(oof_pred, 0, None), np.clip(test_pred, 0, None), weights


def apply_tail_correction(pred, threshold, alpha):
    """Apply smooth upper-tail correction."""
    pred = np.asarray(pred)
    corrected = pred + alpha * np.maximum(pred - threshold, 0)
    return np.clip(corrected, 0, None)


def save_submission_11k(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def make_error_frame(pred, label):
    df = train_raw[[ID_COL, TARGET]].copy()
    df["prediction"] = np.clip(pred, 0, None)
    df["error"] = df["prediction"] - df[TARGET]
    df["abs_error"] = df["error"].abs()
    df["squared_error"] = df["error"] ** 2
    df["model_label"] = label

    df["target_bin_5"] = pd.qcut(
        df[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    return df


def summarize_by_bin(error_frame, group_col):
    return (
        error_frame
        .groupby(group_col)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("prediction", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
        .sort_values("rmse", ascending=False)
    )


def compare_before_after(before_pred, after_pred, label_before, label_after):
    before_df = make_error_frame(before_pred, label_before)
    after_df = make_error_frame(after_pred, label_after)

    before_summary = summarize_by_bin(before_df, "target_bin_5")
    after_summary = summarize_by_bin(after_df, "target_bin_5")

    compare = before_summary.merge(
        after_summary,
        on="target_bin_5",
        suffixes=("_before", "_after")
    )

    compare["rmse_delta_after_minus_before"] = compare["rmse_after"] - compare["rmse_before"]
    compare["bias_delta_after_minus_before"] = compare["bias_after"] - compare["bias_before"]
    compare["mae_delta_after_minus_before"] = compare["mae_after"] - compare["mae_before"]

    print("=" * 90)
    print(f"Overall {label_before} RMSE:", rmse(y, before_pred))
    print(f"Overall {label_after} RMSE:", rmse(y, after_pred))
    print("Delta:", rmse(y, after_pred) - rmse(y, before_pred))
    print("=" * 90)

    display(compare.sort_values("rmse_delta_after_minus_before"))

    return compare


# --------------------------------------------------
# Part 1 - Rebuild old best and best tail candidate
# --------------------------------------------------

old_oof, old_test, old_weights = get_old_best_pred()

print("Old best weights:", old_weights)
print("Old best OOF RMSE:", rmse(y, old_oof))

# Use best_tail from Section 11H if available.
# If unavailable, recompute a simple tail search.
if "best_tail" in globals():
    tail_oof = best_tail["oof_pred"]
    tail_test = best_tail["test_pred"]
    tail_threshold = best_tail.get("threshold", None)
    tail_alpha = best_tail.get("alpha", None)

    print("Using best_tail from Section 11H")
    print("Tail threshold:", tail_threshold)
    print("Tail alpha:", tail_alpha)
    print("Tail OOF RMSE:", rmse(y, tail_oof))

else:
    print("best_tail not found. Recomputing simple tail search...")

    tail_rows = []
    tail_outputs = {}

    thresholds = [
        np.percentile(old_oof, 70),
        np.percentile(old_oof, 75),
        np.percentile(old_oof, 80),
        np.percentile(old_oof, 85),
    ]

    alphas = np.round(np.arange(0.02, 0.31, 0.02), 2)

    for threshold in thresholds:
        for alpha in alphas:
            candidate_oof = apply_tail_correction(old_oof, threshold, alpha)
            candidate_test = apply_tail_correction(old_test, threshold, alpha)
            score = rmse(y, candidate_oof)

            key = f"tail_t{threshold:.2f}_a{alpha:.2f}"
            tail_rows.append({
                "candidate": key,
                "threshold": threshold,
                "alpha": alpha,
                "oof_rmse": score,
                "delta_vs_old_best": score - OLD_BEST_OOF_RMSE,
            })
            tail_outputs[key] = {
                "oof_pred": candidate_oof,
                "test_pred": candidate_test,
                "threshold": threshold,
                "alpha": alpha,
                "oof_rmse": score,
            }

    tail_search_df = pd.DataFrame(tail_rows).sort_values("oof_rmse")
    display(tail_search_df.head(20))

    best_key = tail_search_df.iloc[0]["candidate"]
    best_tail_local = tail_outputs[best_key]

    tail_oof = best_tail_local["oof_pred"]
    tail_test = best_tail_local["test_pred"]
    tail_threshold = best_tail_local["threshold"]
    tail_alpha = best_tail_local["alpha"]

    print("Best recomputed tail:", best_key)
    print("Tail threshold:", tail_threshold)
    print("Tail alpha:", tail_alpha)
    print("Tail OOF RMSE:", rmse(y, tail_oof))


# --------------------------------------------------
# Part 2 - Explain improvement by target range
# --------------------------------------------------

target_bin_compare_df = compare_before_after(
    old_oof,
    tail_oof,
    "old_best",
    "tail_correction"
)

print("Interpretation:")
print("- Negative RMSE delta means tail correction improved that target range.")
print("- Positive RMSE delta means tail correction harmed that target range.")
print("- Pay special attention to very_high because RMSE is sensitive to large errors.")


# --------------------------------------------------
# Part 3 - Explain improvement by prediction range
# --------------------------------------------------
# Tail correction only affects high predictions.
# This checks whether the affected prediction range actually improves.

prediction_range_df = train_raw[[ID_COL, TARGET]].copy()
prediction_range_df["old_pred"] = old_oof
prediction_range_df["tail_pred"] = tail_oof
prediction_range_df["old_error"] = prediction_range_df["old_pred"] - prediction_range_df[TARGET]
prediction_range_df["tail_error"] = prediction_range_df["tail_pred"] - prediction_range_df[TARGET]
prediction_range_df["old_sq_error"] = prediction_range_df["old_error"] ** 2
prediction_range_df["tail_sq_error"] = prediction_range_df["tail_error"] ** 2

prediction_range_df["old_pred_bin_5"] = pd.qcut(
    prediction_range_df["old_pred"],
    q=5,
    labels=["pred_very_low", "pred_low", "pred_medium", "pred_high", "pred_very_high"],
    duplicates="drop"
)

pred_bin_compare = (
    prediction_range_df
    .groupby("old_pred_bin_5")
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),
        old_pred_mean=("old_pred", "mean"),
        tail_pred_mean=("tail_pred", "mean"),
        old_bias=("old_error", "mean"),
        tail_bias=("tail_error", "mean"),
        old_rmse=("old_sq_error", lambda x: np.sqrt(np.mean(x))),
        tail_rmse=("tail_sq_error", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

pred_bin_compare["rmse_delta_tail_minus_old"] = pred_bin_compare["tail_rmse"] - pred_bin_compare["old_rmse"]
pred_bin_compare["bias_delta_tail_minus_old"] = pred_bin_compare["tail_bias"] - pred_bin_compare["old_bias"]

print("Prediction-bin comparison:")
display(pred_bin_compare.sort_values("rmse_delta_tail_minus_old"))

print("Interpretation:")
print("- Tail correction should mainly change pred_high / pred_very_high.")
print("- If low prediction bins get worse, alpha may be too aggressive or threshold too low.")


# --------------------------------------------------
# Part 4 - Segment-level gain/loss
# --------------------------------------------------
# Find where tail correction helps or hurts.

segment_cols_11k = [
    "source_id",
    "has_band_B_spectrum",
    "sampling_strategy",
    "geo_zone_macro",
    "geo_zone_meso",
    "geo_zone_micro",
    "land_cover_type",
    "biome",
    "parent_rock_type",
]

segment_cols_11k = [c for c in segment_cols_11k if c in train_raw.columns]

segment_gain_rows = []

for col in segment_cols_11k:
    temp = train_raw[[col]].copy()
    temp[TARGET] = y.values
    temp["old_pred"] = old_oof
    temp["tail_pred"] = tail_oof
    temp["old_error"] = temp["old_pred"] - temp[TARGET]
    temp["tail_error"] = temp["tail_pred"] - temp[TARGET]
    temp["old_sq_error"] = temp["old_error"] ** 2
    temp["tail_sq_error"] = temp["tail_error"] ** 2

    grouped = (
        temp.groupby(col)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            old_pred_mean=("old_pred", "mean"),
            tail_pred_mean=("tail_pred", "mean"),
            old_bias=("old_error", "mean"),
            tail_bias=("tail_error", "mean"),
            old_rmse=("old_sq_error", lambda x: np.sqrt(np.mean(x))),
            tail_rmse=("tail_sq_error", lambda x: np.sqrt(np.mean(x))),
        )
        .reset_index()
    )

    grouped = grouped[grouped["count"] >= 50].copy()
    grouped["segment_column"] = col
    grouped["segment_value"] = grouped[col].astype(str)
    grouped["rmse_delta_tail_minus_old"] = grouped["tail_rmse"] - grouped["old_rmse"]
    grouped["bias_delta_tail_minus_old"] = grouped["tail_bias"] - grouped["old_bias"]

    segment_gain_rows.append(
        grouped[
            [
                "segment_column",
                "segment_value",
                "count",
                "target_mean",
                "old_pred_mean",
                "tail_pred_mean",
                "old_bias",
                "tail_bias",
                "old_rmse",
                "tail_rmse",
                "rmse_delta_tail_minus_old",
                "bias_delta_tail_minus_old",
            ]
        ]
    )

segment_gain_df = pd.concat(segment_gain_rows, ignore_index=True)

print("Segments most improved by tail correction:")
display(segment_gain_df.sort_values("rmse_delta_tail_minus_old").head(20))

print("Segments most harmed by tail correction:")
display(segment_gain_df.sort_values("rmse_delta_tail_minus_old", ascending=False).head(20))


# --------------------------------------------------
# Part 5 - CV-safe tail parameter check
# --------------------------------------------------
# This checks whether choosing threshold/alpha is stable across calibration folds.

tail_cv_rows = []
tail_cv_oof = np.zeros(len(y))
fold_selected_params = []

threshold_percentiles = [70, 75, 80, 85, 90]
alpha_grid = np.round(np.arange(0.01, 0.21, 0.01), 2)

for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    best_fold = {
        "score": np.inf,
        "threshold_percentile": None,
        "threshold": None,
        "alpha": None,
    }

    # Tune only on calibration-train part
    for p in threshold_percentiles:
        threshold = np.percentile(old_oof[tr_idx], p)

        for alpha in alpha_grid:
            corrected_tr = apply_tail_correction(old_oof[tr_idx], threshold, alpha)
            score = rmse(y.iloc[tr_idx], corrected_tr)

            if score < best_fold["score"]:
                best_fold = {
                    "score": score,
                    "threshold_percentile": p,
                    "threshold": threshold,
                    "alpha": alpha,
                }

    # Apply selected fold params to calibration-validation part
    corrected_val = apply_tail_correction(
        old_oof[val_idx],
        best_fold["threshold"],
        best_fold["alpha"]
    )

    tail_cv_oof[val_idx] = corrected_val

    fold_rmse = rmse(y.iloc[val_idx], corrected_val)

    fold_selected_params.append({
        "fold": fold,
        "selected_threshold_percentile": best_fold["threshold_percentile"],
        "selected_threshold": best_fold["threshold"],
        "selected_alpha": best_fold["alpha"],
        "cal_train_rmse": best_fold["score"],
        "cal_valid_rmse": fold_rmse,
    })

tail_cv_param_df = pd.DataFrame(fold_selected_params)
tail_cv_rmse = rmse(y, tail_cv_oof)

print("CV-safe tail RMSE:", tail_cv_rmse)
print("Old best OOF RMSE:", OLD_BEST_OOF_RMSE)
print("Delta vs old best:", tail_cv_rmse - OLD_BEST_OOF_RMSE)

display(tail_cv_param_df)

target_bin_compare_cv_tail_df = compare_before_after(
    old_oof,
    tail_cv_oof,
    "old_best",
    "cv_safe_tail"
)


# --------------------------------------------------
# Part 6 - Final decision and save CV-safe tail submission
# --------------------------------------------------

# Fit final parameters on all OOF for test-time prediction
final_tail_rows = []

for p in threshold_percentiles:
    threshold = np.percentile(old_oof, p)

    for alpha in alpha_grid:
        corrected_oof = apply_tail_correction(old_oof, threshold, alpha)
        score = rmse(y, corrected_oof)

        final_tail_rows.append({
            "threshold_percentile": p,
            "threshold": threshold,
            "alpha": alpha,
            "oof_rmse": score,
            "delta_vs_old_best": score - OLD_BEST_OOF_RMSE,
        })

final_tail_df = pd.DataFrame(final_tail_rows).sort_values("oof_rmse")
display(final_tail_df.head(20))

best_final_tail = final_tail_df.iloc[0].to_dict()

final_tail_test = apply_tail_correction(
    old_test,
    best_final_tail["threshold"],
    best_final_tail["alpha"]
)

print("Best final tail params:")
print(best_final_tail)

save_submission_11k(
    final_tail_test,
    "submission_tail_cv_checked.csv"
)

print("Decision guide:")
print("- Strong submit if CV-safe tail RMSE < old best OOF RMSE.")
print("- Medium submit if global tail is better and CV-safe tail is close, with very_high bias improvement.")
print("- Do not submit if CV-safe tail is much worse or harms many non-high segments.")

In [ ]:
# ==================================================
# SECTION 11L - Gated Tail Correction
# ==================================================
# Goal:
# Improve tail correction by applying stronger boost only when a sample is likely
# to belong to the high organic-content group.
#
# Why:
# - Plain tail correction improves very_high target range.
# - But it slightly harms some low/medium/high ranges and several segments.
# - Gated tail correction uses a fold-safe high-target classifier to decide
#   how much tail correction should be applied.
#
# No external data. No AutoML.

from lightgbm import LGBMClassifier

OLD_BEST_OOF_RMSE = 11.335988679052521

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def rebuild_old_best_ensemble():
    weights = {
        "catboost": 0.50,
        "lightgbm": 0.25,
        "xgboost": 0.25,
    }

    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(sample_submission.shape[0])

    for model_name, weight in weights.items():
        oof_pred += weight * model_outputs[model_name]["oof"]
        test_pred += weight * model_outputs[model_name]["test_pred"]

    return np.clip(oof_pred, 0, None), np.clip(test_pred, 0, None)


def apply_gated_tail(pred, prob_high, threshold, alpha, power=1.0):
    pred = np.asarray(pred)
    prob_high = np.asarray(prob_high)

    gate = np.power(np.clip(prob_high, 0, 1), power)
    correction = alpha * np.maximum(pred - threshold, 0) * gate

    return np.clip(pred + correction, 0, None)


def save_submission_11l(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def make_target_bin_report_11l(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(pred, 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5")
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
        .sort_values("rmse", ascending=False)
    )

    print("=" * 90)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(pred, 0, None)))
    display(report)

    return report


def compare_old_vs_new_11l(old_pred, new_pred, label):
    old_df = train_raw[[ID_COL, TARGET]].copy()
    old_df["old_pred"] = old_pred
    old_df["new_pred"] = new_pred
    old_df["old_error"] = old_df["old_pred"] - old_df[TARGET]
    old_df["new_error"] = old_df["new_pred"] - old_df[TARGET]
    old_df["old_sq_error"] = old_df["old_error"] ** 2
    old_df["new_sq_error"] = old_df["new_error"] ** 2

    old_df["target_bin_5"] = pd.qcut(
        old_df[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    compare = (
        old_df.groupby("target_bin_5")
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            old_pred_mean=("old_pred", "mean"),
            new_pred_mean=("new_pred", "mean"),
            old_bias=("old_error", "mean"),
            new_bias=("new_error", "mean"),
            old_rmse=("old_sq_error", lambda x: np.sqrt(np.mean(x))),
            new_rmse=("new_sq_error", lambda x: np.sqrt(np.mean(x))),
        )
        .reset_index()
    )

    compare["rmse_delta_new_minus_old"] = compare["new_rmse"] - compare["old_rmse"]
    compare["bias_delta_new_minus_old"] = compare["new_bias"] - compare["old_bias"]

    print("=" * 90)
    print(label)
    print("Old RMSE:", rmse(y, old_pred))
    print("New RMSE:", rmse(y, new_pred))
    print("Delta:", rmse(y, new_pred) - rmse(y, old_pred))
    display(compare.sort_values("rmse_delta_new_minus_old"))

    return compare


# --------------------------------------------------
# Part 1 - Base prediction
# --------------------------------------------------

old_oof_11l, old_test_11l = rebuild_old_best_ensemble()

print("Old best OOF RMSE:", rmse(y, old_oof_11l))

# Define high target label.
# Use top 20% target as high-organic class because very_high bin was the main error source.
high_threshold_target = y.quantile(0.80)
high_target_label = (y >= high_threshold_target).astype(int)

print("High target threshold:", high_threshold_target)
print("High target positive rate:", high_target_label.mean())


# --------------------------------------------------
# Part 2 - Fold-safe high-target probability model
# --------------------------------------------------

X_gate = X_fe.copy()
X_gate_test = X_test_fe.copy()

# Add base prediction as a feature because the gate should know where the old model predicts high.
X_gate["base_pred"] = old_oof_11l
X_gate_test["base_pred"] = old_test_11l

X_gate["base_pred_log1p"] = np.log1p(old_oof_11l)
X_gate_test["base_pred_log1p"] = np.log1p(old_test_11l)

X_gate["base_pred_sq"] = old_oof_11l ** 2
X_gate_test["base_pred_sq"] = old_test_11l ** 2

cat_cols_gate = X_gate.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols_gate = [c for c in X_gate.columns if c not in cat_cols_gate]

gate_oof_prob = np.zeros(len(y))
gate_test_prob = np.zeros(sample_submission.shape[0])

gate_fold_rows = []

for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    X_tr_raw = X_gate.iloc[tr_idx]
    X_val_raw = X_gate.iloc[val_idx]
    X_te_raw = X_gate_test

    y_tr_cls = high_target_label.iloc[tr_idx]
    y_val_cls = high_target_label.iloc[val_idx]

    X_tr, X_val, X_te, feature_names = prepare_encoded_fold(
        X_tr_raw,
        X_val_raw,
        X_te_raw,
        cat_cols_gate,
        num_cols_gate
    )

    clf = LGBMClassifier(
        objective="binary",
        n_estimators=1500,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=-1,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.5,
        reg_lambda=5.0,
        min_child_samples=80,
        random_state=RANDOM_STATE + 2000 + fold,
        n_jobs=-1,
        verbosity=-1,
    )

    try:
        clf.fit(
            X_tr,
            y_tr_cls,
            eval_set=[(X_val, y_val_cls)],
            eval_metric="binary_logloss",
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(0)
            ],
        )
    except TypeError:
        clf.fit(
            X_tr,
            y_tr_cls,
            eval_set=[(X_val, y_val_cls)],
            eval_metric="binary_logloss",
            early_stopping_rounds=100,
            verbose=False,
        )

    val_prob = clf.predict_proba(X_val)[:, 1]
    test_prob = clf.predict_proba(X_te)[:, 1]

    gate_oof_prob[val_idx] = val_prob
    gate_test_prob += test_prob / len(folds)

    # Simple diagnostic by fold
    top_prob_rate = (val_prob >= 0.5).mean()

    gate_fold_rows.append({
        "fold": fold,
        "val_positive_rate": y_val_cls.mean(),
        "pred_prob_mean": val_prob.mean(),
        "pred_prob_std": val_prob.std(),
        "prob_ge_0_5_rate": top_prob_rate,
    })

gate_fold_df = pd.DataFrame(gate_fold_rows)
display(gate_fold_df)

print("OOF gate probability summary:")
display(pd.Series(gate_oof_prob).describe().to_frame("gate_oof_prob").T)

print("Test gate probability summary:")
display(pd.Series(gate_test_prob).describe().to_frame("gate_test_prob").T)


# --------------------------------------------------
# Part 3 - Search gated tail correction
# --------------------------------------------------

threshold_percentiles = [70, 75, 80, 85, 90]
alpha_grid = np.round(np.arange(0.02, 0.31, 0.02), 2)
power_grid = [0.5, 1.0, 1.5, 2.0]

gated_rows = []
gated_outputs = {}

for p in threshold_percentiles:
    threshold = np.percentile(old_oof_11l, p)

    for alpha in alpha_grid:
        for power in power_grid:
            corrected_oof = apply_gated_tail(
                old_oof_11l,
                gate_oof_prob,
                threshold,
                alpha,
                power
            )

            score = rmse(y, corrected_oof)

            candidate_name = f"gated_tail_p{p}_a{alpha:.2f}_pow{power}"

            gated_rows.append({
                "candidate": candidate_name,
                "threshold_percentile": p,
                "threshold": threshold,
                "alpha": alpha,
                "power": power,
                "oof_rmse": score,
                "delta_vs_old_best": score - OLD_BEST_OOF_RMSE,
            })

            corrected_test = apply_gated_tail(
                old_test_11l,
                gate_test_prob,
                threshold,
                alpha,
                power
            )

            gated_outputs[candidate_name] = {
                "oof_pred": corrected_oof,
                "test_pred": corrected_test,
                "oof_rmse": score,
                "threshold_percentile": p,
                "threshold": threshold,
                "alpha": alpha,
                "power": power,
            }

gated_tail_df = pd.DataFrame(gated_rows).sort_values("oof_rmse")
display(gated_tail_df.head(30))

best_gated_name = gated_tail_df.iloc[0]["candidate"]
best_gated = gated_outputs[best_gated_name]

print("Best gated tail candidate:", best_gated_name)
print("Best gated tail OOF RMSE:", best_gated["oof_rmse"])
print("Old best OOF RMSE:", OLD_BEST_OOF_RMSE)
print("Delta vs old best:", best_gated["oof_rmse"] - OLD_BEST_OOF_RMSE)

make_target_bin_report_11l(
    best_gated["oof_pred"],
    f"Best gated tail: {best_gated_name}"
)

compare_old_vs_new_11l(
    old_oof_11l,
    best_gated["oof_pred"],
    "Old best vs gated tail"
)


# --------------------------------------------------
# Part 4 - Compare gated tail vs plain tail if available
# --------------------------------------------------

candidate_comparison_rows = [
    {
        "candidate": "old_best",
        "oof_rmse": rmse(y, old_oof_11l),
        "delta_vs_old_best": 0.0,
    },
    {
        "candidate": best_gated_name,
        "oof_rmse": best_gated["oof_rmse"],
        "delta_vs_old_best": best_gated["oof_rmse"] - OLD_BEST_OOF_RMSE,
    },
]

if "tail_cv_rmse" in globals():
    candidate_comparison_rows.append({
        "candidate": "cv_safe_tail_from_11K",
        "oof_rmse": tail_cv_rmse,
        "delta_vs_old_best": tail_cv_rmse - OLD_BEST_OOF_RMSE,
    })

if "best_tail" in globals():
    candidate_comparison_rows.append({
        "candidate": "global_best_tail_from_11H",
        "oof_rmse": rmse(y, best_tail["oof_pred"]),
        "delta_vs_old_best": rmse(y, best_tail["oof_pred"]) - OLD_BEST_OOF_RMSE,
    })

candidate_comparison_11l = pd.DataFrame(candidate_comparison_rows).sort_values("oof_rmse")
display(candidate_comparison_11l)


# --------------------------------------------------
# Part 5 - Save best gated tail submission
# --------------------------------------------------

save_submission_11l(
    best_gated["test_pred"],
    "submission_gated_tail_correction.csv"
)

print("Decision guide:")
print("- Strong submit if gated tail OOF < plain tail / cv-safe tail.")
print("- Medium submit if gated tail is better than old best and harms fewer low/medium bins.")
print("- Do not submit if gated tail is worse than old best or only improves by a tiny amount with unstable distribution.")

In [ ]:
# ==================================================
# SECTION 11M - Multi-Strategy Stacked Correction
# ==================================================
# Requires: all cells from Section 1-11L must be run first.
# Dependencies: model_outputs, y, folds, sample_submission, train_raw,
#               X_fe, X_test_fe, best_tail (from 11H), best_gated (from 11L)

from scipy.optimize import minimize_scalar
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

OLD_BEST = 11.335988679052521
TAIL_BEST = 11.31590

# --- Helpers ---
def rebuild_ens():
    w = {"catboost": 0.50, "lightgbm": 0.25, "xgboost": 0.25}
    oof = sum(w[n] * model_outputs[n]["oof"] for n in w)
    tst = sum(w[n] * model_outputs[n]["test_pred"] for n in w)
    return np.clip(oof, 0, None), np.clip(tst, 0, None)

def rpt(pred, lbl):
    t = train_raw[[ID_COL, TARGET]].copy()
    t["p"] = np.clip(pred, 0, None)
    t["e"] = t["p"] - t[TARGET]
    t["se"] = t["e"] ** 2
    t["bin"] = pd.qcut(t[TARGET], 5, labels=["very_low","low","medium","high","very_high"], duplicates="drop")
    r = t.groupby("bin").agg(n=(TARGET,"count"), tgt_mean=(TARGET,"mean"),
        pred_mean=("p","mean"), bias=("e","mean"),
        seg_rmse=("se", lambda x: np.sqrt(np.mean(x)))).reset_index()
    ov = rmse(y, np.clip(pred, 0, None))
    print(f"{lbl} | Overall RMSE: {ov:.5f}")
    display(r.sort_values("seg_rmse", ascending=False))
    return ov

base_oof, base_test = rebuild_ens()
print("Base RMSE:", rmse(y, base_oof))

# ============================================================
# PART 1: Dual-Tail Correction (upper + lower)
# ============================================================
# Insight: very_low has bias +5.18 (overprediction) - never addressed before.
print("\n" + "=" * 80)
print("PART 1: Dual-Tail Correction")

dt_best = {"score": np.inf}
for pU in [70, 75, 80]:
    tU = np.percentile(base_oof, pU)
    for pL in [20, 25, 30]:
        tL = np.percentile(base_oof, pL)
        for aU in np.arange(0.02, 0.16, 0.02):
            for aL in np.arange(0.01, 0.08, 0.01):
                adj = base_oof + aU * np.maximum(base_oof - tU, 0) - aL * np.maximum(tL - base_oof, 0)
                s = rmse(y, np.clip(adj, 0, None))
                if s < dt_best["score"]:
                    dt_best = {"score": s, "pU": pU, "pL": pL,
                               "aU": round(float(aU), 2), "aL": round(float(aL), 2),
                               "tU": tU, "tL": tL}

dual_oof = np.clip(base_oof + dt_best["aU"] * np.maximum(base_oof - dt_best["tU"], 0)
                   - dt_best["aL"] * np.maximum(dt_best["tL"] - base_oof, 0), 0, None)
dual_test = np.clip(base_test + dt_best["aU"] * np.maximum(base_test - dt_best["tU"], 0)
                    - dt_best["aL"] * np.maximum(dt_best["tL"] - base_test, 0), 0, None)

print(f"Best dual-tail: pU={dt_best['pU']}, pL={dt_best['pL']}, aU={dt_best['aU']}, aL={dt_best['aL']}")
print(f"RMSE: {dt_best['score']:.5f} (vs tail_best: {dt_best['score'] - TAIL_BEST:+.5f})")
rpt(dual_oof, "Dual-tail")

# ============================================================
# PART 2: Ridge Stacking
# ============================================================
print("\n" + "=" * 80)
print("PART 2: Ridge Stacking")

S_oof = np.column_stack([model_outputs[n]["oof"] for n in ["catboost", "lightgbm", "xgboost"]])
S_test = np.column_stack([model_outputs[n]["test_pred"] for n in ["catboost", "lightgbm", "xgboost"]])
S_oof_ext = np.column_stack([S_oof, S_oof ** 2,
    S_oof[:, 0] * S_oof[:, 1], S_oof[:, 0] * S_oof[:, 2], S_oof[:, 1] * S_oof[:, 2]])
S_test_ext = np.column_stack([S_test, S_test ** 2,
    S_test[:, 0] * S_test[:, 1], S_test[:, 0] * S_test[:, 2], S_test[:, 1] * S_test[:, 2]])

ridge_results = []
for alpha in [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0]:
    r_oof = np.zeros(len(y))
    r_test = np.zeros(len(base_test))
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        sc = StandardScaler()
        Xtr = sc.fit_transform(S_oof_ext[tr_idx])
        Xva = sc.transform(S_oof_ext[val_idx])
        Xte = sc.transform(S_test_ext)
        mdl = Ridge(alpha=alpha, random_state=RANDOM_STATE)
        mdl.fit(Xtr, y.values[tr_idx])
        r_oof[val_idx] = mdl.predict(Xva)
        r_test += mdl.predict(Xte) / len(folds)
    r_oof = np.clip(r_oof, 0, None)
    r_test = np.clip(r_test, 0, None)
    s = rmse(y, r_oof)
    ridge_results.append({"alpha": alpha, "rmse": s, "oof": r_oof, "test": r_test})
    print(f"  alpha={alpha}: RMSE={s:.5f}")

best_r = min(ridge_results, key=lambda x: x["rmse"])
ridge_oof, ridge_test = best_r["oof"], best_r["test"]
print(f"Best Ridge alpha={best_r['alpha']}, RMSE={best_r['rmse']:.5f}")
rpt(ridge_oof, "Ridge stack")

# ============================================================
# PART 3: Tail + Dual-Tail on Ridge
# ============================================================
print("\n" + "=" * 80)
print("PART 3: Tail on Ridge Stack")

rt_best = {"score": np.inf}
for p in [70, 75, 80, 85]:
    th = np.percentile(ridge_oof, p)
    for a in np.arange(0.02, 0.16, 0.02):
        adj = ridge_oof + a * np.maximum(ridge_oof - th, 0)
        s = rmse(y, np.clip(adj, 0, None))
        if s < rt_best["score"]:
            rt_best = {"score": s, "p": p, "a": round(float(a), 2), "th": th}

rt_oof = np.clip(ridge_oof + rt_best["a"] * np.maximum(ridge_oof - rt_best["th"], 0), 0, None)
rt_test = np.clip(ridge_test + rt_best["a"] * np.maximum(ridge_test - rt_best["th"], 0), 0, None)
print(f"Ridge+Tail: p={rt_best['p']}, a={rt_best['a']}, RMSE={rt_best['score']:.5f}")
rpt(rt_oof, "Ridge+Tail")

print("\n--- Dual-Tail on Ridge ---")
rd_best = {"score": np.inf}
for pU in [70, 75, 80]:
    tU = np.percentile(ridge_oof, pU)
    for pL in [20, 25, 30]:
        tL = np.percentile(ridge_oof, pL)
        for aU in np.arange(0.02, 0.12, 0.02):
            for aL in np.arange(0.01, 0.06, 0.01):
                adj = ridge_oof + aU * np.maximum(ridge_oof - tU, 0) - aL * np.maximum(tL - ridge_oof, 0)
                s = rmse(y, np.clip(adj, 0, None))
                if s < rd_best["score"]:
                    rd_best = {"score": s, "pU": pU, "pL": pL,
                               "aU": round(float(aU), 2), "aL": round(float(aL), 2),
                               "tU": tU, "tL": tL}

rd_oof = np.clip(ridge_oof + rd_best["aU"] * np.maximum(ridge_oof - rd_best["tU"], 0)
                 - rd_best["aL"] * np.maximum(rd_best["tL"] - ridge_oof, 0), 0, None)
rd_test = np.clip(ridge_test + rd_best["aU"] * np.maximum(ridge_test - rd_best["tU"], 0)
                  - rd_best["aL"] * np.maximum(rd_best["tL"] - ridge_test, 0), 0, None)
print(f"Ridge+Dual: pU={rd_best['pU']}, pL={rd_best['pL']}, aU={rd_best['aU']}, aL={rd_best['aL']}")
print(f"RMSE={rd_best['score']:.5f} (vs tail: {rd_best['score'] - TAIL_BEST:+.5f})")
rpt(rd_oof, "Ridge+Dual-Tail")

# ============================================================
# PART 4: Final Comparison, Blend & Save
# ============================================================
print("\n" + "=" * 80)
print("PART 4: Final Comparison & Blend")

cands = {
    "old_ensemble": (base_oof, base_test),
    "dual_tail": (dual_oof, dual_test),
    "ridge_stack": (ridge_oof, ridge_test),
    "ridge_tail": (rt_oof, rt_test),
    "ridge_dual": (rd_oof, rd_test),
}
if "best_tail" in dir() or "best_tail" in globals():
    cands["global_tail_11H"] = (best_tail["oof_pred"], best_tail["test_pred"])
if "best_gated" in dir() or "best_gated" in globals():
    cands["gated_tail_11L"] = (best_gated["oof_pred"], best_gated["test_pred"])

comp = []
for name, (o, t) in cands.items():
    s = rmse(y, np.clip(o, 0, None))
    comp.append({"candidate": name, "oof_rmse": s,
                 "delta_vs_old": s - OLD_BEST, "delta_vs_tail": s - TAIL_BEST})
comp_df = pd.DataFrame(comp).sort_values("oof_rmse")
print("All candidates:")
display(comp_df)

# Blend top 3
top3 = comp_df["candidate"].tolist()[:3]
print(f"\nBlending top 3: {top3}")
blend_best = {"score": np.inf, "w": None}
step = 0.05
for w1 in np.arange(0, 1 + step, step):
    for w2 in np.arange(0, 1 + step - w1, step):
        w3 = round(1 - w1 - w2, 2)
        if w3 < -0.001 or w3 > 1.001:
            continue
        w3 = max(w3, 0.0)
        bo = w1 * cands[top3[0]][0] + w2 * cands[top3[1]][0] + w3 * cands[top3[2]][0]
        s = rmse(y, np.clip(bo, 0, None))
        if s < blend_best["score"]:
            blend_best = {"score": s,
                          "w": {top3[0]: round(float(w1), 2),
                                top3[1]: round(float(w2), 2),
                                top3[2]: round(float(w3), 2)}}

print(f"Best blend weights: {blend_best['w']}")
print(f"Blend RMSE: {blend_best['score']:.5f} (vs tail: {blend_best['score'] - TAIL_BEST:+.5f})")

bl_oof = sum(w * cands[n][0] for n, w in blend_best["w"].items())
bl_test = sum(w * cands[n][1] for n, w in blend_best["w"].items())
bl_oof, bl_test = np.clip(bl_oof, 0, None), np.clip(bl_test, 0, None)
rpt(bl_oof, "Best Blend")

# --- Save winner ---
all_11m = {
    "dual_tail": (dual_oof, dual_test, dt_best["score"]),
    "ridge_tail": (rt_oof, rt_test, rt_best["score"]),
    "ridge_dual": (rd_oof, rd_test, rd_best["score"]),
    "blend": (bl_oof, bl_test, blend_best["score"]),
}
winner_name = min(all_11m, key=lambda k: all_11m[k][2])
w_oof, w_test, w_score = all_11m[winner_name]
print(f"\n{'='*80}")
print(f"11M WINNER: {winner_name}, RMSE: {w_score:.5f}")
print(f"Delta vs old ensemble: {w_score - OLD_BEST:+.5f}")
print(f"Delta vs best tail 11H: {w_score - TAIL_BEST:+.5f}")

sub = sample_submission.copy()
sub[TARGET] = np.clip(w_test, 0, None)
assert sub[TARGET].isna().sum() == 0 and (sub[TARGET] >= 0).all()
sub.to_csv("submission_11m_best.csv", index=False)
print("Saved submission_11m_best.csv")
display(sub[TARGET].describe().to_frame().T)

sub2 = sample_submission.copy()
sub2[TARGET] = np.clip(bl_test, 0, None)
sub2.to_csv("submission_11m_blend.csv", index=False)
print("Saved submission_11m_blend.csv")


In [ ]:
# ==================================================
# SECTION 11N - Log-Target Retraining + Expanded Stacking
# ==================================================
# Root cause from 11M:
# 1. Ridge stacking (11.31405) is best, but very_high bias still -7.4
# 2. Tree models compress predictions toward mean (regression to mean)
# 3. Post-processing on Ridge fails (over-correction)
#
# Fix: retrain models with log1p(target) to naturally decompress high values.
# Then Ridge-stack ALL 6 model OOFs (3 original + 3 log-target) together.
# The stacker can learn to use log-target models more for high predictions.
#
# No external data. No AutoML.

OLD_BEST = 11.335988679052521
RIDGE_BEST = 11.31405

# --------------------------------------------------
# PART 1: Train log1p(target) models
# --------------------------------------------------
print("=" * 80)
print("PART 1: Log-target model training")
print("=" * 80)

y_log = np.log1p(y)

# --- CatBoost log-target ---
print("\nTraining CatBoost on log1p(target)...")
cb_log_oof = np.zeros(len(y))
cb_log_test = np.zeros(len(test_raw))

X_cb_full = prepare_catboost_frame(X_fe, cat_features)
X_test_cb = prepare_catboost_frame(X_test_fe, cat_features)
cat_indices = [X_cb_full.columns.get_loc(c) for c in cat_features if c in X_cb_full.columns]

for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    X_tr, X_val = X_cb_full.iloc[tr_idx], X_cb_full.iloc[val_idx]
    y_tr_log, y_val_log = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    mdl = CatBoostRegressor(
        loss_function="RMSE", eval_metric="RMSE",
        iterations=MAIN_N_ESTIMATORS, learning_rate=0.03,
        depth=6, l2_leaf_reg=5, random_strength=1.0,
        bagging_temperature=0.5, random_seed=RANDOM_STATE + fold,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=200, allow_writing_files=False,
    )
    mdl.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log),
            cat_features=cat_indices, use_best_model=True)

    cb_log_oof[val_idx] = np.expm1(mdl.predict(X_val))
    cb_log_test += np.expm1(mdl.predict(X_test_cb)) / len(folds)
    fold_rmse_val = rmse(y.iloc[val_idx], np.clip(cb_log_oof[val_idx], 0, None))
    print(f"  CatBoost-log fold {fold}: RMSE (original scale) = {fold_rmse_val:.5f}")

cb_log_oof = np.clip(cb_log_oof, 0, None)
cb_log_test = np.clip(cb_log_test, 0, None)
print(f"CatBoost-log OOF RMSE: {rmse(y, cb_log_oof):.5f}")

# --- LightGBM log-target ---
print("\nTraining LightGBM on log1p(target)...")
lgb_log_oof = np.zeros(len(y))
lgb_log_test = np.zeros(len(test_raw))

for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    X_tr_raw, X_val_raw = X_fe.iloc[tr_idx], X_fe.iloc[val_idx]
    y_tr_log, y_val_log = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    X_tr, X_val, X_te, fn = prepare_encoded_fold(
        X_tr_raw, X_val_raw, X_test_fe, cat_features, num_features)

    mdl = LGBMRegressor(
        objective="regression", metric="rmse",
        n_estimators=MAIN_N_ESTIMATORS, learning_rate=0.03,
        num_leaves=31, max_depth=-1, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        min_child_samples=20, random_state=RANDOM_STATE + fold,
        n_jobs=-1, verbosity=-1,
    )
    try:
        mdl.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)],
                eval_metric="rmse",
                callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)])
    except TypeError:
        mdl.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)],
                eval_metric="rmse",
                early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)

    lgb_log_oof[val_idx] = np.expm1(mdl.predict(X_val))
    lgb_log_test += np.expm1(mdl.predict(X_te)) / len(folds)
    fold_rmse_val = rmse(y.iloc[val_idx], np.clip(lgb_log_oof[val_idx], 0, None))
    print(f"  LightGBM-log fold {fold}: RMSE = {fold_rmse_val:.5f}")

lgb_log_oof = np.clip(lgb_log_oof, 0, None)
lgb_log_test = np.clip(lgb_log_test, 0, None)
print(f"LightGBM-log OOF RMSE: {rmse(y, lgb_log_oof):.5f}")

# --- XGBoost log-target ---
print("\nTraining XGBoost on log1p(target)...")
xgb_log_oof = np.zeros(len(y))
xgb_log_test = np.zeros(len(test_raw))

for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    X_tr_raw, X_val_raw = X_fe.iloc[tr_idx], X_fe.iloc[val_idx]
    y_tr_log, y_val_log = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    X_tr, X_val, X_te, fn = prepare_encoded_fold(
        X_tr_raw, X_val_raw, X_test_fe, cat_features, num_features)

    xgb_params = dict(
        objective="reg:squarederror", eval_metric="rmse",
        n_estimators=MAIN_N_ESTIMATORS, learning_rate=0.03,
        max_depth=6, min_child_weight=20, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE + fold, tree_method="hist", n_jobs=-1,
    )
    try:
        mdl = XGBRegressor(**xgb_params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        mdl.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)], verbose=0)
    except TypeError:
        mdl = XGBRegressor(**xgb_params)
        mdl.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)],
                early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=0)

    xgb_log_oof[val_idx] = np.expm1(mdl.predict(X_val))
    xgb_log_test += np.expm1(mdl.predict(X_te)) / len(folds)
    fold_rmse_val = rmse(y.iloc[val_idx], np.clip(xgb_log_oof[val_idx], 0, None))
    print(f"  XGBoost-log fold {fold}: RMSE = {fold_rmse_val:.5f}")

xgb_log_oof = np.clip(xgb_log_oof, 0, None)
xgb_log_test = np.clip(xgb_log_test, 0, None)
print(f"XGBoost-log OOF RMSE: {rmse(y, xgb_log_oof):.5f}")

# --------------------------------------------------
# PART 2: Expanded Ridge Stacking (6 models)
# --------------------------------------------------
print("\n" + "=" * 80)
print("PART 2: Expanded Ridge Stacking (3 original + 3 log-target)")
print("=" * 80)

S6_oof = np.column_stack([
    model_outputs["catboost"]["oof"],
    model_outputs["lightgbm"]["oof"],
    model_outputs["xgboost"]["oof"],
    cb_log_oof,
    lgb_log_oof,
    xgb_log_oof,
])

S6_test = np.column_stack([
    model_outputs["catboost"]["test_pred"],
    model_outputs["lightgbm"]["test_pred"],
    model_outputs["xgboost"]["test_pred"],
    cb_log_test,
    lgb_log_test,
    xgb_log_test,
])

model_labels = ["cat", "lgb", "xgb", "cat_log", "lgb_log", "xgb_log"]

# Also build extended features (squared + pairwise for original 3 + log 3)
S6_ext = np.column_stack([S6_oof, S6_oof ** 2])
S6_test_ext = np.column_stack([S6_test, S6_test ** 2])

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler

# Test Ridge and ElasticNet
stack6_rows = []

for alpha in [1.0, 5.0, 10.0, 20.0, 50.0, 100.0]:
    for use_ext in [False, True]:
        feat_oof = S6_ext if use_ext else S6_oof
        feat_test = S6_test_ext if use_ext else S6_test

        r_oof = np.zeros(len(y))
        r_test = np.zeros(len(base_test))

        for fold_idx, (tr_idx, val_idx) in enumerate(folds):
            sc = StandardScaler()
            Xtr = sc.fit_transform(feat_oof[tr_idx])
            Xva = sc.transform(feat_oof[val_idx])
            Xte = sc.transform(feat_test)
            mdl = Ridge(alpha=alpha, random_state=RANDOM_STATE)
            mdl.fit(Xtr, y.values[tr_idx])
            r_oof[val_idx] = mdl.predict(Xva)
            r_test += mdl.predict(Xte) / len(folds)

        r_oof = np.clip(r_oof, 0, None)
        r_test = np.clip(r_test, 0, None)
        s = rmse(y, r_oof)
        ext_label = "ext" if use_ext else "raw"
        stack6_rows.append({
            "model": f"Ridge6_{ext_label}_a{alpha}",
            "oof_rmse": s,
            "delta_vs_old": s - OLD_BEST,
            "delta_vs_ridge3": s - RIDGE_BEST,
            "oof": r_oof, "test": r_test,
        })
        print(f"  Ridge6 {ext_label} alpha={alpha}: RMSE={s:.5f} (vs Ridge3: {s - RIDGE_BEST:+.5f})")

stack6_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["oof", "test"]}
                          for r in stack6_rows]).sort_values("oof_rmse")
print("\nAll 6-model stacking results:")
display(stack6_df)

best_s6 = min(stack6_rows, key=lambda x: x["oof_rmse"])
s6_oof, s6_test = best_s6["oof"], best_s6["test"]
print(f"\nBest 6-model stack: {best_s6['model']}, RMSE={best_s6['oof_rmse']:.5f}")

# Bin report
t = train_raw[[ID_COL, TARGET]].copy()
t["p"] = s6_oof
t["e"] = t["p"] - t[TARGET]
t["se"] = t["e"] ** 2
t["bin"] = pd.qcut(t[TARGET], 5, labels=["very_low","low","medium","high","very_high"], duplicates="drop")
r = t.groupby("bin").agg(n=(TARGET,"count"), tgt_mean=(TARGET,"mean"),
    pred_mean=("p","mean"), bias=("e","mean"),
    seg_rmse=("se", lambda x: np.sqrt(np.mean(x)))).reset_index()
print(f"Best 6-model stack | Overall RMSE: {best_s6['oof_rmse']:.5f}")
display(r.sort_values("seg_rmse", ascending=False))

# --------------------------------------------------
# PART 3: Simple Blend - Ridge3 vs Ridge6
# --------------------------------------------------
print("\n" + "=" * 80)
print("PART 3: Blend Ridge3 vs Ridge6")
print("=" * 80)

# Get Ridge3 predictions from 11M
ridge3_oof = best_r["oof"] if "best_r" in dir() else ridge_oof
ridge3_test = best_r["test"] if "best_r" in dir() else ridge_test

blend_n_rows = []
for w6 in np.arange(0, 1.01, 0.05):
    w3 = round(1 - w6, 2)
    bl = np.clip(w6 * s6_oof + w3 * ridge3_oof, 0, None)
    s = rmse(y, bl)
    blend_n_rows.append({"w_ridge6": round(float(w6), 2), "w_ridge3": w3, "oof_rmse": s})

blend_n_df = pd.DataFrame(blend_n_rows).sort_values("oof_rmse")
print("Ridge3 vs Ridge6 blend:")
display(blend_n_df.head(10))

best_bn = blend_n_df.iloc[0]
bn_oof = np.clip(best_bn["w_ridge6"] * s6_oof + best_bn["w_ridge3"] * ridge3_oof, 0, None)
bn_test = np.clip(best_bn["w_ridge6"] * s6_test + best_bn["w_ridge3"] * ridge3_test, 0, None)
print(f"Best blend: w6={best_bn['w_ridge6']}, w3={best_bn['w_ridge3']}, RMSE={best_bn['oof_rmse']:.5f}")

# --------------------------------------------------
# PART 4: Also try blending with global tail from 11H
# --------------------------------------------------
if "best_tail" in dir() or "best_tail" in globals():
    print("\n" + "=" * 80)
    print("PART 4: Blend with Global Tail 11H")
    print("=" * 80)

    tail_oof_11h = best_tail["oof_pred"]
    tail_test_11h = best_tail["test_pred"]

    tri_blend_rows = []
    step = 0.05
    for w1 in np.arange(0, 1 + step, step):
        for w2 in np.arange(0, 1 + step - w1, step):
            w3 = round(1 - w1 - w2, 2)
            if w3 < -0.001 or w3 > 1.001:
                continue
            w3 = max(w3, 0.0)
            bl = np.clip(w1 * s6_oof + w2 * ridge3_oof + w3 * tail_oof_11h, 0, None)
            s = rmse(y, bl)
            tri_blend_rows.append({
                "w_ridge6": round(float(w1), 2),
                "w_ridge3": round(float(w2), 2),
                "w_tail11h": round(float(w3), 2),
                "oof_rmse": s,
            })

    tri_df = pd.DataFrame(tri_blend_rows).sort_values("oof_rmse")
    print("3-way blend (Ridge6 + Ridge3 + Tail11H):")
    display(tri_df.head(10))

    best_tri = tri_df.iloc[0]
    tri_oof = np.clip(best_tri["w_ridge6"] * s6_oof + best_tri["w_ridge3"] * ridge3_oof
                      + best_tri["w_tail11h"] * tail_oof_11h, 0, None)
    tri_test = np.clip(best_tri["w_ridge6"] * s6_test + best_tri["w_ridge3"] * ridge3_test
                       + best_tri["w_tail11h"] * tail_test_11h, 0, None)
    print(f"Best 3-way: w6={best_tri['w_ridge6']}, w3={best_tri['w_ridge3']}, "
          f"wT={best_tri['w_tail11h']}, RMSE={best_tri['oof_rmse']:.5f}")

# --------------------------------------------------
# PART 5: Final Comparison & Save
# --------------------------------------------------
print("\n" + "=" * 80)
print("PART 5: Final 11N Comparison")
print("=" * 80)

final_cands = {
    "ridge3_11M": (ridge3_oof, ridge3_test),
    "ridge6_11N": (s6_oof, s6_test),
    "blend_r3_r6": (bn_oof, bn_test),
}
if "tri_oof" in dir():
    final_cands["blend_r6_r3_tail"] = (tri_oof, tri_test)
if "best_tail" in dir() or "best_tail" in globals():
    final_cands["global_tail_11H"] = (best_tail["oof_pred"], best_tail["test_pred"])

final_rows = []
for name, (o, t) in final_cands.items():
    s = rmse(y, np.clip(o, 0, None))
    final_rows.append({"candidate": name, "oof_rmse": s,
                       "delta_vs_old": s - OLD_BEST, "delta_vs_ridge3": s - RIDGE_BEST})
final_df = pd.DataFrame(final_rows).sort_values("oof_rmse")
print("All 11N candidates:")
display(final_df)

# Save winner
winner = final_df.iloc[0]["candidate"]
w_oof, w_test = final_cands[winner]
w_score = final_df.iloc[0]["oof_rmse"]
print(f"\n11N WINNER: {winner}, RMSE: {w_score:.5f}")
print(f"Delta vs old ensemble: {w_score - OLD_BEST:+.5f}")
print(f"Delta vs Ridge3 (11M): {w_score - RIDGE_BEST:+.5f}")

sub = sample_submission.copy()
sub[TARGET] = np.clip(w_test, 0, None)
assert sub[TARGET].isna().sum() == 0 and (sub[TARGET] >= 0).all()
sub.to_csv("submission_11n_best.csv", index=False)
print("Saved submission_11n_best.csv")
display(sub[TARGET].describe().to_frame().T)

# Also save Ridge6 standalone
sub2 = sample_submission.copy()
sub2[TARGET] = np.clip(s6_test, 0, None)
sub2.to_csv("submission_11n_ridge6.csv", index=False)
print("Saved submission_11n_ridge6.csv")


In [ ]:
# ==================================================
# SECTION 11O - Fold-Safe Prediction-Bin Adaptive Blending
# ==================================================
# Goal:
# Improve 11N by allowing different blend weights for different prediction ranges.
#
# Why:
# - 11N improved strongly because Ridge6 fixed low-target behavior.
# - Tail correction helps high-target underprediction.
# - A single global blend weight may be suboptimal across all prediction ranges.
#
# Strategy:
# - Use available strong candidates: Ridge6, Ridge3, Tail11H, and optional GatedTail.
# - Split data into bins based on prediction level.
# - Learn blend weights per bin using fold-safe validation.
# - Save submission only if OOF improves over 11N.
#
# No external data. No AutoML.

OLD_BEST_OOF = 11.335988679052521
RIDGE3_OOF_SCORE = 11.31405
BEST_11N_OOF = 11.268047  # from your 11N output

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def save_submission_11o(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)
    return sub


def make_bin_report_11o(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(pred, 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5")
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
        .sort_values("rmse", ascending=False)
    )

    print("=" * 90)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(pred, 0, None)))
    display(report)
    return report


def weighted_stack_matrix(P, weights):
    weights = np.asarray(weights)
    return np.clip(P @ weights, 0, None)


def make_simplex_grid(k, step=0.05):
    values = np.round(np.arange(0, 1 + step, step), 2)
    weights = []

    def rec(prefix, remaining, depth):
        if depth == k - 1:
            weights.append(prefix + [round(remaining, 2)])
            return

        for v in values:
            if v <= remaining + 1e-9:
                rec(prefix + [float(v)], round(remaining - v, 2), depth + 1)

    rec([], 1.0, 0)
    return np.array(weights)


def best_weights_for_subset(P, target, weight_grid):
    best_score = np.inf
    best_w = None

    for w in weight_grid:
        pred = weighted_stack_matrix(P, w)
        score = rmse(target, pred)

        if score < best_score:
            best_score = score
            best_w = w

    return best_w, best_score


def assign_bins_by_reference(ref_pred, percentiles):
    cuts = np.percentile(ref_pred, percentiles)
    # Example bins:
    # (-inf, p20], (p20,p40], ..., (p80,inf)
    return np.digitize(ref_pred, cuts, right=True), cuts


# --------------------------------------------------
# Part 1 - Build candidate dictionary
# --------------------------------------------------

candidate_preds = {}

# Ridge6 from 11N
if "s6_oof" in globals() and "s6_test" in globals():
    candidate_preds["ridge6_11N"] = {
        "oof": np.clip(s6_oof, 0, None),
        "test": np.clip(s6_test, 0, None),
    }

# Ridge3 from 11M
if "ridge3_oof" in globals() and "ridge3_test" in globals():
    candidate_preds["ridge3_11M"] = {
        "oof": np.clip(ridge3_oof, 0, None),
        "test": np.clip(ridge3_test, 0, None),
    }
elif "best_r" in globals():
    candidate_preds["ridge3_11M"] = {
        "oof": np.clip(best_r["oof"], 0, None),
        "test": np.clip(best_r["test"], 0, None),
    }

# Global tail from 11H
if "best_tail" in globals():
    candidate_preds["global_tail_11H"] = {
        "oof": np.clip(best_tail["oof_pred"], 0, None),
        "test": np.clip(best_tail["test_pred"], 0, None),
    }

# Gated tail from 11L
if "best_gated" in globals():
    candidate_preds["gated_tail_11L"] = {
        "oof": np.clip(best_gated["oof_pred"], 0, None),
        "test": np.clip(best_gated["test_pred"], 0, None),
    }

# Old ensemble as fallback / anchor
old_ensemble_oof = (
    0.50 * model_outputs["catboost"]["oof"] +
    0.25 * model_outputs["lightgbm"]["oof"] +
    0.25 * model_outputs["xgboost"]["oof"]
)

old_ensemble_test = (
    0.50 * model_outputs["catboost"]["test_pred"] +
    0.25 * model_outputs["lightgbm"]["test_pred"] +
    0.25 * model_outputs["xgboost"]["test_pred"]
)

candidate_preds["old_ensemble"] = {
    "oof": np.clip(old_ensemble_oof, 0, None),
    "test": np.clip(old_ensemble_test, 0, None),
}

# Keep strongest/relevant candidates only.
preferred_order = [
    "ridge6_11N",
    "ridge3_11M",
    "global_tail_11H",
    "gated_tail_11L",
    "old_ensemble",
]

candidate_names = [c for c in preferred_order if c in candidate_preds]

print("Candidates used in 11O:")
print(candidate_names)

P_oof = np.column_stack([candidate_preds[c]["oof"] for c in candidate_names])
P_test = np.column_stack([candidate_preds[c]["test"] for c in candidate_names])

candidate_score_df = pd.DataFrame({
    "candidate": candidate_names,
    "oof_rmse": [rmse(y, candidate_preds[c]["oof"]) for c in candidate_names],
}).sort_values("oof_rmse")

display(candidate_score_df)


# --------------------------------------------------
# Part 2 - Global blend re-check
# --------------------------------------------------

# Coarser grid if candidates >= 5 to avoid too much search.
grid_step = 0.05 if len(candidate_names) <= 4 else 0.10
weight_grid = make_simplex_grid(len(candidate_names), step=grid_step)

global_w, global_score = best_weights_for_subset(P_oof, y.values, weight_grid)
global_oof = weighted_stack_matrix(P_oof, global_w)
global_test = weighted_stack_matrix(P_test, global_w)

global_weight_df = pd.DataFrame({
    "candidate": candidate_names,
    "global_weight": global_w,
})

print("Best global blend score:", global_score)
display(global_weight_df)

make_bin_report_11o(global_oof, "11O global blend re-check")


# --------------------------------------------------
# Part 3 - Fold-safe prediction-bin adaptive blend
# --------------------------------------------------

# Reference prediction for binning:
# Use Ridge6 when available because it is the strongest single stacked component.
# Otherwise use the global blend.
if "ridge6_11N" in candidate_names:
    ref_oof = candidate_preds["ridge6_11N"]["oof"]
    ref_test = candidate_preds["ridge6_11N"]["test"]
else:
    ref_oof = global_oof
    ref_test = global_test

bin_percentiles = [20, 40, 60, 80]
oof_bins, bin_cuts = assign_bins_by_reference(ref_oof, bin_percentiles)
test_bins = np.digitize(ref_test, bin_cuts, right=True)

print("Prediction-bin cuts:", bin_cuts)

adaptive_oof = np.zeros(len(y))
fold_bin_rows = []

# Shrinkage means:
# 0.0 = use global weights
# 1.0 = use local bin weights fully
shrinkage_grid = [0.25, 0.50, 0.75, 1.00]

MIN_BIN_TRAIN_COUNT = 300
MIN_BIN_VALID_COUNT = 50

for fold_id, (tr_idx, val_idx) in enumerate(folds, 1):
    train_bins = oof_bins[tr_idx]
    valid_bins = oof_bins[val_idx]

    P_train_all = P_oof[tr_idx]
    y_train_all = y.values[tr_idx]

    P_valid_all = P_oof[val_idx]

    for bin_id in sorted(np.unique(oof_bins)):
        tr_mask = train_bins == bin_id
        val_mask = valid_bins == bin_id

        tr_count = int(tr_mask.sum())
        val_count = int(val_mask.sum())

        if val_count == 0:
            continue

        # Default: global weights
        chosen_w = global_w.copy()
        chosen_shrink = 0.0
        chosen_train_score = np.nan

        if tr_count >= MIN_BIN_TRAIN_COUNT and val_count >= MIN_BIN_VALID_COUNT:
            local_w, local_train_score = best_weights_for_subset(
                P_train_all[tr_mask],
                y_train_all[tr_mask],
                weight_grid
            )

            # Select shrinkage on training subset only
            best_shrink_score = np.inf
            best_shrink = 0.0
            best_blended_w = global_w.copy()

            for shrink in shrinkage_grid:
                blended_w = (1 - shrink) * global_w + shrink * local_w
                pred_train_bin = weighted_stack_matrix(P_train_all[tr_mask], blended_w)
                score_train_bin = rmse(y_train_all[tr_mask], pred_train_bin)

                if score_train_bin < best_shrink_score:
                    best_shrink_score = score_train_bin
                    best_shrink = shrink
                    best_blended_w = blended_w

            chosen_w = best_blended_w
            chosen_shrink = best_shrink
            chosen_train_score = best_shrink_score

        pred_val_bin = weighted_stack_matrix(P_valid_all[val_mask], chosen_w)

        # Write back to correct validation positions
        val_indices_bin = val_idx[val_mask]
        adaptive_oof[val_indices_bin] = pred_val_bin

        fold_bin_rows.append({
            "fold": fold_id,
            "bin_id": int(bin_id),
            "train_count": tr_count,
            "valid_count": val_count,
            "chosen_shrink": chosen_shrink,
            "train_bin_score": chosen_train_score,
            "valid_bin_score": rmse(y.values[val_indices_bin], pred_val_bin),
            **{f"w_{name}": chosen_w[i] for i, name in enumerate(candidate_names)}
        })

adaptive_oof = np.clip(adaptive_oof, 0, None)
adaptive_score = rmse(y, adaptive_oof)

fold_bin_df = pd.DataFrame(fold_bin_rows)
print("Adaptive OOF RMSE:", adaptive_score)
print("Delta vs old ensemble:", adaptive_score - OLD_BEST_OOF)
print("Delta vs 11N best:", adaptive_score - BEST_11N_OOF)

display(fold_bin_df)

make_bin_report_11o(adaptive_oof, "11O fold-safe adaptive blend")


# --------------------------------------------------
# Part 4 - Fit final adaptive weights on all OOF and apply to test
# --------------------------------------------------

final_adaptive_test = np.zeros(sample_submission.shape[0])
final_bin_rows = []

for bin_id in sorted(np.unique(oof_bins)):
    train_mask = oof_bins == bin_id
    test_mask = test_bins == bin_id

    if test_mask.sum() == 0:
        continue

    if train_mask.sum() >= MIN_BIN_TRAIN_COUNT:
        local_w, local_score = best_weights_for_subset(
            P_oof[train_mask],
            y.values[train_mask],
            weight_grid
        )

        # Use average shrinkage selected in CV for this bin.
        # If no shrinkage info, default to 0.50.
        bin_cv_rows = fold_bin_df[fold_bin_df["bin_id"] == bin_id]
        if len(bin_cv_rows) > 0:
            avg_shrink = float(bin_cv_rows["chosen_shrink"].mean())
        else:
            avg_shrink = 0.50

        final_w = (1 - avg_shrink) * global_w + avg_shrink * local_w
    else:
        local_score = np.nan
        avg_shrink = 0.0
        final_w = global_w.copy()

    final_adaptive_test[test_mask] = weighted_stack_matrix(P_test[test_mask], final_w)

    final_bin_rows.append({
        "bin_id": int(bin_id),
        "train_count": int(train_mask.sum()),
        "test_count": int(test_mask.sum()),
        "avg_cv_shrink": avg_shrink,
        "local_train_score": local_score,
        **{f"w_{name}": final_w[i] for i, name in enumerate(candidate_names)}
    })

final_adaptive_test = np.clip(final_adaptive_test, 0, None)

final_bin_weight_df = pd.DataFrame(final_bin_rows)
print("Final adaptive test-bin weights:")
display(final_bin_weight_df)


# --------------------------------------------------
# Part 5 - Compare and save
# --------------------------------------------------

comparison_11o = pd.DataFrame([
    {
        "candidate": "old_ensemble",
        "oof_rmse": rmse(y, old_ensemble_oof),
        "delta_vs_old": 0.0,
        "delta_vs_11n_best": rmse(y, old_ensemble_oof) - BEST_11N_OOF,
    },
    {
        "candidate": "11O_global_blend",
        "oof_rmse": global_score,
        "delta_vs_old": global_score - OLD_BEST_OOF,
        "delta_vs_11n_best": global_score - BEST_11N_OOF,
    },
    {
        "candidate": "11O_adaptive_blend",
        "oof_rmse": adaptive_score,
        "delta_vs_old": adaptive_score - OLD_BEST_OOF,
        "delta_vs_11n_best": adaptive_score - BEST_11N_OOF,
    },
])

# Include existing 11N best if available
if "final_df" in globals():
    try:
        best_11n_score = float(final_df.iloc[0]["oof_rmse"])
        comparison_11o = pd.concat([
            comparison_11o,
            pd.DataFrame([{
                "candidate": "11N_best_from_final_df",
                "oof_rmse": best_11n_score,
                "delta_vs_old": best_11n_score - OLD_BEST_OOF,
                "delta_vs_11n_best": best_11n_score - BEST_11N_OOF,
            }])
        ], ignore_index=True)
    except Exception:
        pass

comparison_11o = comparison_11o.sort_values("oof_rmse")
display(comparison_11o)

# Save both global and adaptive candidates.
save_submission_11o(global_test, "submission_11o_global_blend.csv")
save_submission_11o(final_adaptive_test, "submission_11o_adaptive_blend.csv")

print("Decision guide:")
print("- Strong submit if 11O adaptive/global < 11N best OOF.")
print("- Submit 11O global if it improves or matches 11N with simpler behavior.")
print("- Do not submit 11O adaptive if it improves only tiny amount but has unstable bin weights.")

In [ ]:
# ==================================================
# SECTION 11P - High / Very-High Specialist + CV-Safe Meta Stacking
# ==================================================
# Goal:
# 1. Target remaining error in high and very_high bins.
# 2. Train high-target specialist models using target-based sample weights.
# 3. Stack old strong candidates + high specialists using CV-safe Ridge meta model.
# 4. Blend with 11O adaptive to avoid over-aggressive high-tail correction.
#
# Paste this cell BELOW Section 11O.
# No external data. No AutoML.

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    print("LightGBM import issue:", e)

try:
    from catboost import CatBoostRegressor
except Exception as e:
    print("CatBoost import issue:", e)


# --------------------------------------------------
# Config
# --------------------------------------------------

RUN_11P_WEIGHTED_LGBM = True
RUN_11P_WEIGHTED_CATBOOST = True

# Kalau runtime kelamaan, ubah jadi False dulu.
# RUN_11P_WEIGHTED_CATBOOST = False

OLD_ENSEMBLE_OOF = 11.335988679052521

if "adaptive_oof" in globals():
    BEST_11O_OOF = rmse(y, np.clip(adaptive_oof, 0, None))
else:
    BEST_11O_OOF = 11.26696803195285

print("BEST_11O_OOF used:", BEST_11O_OOF)


# --------------------------------------------------
# Safety check
# --------------------------------------------------

required_vars_11p = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "X_fe",
    "X_test_fe",
    "cat_features",
    "num_features",
    "prepare_encoded_fold",
    "rmse",
]

missing_11p = [v for v in required_vars_11p if v not in globals()]

if len(missing_11p) > 0:
    raise NameError(f"Missing required variables for 11P: {missing_11p}")

print("All required variables for 11P are available.")


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def save_submission_11p(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 90)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def make_target_bin_report_11p(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(np.asarray(pred), 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()
    report = report.sort_values("rmse", ascending=False)

    print("=" * 90)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(np.asarray(pred), 0, None)))
    display(report)

    return report


def make_high_target_weights(y_values, mode="mild"):
    """
    More weight for high and very_high samples.
    q80 = high zone
    q90 = very_high zone
    q95 = extreme very_high zone
    """
    y_arr = np.asarray(y_values)

    q80 = np.quantile(y_arr, 0.80)
    q90 = np.quantile(y_arr, 0.90)
    q95 = np.quantile(y_arr, 0.95)

    w = np.ones(len(y_arr), dtype=float)

    if mode == "mild":
        w += 0.75 * (y_arr >= q80)
        w += 1.25 * (y_arr >= q90)
        w += 1.50 * (y_arr >= q95)

    elif mode == "medium":
        w += 1.25 * (y_arr >= q80)
        w += 2.00 * (y_arr >= q90)
        w += 2.50 * (y_arr >= q95)

    elif mode == "strong":
        w += 1.75 * (y_arr >= q80)
        w += 3.00 * (y_arr >= q90)
        w += 4.00 * (y_arr >= q95)

    else:
        raise ValueError("Unknown weight mode")

    return w


def fit_lgbm_11p(model, X_tr, y_tr, X_val, y_val, sample_weight=None):
    try:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(200, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
    except TypeError:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=200,
            verbose=False,
        )

    return model


# --------------------------------------------------
# Part 0 - Baseline target-bin check
# --------------------------------------------------

print("=" * 90)
print("Baseline target-bin check")

if "adaptive_oof" in globals():
    baseline_11p_report = make_target_bin_report_11p(
        adaptive_oof,
        "11O adaptive baseline"
    )
elif "global_oof" in globals():
    baseline_11p_report = make_target_bin_report_11p(
        global_oof,
        "11O global baseline"
    )
else:
    print("No 11O OOF baseline found. Continuing with available candidates.")


# --------------------------------------------------
# Part 1 - Collect existing strong candidates
# --------------------------------------------------

candidate_preds_11p = {}

def add_candidate_11p(name, oof_pred, test_pred):
    oof_pred = np.clip(np.asarray(oof_pred), 0, None)
    test_pred = np.clip(np.asarray(test_pred), 0, None)

    if len(oof_pred) != len(y):
        print(f"Skip {name}: invalid OOF length")
        return

    if len(test_pred) != sample_submission.shape[0]:
        print(f"Skip {name}: invalid test length")
        return

    candidate_preds_11p[name] = {
        "oof": oof_pred,
        "test": test_pred,
    }

    print(f"Added candidate: {name} | OOF RMSE = {rmse(y, oof_pred):.6f}")


# 11O adaptive
if "adaptive_oof" in globals() and "final_adaptive_test" in globals():
    add_candidate_11p("adaptive_11O", adaptive_oof, final_adaptive_test)

# 11O global
if "global_oof" in globals() and "global_test" in globals():
    add_candidate_11p("global_11O", global_oof, global_test)

# 11N ridge6
if "s6_oof" in globals() and "s6_test" in globals():
    add_candidate_11p("ridge6_11N", s6_oof, s6_test)

# 11M ridge3
if "ridge3_oof" in globals() and "ridge3_test" in globals():
    add_candidate_11p("ridge3_11M", ridge3_oof, ridge3_test)
elif "best_r" in globals():
    if isinstance(best_r, dict) and "oof" in best_r and "test" in best_r:
        add_candidate_11p("ridge3_11M", best_r["oof"], best_r["test"])

# global tail
if "best_tail" in globals():
    if isinstance(best_tail, dict):
        if "oof_pred" in best_tail and "test_pred" in best_tail:
            add_candidate_11p(
                "global_tail_11H",
                best_tail["oof_pred"],
                best_tail["test_pred"],
            )

# gated tail
if "best_gated" in globals():
    if isinstance(best_gated, dict):
        if "oof_pred" in best_gated and "test_pred" in best_gated:
            add_candidate_11p(
                "gated_tail_11L",
                best_gated["oof_pred"],
                best_gated["test_pred"],
            )

# old ensemble
if "model_outputs" in globals():
    needed_models = ["catboost", "lightgbm", "xgboost"]
    if all(m in model_outputs for m in needed_models):
        old_ensemble_oof_11p = (
            0.50 * model_outputs["catboost"]["oof"] +
            0.25 * model_outputs["lightgbm"]["oof"] +
            0.25 * model_outputs["xgboost"]["oof"]
        )

        old_ensemble_test_11p = (
            0.50 * model_outputs["catboost"]["test_pred"] +
            0.25 * model_outputs["lightgbm"]["test_pred"] +
            0.25 * model_outputs["xgboost"]["test_pred"]
        )

        add_candidate_11p(
            "old_ensemble",
            old_ensemble_oof_11p,
            old_ensemble_test_11p,
        )

if len(candidate_preds_11p) == 0:
    raise ValueError("No existing candidates found for 11P. Run previous sections first.")

candidate_score_initial_11p = pd.DataFrame({
    "candidate": list(candidate_preds_11p.keys()),
    "oof_rmse": [rmse(y, v["oof"]) for v in candidate_preds_11p.values()],
}).sort_values("oof_rmse")

print("=" * 90)
print("Initial candidates for 11P")
display(candidate_score_initial_11p)


# --------------------------------------------------
# Part 2 - Train high-target specialist LightGBM models
# --------------------------------------------------

weight_modes_lgb_11p = ["mild", "medium", "strong"]

if RUN_11P_WEIGHTED_LGBM:
    for weight_mode in weight_modes_lgb_11p:
        print("=" * 100)
        print(f"Training high-target specialist LightGBM: {weight_mode}")

        sample_weights_all = make_high_target_weights(y.values, mode=weight_mode)

        lgb_oof = np.zeros(len(y))
        lgb_test = np.zeros(sample_submission.shape[0])
        lgb_fold_scores = []

        for fold, (tr_idx, val_idx) in enumerate(folds, 1):
            X_tr_raw = X_fe.iloc[tr_idx]
            X_val_raw = X_fe.iloc[val_idx]
            X_te_raw = X_test_fe

            y_tr = y.iloc[tr_idx]
            y_val = y.iloc[val_idx]
            w_tr = sample_weights_all[tr_idx]

            X_tr, X_val, X_te, feature_names = prepare_encoded_fold(
                X_tr_raw,
                X_val_raw,
                X_te_raw,
                cat_features,
                num_features,
            )

            model = LGBMRegressor(
                objective="regression",
                metric="rmse",
                n_estimators=3500,
                learning_rate=0.02,
                num_leaves=31,
                max_depth=-1,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.5,
                reg_lambda=8.0,
                min_child_samples=60,
                random_state=RANDOM_STATE + 3000 + fold,
                n_jobs=-1,
                verbosity=-1,
            )

            model = fit_lgbm_11p(
                model,
                X_tr,
                y_tr,
                X_val,
                y_val,
                sample_weight=w_tr,
            )

            val_pred = model.predict(X_val)
            test_pred = model.predict(X_te)

            lgb_oof[val_idx] = val_pred
            lgb_test += test_pred / len(folds)

            fold_rmse = rmse(y_val, np.clip(val_pred, 0, None))
            lgb_fold_scores.append(fold_rmse)

            print(f"LGB-high {weight_mode} fold {fold}: RMSE={fold_rmse:.5f}")

        lgb_oof = np.clip(lgb_oof, 0, None)
        lgb_test = np.clip(lgb_test, 0, None)

        key = f"lgb_high_{weight_mode}_11P"
        add_candidate_11p(key, lgb_oof, lgb_test)

        make_target_bin_report_11p(lgb_oof, key)


# --------------------------------------------------
# Part 3 - Train high-target specialist CatBoost models
# --------------------------------------------------

if RUN_11P_WEIGHTED_CATBOOST:
    X_cb_all = X_fe.copy()
    X_cb_test = X_test_fe.copy()

    cat_features_cb = [c for c in cat_features if c in X_cb_all.columns]

    for c in cat_features_cb:
        X_cb_all[c] = X_cb_all[c].astype(str).fillna("Missing")
        X_cb_test[c] = X_cb_test[c].astype(str).fillna("Missing")

    cat_feature_indices_cb = [X_cb_all.columns.get_loc(c) for c in cat_features_cb]

    for weight_mode in ["mild", "medium"]:
        print("=" * 100)
        print(f"Training high-target specialist CatBoost: {weight_mode}")

        sample_weights_all = make_high_target_weights(y.values, mode=weight_mode)

        cb_oof = np.zeros(len(y))
        cb_test = np.zeros(sample_submission.shape[0])
        cb_fold_scores = []

        for fold, (tr_idx, val_idx) in enumerate(folds, 1):
            X_tr = X_cb_all.iloc[tr_idx]
            X_val = X_cb_all.iloc[val_idx]
            X_te = X_cb_test

            y_tr = y.iloc[tr_idx]
            y_val = y.iloc[val_idx]
            w_tr = sample_weights_all[tr_idx]

            model = CatBoostRegressor(
                loss_function="RMSE",
                eval_metric="RMSE",
                iterations=3000,
                learning_rate=0.025,
                depth=6,
                l2_leaf_reg=10.0,
                random_seed=RANDOM_STATE + 4000 + fold,
                od_type="Iter",
                od_wait=250,
                verbose=False,
                allow_writing_files=False,
            )

            model.fit(
                X_tr,
                y_tr,
                sample_weight=w_tr,
                cat_features=cat_feature_indices_cb,
                eval_set=(X_val, y_val),
                use_best_model=True,
            )

            val_pred = model.predict(X_val)
            test_pred = model.predict(X_te)

            cb_oof[val_idx] = val_pred
            cb_test += test_pred / len(folds)

            fold_rmse = rmse(y_val, np.clip(val_pred, 0, None))
            cb_fold_scores.append(fold_rmse)

            print(f"CB-high {weight_mode} fold {fold}: RMSE={fold_rmse:.5f}")

        cb_oof = np.clip(cb_oof, 0, None)
        cb_test = np.clip(cb_test, 0, None)

        key = f"cat_high_{weight_mode}_11P"
        add_candidate_11p(key, cb_oof, cb_test)

        make_target_bin_report_11p(cb_oof, key)


# --------------------------------------------------
# Part 4 - Candidate score after specialist models
# --------------------------------------------------

candidate_score_11p = pd.DataFrame({
    "candidate": list(candidate_preds_11p.keys()),
    "oof_rmse": [rmse(y, v["oof"]) for v in candidate_preds_11p.values()],
}).sort_values("oof_rmse")

print("=" * 90)
print("All candidates after high-target specialists")
display(candidate_score_11p)


# --------------------------------------------------
# Part 5 - CV-safe meta stacking
# --------------------------------------------------

# Keep strongest candidates, but force-keep specialist models.
base_keep = candidate_score_11p["candidate"].head(10).tolist()

specialist_keep = [
    c for c in candidate_preds_11p.keys()
    if ("high" in c and "11P" in c)
]

candidate_names_11p = []

for c in base_keep + specialist_keep:
    if c not in candidate_names_11p:
        candidate_names_11p.append(c)

print("=" * 90)
print("Candidates used for 11P meta stack:")
for c in candidate_names_11p:
    print("-", c, "| RMSE:", rmse(y, candidate_preds_11p[c]["oof"]))

P_oof_raw = np.column_stack([
    candidate_preds_11p[c]["oof"] for c in candidate_names_11p
])

P_test_raw = np.column_stack([
    candidate_preds_11p[c]["test"] for c in candidate_names_11p
])


def make_meta_features_11p(P):
    """
    Meta features:
    1. raw predictions
    2. log1p predictions
    3. squared predictions
    4. simple high confidence signals
    5. pairwise interactions
    """
    P = np.clip(np.asarray(P), 0, None)

    feats = []
    feats.append(P)
    feats.append(np.log1p(P))
    feats.append(P ** 2)

    row_mean = P.mean(axis=1).reshape(-1, 1)
    row_std = P.std(axis=1).reshape(-1, 1)
    row_min = P.min(axis=1).reshape(-1, 1)
    row_max = P.max(axis=1).reshape(-1, 1)
    row_range = row_max - row_min

    feats.extend([row_mean, row_std, row_min, row_max, row_range])

    interactions = []

    for i in range(P.shape[1]):
        for j in range(i + 1, P.shape[1]):
            interactions.append((P[:, i] * P[:, j]).reshape(-1, 1))

    if len(interactions) > 0:
        feats.append(np.hstack(interactions))

    return np.hstack(feats)


META_oof = make_meta_features_11p(P_oof_raw)
META_test = make_meta_features_11p(P_test_raw)

print("Meta OOF shape:", META_oof.shape)
print("Meta test shape:", META_test.shape)

meta_alpha_grid = [0.1, 0.3, 1.0, 3.0, 5.0, 10.0, 20.0, 50.0, 100.0]
meta_weight_modes = ["none", "mild_high", "medium_high"]

meta_rows = []
meta_outputs = {}

for weight_mode in meta_weight_modes:
    if weight_mode == "none":
        meta_weights_all = None
    elif weight_mode == "mild_high":
        meta_weights_all = make_high_target_weights(y.values, mode="mild")
    elif weight_mode == "medium_high":
        meta_weights_all = make_high_target_weights(y.values, mode="medium")
    else:
        raise ValueError("Unknown meta weight mode")

    for alpha in meta_alpha_grid:
        meta_oof_pred = np.zeros(len(y))
        fold_scores = []

        for fold, (tr_idx, val_idx) in enumerate(folds, 1):
            X_meta_tr = META_oof[tr_idx]
            X_meta_val = META_oof[val_idx]

            y_meta_tr = y.values[tr_idx]
            y_meta_val = y.values[val_idx]

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=alpha, random_state=RANDOM_STATE)),
            ])

            if meta_weights_all is None:
                pipe.fit(X_meta_tr, y_meta_tr)
            else:
                pipe.fit(
                    X_meta_tr,
                    y_meta_tr,
                    ridge__sample_weight=meta_weights_all[tr_idx],
                )

            val_pred = pipe.predict(X_meta_val)
            val_pred = np.clip(val_pred, 0, None)

            meta_oof_pred[val_idx] = val_pred
            fold_scores.append(rmse(y_meta_val, val_pred))

        meta_score = rmse(y, meta_oof_pred)

        meta_name = f"meta_{weight_mode}_alpha_{alpha}"

        meta_rows.append({
            "candidate": meta_name,
            "weight_mode": weight_mode,
            "alpha": alpha,
            "oof_rmse": meta_score,
            "fold_std": np.std(fold_scores),
            "delta_vs_11O": meta_score - BEST_11O_OOF,
            "delta_vs_old": meta_score - OLD_ENSEMBLE_OOF,
        })

        meta_outputs[meta_name] = {
            "oof": meta_oof_pred,
            "alpha": alpha,
            "weight_mode": weight_mode,
            "fold_scores": fold_scores,
        }

meta_result_df = pd.DataFrame(meta_rows).sort_values("oof_rmse")

print("=" * 90)
print("11P meta stack results")
display(meta_result_df.head(30))

best_meta_name = meta_result_df.iloc[0]["candidate"]
best_meta_info = meta_outputs[best_meta_name]

print("Best 11P meta candidate:", best_meta_name)
print("Best 11P meta OOF RMSE:", rmse(y, best_meta_info["oof"]))
print("Delta vs 11O:", rmse(y, best_meta_info["oof"]) - BEST_11O_OOF)

best_meta_bin_report_11p = make_target_bin_report_11p(
    best_meta_info["oof"],
    f"Best 11P meta stack: {best_meta_name}",
)


# --------------------------------------------------
# Part 6 - Fit final meta model on full OOF and predict test
# --------------------------------------------------

best_alpha = best_meta_info["alpha"]
best_weight_mode = best_meta_info["weight_mode"]

if best_weight_mode == "none":
    final_meta_weights = None
elif best_weight_mode == "mild_high":
    final_meta_weights = make_high_target_weights(y.values, mode="mild")
elif best_weight_mode == "medium_high":
    final_meta_weights = make_high_target_weights(y.values, mode="medium")
else:
    raise ValueError("Unknown best meta weight mode")

final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best_alpha, random_state=RANDOM_STATE)),
])

if final_meta_weights is None:
    final_pipe.fit(META_oof, y.values)
else:
    final_pipe.fit(
        META_oof,
        y.values,
        ridge__sample_weight=final_meta_weights,
    )

final_11p_meta_test_pred = np.clip(final_pipe.predict(META_test), 0, None)


# --------------------------------------------------
# Part 7 - Conservative blend with 11O adaptive
# --------------------------------------------------

if "adaptive_11O" in candidate_preds_11p:
    adaptive_oof_ref = candidate_preds_11p["adaptive_11O"]["oof"]
    adaptive_test_ref = candidate_preds_11p["adaptive_11O"]["test"]
else:
    # fallback to best available candidate
    fallback_name = candidate_score_11p.iloc[0]["candidate"]
    adaptive_oof_ref = candidate_preds_11p[fallback_name]["oof"]
    adaptive_test_ref = candidate_preds_11p[fallback_name]["test"]

    print(f"No adaptive_11O found. Using fallback reference: {fallback_name}")

blend_rows = []
blend_outputs = {}

for w_meta in np.round(np.arange(0.30, 1.01, 0.05), 2):
    w_ref = 1.0 - w_meta

    blend_oof = np.clip(
        w_meta * best_meta_info["oof"] + w_ref * adaptive_oof_ref,
        0,
        None,
    )

    blend_test = np.clip(
        w_meta * final_11p_meta_test_pred + w_ref * adaptive_test_ref,
        0,
        None,
    )

    score = rmse(y, blend_oof)

    blend_rows.append({
        "w_11p_meta": w_meta,
        "w_reference": w_ref,
        "oof_rmse": score,
        "delta_vs_11O": score - BEST_11O_OOF,
    })

    blend_outputs[w_meta] = {
        "oof": blend_oof,
        "test": blend_test,
        "score": score,
    }

blend_11p_df = pd.DataFrame(blend_rows).sort_values("oof_rmse")

print("=" * 90)
print("11P conservative blend results")
display(blend_11p_df)

best_blend_w = float(blend_11p_df.iloc[0]["w_11p_meta"])

best_11p_final_oof = blend_outputs[best_blend_w]["oof"]
best_11p_final_test = blend_outputs[best_blend_w]["test"]
best_11p_final_score = blend_outputs[best_blend_w]["score"]

print("Best final 11P blend weight:", best_blend_w)
print("Best final 11P OOF RMSE:", best_11p_final_score)
print("Delta vs 11O:", best_11p_final_score - BEST_11O_OOF)

best_11p_final_bin_report = make_target_bin_report_11p(
    best_11p_final_oof,
    "Best final 11P prediction",
)


# --------------------------------------------------
# Part 8 - Final comparison and save submissions
# --------------------------------------------------

comparison_11p = pd.DataFrame([
    {
        "candidate": "old_ensemble",
        "oof_rmse": OLD_ENSEMBLE_OOF,
        "delta_vs_11O": OLD_ENSEMBLE_OOF - BEST_11O_OOF,
    },
    {
        "candidate": "best_11O_adaptive",
        "oof_rmse": BEST_11O_OOF,
        "delta_vs_11O": 0.0,
    },
    {
        "candidate": "best_11P_meta",
        "oof_rmse": rmse(y, best_meta_info["oof"]),
        "delta_vs_11O": rmse(y, best_meta_info["oof"]) - BEST_11O_OOF,
    },
    {
        "candidate": "best_11P_final_blend",
        "oof_rmse": best_11p_final_score,
        "delta_vs_11O": best_11p_final_score - BEST_11O_OOF,
    },
]).sort_values("oof_rmse")

print("=" * 90)
print("Final 11P comparison")
display(comparison_11p)

sub_11p_meta = save_submission_11p(
    final_11p_meta_test_pred,
    "submission_11p_meta_stack.csv",
)

sub_11p_final = save_submission_11p(
    best_11p_final_test,
    "submission_11p_final_blend.csv",
)


# --------------------------------------------------
# Part 9 - Decision guide
# --------------------------------------------------

print("=" * 90)
print("11P Decision Guide")
print(f"11O baseline OOF        : {BEST_11O_OOF:.6f}")
print(f"11P meta OOF           : {rmse(y, best_meta_info['oof']):.6f}")
print(f"11P final blend OOF    : {best_11p_final_score:.6f}")
print(f"Delta final vs 11O     : {best_11p_final_score - BEST_11O_OOF:.6f}")

if best_11p_final_score < BEST_11O_OOF - 0.005:
    print("Decision: STRONG SUBMIT submission_11p_final_blend.csv")
elif best_11p_final_score < BEST_11O_OOF:
    print("Decision: MEDIUM SUBMIT. Submit only if high / very_high bin improves clearly.")
else:
    print("Decision: DO NOT FORCE. Keep 11O adaptive unless public score testing says otherwise.")

print("=" * 90)
print("Please screenshot these outputs:")
print("1. comparison_11p")
print("2. meta_result_df.head(10)")
print("3. best_11p_final_bin_report")
print("4. blend_11p_df")

In [ ]:
# SECTION 11Q - sqrt-target + quantile + deeper trees diversity
import warnings; warnings.filterwarnings("ignore")
BEST_11O = 11.26697

y_sqrt = np.sqrt(y)

# === PART 1: sqrt-target models ===
print("="*80, "\nPART 1: sqrt-target models")

cb_sqrt_oof, cb_sqrt_test = np.zeros(len(y)), np.zeros(len(test_raw))
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    mdl = CatBoostRegressor(loss_function="RMSE", iterations=MAIN_N_ESTIMATORS, learning_rate=0.03,
        depth=6, l2_leaf_reg=5, random_seed=RANDOM_STATE+fold, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=0, allow_writing_files=False)
    mdl.fit(X_cb_full.iloc[tr_idx], y_sqrt.iloc[tr_idx],
            eval_set=(X_cb_full.iloc[val_idx], y_sqrt.iloc[val_idx]), cat_features=cat_indices, use_best_model=True)
    cb_sqrt_oof[val_idx] = mdl.predict(X_cb_full.iloc[val_idx])**2
    cb_sqrt_test += mdl.predict(X_test_cb)**2 / len(folds)
    print(f"  CB-sqrt fold {fold}: {rmse(y.iloc[val_idx], np.clip(cb_sqrt_oof[val_idx],0,None)):.5f}")
cb_sqrt_oof, cb_sqrt_test = np.clip(cb_sqrt_oof,0,None), np.clip(cb_sqrt_test,0,None)
print(f"CB-sqrt OOF: {rmse(y,cb_sqrt_oof):.5f}")

lgb_sqrt_oof, lgb_sqrt_test = np.zeros(len(y)), np.zeros(len(test_raw))
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    Xtr,Xva,Xte,fn = prepare_encoded_fold(X_fe.iloc[tr_idx],X_fe.iloc[val_idx],X_test_fe,cat_features,num_features)
    mdl = LGBMRegressor(objective="regression",n_estimators=MAIN_N_ESTIMATORS,learning_rate=0.03,
        num_leaves=31,subsample=0.8,colsample_bytree=0.8,reg_alpha=0.1,reg_lambda=1.0,
        min_child_samples=20,random_state=RANDOM_STATE+fold,n_jobs=-1,verbosity=-1)
    try: mdl.fit(Xtr,y_sqrt.values[tr_idx],eval_set=[(Xva,y_sqrt.values[val_idx])],eval_metric="rmse",
                 callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS),lgb.log_evaluation(0)])
    except: mdl.fit(Xtr,y_sqrt.values[tr_idx],eval_set=[(Xva,y_sqrt.values[val_idx])],
                     early_stopping_rounds=EARLY_STOPPING_ROUNDS,verbose=False)
    lgb_sqrt_oof[val_idx] = mdl.predict(Xva)**2
    lgb_sqrt_test += mdl.predict(Xte)**2 / len(folds)
lgb_sqrt_oof, lgb_sqrt_test = np.clip(lgb_sqrt_oof,0,None), np.clip(lgb_sqrt_test,0,None)
print(f"LGB-sqrt OOF: {rmse(y,lgb_sqrt_oof):.5f}")

xgb_sqrt_oof, xgb_sqrt_test = np.zeros(len(y)), np.zeros(len(test_raw))
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    Xtr,Xva,Xte,fn = prepare_encoded_fold(X_fe.iloc[tr_idx],X_fe.iloc[val_idx],X_test_fe,cat_features,num_features)
    try:
        mdl = XGBRegressor(objective="reg:squarederror",n_estimators=MAIN_N_ESTIMATORS,learning_rate=0.03,
            max_depth=6,min_child_weight=20,subsample=0.8,colsample_bytree=0.8,reg_alpha=0.1,reg_lambda=1.0,
            random_state=RANDOM_STATE+fold,tree_method="hist",n_jobs=-1,early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        mdl.fit(Xtr,y_sqrt.values[tr_idx],eval_set=[(Xva,y_sqrt.values[val_idx])],verbose=0)
    except:
        mdl = XGBRegressor(objective="reg:squarederror",n_estimators=MAIN_N_ESTIMATORS,learning_rate=0.03,
            max_depth=6,min_child_weight=20,subsample=0.8,colsample_bytree=0.8,random_state=RANDOM_STATE+fold,
            tree_method="hist",n_jobs=-1)
        mdl.fit(Xtr,y_sqrt.values[tr_idx],eval_set=[(Xva,y_sqrt.values[val_idx])],
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,verbose=0)
    xgb_sqrt_oof[val_idx] = mdl.predict(Xva)**2
    xgb_sqrt_test += mdl.predict(Xte)**2 / len(folds)
xgb_sqrt_oof, xgb_sqrt_test = np.clip(xgb_sqrt_oof,0,None), np.clip(xgb_sqrt_test,0,None)
print(f"XGB-sqrt OOF: {rmse(y,xgb_sqrt_oof):.5f}")

# === PART 2: Quantile 0.55 models ===
print("\n"+"="*80, "\nPART 2: Quantile 0.55")

cb_q55_oof, cb_q55_test = np.zeros(len(y)), np.zeros(len(test_raw))
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    mdl = CatBoostRegressor(loss_function="Quantile:alpha=0.55", iterations=MAIN_N_ESTIMATORS, learning_rate=0.03,
        depth=6, l2_leaf_reg=5, random_seed=RANDOM_STATE+fold, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=0, allow_writing_files=False)
    mdl.fit(X_cb_full.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=(X_cb_full.iloc[val_idx], y.iloc[val_idx]), cat_features=cat_indices, use_best_model=True)
    cb_q55_oof[val_idx] = mdl.predict(X_cb_full.iloc[val_idx])
    cb_q55_test += mdl.predict(X_test_cb) / len(folds)
cb_q55_oof, cb_q55_test = np.clip(cb_q55_oof,0,None), np.clip(cb_q55_test,0,None)
print(f"CB-Q55 OOF: {rmse(y,cb_q55_oof):.5f}")

lgb_q55_oof, lgb_q55_test = np.zeros(len(y)), np.zeros(len(test_raw))
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    Xtr,Xva,Xte,fn = prepare_encoded_fold(X_fe.iloc[tr_idx],X_fe.iloc[val_idx],X_test_fe,cat_features,num_features)
    mdl = LGBMRegressor(objective="quantile",alpha=0.55,n_estimators=MAIN_N_ESTIMATORS,learning_rate=0.03,
        num_leaves=31,subsample=0.8,colsample_bytree=0.8,reg_alpha=0.1,reg_lambda=1.0,
        min_child_samples=20,random_state=RANDOM_STATE+fold,n_jobs=-1,verbosity=-1)
    try: mdl.fit(Xtr,y.values[tr_idx],eval_set=[(Xva,y.values[val_idx])],
                 callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS),lgb.log_evaluation(0)])
    except: mdl.fit(Xtr,y.values[tr_idx],eval_set=[(Xva,y.values[val_idx])],
                     early_stopping_rounds=EARLY_STOPPING_ROUNDS,verbose=False)
    lgb_q55_oof[val_idx] = mdl.predict(Xva)
    lgb_q55_test += mdl.predict(Xte) / len(folds)
lgb_q55_oof, lgb_q55_test = np.clip(lgb_q55_oof,0,None), np.clip(lgb_q55_test,0,None)
print(f"LGB-Q55 OOF: {rmse(y,lgb_q55_oof):.5f}")

# === PART 3: Deeper CatBoost depth=8 ===
print("\n"+"="*80, "\nPART 3: CatBoost depth=8")

cb_d8_oof, cb_d8_test = np.zeros(len(y)), np.zeros(len(test_raw))
for fold, (tr_idx, val_idx) in enumerate(folds, 1):
    mdl = CatBoostRegressor(loss_function="RMSE", iterations=MAIN_N_ESTIMATORS, learning_rate=0.03,
        depth=8, l2_leaf_reg=10, random_strength=1.5, bagging_temperature=0.7,
        random_seed=RANDOM_STATE+fold, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=0, allow_writing_files=False, min_data_in_leaf=30)
    mdl.fit(X_cb_full.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=(X_cb_full.iloc[val_idx], y.iloc[val_idx]), cat_features=cat_indices, use_best_model=True)
    cb_d8_oof[val_idx] = mdl.predict(X_cb_full.iloc[val_idx])
    cb_d8_test += mdl.predict(X_test_cb) / len(folds)
cb_d8_oof, cb_d8_test = np.clip(cb_d8_oof,0,None), np.clip(cb_d8_test,0,None)
print(f"CB-deep8 OOF: {rmse(y,cb_d8_oof):.5f}")

# === PART 4: Ridge stacking with all models ===
print("\n"+"="*80, "\nPART 4: Expanded Ridge Stacking")

models_all = {
    "cat": (model_outputs["catboost"]["oof"], model_outputs["catboost"]["test_pred"]),
    "lgb": (model_outputs["lightgbm"]["oof"], model_outputs["lightgbm"]["test_pred"]),
    "xgb": (model_outputs["xgboost"]["oof"], model_outputs["xgboost"]["test_pred"]),
    "cat_log": (cb_log_oof, cb_log_test), "lgb_log": (lgb_log_oof, lgb_log_test),
    "xgb_log": (xgb_log_oof, xgb_log_test),
    "cat_sqrt": (cb_sqrt_oof, cb_sqrt_test), "lgb_sqrt": (lgb_sqrt_oof, lgb_sqrt_test),
    "xgb_sqrt": (xgb_sqrt_oof, xgb_sqrt_test),
    "cat_q55": (cb_q55_oof, cb_q55_test), "lgb_q55": (lgb_q55_oof, lgb_q55_test),
    "cat_d8": (cb_d8_oof, cb_d8_test),
}

configs = {
    "ridge6": ["cat","lgb","xgb","cat_log","lgb_log","xgb_log"],
    "ridge9": ["cat","lgb","xgb","cat_log","lgb_log","xgb_log","cat_sqrt","lgb_sqrt","xgb_sqrt"],
    "ridge11": ["cat","lgb","xgb","cat_log","lgb_log","xgb_log","cat_sqrt","lgb_sqrt","xgb_sqrt","cat_q55","lgb_q55"],
    "ridge12": list(models_all.keys()),
}

stack_rows = []
stack_preds = {}
for cfg, names in configs.items():
    So = np.column_stack([models_all[n][0] for n in names])
    St = np.column_stack([models_all[n][1] for n in names])
    So_ext = np.column_stack([So, So**2]); St_ext = np.column_stack([St, St**2])
    for alpha in [1.0, 3.0, 5.0, 10.0, 20.0, 50.0]:
        ro, rt = np.zeros(len(y)), np.zeros(len(base_test))
        for fi, (tr_idx, val_idx) in enumerate(folds):
            sc = StandardScaler(); Xtr=sc.fit_transform(So_ext[tr_idx])
            Xva=sc.transform(So_ext[val_idx]); Xte=sc.transform(St_ext)
            m = Ridge(alpha=alpha, random_state=RANDOM_STATE); m.fit(Xtr, y.values[tr_idx])
            ro[val_idx]=m.predict(Xva); rt+=m.predict(Xte)/len(folds)
        ro,rt = np.clip(ro,0,None), np.clip(rt,0,None)
        s = rmse(y,ro); key = f"{cfg}_a{alpha}"
        stack_rows.append({"key":key,"config":cfg,"alpha":alpha,"rmse":s,"d_11O":s-BEST_11O})
        stack_preds[key] = (ro, rt)
        print(f"  {key}: RMSE={s:.5f} (vs 11O: {s-BEST_11O:+.5f})")

sdf = pd.DataFrame(stack_rows).sort_values("rmse")
print("\nTop stacking results:"); display(sdf.head(15))

best_key = sdf.iloc[0]["key"]
best_q_oof, best_q_test = stack_preds[best_key]
print(f"\nBest: {best_key}, RMSE={sdf.iloc[0]['rmse']:.5f}")

# Bin report
t = train_raw[[ID_COL, TARGET]].copy(); t["p"]=best_q_oof; t["e"]=t["p"]-t[TARGET]; t["se"]=t["e"]**2
t["bin"] = pd.qcut(t[TARGET],5,labels=["very_low","low","medium","high","very_high"],duplicates="drop")
r = t.groupby("bin").agg(n=(TARGET,"count"),tgt_mean=(TARGET,"mean"),pred_mean=("p","mean"),
    bias=("e","mean"),seg_rmse=("se",lambda x:np.sqrt(np.mean(x)))).reset_index()
print(f"Best 11Q stack | RMSE: {sdf.iloc[0]['rmse']:.5f}"); display(r.sort_values("seg_rmse",ascending=False))

# === PART 5: Save ===
print("\n"+"="*80, "\nPART 5: Save")
sub = sample_submission.copy(); sub[TARGET] = np.clip(best_q_test,0,None)
assert sub[TARGET].isna().sum()==0 and (sub[TARGET]>=0).all()
sub.to_csv("submission_11q_stack.csv", index=False)
print("Saved submission_11q_stack.csv"); display(sub[TARGET].describe().to_frame().T)


In [ ]:
# ==================================================
# SECTION 11R - Missed-Tail Residual Correction
# ==================================================
# Goal:
# 1. Keep 11O adaptive as the base prediction.
# 2. Detect samples that are likely underpredicted in high / very_high area.
# 3. Apply small positive-only residual correction to top-k suspicious samples only.
# 4. Avoid global upward correction that damaged 11P and 11Q.
#
# Main idea:
# final_pred = base_pred + capped_positive_correction
#
# No external data. No AutoML.

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import roc_auc_score
from scipy import sparse

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier, LGBMRegressor
except Exception as e:
    print("LightGBM import issue:", e)


# --------------------------------------------------
# Config
# --------------------------------------------------

BEST_11O_OOF = 11.26696803195285

if "adaptive_oof" in globals():
    base_oof_11r = np.clip(np.asarray(adaptive_oof), 0, None)
else:
    raise NameError("adaptive_oof not found. Run 11O first.")

if "final_adaptive_test" in globals():
    base_test_11r = np.clip(np.asarray(final_adaptive_test), 0, None)
else:
    raise NameError("final_adaptive_test not found. Run 11O first.")

N_EST_11R = globals().get("MAIN_N_ESTIMATORS", 3000)
ES_11R = globals().get("EARLY_STOPPING_ROUNDS", 200)
SEED_11R = globals().get("RANDOM_STATE", 42)

print("=" * 90)
print("SECTION 11R - Missed-Tail Residual Correction")
print("Base 11O adaptive OOF:", rmse(y, base_oof_11r))
print("Target to beat:", BEST_11O_OOF)


# --------------------------------------------------
# Safety checks
# --------------------------------------------------

required_vars_11r = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "X_fe",
    "X_test_fe",
    "cat_features",
    "num_features",
    "prepare_encoded_fold",
    "rmse",
]

missing_11r = [v for v in required_vars_11r if v not in globals()]
if len(missing_11r) > 0:
    raise NameError(f"Missing required variables for 11R: {missing_11r}")

print("All required variables for 11R are available.")


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def save_submission_11r(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 90)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def make_target_bin_report_11r(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(np.asarray(pred), 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 90)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(np.asarray(pred), 0, None)))
    display(report.sort_values("rmse", ascending=False))

    return report


def rank_against_train(train_values, values):
    """
    Percentile rank of values against train_values.
    Safe for test because it only uses prediction distribution.
    """
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)

    ranks = np.searchsorted(train_sorted, values, side="right") / len(train_sorted)
    return ranks


def append_meta_features_11r(X_base, meta_features):
    """
    Append meta prediction features to encoded matrix.
    Works for dense, pandas, and sparse matrices.
    """
    meta_features = np.asarray(meta_features, dtype=float)

    if sparse.issparse(X_base):
        return sparse.hstack([X_base, sparse.csr_matrix(meta_features)]).tocsr()

    if isinstance(X_base, pd.DataFrame):
        return np.hstack([X_base.values, meta_features])

    return np.hstack([np.asarray(X_base), meta_features])


def fit_lgbm_classifier_11r(model, X_tr, y_tr, X_val, y_val):
    try:
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(ES_11R, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
    except TypeError:
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            early_stopping_rounds=ES_11R,
            verbose=False,
        )

    return model


def fit_lgbm_regressor_11r(model, X_tr, y_tr, X_val, y_val, sample_weight=None):
    try:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(ES_11R, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
    except TypeError:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=ES_11R,
            verbose=False,
        )

    return model


# --------------------------------------------------
# Part 0 - Base report
# --------------------------------------------------

base_report_11r = make_target_bin_report_11r(
    base_oof_11r,
    "11R base: 11O adaptive"
)

base_residual_11r = y.values - base_oof_11r
positive_residual_11r = np.maximum(base_residual_11r, 0)

print("=" * 90)
print("Base residual summary")
display(pd.Series(base_residual_11r).describe().to_frame("base_residual").T)
display(pd.Series(positive_residual_11r).describe().to_frame("positive_residual").T)


# --------------------------------------------------
# Part 1 - Build meta prediction features
# --------------------------------------------------

meta_cols_oof_11r = []
meta_cols_test_11r = []
meta_names_11r = []

def add_meta_col_11r(name, oof_col, test_col):
    oof_col = np.clip(np.asarray(oof_col), 0, None)
    test_col = np.clip(np.asarray(test_col), 0, None)

    if len(oof_col) != len(y):
        print(f"Skip meta {name}: invalid OOF length")
        return

    if len(test_col) != sample_submission.shape[0]:
        print(f"Skip meta {name}: invalid test length")
        return

    meta_cols_oof_11r.append(oof_col.reshape(-1, 1))
    meta_cols_test_11r.append(test_col.reshape(-1, 1))
    meta_names_11r.append(name)

    print(f"Added meta feature: {name}")


# Main base
add_meta_col_11r("base_11O_adaptive", base_oof_11r, base_test_11r)

# Existing model outputs
if "model_outputs" in globals():
    for model_name in ["catboost", "lightgbm", "xgboost"]:
        if model_name in model_outputs:
            if "oof" in model_outputs[model_name] and "test_pred" in model_outputs[model_name]:
                add_meta_col_11r(
                    f"base_{model_name}",
                    model_outputs[model_name]["oof"],
                    model_outputs[model_name]["test_pred"],
                )

# Previous useful predictions if available
optional_prediction_pairs_11r = [
    ("global_11O", "global_oof", "global_test"),
    ("ridge6_11N", "s6_oof", "s6_test"),
    ("ridge3_11M", "ridge3_oof", "ridge3_test"),
    ("cb_log", "cb_log_oof", "cb_log_test"),
    ("lgb_log", "lgb_log_oof", "lgb_log_test"),
    ("xgb_log", "xgb_log_oof", "xgb_log_test"),
]

for name, oof_name, test_name in optional_prediction_pairs_11r:
    if oof_name in globals() and test_name in globals():
        add_meta_col_11r(name, globals()[oof_name], globals()[test_name])

# best_r fallback
if "best_r" in globals():
    if isinstance(best_r, dict) and "oof" in best_r and "test" in best_r:
        add_meta_col_11r("best_r", best_r["oof"], best_r["test"])

# Tail candidates if available
if "best_tail" in globals():
    if isinstance(best_tail, dict):
        if "oof_pred" in best_tail and "test_pred" in best_tail:
            add_meta_col_11r("best_tail", best_tail["oof_pred"], best_tail["test_pred"])

if "best_gated" in globals():
    if isinstance(best_gated, dict):
        if "oof_pred" in best_gated and "test_pred" in best_gated:
            add_meta_col_11r("best_gated", best_gated["oof_pred"], best_gated["test_pred"])

M_oof_raw_11r = np.hstack(meta_cols_oof_11r)
M_test_raw_11r = np.hstack(meta_cols_test_11r)

# Summary meta features
M_oof_summary_11r = np.column_stack([
    M_oof_raw_11r.mean(axis=1),
    M_oof_raw_11r.std(axis=1),
    M_oof_raw_11r.min(axis=1),
    M_oof_raw_11r.max(axis=1),
    M_oof_raw_11r.max(axis=1) - M_oof_raw_11r.min(axis=1),
    rank_against_train(base_oof_11r, base_oof_11r),
    np.log1p(base_oof_11r),
    base_oof_11r ** 2,
])

M_test_summary_11r = np.column_stack([
    M_test_raw_11r.mean(axis=1),
    M_test_raw_11r.std(axis=1),
    M_test_raw_11r.min(axis=1),
    M_test_raw_11r.max(axis=1),
    M_test_raw_11r.max(axis=1) - M_test_raw_11r.min(axis=1),
    rank_against_train(base_oof_11r, base_test_11r),
    np.log1p(base_test_11r),
    base_test_11r ** 2,
])

M_oof_11r = np.hstack([M_oof_raw_11r, M_oof_summary_11r])
M_test_11r = np.hstack([M_test_raw_11r, M_test_summary_11r])

print("=" * 90)
print("Meta features for 11R")
print("Raw prediction features:", meta_names_11r)
print("M_oof_11r shape:", M_oof_11r.shape)
print("M_test_11r shape:", M_test_11r.shape)


# --------------------------------------------------
# Part 2 - 11R-A: Fold-safe prediction-rank calibration
# --------------------------------------------------
# This is the simple calibration route.
# It applies small positive correction based on base prediction percentile only.

print("=" * 90)
print("PART 2 - 11R-A: Fold-safe prediction-rank calibration")

rank_cal_rows_11r = []
rank_cal_outputs_11r = {}

bin_sets_11r = {
    "coarse": [0.00, 0.50, 0.65, 0.75, 0.85, 0.92, 0.97, 1.00],
    "tail_focused": [0.00, 0.60, 0.70, 0.80, 0.88, 0.94, 0.98, 1.00],
    "very_tail": [0.00, 0.70, 0.80, 0.88, 0.94, 0.97, 0.99, 1.00],
}

for bin_name, bin_quantiles in bin_sets_11r.items():
    for min_base_q in [0.65, 0.70, 0.75, 0.80, 0.85]:
        for damp in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
            for cap in [0.50, 0.75, 1.00, 1.50, 2.00, 2.50]:
                cal_oof = np.zeros(len(y))
                cal_test = np.zeros(sample_submission.shape[0])

                for fold, (tr_idx, val_idx) in enumerate(folds, 1):
                    base_tr = base_oof_11r[tr_idx]
                    base_val = base_oof_11r[val_idx]
                    y_tr = y.values[tr_idx]

                    residual_tr = y_tr - base_tr

                    edges = np.quantile(base_tr, bin_quantiles)
                    edges = np.unique(edges)

                    if len(edges) < 3:
                        cal_oof[val_idx] = base_val
                        cal_test += base_test_11r / len(folds)
                        continue

                    tr_bin = np.digitize(base_tr, edges[1:-1], right=True)
                    val_bin = np.digitize(base_val, edges[1:-1], right=True)
                    test_bin = np.digitize(base_test_11r, edges[1:-1], right=True)

                    min_base_threshold = np.quantile(base_tr, min_base_q)

                    bin_bias = {}

                    for b in np.unique(tr_bin):
                        mask_b = tr_bin == b

                        if mask_b.sum() < 80:
                            bin_bias[b] = 0.0
                            continue

                        raw_bias = np.mean(residual_tr[mask_b])

                        # Positive-only correction.
                        # If a bin is overpredicted, do not push it further.
                        bin_bias[b] = max(raw_bias, 0.0)

                    val_corr = np.array([bin_bias.get(b, 0.0) for b in val_bin])
                    test_corr = np.array([bin_bias.get(b, 0.0) for b in test_bin])

                    val_corr = np.clip(damp * val_corr, 0, cap)
                    test_corr = np.clip(damp * test_corr, 0, cap)

                    val_corr = np.where(base_val >= min_base_threshold, val_corr, 0.0)
                    test_corr = np.where(base_test_11r >= min_base_threshold, test_corr, 0.0)

                    cal_oof[val_idx] = base_val + val_corr
                    cal_test += (base_test_11r + test_corr) / len(folds)

                cal_oof = np.clip(cal_oof, 0, None)
                cal_test = np.clip(cal_test, 0, None)

                score = rmse(y, cal_oof)
                key = f"rankcal_{bin_name}_minq{min_base_q}_d{damp}_cap{cap}"

                rank_cal_rows_11r.append({
                    "key": key,
                    "method": "rank_calibration",
                    "bin_set": bin_name,
                    "min_base_q": min_base_q,
                    "damp": damp,
                    "cap": cap,
                    "rmse": score,
                    "delta_vs_11O": score - BEST_11O_OOF,
                    "mean_correction": np.mean(cal_oof - base_oof_11r),
                    "max_correction": np.max(cal_oof - base_oof_11r),
                    "corrected_rate": np.mean((cal_oof - base_oof_11r) > 0),
                })

                rank_cal_outputs_11r[key] = {
                    "oof": cal_oof,
                    "test": cal_test,
                    "score": score,
                }

rank_cal_df_11r = pd.DataFrame(rank_cal_rows_11r).sort_values("rmse")

print("Top rank calibration results")
display(rank_cal_df_11r.head(20))

best_rank_key_11r = rank_cal_df_11r.iloc[0]["key"]
best_rank_oof_11r = rank_cal_outputs_11r[best_rank_key_11r]["oof"]
best_rank_test_11r = rank_cal_outputs_11r[best_rank_key_11r]["test"]

print("Best rank calibration:", best_rank_key_11r)
print("Best rank calibration RMSE:", rmse(y, best_rank_oof_11r))
rank_report_11r = make_target_bin_report_11r(
    best_rank_oof_11r,
    f"Best 11R-A rank calibration: {best_rank_key_11r}"
)


# --------------------------------------------------
# Part 3 - 11R-B: Missed-tail classifier + residual model
# --------------------------------------------------
# This is the main route.
# Label = high target AND strongly underpredicted by 11O base.

print("=" * 90)
print("PART 3 - 11R-B: Missed-tail classifier + residual model")

target_q80_11r = np.quantile(y.values, 0.80)
target_q85_11r = np.quantile(y.values, 0.85)
target_q90_11r = np.quantile(y.values, 0.90)

label_configs_11r = [
    {
        "name": "q80_res5",
        "target_threshold": target_q80_11r,
        "residual_threshold": 5.0,
    },
    {
        "name": "q80_res8",
        "target_threshold": target_q80_11r,
        "residual_threshold": 8.0,
    },
    {
        "name": "q85_res5",
        "target_threshold": target_q85_11r,
        "residual_threshold": 5.0,
    },
    {
        "name": "q85_res8",
        "target_threshold": target_q85_11r,
        "residual_threshold": 8.0,
    },
    {
        "name": "q90_res5",
        "target_threshold": target_q90_11r,
        "residual_threshold": 5.0,
    },
    {
        "name": "q90_res8",
        "target_threshold": target_q90_11r,
        "residual_threshold": 8.0,
    },
]

missed_tail_outputs_11r = {}
missed_tail_train_rows_11r = []

for cfg in label_configs_11r:
    label_name = cfg["name"]
    target_threshold = cfg["target_threshold"]
    residual_threshold = cfg["residual_threshold"]

    tail_label = (
        (y.values >= target_threshold) &
        ((y.values - base_oof_11r) >= residual_threshold)
    ).astype(int)

    pos_count = int(tail_label.sum())
    pos_rate = float(tail_label.mean())

    print("=" * 90)
    print(f"Training missed-tail models: {label_name}")
    print("Positive count:", pos_count)
    print("Positive rate:", pos_rate)

    if pos_count < 50:
        print(f"Skip {label_name}: too few positive samples.")
        continue

    gate_oof = np.zeros(len(y))
    gate_test = np.zeros(sample_submission.shape[0])

    resid_oof = np.zeros(len(y))
    resid_test = np.zeros(sample_submission.shape[0])

    auc_scores = []
    resid_fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw = X_fe.iloc[tr_idx]
        X_val_raw = X_fe.iloc[val_idx]
        X_te_raw = X_test_fe

        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(
            X_tr_raw,
            X_val_raw,
            X_te_raw,
            cat_features,
            num_features,
        )

        X_tr_ext = append_meta_features_11r(X_tr, M_oof_11r[tr_idx])
        X_val_ext = append_meta_features_11r(X_val, M_oof_11r[val_idx])
        X_te_ext = append_meta_features_11r(X_te, M_test_11r)

        y_clf_tr = tail_label[tr_idx]
        y_clf_val = tail_label[val_idx]

        pos_tr = y_clf_tr.sum()
        neg_tr = len(y_clf_tr) - pos_tr
        scale_pos_weight = max(neg_tr / max(pos_tr, 1), 1.0)

        clf = LGBMClassifier(
            objective="binary",
            n_estimators=N_EST_11R,
            learning_rate=0.025,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.5,
            reg_lambda=5.0,
            min_child_samples=40,
            scale_pos_weight=scale_pos_weight,
            random_state=SEED_11R + 5100 + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        clf = fit_lgbm_classifier_11r(
            clf,
            X_tr_ext,
            y_clf_tr,
            X_val_ext,
            y_clf_val,
        )

        val_prob = clf.predict_proba(X_val_ext)[:, 1]
        test_prob = clf.predict_proba(X_te_ext)[:, 1]

        gate_oof[val_idx] = val_prob
        gate_test += test_prob / len(folds)

        if len(np.unique(y_clf_val)) > 1:
            fold_auc = roc_auc_score(y_clf_val, val_prob)
            auc_scores.append(fold_auc)
            print(f"  Fold {fold} gate AUC: {fold_auc:.5f}")
        else:
            print(f"  Fold {fold} gate AUC: skipped, one class in validation")

        # Residual target
        # We learn positive residual only, not full target.
        y_resid_tr = positive_residual_11r[tr_idx]
        y_resid_val = positive_residual_11r[val_idx]

        # Emphasize missed-tail samples, but still train on all rows.
        sample_weight_resid = (
            1.0
            + 4.0 * tail_label[tr_idx]
            + 1.5 * (y.values[tr_idx] >= target_q80_11r)
            + 0.5 * (base_oof_11r[tr_idx] >= np.quantile(base_oof_11r, 0.75))
        )

        reg = LGBMRegressor(
            objective="regression",
            n_estimators=N_EST_11R,
            learning_rate=0.025,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.5,
            reg_lambda=8.0,
            min_child_samples=50,
            random_state=SEED_11R + 6100 + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        reg = fit_lgbm_regressor_11r(
            reg,
            X_tr_ext,
            y_resid_tr,
            X_val_ext,
            y_resid_val,
            sample_weight=sample_weight_resid,
        )

        val_resid_pred = np.clip(reg.predict(X_val_ext), 0, None)
        test_resid_pred = np.clip(reg.predict(X_te_ext), 0, None)

        resid_oof[val_idx] = val_resid_pred
        resid_test += test_resid_pred / len(folds)

        resid_score = rmse(y_resid_val, val_resid_pred)
        resid_fold_scores.append(resid_score)
        print(f"  Fold {fold} residual RMSE: {resid_score:.5f}")

    missed_tail_outputs_11r[label_name] = {
        "gate_oof": gate_oof,
        "gate_test": gate_test,
        "resid_oof": np.clip(resid_oof, 0, None),
        "resid_test": np.clip(resid_test, 0, None),
        "tail_label": tail_label,
        "pos_count": pos_count,
        "pos_rate": pos_rate,
        "auc_mean": np.mean(auc_scores) if len(auc_scores) > 0 else np.nan,
        "auc_std": np.std(auc_scores) if len(auc_scores) > 0 else np.nan,
        "resid_rmse_mean": np.mean(resid_fold_scores),
    }

    missed_tail_train_rows_11r.append({
        "label_name": label_name,
        "pos_count": pos_count,
        "pos_rate": pos_rate,
        "auc_mean": np.mean(auc_scores) if len(auc_scores) > 0 else np.nan,
        "auc_std": np.std(auc_scores) if len(auc_scores) > 0 else np.nan,
        "resid_rmse_mean": np.mean(resid_fold_scores),
    })

missed_tail_train_df_11r = pd.DataFrame(missed_tail_train_rows_11r)
print("=" * 90)
print("Missed-tail model training summary")
display(missed_tail_train_df_11r)


# --------------------------------------------------
# Part 4 - Grid search selective residual correction
# --------------------------------------------------

print("=" * 90)
print("PART 4 - Grid search selective residual correction")

correction_rows_11r = []
correction_outputs_11r = {}

for label_name, obj in missed_tail_outputs_11r.items():
    gate_oof = obj["gate_oof"]
    gate_test = obj["gate_test"]
    resid_oof = obj["resid_oof"]
    resid_test = obj["resid_test"]

    # Main score: high probability AND predicted missing amount.
    score_oof = gate_oof * resid_oof
    score_test = gate_test * resid_test

    for min_base_q in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
        base_threshold_oof = np.quantile(base_oof_11r, min_base_q)
        base_threshold_test = np.quantile(base_test_11r, min_base_q)

        eligible_oof = base_oof_11r >= base_threshold_oof
        eligible_test = base_test_11r >= base_threshold_test

        for top_frac in [0.005, 0.01, 0.015, 0.02, 0.03, 0.05, 0.08, 0.10]:
            if eligible_oof.sum() < 20:
                continue

            if eligible_test.sum() < 5:
                continue

            score_oof_eligible = score_oof[eligible_oof]
            score_test_eligible = score_test[eligible_test]

            if np.max(score_oof_eligible) <= 0:
                continue

            if np.max(score_test_eligible) <= 0:
                continue

            # Top-k threshold separately for OOF and test to keep correction budget consistent.
            thr_oof = np.quantile(score_oof_eligible, max(0.0, 1.0 - top_frac))
            thr_test = np.quantile(score_test_eligible, max(0.0, 1.0 - top_frac))

            top_mask_oof = eligible_oof & (score_oof >= thr_oof) & (score_oof > 0)
            top_mask_test = eligible_test & (score_test >= thr_test) & (score_test > 0)

            for strength in [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30, 0.35]:
                for cap in [0.50, 0.75, 1.00, 1.25, 1.50, 2.00, 2.50, 3.00]:
                    correction_oof = strength * gate_oof * resid_oof
                    correction_test = strength * gate_test * resid_test

                    correction_oof = np.clip(correction_oof, 0, cap)
                    correction_test = np.clip(correction_test, 0, cap)

                    correction_oof = np.where(top_mask_oof, correction_oof, 0.0)
                    correction_test = np.where(top_mask_test, correction_test, 0.0)

                    final_oof = np.clip(base_oof_11r + correction_oof, 0, None)
                    final_test = np.clip(base_test_11r + correction_test, 0, None)

                    score = rmse(y, final_oof)

                    key = (
                        f"missed_{label_name}"
                        f"_minq{min_base_q}"
                        f"_top{top_frac}"
                        f"_s{strength}"
                        f"_cap{cap}"
                    )

                    correction_rows_11r.append({
                        "key": key,
                        "method": "missed_tail_residual",
                        "label_name": label_name,
                        "min_base_q": min_base_q,
                        "top_frac": top_frac,
                        "strength": strength,
                        "cap": cap,
                        "rmse": score,
                        "delta_vs_11O": score - BEST_11O_OOF,
                        "corrected_rate_oof": top_mask_oof.mean(),
                        "corrected_count_oof": int(top_mask_oof.sum()),
                        "mean_correction": correction_oof.mean(),
                        "max_correction": correction_oof.max(),
                        "gate_mean_top": gate_oof[top_mask_oof].mean() if top_mask_oof.sum() > 0 else 0,
                        "resid_pred_mean_top": resid_oof[top_mask_oof].mean() if top_mask_oof.sum() > 0 else 0,
                    })

                    correction_outputs_11r[key] = {
                        "oof": final_oof,
                        "test": final_test,
                        "correction_oof": correction_oof,
                        "correction_test": correction_test,
                        "score": score,
                    }

correction_df_11r = pd.DataFrame(correction_rows_11r).sort_values("rmse")

print("Top missed-tail residual correction results")
display(correction_df_11r.head(30))


# --------------------------------------------------
# Part 5 - Combine all 11R candidates
# --------------------------------------------------

all_11r_rows = []
all_11r_outputs = {}

# Base
all_11r_rows.append({
    "key": "base_11O_adaptive",
    "method": "base",
    "rmse": rmse(y, base_oof_11r),
    "delta_vs_11O": rmse(y, base_oof_11r) - BEST_11O_OOF,
})
all_11r_outputs["base_11O_adaptive"] = {
    "oof": base_oof_11r,
    "test": base_test_11r,
}

# Rank calibration candidates
for _, row in rank_cal_df_11r.head(50).iterrows():
    key = row["key"]
    all_11r_rows.append({
        "key": key,
        "method": "rank_calibration",
        "rmse": row["rmse"],
        "delta_vs_11O": row["delta_vs_11O"],
    })
    all_11r_outputs[key] = {
        "oof": rank_cal_outputs_11r[key]["oof"],
        "test": rank_cal_outputs_11r[key]["test"],
    }

# Missed-tail candidates
for _, row in correction_df_11r.head(100).iterrows():
    key = row["key"]
    all_11r_rows.append({
        "key": key,
        "method": "missed_tail_residual",
        "rmse": row["rmse"],
        "delta_vs_11O": row["delta_vs_11O"],
    })
    all_11r_outputs[key] = {
        "oof": correction_outputs_11r[key]["oof"],
        "test": correction_outputs_11r[key]["test"],
    }

all_11r_df = pd.DataFrame(all_11r_rows).sort_values("rmse")

print("=" * 90)
print("Final 11R candidate comparison")
display(all_11r_df.head(30))

best_11r_key = all_11r_df.iloc[0]["key"]
best_11r_method = all_11r_df.iloc[0]["method"]
best_11r_oof = all_11r_outputs[best_11r_key]["oof"]
best_11r_test = all_11r_outputs[best_11r_key]["test"]
best_11r_score = rmse(y, best_11r_oof)

print("Best 11R key:", best_11r_key)
print("Best 11R method:", best_11r_method)
print("Best 11R RMSE:", best_11r_score)
print("Delta vs 11O:", best_11r_score - BEST_11O_OOF)

best_11r_report = make_target_bin_report_11r(
    best_11r_oof,
    f"Best 11R final: {best_11r_key}"
)


# --------------------------------------------------
# Part 6 - Compare base vs best 11R by bin
# --------------------------------------------------

print("=" * 90)
print("Bin comparison: 11O base vs best 11R")

base_bin = train_raw[[ID_COL, TARGET]].copy()
base_bin["base_pred"] = base_oof_11r
base_bin["best_pred"] = best_11r_oof
base_bin["base_error"] = base_bin["base_pred"] - base_bin[TARGET]
base_bin["best_error"] = base_bin["best_pred"] - base_bin[TARGET]
base_bin["base_se"] = base_bin["base_error"] ** 2
base_bin["best_se"] = base_bin["best_error"] ** 2

base_bin["target_bin_5"] = pd.qcut(
    base_bin[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
)

bin_compare_11r = (
    base_bin.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),
        base_pred_mean=("base_pred", "mean"),
        best_pred_mean=("best_pred", "mean"),
        base_bias=("base_error", "mean"),
        best_bias=("best_error", "mean"),
        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        best_rmse=("best_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11r["rmse_delta_best_minus_base"] = (
    bin_compare_11r["best_rmse"] - bin_compare_11r["base_rmse"]
)

bin_compare_11r["bias_abs_delta_best_minus_base"] = (
    bin_compare_11r["best_bias"].abs() - bin_compare_11r["base_bias"].abs()
)

display(bin_compare_11r.sort_values("base_rmse", ascending=False))


# --------------------------------------------------
# Part 7 - Save submissions
# --------------------------------------------------

print("=" * 90)
print("PART 7 - Save submissions")

sub_11r_best = save_submission_11r(
    best_11r_test,
    "submission_11r_best.csv"
)

# Also save base again for safety comparison.
sub_11o_ref = save_submission_11r(
    base_test_11r,
    "submission_11o_adaptive_reference.csv"
)


# --------------------------------------------------
# Part 8 - Decision guide
# --------------------------------------------------

print("=" * 90)
print("11R Decision Guide")
print(f"11O adaptive OOF : {rmse(y, base_oof_11r):.6f}")
print(f"Best 11R OOF     : {best_11r_score:.6f}")
print(f"Delta vs 11O     : {best_11r_score - BEST_11O_OOF:.6f}")

if best_11r_score < BEST_11O_OOF - 0.003:
    print("Decision: STRONG CANDIDATE. Submit submission_11r_best.csv.")
elif best_11r_score < BEST_11O_OOF:
    print("Decision: MEDIUM CANDIDATE. Submit only if very_high improves and lower bins are stable.")
else:
    print("Decision: DO NOT FORCE. Keep 11O adaptive as primary.")

print("=" * 90)
print("Please screenshot these outputs:")
print("1. all_11r_df.head(30)")
print("2. best_11r_report")
print("3. bin_compare_11r")
print("4. correction_df_11r.head(30)")
print("5. rank_cal_df_11r.head(20)")

In [ ]:
# ==================================================
# SECTION 11S - Stable Missed-Tail Correction
# ==================================================
# Contains:
# A. Correction blend
# B. Empirical residual lookup
# C. Robust final selection
#
# Goal:
# 1. Keep 11O adaptive as base.
# 2. Reuse 11R missed-tail signals.
# 3. Improve correction magnitude stability.
# 4. Avoid overfitting to tiny corrected-count configs.
#
# Paste this cell AFTER Section 11R.
# No external data. No AutoML.

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict


# --------------------------------------------------
# Config
# --------------------------------------------------

BEST_11O_OOF = 11.26696803195285
SEED_11S = globals().get("RANDOM_STATE", 42)

print("=" * 100)
print("SECTION 11S - Stable Missed-Tail Correction")
print("Target to beat:", BEST_11O_OOF)


# --------------------------------------------------
# Safety checks
# --------------------------------------------------

required_vars_11s = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "correction_df_11r",
    "correction_outputs_11r",
    "missed_tail_outputs_11r",
]

missing_11s = [v for v in required_vars_11s if v not in globals()]
if len(missing_11s) > 0:
    raise NameError(f"Missing required variables for 11S: {missing_11s}")

base_oof_11s = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11s = np.clip(np.asarray(final_adaptive_test), 0, None)

print("Base 11O adaptive OOF:", rmse(y, base_oof_11s))

if abs(rmse(y, base_oof_11s) - BEST_11O_OOF) > 0.01:
    print("Warning: BEST_11O_OOF differs from current adaptive_oof RMSE.")

base_residual_11s = y.values - base_oof_11s
positive_residual_11s = np.maximum(base_residual_11s, 0)


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def save_submission_11s(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def make_target_bin_report_11s(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(np.asarray(pred), 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(np.asarray(pred), 0, None)))
    display(report.sort_values("rmse", ascending=False))

    return report


def rank_against_train_11s(train_values, values):
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)
    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def get_11r_correction_by_key(key):
    obj = correction_outputs_11r[key]

    if "correction_oof" in obj and "correction_test" in obj:
        corr_oof = np.asarray(obj["correction_oof"])
        corr_test = np.asarray(obj["correction_test"])
    else:
        corr_oof = np.asarray(obj["oof"]) - base_oof_11s
        corr_test = np.asarray(obj["test"]) - base_test_11s

    corr_oof = np.clip(corr_oof, 0, None)
    corr_test = np.clip(corr_test, 0, None)

    return corr_oof, corr_test


def get_candidate_metrics_11s(pred, correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    overall = rmse(y, pred)

    temp = pd.DataFrame({
        "y": y.values,
        "pred": pred,
    })

    temp["err"] = temp["pred"] - temp["y"]
    temp["se"] = temp["err"] ** 2

    temp["bin"] = pd.qcut(
        temp["y"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )

    bin_rmse = temp.groupby("bin", observed=False)["se"].apply(lambda x: np.sqrt(np.mean(x)))
    bin_bias = temp.groupby("bin", observed=False)["err"].mean()

    if correction is None:
        correction = pred - base_oof_11s

    correction = np.asarray(correction)

    metrics = {
        "rmse": overall,
        "very_high_rmse": float(bin_rmse.get("very_high", np.nan)),
        "high_rmse": float(bin_rmse.get("high", np.nan)),
        "medium_rmse": float(bin_rmse.get("medium", np.nan)),
        "low_rmse": float(bin_rmse.get("low", np.nan)),
        "very_low_rmse": float(bin_rmse.get("very_low", np.nan)),
        "very_high_bias": float(bin_bias.get("very_high", np.nan)),
        "high_bias": float(bin_bias.get("high", np.nan)),
        "medium_bias": float(bin_bias.get("medium", np.nan)),
        "corrected_count": int((correction > 1e-12).sum()),
        "corrected_rate": float(np.mean(correction > 1e-12)),
        "mean_correction": float(np.mean(correction)),
        "max_correction": float(np.max(correction)),
    }

    return metrics


def calc_positive_benefit_rate_11s(pred):
    pred = np.clip(np.asarray(pred), 0, None)

    base_se = (base_oof_11s - y.values) ** 2
    new_se = (pred - y.values) ** 2

    benefit = base_se - new_se
    corrected = np.abs(pred - base_oof_11s) > 1e-12

    if corrected.sum() == 0:
        return 0.0, 0.0, 0

    return (
        float(np.mean(benefit[corrected] > 0)),
        float(np.mean(benefit[corrected])),
        int(corrected.sum()),
    )


base_metrics_11s = get_candidate_metrics_11s(base_oof_11s, np.zeros_like(base_oof_11s))

print("=" * 100)
print("Base 11O metrics")
display(pd.DataFrame([base_metrics_11s]))

base_report_11s = make_target_bin_report_11s(
    base_oof_11s,
    "11S base: 11O adaptive"
)


# ==================================================
# PART A - Correction Blend
# ==================================================
# Blend narrow and stable correction candidates from 11R.
#
# Narrow:
# - Highest OOF but tiny corrected_count.
#
# Stable:
# - Slightly worse OOF but broader correction and lower cap.

print("=" * 100)
print("PART A - Correction Blend")

corr_df = correction_df_11r.copy()

# Defensive sorting
corr_df = corr_df.sort_values("rmse").reset_index(drop=True)

# Narrow candidates: close to best 11R pattern
narrow_df_11s = corr_df[
    (corr_df["method"] == "missed_tail_residual") &
    (corr_df["label_name"].astype(str).str.contains("q85_res5")) &
    (np.isclose(corr_df["min_base_q"].astype(float), 0.85)) &
    (np.isclose(corr_df["top_frac"].astype(float), 0.005))
].sort_values("rmse").head(12)

# Stable candidates: wider coverage, smaller cap
stable_df_11s = corr_df[
    (corr_df["method"] == "missed_tail_residual") &
    (corr_df["label_name"].astype(str).str.contains("q85_res8")) &
    (corr_df["corrected_count_oof"].astype(float) >= 100) &
    (corr_df["corrected_count_oof"].astype(float) <= 400) &
    (corr_df["cap"].astype(float) <= 1.0)
].sort_values("rmse").head(12)

# Fallbacks if filters are too strict
if len(narrow_df_11s) == 0:
    print("Narrow fallback: using top 12 overall correction candidates.")
    narrow_df_11s = corr_df.sort_values("rmse").head(12)

if len(stable_df_11s) == 0:
    print("Stable fallback: using top candidates with corrected_count >= 50.")
    stable_df_11s = corr_df[
        corr_df["corrected_count_oof"].astype(float) >= 50
    ].sort_values("rmse").head(12)

print("Narrow candidates")
display(narrow_df_11s[["key", "rmse", "delta_vs_11O", "corrected_count_oof", "mean_correction", "max_correction"]])

print("Stable candidates")
display(stable_df_11s[["key", "rmse", "delta_vs_11O", "corrected_count_oof", "mean_correction", "max_correction"]])

blend_rows_11s = []
blend_outputs_11s = {}

w_narrow_grid_11s = [0.00, 0.25, 0.50, 0.75, 1.00, 1.25]
w_stable_grid_11s = [0.00, 0.25, 0.50, 0.75, 1.00, 1.25]
total_cap_grid_11s = [0.50, 0.75, 1.00, 1.50, 2.00, 2.50, 3.00, 4.00]

for _, nrow in narrow_df_11s.iterrows():
    nkey = nrow["key"]
    n_corr_oof, n_corr_test = get_11r_correction_by_key(nkey)

    for _, srow in stable_df_11s.iterrows():
        skey = srow["key"]
        s_corr_oof, s_corr_test = get_11r_correction_by_key(skey)

        for wn in w_narrow_grid_11s:
            for ws in w_stable_grid_11s:
                if wn == 0 and ws == 0:
                    continue

                raw_corr_oof = wn * n_corr_oof + ws * s_corr_oof
                raw_corr_test = wn * n_corr_test + ws * s_corr_test

                for total_cap in total_cap_grid_11s:
                    corr_oof = np.clip(raw_corr_oof, 0, total_cap)
                    corr_test = np.clip(raw_corr_test, 0, total_cap)

                    pred_oof = np.clip(base_oof_11s + corr_oof, 0, None)
                    pred_test = np.clip(base_test_11s + corr_test, 0, None)

                    score = rmse(y, pred_oof)
                    benefit_rate, avg_benefit, corrected_count = calc_positive_benefit_rate_11s(pred_oof)

                    key = (
                        f"blend_n[{nkey}]"
                        f"_s[{skey}]"
                        f"_wn{wn}_ws{ws}_cap{total_cap}"
                    )

                    blend_rows_11s.append({
                        "key": key,
                        "method": "A_correction_blend",
                        "narrow_key": nkey,
                        "stable_key": skey,
                        "w_narrow": wn,
                        "w_stable": ws,
                        "total_cap": total_cap,
                        "rmse": score,
                        "delta_vs_11O": score - BEST_11O_OOF,
                        "corrected_count": int((corr_oof > 1e-12).sum()),
                        "corrected_rate": float(np.mean(corr_oof > 1e-12)),
                        "mean_correction": float(np.mean(corr_oof)),
                        "max_correction": float(np.max(corr_oof)),
                        "benefit_positive_rate": benefit_rate,
                        "avg_benefit_corrected": avg_benefit,
                    })

                    blend_outputs_11s[key] = {
                        "oof": pred_oof,
                        "test": pred_test,
                        "correction_oof": corr_oof,
                        "correction_test": corr_test,
                    }

blend_df_11s = pd.DataFrame(blend_rows_11s).sort_values("rmse")

print("=" * 100)
print("Top PART A correction blend results")
display(blend_df_11s.head(30))


# ==================================================
# PART B - Empirical Residual Lookup
# ==================================================
# Instead of predicting residual magnitude directly,
# estimate typical residual correction by risk groups.
#
# Fold-safe:
# - For each validation fold, lookup table is built using training fold only.
# - Test correction is averaged across folds.

print("=" * 100)
print("PART B - Empirical Residual Lookup")


def make_edges_11s(values, quantiles):
    values = np.asarray(values)
    edges = np.quantile(values, quantiles)
    edges = np.unique(edges)
    return edges


def bin_with_edges_11s(values, edges):
    values = np.asarray(values)

    if len(edges) <= 2:
        return np.zeros(len(values), dtype=int)

    return np.digitize(values, edges[1:-1], right=True)


def make_group_codes_11s(arrays):
    stacked = np.column_stack(arrays)
    return [tuple(row) for row in stacked]


def trimmed_mean_11s(values, low_q=0.10, high_q=0.90):
    values = np.asarray(values)

    if len(values) == 0:
        return 0.0

    lo = np.quantile(values, low_q)
    hi = np.quantile(values, high_q)
    clipped = np.clip(values, lo, hi)

    return float(np.mean(clipped))


# Meta disagreement from 11R if available
if "M_oof_raw_11r" in globals() and "M_test_raw_11r" in globals():
    disagreement_oof_11s = np.std(np.asarray(M_oof_raw_11r), axis=1)
    disagreement_test_11s = np.std(np.asarray(M_test_raw_11r), axis=1)
else:
    print("M_oof_raw_11r not found. Using zero disagreement.")
    disagreement_oof_11s = np.zeros(len(y))
    disagreement_test_11s = np.zeros(sample_submission.shape[0])

base_rank_oof_11s = rank_against_train_11s(base_oof_11s, base_oof_11s)
base_rank_test_11s = rank_against_train_11s(base_oof_11s, base_test_11s)

lookup_label_names_11s = [
    "q85_res5",
    "q85_res8",
    "q90_res5",
    "q90_res8",
]

lookup_label_names_11s = [
    name for name in lookup_label_names_11s
    if name in missed_tail_outputs_11r
]

if len(lookup_label_names_11s) == 0:
    raise ValueError("No missed-tail outputs found for empirical lookup.")

bin_schemes_11s = {
    "base_gate": ["base_rank", "gate"],
    "base_risk": ["base_rank", "risk"],
    "base_gate_risk": ["base_rank", "gate", "risk"],
    "base_risk_disagree": ["base_rank", "risk", "disagree"],
}

quantile_map_11s = {
    "base_rank": [0.00, 0.50, 0.70, 0.82, 0.90, 0.96, 1.00],
    "gate": [0.00, 0.50, 0.70, 0.85, 0.94, 0.98, 1.00],
    "risk": [0.00, 0.50, 0.70, 0.85, 0.94, 0.98, 1.00],
    "disagree": [0.00, 0.50, 0.75, 0.90, 0.97, 1.00],
}

lookup_rows_11s = []
lookup_outputs_11s = {}

lookup_raw_cache_11s = {}

for label_name in lookup_label_names_11s:
    obj = missed_tail_outputs_11r[label_name]

    gate_oof = np.asarray(obj["gate_oof"])
    gate_test = np.asarray(obj["gate_test"])
    resid_pred_oof = np.clip(np.asarray(obj["resid_oof"]), 0, None)
    resid_pred_test = np.clip(np.asarray(obj["resid_test"]), 0, None)

    risk_oof = gate_oof * resid_pred_oof
    risk_test = gate_test * resid_pred_test

    feature_bank_oof = {
        "base_rank": base_rank_oof_11s,
        "gate": gate_oof,
        "risk": risk_oof,
        "disagree": disagreement_oof_11s,
    }

    feature_bank_test = {
        "base_rank": base_rank_test_11s,
        "gate": gate_test,
        "risk": risk_test,
        "disagree": disagreement_test_11s,
    }

    for scheme_name, scheme_features in bin_schemes_11s.items():
        for min_group_count in [40, 60, 100, 150]:
            for stat_method in ["mean", "trimmed"]:
                for shrink_prior in [30, 60, 120]:
                    raw_key = (
                        f"lookup_raw_{label_name}"
                        f"_{scheme_name}"
                        f"_ming{min_group_count}"
                        f"_{stat_method}"
                        f"_prior{shrink_prior}"
                    )

                    raw_corr_oof = np.zeros(len(y))
                    raw_corr_test = np.zeros(sample_submission.shape[0])

                    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
                        # Build fold-specific binned group codes
                        tr_bins = []
                        val_bins = []
                        test_bins = []

                        for feat in scheme_features:
                            tr_values = feature_bank_oof[feat][tr_idx]
                            val_values = feature_bank_oof[feat][val_idx]
                            test_values = feature_bank_test[feat]

                            edges = make_edges_11s(
                                tr_values,
                                quantile_map_11s[feat]
                            )

                            tr_bins.append(bin_with_edges_11s(tr_values, edges))
                            val_bins.append(bin_with_edges_11s(val_values, edges))
                            test_bins.append(bin_with_edges_11s(test_values, edges))

                        tr_codes = make_group_codes_11s(tr_bins)
                        val_codes = make_group_codes_11s(val_bins)
                        test_codes = make_group_codes_11s(test_bins)

                        residual_tr = base_residual_11s[tr_idx]

                        group_values = defaultdict(list)

                        for code, res in zip(tr_codes, residual_tr):
                            group_values[code].append(res)

                        group_corr = {}

                        for code, vals in group_values.items():
                            vals = np.asarray(vals)

                            if len(vals) < min_group_count:
                                group_corr[code] = 0.0
                                continue

                            if stat_method == "mean":
                                raw_stat = float(np.mean(vals))
                            elif stat_method == "trimmed":
                                raw_stat = trimmed_mean_11s(vals)
                            else:
                                raise ValueError("Unknown stat method")

                            # Positive-only empirical residual correction
                            raw_stat = max(raw_stat, 0.0)

                            positive_rate = np.mean(vals > 0)

                            # Reliability:
                            # - larger group count = more trust
                            # - positive residual dominance = more trust
                            count_reliability = len(vals) / (len(vals) + shrink_prior)
                            sign_reliability = max((positive_rate - 0.50) * 2.0, 0.0)

                            corr = raw_stat * count_reliability * sign_reliability

                            group_corr[code] = corr

                        val_corr = np.array([group_corr.get(code, 0.0) for code in val_codes])
                        test_corr = np.array([group_corr.get(code, 0.0) for code in test_codes])

                        raw_corr_oof[val_idx] = val_corr
                        raw_corr_test += test_corr / len(folds)

                    raw_corr_oof = np.clip(raw_corr_oof, 0, None)
                    raw_corr_test = np.clip(raw_corr_test, 0, None)

                    lookup_raw_cache_11s[raw_key] = {
                        "raw_corr_oof": raw_corr_oof,
                        "raw_corr_test": raw_corr_test,
                        "label_name": label_name,
                        "scheme_name": scheme_name,
                        "min_group_count": min_group_count,
                        "stat_method": stat_method,
                        "shrink_prior": shrink_prior,
                        "gate_oof": gate_oof,
                        "gate_test": gate_test,
                        "risk_oof": risk_oof,
                        "risk_test": risk_test,
                    }

                    # Grid final application
                    for min_base_q in [0.70, 0.75, 0.80, 0.85]:
                        base_thr_oof = np.quantile(base_oof_11s, min_base_q)
                        base_thr_test = np.quantile(base_test_11s, min_base_q)

                        eligible_oof = base_oof_11s >= base_thr_oof
                        eligible_test = base_test_11s >= base_thr_test

                        score_oof = risk_oof * raw_corr_oof
                        score_test = risk_test * raw_corr_test

                        for top_frac in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]:
                            if eligible_oof.sum() < 20:
                                continue

                            score_eligible_oof = score_oof[eligible_oof]
                            score_eligible_test = score_test[eligible_test]

                            if len(score_eligible_oof) == 0 or np.max(score_eligible_oof) <= 0:
                                continue

                            if len(score_eligible_test) == 0 or np.max(score_eligible_test) <= 0:
                                continue

                            thr_oof = np.quantile(score_eligible_oof, max(0.0, 1.0 - top_frac))
                            thr_test = np.quantile(score_eligible_test, max(0.0, 1.0 - top_frac))

                            top_mask_oof = eligible_oof & (score_oof >= thr_oof) & (score_oof > 0)
                            top_mask_test = eligible_test & (score_test >= thr_test) & (score_test > 0)

                            for strength in [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30]:
                                for cap in [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 2.50, 3.00]:
                                    corr_oof = np.clip(strength * raw_corr_oof, 0, cap)
                                    corr_test = np.clip(strength * raw_corr_test, 0, cap)

                                    corr_oof = np.where(top_mask_oof, corr_oof, 0.0)
                                    corr_test = np.where(top_mask_test, corr_test, 0.0)

                                    pred_oof = np.clip(base_oof_11s + corr_oof, 0, None)
                                    pred_test = np.clip(base_test_11s + corr_test, 0, None)

                                    score = rmse(y, pred_oof)
                                    benefit_rate, avg_benefit, corrected_count = calc_positive_benefit_rate_11s(pred_oof)

                                    key = (
                                        f"lookup_{label_name}"
                                        f"_{scheme_name}"
                                        f"_ming{min_group_count}"
                                        f"_{stat_method}"
                                        f"_prior{shrink_prior}"
                                        f"_minq{min_base_q}"
                                        f"_top{top_frac}"
                                        f"_s{strength}"
                                        f"_cap{cap}"
                                    )

                                    lookup_rows_11s.append({
                                        "key": key,
                                        "method": "B_empirical_lookup",
                                        "label_name": label_name,
                                        "scheme_name": scheme_name,
                                        "min_group_count": min_group_count,
                                        "stat_method": stat_method,
                                        "shrink_prior": shrink_prior,
                                        "min_base_q": min_base_q,
                                        "top_frac": top_frac,
                                        "strength": strength,
                                        "cap": cap,
                                        "rmse": score,
                                        "delta_vs_11O": score - BEST_11O_OOF,
                                        "corrected_count": int((corr_oof > 1e-12).sum()),
                                        "corrected_rate": float(np.mean(corr_oof > 1e-12)),
                                        "mean_correction": float(np.mean(corr_oof)),
                                        "max_correction": float(np.max(corr_oof)),
                                        "benefit_positive_rate": benefit_rate,
                                        "avg_benefit_corrected": avg_benefit,
                                        "raw_key": raw_key,
                                    })

                                    lookup_outputs_11s[key] = {
                                        "oof": pred_oof,
                                        "test": pred_test,
                                        "correction_oof": corr_oof,
                                        "correction_test": corr_test,
                                    }

lookup_df_11s = pd.DataFrame(lookup_rows_11s).sort_values("rmse")

print("=" * 100)
print("Top PART B empirical residual lookup results")
display(lookup_df_11s.head(30))


# ==================================================
# PART C - Robust Final Selection
# ==================================================
# Select not only by raw RMSE, but also stability:
# - corrected_count not too tiny
# - very_high should improve
# - high / medium should not degrade too much
# - corrected samples should have positive benefit rate

print("=" * 100)
print("PART C - Robust Final Selection")


def add_candidate_to_pool_11s(pool_rows, pool_outputs, key, method, pred_oof, pred_test, corr_oof):
    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)
    corr_oof = np.clip(np.asarray(corr_oof), 0, None)

    metrics = get_candidate_metrics_11s(pred_oof, corr_oof)
    benefit_rate, avg_benefit, corrected_count = calc_positive_benefit_rate_11s(pred_oof)

    row = {
        "key": key,
        "method": method,
        **metrics,
        "delta_vs_11O": metrics["rmse"] - BEST_11O_OOF,
        "benefit_positive_rate": benefit_rate,
        "avg_benefit_corrected": avg_benefit,
    }

    pool_rows.append(row)

    pool_outputs[key] = {
        "oof": pred_oof,
        "test": pred_test,
        "correction_oof": corr_oof,
    }


candidate_rows_11s = []
candidate_outputs_11s = {}

# Base
add_candidate_to_pool_11s(
    candidate_rows_11s,
    candidate_outputs_11s,
    "base_11O_adaptive",
    "base",
    base_oof_11s,
    base_test_11s,
    np.zeros_like(base_oof_11s),
)

# Include best original 11R candidates
for _, row in correction_df_11r.sort_values("rmse").head(100).iterrows():
    key = row["key"]
    corr_oof, corr_test = get_11r_correction_by_key(key)

    pred_oof = np.clip(base_oof_11s + corr_oof, 0, None)
    pred_test = np.clip(base_test_11s + corr_test, 0, None)

    add_candidate_to_pool_11s(
        candidate_rows_11s,
        candidate_outputs_11s,
        f"orig11R_{key}",
        "orig11R",
        pred_oof,
        pred_test,
        corr_oof,
    )

# Include top A
for _, row in blend_df_11s.sort_values("rmse").head(200).iterrows():
    key = row["key"]
    obj = blend_outputs_11s[key]

    add_candidate_to_pool_11s(
        candidate_rows_11s,
        candidate_outputs_11s,
        f"A_{key}",
        "A_correction_blend",
        obj["oof"],
        obj["test"],
        obj["correction_oof"],
    )

# Include top B
for _, row in lookup_df_11s.sort_values("rmse").head(250).iterrows():
    key = row["key"]
    obj = lookup_outputs_11s[key]

    add_candidate_to_pool_11s(
        candidate_rows_11s,
        candidate_outputs_11s,
        f"B_{key}",
        "B_empirical_lookup",
        obj["oof"],
        obj["test"],
        obj["correction_oof"],
    )

candidate_df_11s = pd.DataFrame(candidate_rows_11s)

# Base bin references
base_vh_rmse = base_metrics_11s["very_high_rmse"]
base_high_rmse = base_metrics_11s["high_rmse"]
base_medium_rmse = base_metrics_11s["medium_rmse"]
base_low_rmse = base_metrics_11s["low_rmse"]
base_vlow_rmse = base_metrics_11s["very_low_rmse"]


def robust_penalty_11s(row):
    penalty = 0.0

    corrected_count = row["corrected_count"]

    # Too tiny = likely overfit to a handful of OOF rows
    if corrected_count > 0 and corrected_count < 20:
        penalty += 0.0010
    elif corrected_count >= 20 and corrected_count < 50:
        penalty += 0.0004

    # Too broad = possible leakage to non-tail bins
    if corrected_count > 600:
        penalty += 0.0005
    if corrected_count > 1000:
        penalty += 0.0010

    # If very_high does not improve, penalize
    if row["very_high_rmse"] > base_vh_rmse:
        penalty += 0.0005

    # Guardrails for other bins
    if row["high_rmse"] > base_high_rmse + 0.04:
        penalty += 0.0004

    if row["medium_rmse"] > base_medium_rmse + 0.06:
        penalty += 0.0005

    if row["low_rmse"] > base_low_rmse + 0.04:
        penalty += 0.0004

    if row["very_low_rmse"] > base_vlow_rmse + 0.04:
        penalty += 0.0004

    # Corrected rows should mostly help
    if corrected_count > 0:
        if row["benefit_positive_rate"] < 0.52:
            penalty += 0.0005
        if row["avg_benefit_corrected"] < 0:
            penalty += 0.0005

    return penalty


candidate_df_11s["robust_penalty"] = candidate_df_11s.apply(robust_penalty_11s, axis=1)
candidate_df_11s["selection_score"] = candidate_df_11s["rmse"] + candidate_df_11s["robust_penalty"]

candidate_df_11s = candidate_df_11s.sort_values("rmse").reset_index(drop=True)

print("=" * 100)
print("Top 11S candidates by raw RMSE")
display(candidate_df_11s.head(30))

candidate_df_11s_robust = candidate_df_11s.sort_values("selection_score").reset_index(drop=True)

print("=" * 100)
print("Top 11S candidates by robust selection score")
display(candidate_df_11s_robust.head(30))


# --------------------------------------------------
# Pick raw best and robust best
# --------------------------------------------------

raw_best_key_11s = candidate_df_11s.iloc[0]["key"]
robust_best_key_11s = candidate_df_11s_robust.iloc[0]["key"]

raw_best_oof_11s = candidate_outputs_11s[raw_best_key_11s]["oof"]
raw_best_test_11s = candidate_outputs_11s[raw_best_key_11s]["test"]

robust_best_oof_11s = candidate_outputs_11s[robust_best_key_11s]["oof"]
robust_best_test_11s = candidate_outputs_11s[robust_best_key_11s]["test"]

print("=" * 100)
print("11S selected candidates")
print("Raw best key    :", raw_best_key_11s)
print("Raw best RMSE   :", rmse(y, raw_best_oof_11s))
print("Raw delta 11O   :", rmse(y, raw_best_oof_11s) - BEST_11O_OOF)
print("-" * 100)
print("Robust best key :", robust_best_key_11s)
print("Robust best RMSE:", rmse(y, robust_best_oof_11s))
print("Robust delta 11O:", rmse(y, robust_best_oof_11s) - BEST_11O_OOF)


raw_best_report_11s = make_target_bin_report_11s(
    raw_best_oof_11s,
    f"11S raw best: {raw_best_key_11s}"
)

robust_best_report_11s = make_target_bin_report_11s(
    robust_best_oof_11s,
    f"11S robust best: {robust_best_key_11s}"
)


# --------------------------------------------------
# Bin comparison
# --------------------------------------------------

print("=" * 100)
print("Bin comparison: 11O base vs 11S raw best vs 11S robust best")

compare_df_11s = train_raw[[ID_COL, TARGET]].copy()
compare_df_11s["base_pred"] = base_oof_11s
compare_df_11s["raw_best_pred"] = raw_best_oof_11s
compare_df_11s["robust_best_pred"] = robust_best_oof_11s

compare_df_11s["base_error"] = compare_df_11s["base_pred"] - compare_df_11s[TARGET]
compare_df_11s["raw_error"] = compare_df_11s["raw_best_pred"] - compare_df_11s[TARGET]
compare_df_11s["robust_error"] = compare_df_11s["robust_best_pred"] - compare_df_11s[TARGET]

compare_df_11s["base_se"] = compare_df_11s["base_error"] ** 2
compare_df_11s["raw_se"] = compare_df_11s["raw_error"] ** 2
compare_df_11s["robust_se"] = compare_df_11s["robust_error"] ** 2

compare_df_11s["target_bin_5"] = pd.qcut(
    compare_df_11s[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
)

bin_compare_11s = (
    compare_df_11s.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        raw_pred_mean=("raw_best_pred", "mean"),
        robust_pred_mean=("robust_best_pred", "mean"),

        base_bias=("base_error", "mean"),
        raw_bias=("raw_error", "mean"),
        robust_bias=("robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        raw_rmse=("raw_se", lambda x: np.sqrt(np.mean(x))),
        robust_rmse=("robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11s["raw_rmse_delta"] = bin_compare_11s["raw_rmse"] - bin_compare_11s["base_rmse"]
bin_compare_11s["robust_rmse_delta"] = bin_compare_11s["robust_rmse"] - bin_compare_11s["base_rmse"]

display(bin_compare_11s.sort_values("base_rmse", ascending=False))


# --------------------------------------------------
# Save submissions
# --------------------------------------------------

print("=" * 100)
print("Save 11S submissions")

sub_11s_raw_best = save_submission_11s(
    raw_best_test_11s,
    "submission_11s_raw_best.csv"
)

sub_11s_robust_best = save_submission_11s(
    robust_best_test_11s,
    "submission_11s_robust_best.csv"
)

sub_11o_ref_11s = save_submission_11s(
    base_test_11s,
    "submission_11o_reference_from_11s.csv"
)


# --------------------------------------------------
# Decision guide
# --------------------------------------------------

print("=" * 100)
print("11S Decision Guide")

raw_score = rmse(y, raw_best_oof_11s)
robust_score = rmse(y, robust_best_oof_11s)

print(f"11O adaptive OOF   : {rmse(y, base_oof_11s):.6f}")
print(f"11S raw best OOF   : {raw_score:.6f}")
print(f"11S robust best OOF: {robust_score:.6f}")
print(f"Raw delta vs 11O   : {raw_score - BEST_11O_OOF:.6f}")
print(f"Robust delta vs 11O: {robust_score - BEST_11O_OOF:.6f}")

if raw_score < BEST_11O_OOF - 0.003:
    print("Raw best decision: STRONG candidate.")
elif raw_score < BEST_11O_OOF:
    print("Raw best decision: WEAK/MEDIUM candidate. Submit only if bins look safe.")
else:
    print("Raw best decision: DO NOT FORCE.")

if robust_score < BEST_11O_OOF - 0.003:
    print("Robust best decision: STRONG candidate.")
elif robust_score < BEST_11O_OOF:
    print("Robust best decision: WEAK/MEDIUM candidate. Safer than raw if bins are stable.")
else:
    print("Robust best decision: DO NOT FORCE.")

print("=" * 100)
print("Please screenshot these outputs:")
print("1. blend_df_11s.head(30)")
print("2. lookup_df_11s.head(30)")
print("3. candidate_df_11s.head(30)")
print("4. candidate_df_11s_robust.head(30)")
print("5. raw_best_report_11s")
print("6. robust_best_report_11s")
print("7. bin_compare_11s")

In [ ]:
# ==================================================
# SECTION 11S LITE - Stable Missed-Tail Correction
# ==================================================
# Memory-safe replacement for full 11S.
#
# Contains:
# A. Correction blend from 11R narrow + stable candidates
# B. Empirical residual lookup with limited grid
# C. Robust final selection without storing all candidate arrays
#
# Paste AFTER SECTION 11R.
# Skip SECTION 11P and SECTION 11Q.
#
# Key memory rule:
# - Do NOT store every OOF/test prediction.
# - Store only scalar rows + best raw/robust candidates.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict


# ==================================================
# PART 0 - Setup and cleanup
# ==================================================

print("=" * 100)
print("SECTION 11S LITE - Stable Missed-Tail Correction")

BEST_11O_OOF = 11.26696803195285

# Clean old heavy 11S objects if they exist
for var in [
    "blend_outputs_11s",
    "lookup_outputs_11s",
    "candidate_outputs_11s",
    "lookup_raw_cache_11s",
    "blend_df_11s",
    "lookup_df_11s",
    "candidate_df_11s",
    "candidate_df_11s_robust",
]:
    if var in globals():
        del globals()[var]

gc.collect()

required_vars_11s_lite = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "correction_df_11r",
    "correction_outputs_11r",
    "missed_tail_outputs_11r",
]

missing_11s_lite = [v for v in required_vars_11s_lite if v not in globals()]
if len(missing_11s_lite) > 0:
    raise NameError(f"Missing required variables for 11S Lite: {missing_11s_lite}")

base_oof_11s = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11s = np.clip(np.asarray(final_adaptive_test), 0, None)

base_score_11s = rmse(y, base_oof_11s)

print("Base 11O adaptive OOF:", base_score_11s)
print("Target to beat:", BEST_11O_OOF)

base_residual_11s = y.values - base_oof_11s
positive_residual_11s = np.maximum(base_residual_11s, 0)


# ==================================================
# Helpers
# ==================================================

def save_submission_11s_lite(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11s_lite(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(np.asarray(pred), 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(np.asarray(pred), 0, None)))
    display(report.sort_values("rmse", ascending=False))

    return report


def rank_against_train_11s_lite(train_values, values):
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)
    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def get_11r_correction_lite(key):
    obj = correction_outputs_11r[key]

    if "correction_oof" in obj and "correction_test" in obj:
        corr_oof = np.asarray(obj["correction_oof"])
        corr_test = np.asarray(obj["correction_test"])
    else:
        corr_oof = np.asarray(obj["oof"]) - base_oof_11s
        corr_test = np.asarray(obj["test"]) - base_test_11s

    corr_oof = np.clip(corr_oof, 0, None)
    corr_test = np.clip(corr_test, 0, None)

    return corr_oof, corr_test


def calc_bin_metrics_11s_lite(pred, correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = pd.DataFrame({
        "y": y.values,
        "pred": pred,
    })

    temp["err"] = temp["pred"] - temp["y"]
    temp["se"] = temp["err"] ** 2

    temp["bin"] = pd.qcut(
        temp["y"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    bin_rmse = temp.groupby("bin", observed=False)["se"].apply(lambda x: np.sqrt(np.mean(x)))
    bin_bias = temp.groupby("bin", observed=False)["err"].mean()

    if correction is None:
        correction = pred - base_oof_11s

    correction = np.asarray(correction)

    return {
        "rmse": rmse(y, pred),
        "very_high_rmse": float(bin_rmse.get("very_high", np.nan)),
        "high_rmse": float(bin_rmse.get("high", np.nan)),
        "medium_rmse": float(bin_rmse.get("medium", np.nan)),
        "low_rmse": float(bin_rmse.get("low", np.nan)),
        "very_low_rmse": float(bin_rmse.get("very_low", np.nan)),
        "very_high_bias": float(bin_bias.get("very_high", np.nan)),
        "high_bias": float(bin_bias.get("high", np.nan)),
        "medium_bias": float(bin_bias.get("medium", np.nan)),
        "corrected_count": int((correction > 1e-12).sum()),
        "corrected_rate": float(np.mean(correction > 1e-12)),
        "mean_correction": float(np.mean(correction)),
        "max_correction": float(np.max(correction)),
    }


def calc_benefit_metrics_11s_lite(pred):
    pred = np.clip(np.asarray(pred), 0, None)

    base_se = (base_oof_11s - y.values) ** 2
    new_se = (pred - y.values) ** 2

    benefit = base_se - new_se
    corrected = np.abs(pred - base_oof_11s) > 1e-12

    if corrected.sum() == 0:
        return 0.0, 0.0, 0

    return (
        float(np.mean(benefit[corrected] > 0)),
        float(np.mean(benefit[corrected])),
        int(corrected.sum()),
    )


base_metrics_11s = calc_bin_metrics_11s_lite(
    base_oof_11s,
    correction=np.zeros_like(base_oof_11s),
)

base_vh_rmse = base_metrics_11s["very_high_rmse"]
base_high_rmse = base_metrics_11s["high_rmse"]
base_medium_rmse = base_metrics_11s["medium_rmse"]
base_low_rmse = base_metrics_11s["low_rmse"]
base_vlow_rmse = base_metrics_11s["very_low_rmse"]


def robust_penalty_11s_lite(metrics):
    penalty = 0.0
    corrected_count = metrics["corrected_count"]

    # Tiny correction can overfit OOF.
    if corrected_count > 0 and corrected_count < 20:
        penalty += 0.0010
    elif corrected_count >= 20 and corrected_count < 50:
        penalty += 0.0004

    # Too wide can leak to non-tail rows.
    if corrected_count > 600:
        penalty += 0.0005
    if corrected_count > 1000:
        penalty += 0.0010

    # Very high should not worsen.
    if metrics["very_high_rmse"] > base_vh_rmse:
        penalty += 0.0005

    # Guardrails.
    if metrics["high_rmse"] > base_high_rmse + 0.04:
        penalty += 0.0004

    if metrics["medium_rmse"] > base_medium_rmse + 0.06:
        penalty += 0.0005

    if metrics["low_rmse"] > base_low_rmse + 0.04:
        penalty += 0.0004

    if metrics["very_low_rmse"] > base_vlow_rmse + 0.04:
        penalty += 0.0004

    if corrected_count > 0:
        if metrics["benefit_positive_rate"] < 0.52:
            penalty += 0.0005
        if metrics["avg_benefit_corrected"] < 0:
            penalty += 0.0005

    return penalty


# Best candidate holders
best_raw = {
    "key": "base_11O_adaptive",
    "method": "base",
    "rmse": base_score_11s,
    "selection_score": base_score_11s,
    "oof": base_oof_11s.copy(),
    "test": base_test_11s.copy(),
    "correction_oof": np.zeros_like(base_oof_11s),
}

best_robust = best_raw.copy()

candidate_rows_11s_lite = []


def evaluate_candidate_11s_lite(key, method, pred_oof, pred_test, corr_oof):
    global best_raw, best_robust

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)
    corr_oof = np.clip(np.asarray(corr_oof), 0, None)

    metrics = calc_bin_metrics_11s_lite(pred_oof, corr_oof)
    benefit_rate, avg_benefit, corrected_count = calc_benefit_metrics_11s_lite(pred_oof)

    metrics["benefit_positive_rate"] = benefit_rate
    metrics["avg_benefit_corrected"] = avg_benefit
    metrics["corrected_count"] = corrected_count

    metrics["delta_vs_11O"] = metrics["rmse"] - BEST_11O_OOF
    metrics["robust_penalty"] = robust_penalty_11s_lite(metrics)
    metrics["selection_score"] = metrics["rmse"] + metrics["robust_penalty"]

    row = {
        "key": key,
        "method": method,
        **metrics,
    }

    candidate_rows_11s_lite.append(row)

    if metrics["rmse"] < best_raw["rmse"]:
        best_raw = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": corr_oof.copy(),
        }

    if metrics["selection_score"] < best_robust["selection_score"]:
        best_robust = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": corr_oof.copy(),
        }


# Add base to rows
evaluate_candidate_11s_lite(
    key="base_11O_adaptive",
    method="base",
    pred_oof=base_oof_11s,
    pred_test=base_test_11s,
    corr_oof=np.zeros_like(base_oof_11s),
)


print("=" * 100)
print("Base report")
target_bin_report_11s_lite(base_oof_11s, "11S Lite base: 11O adaptive")


# ==================================================
# PART A - Correction Blend Lite
# ==================================================

print("=" * 100)
print("PART A - Correction Blend Lite")

corr_df = correction_df_11r.copy().sort_values("rmse").reset_index(drop=True)

narrow_df = corr_df[
    (corr_df["method"] == "missed_tail_residual") &
    (corr_df["label_name"].astype(str).str.contains("q85_res5")) &
    (np.isclose(corr_df["min_base_q"].astype(float), 0.85)) &
    (np.isclose(corr_df["top_frac"].astype(float), 0.005))
].sort_values("rmse").head(5)

stable_df = corr_df[
    (corr_df["method"] == "missed_tail_residual") &
    (corr_df["label_name"].astype(str).str.contains("q85_res8")) &
    (corr_df["corrected_count_oof"].astype(float) >= 100) &
    (corr_df["corrected_count_oof"].astype(float) <= 400) &
    (corr_df["cap"].astype(float) <= 1.0)
].sort_values("rmse").head(5)

if len(narrow_df) == 0:
    narrow_df = corr_df.sort_values("rmse").head(5)

if len(stable_df) == 0:
    stable_df = corr_df[
        corr_df["corrected_count_oof"].astype(float) >= 50
    ].sort_values("rmse").head(5)

print("Narrow candidates:")
display(narrow_df[["key", "rmse", "delta_vs_11O", "corrected_count_oof", "mean_correction", "max_correction"]])

print("Stable candidates:")
display(stable_df[["key", "rmse", "delta_vs_11O", "corrected_count_oof", "mean_correction", "max_correction"]])

w_narrow_grid = [0.00, 0.50, 0.75, 1.00, 1.25]
w_stable_grid = [0.00, 0.25, 0.50, 0.75, 1.00]
total_cap_grid = [0.50, 0.75, 1.00, 1.50, 2.00, 3.00]

part_a_rows = []

for _, nrow in narrow_df.iterrows():
    nkey = nrow["key"]
    n_corr_oof, n_corr_test = get_11r_correction_lite(nkey)

    for _, srow in stable_df.iterrows():
        skey = srow["key"]
        s_corr_oof, s_corr_test = get_11r_correction_lite(skey)

        for wn in w_narrow_grid:
            for ws in w_stable_grid:
                if wn == 0 and ws == 0:
                    continue

                raw_corr_oof = wn * n_corr_oof + ws * s_corr_oof
                raw_corr_test = wn * n_corr_test + ws * s_corr_test

                for total_cap in total_cap_grid:
                    corr_oof = np.clip(raw_corr_oof, 0, total_cap)
                    corr_test = np.clip(raw_corr_test, 0, total_cap)

                    pred_oof = np.clip(base_oof_11s + corr_oof, 0, None)
                    pred_test = np.clip(base_test_11s + corr_test, 0, None)

                    key = f"A_blend_wn{wn}_ws{ws}_cap{total_cap}_n{nkey}_s{skey}"

                    evaluate_candidate_11s_lite(
                        key=key,
                        method="A_correction_blend",
                        pred_oof=pred_oof,
                        pred_test=pred_test,
                        corr_oof=corr_oof,
                    )

                    part_a_rows.append(candidate_rows_11s_lite[-1])

                del corr_oof, corr_test, pred_oof, pred_test

        del s_corr_oof, s_corr_test

    del n_corr_oof, n_corr_test
    gc.collect()

part_a_df = pd.DataFrame(part_a_rows).sort_values("rmse")

print("=" * 100)
print("Top PART A results")
display(part_a_df.head(20))


# ==================================================
# PART B - Empirical Residual Lookup Lite
# ==================================================

print("=" * 100)
print("PART B - Empirical Residual Lookup Lite")


def make_edges_11s_lite(values, quantiles):
    edges = np.quantile(np.asarray(values), quantiles)
    return np.unique(edges)


def bin_with_edges_11s_lite(values, edges):
    if len(edges) <= 2:
        return np.zeros(len(values), dtype=int)
    return np.digitize(np.asarray(values), edges[1:-1], right=True)


def group_codes_11s_lite(arrays):
    stacked = np.column_stack(arrays)
    return [tuple(row) for row in stacked]


def trimmed_mean_11s_lite(values, low_q=0.10, high_q=0.90):
    values = np.asarray(values)

    if len(values) == 0:
        return 0.0

    lo = np.quantile(values, low_q)
    hi = np.quantile(values, high_q)

    return float(np.mean(np.clip(values, lo, hi)))


if "M_oof_raw_11r" in globals() and "M_test_raw_11r" in globals():
    disagreement_oof = np.std(np.asarray(M_oof_raw_11r), axis=1)
    disagreement_test = np.std(np.asarray(M_test_raw_11r), axis=1)
else:
    disagreement_oof = np.zeros(len(y))
    disagreement_test = np.zeros(sample_submission.shape[0])

base_rank_oof = rank_against_train_11s_lite(base_oof_11s, base_oof_11s)
base_rank_test = rank_against_train_11s_lite(base_oof_11s, base_test_11s)

label_names = [
    name for name in ["q85_res5", "q85_res8", "q90_res5"]
    if name in missed_tail_outputs_11r
]

if len(label_names) == 0:
    raise ValueError("No usable missed_tail_outputs_11r label found.")

bin_schemes = {
    "base_risk": ["base_rank", "risk"],
    "base_gate_risk": ["base_rank", "gate", "risk"],
    "base_risk_disagree": ["base_rank", "risk", "disagree"],
}

quantile_map = {
    "base_rank": [0.00, 0.60, 0.75, 0.85, 0.93, 0.98, 1.00],
    "gate": [0.00, 0.60, 0.80, 0.92, 0.98, 1.00],
    "risk": [0.00, 0.60, 0.80, 0.92, 0.98, 1.00],
    "disagree": [0.00, 0.60, 0.85, 0.95, 1.00],
}

part_b_rows = []

for label_name in label_names:
    obj = missed_tail_outputs_11r[label_name]

    gate_oof = np.asarray(obj["gate_oof"])
    gate_test = np.asarray(obj["gate_test"])
    resid_oof = np.clip(np.asarray(obj["resid_oof"]), 0, None)
    resid_test = np.clip(np.asarray(obj["resid_test"]), 0, None)

    risk_oof = gate_oof * resid_oof
    risk_test = gate_test * resid_test

    feature_oof = {
        "base_rank": base_rank_oof,
        "gate": gate_oof,
        "risk": risk_oof,
        "disagree": disagreement_oof,
    }

    feature_test = {
        "base_rank": base_rank_test,
        "gate": gate_test,
        "risk": risk_test,
        "disagree": disagreement_test,
    }

    for scheme_name, scheme_features in bin_schemes.items():
        for min_group_count in [60, 100]:
            for stat_method in ["mean", "trimmed"]:
                for shrink_prior in [60, 120]:

                    raw_corr_oof = np.zeros(len(y))
                    raw_corr_test = np.zeros(sample_submission.shape[0])

                    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
                        tr_bins = []
                        val_bins = []
                        test_bins = []

                        for feat in scheme_features:
                            tr_values = feature_oof[feat][tr_idx]
                            val_values = feature_oof[feat][val_idx]
                            test_values = feature_test[feat]

                            edges = make_edges_11s_lite(
                                tr_values,
                                quantile_map[feat],
                            )

                            tr_bins.append(bin_with_edges_11s_lite(tr_values, edges))
                            val_bins.append(bin_with_edges_11s_lite(val_values, edges))
                            test_bins.append(bin_with_edges_11s_lite(test_values, edges))

                        tr_codes = group_codes_11s_lite(tr_bins)
                        val_codes = group_codes_11s_lite(val_bins)
                        test_codes = group_codes_11s_lite(test_bins)

                        residual_tr = base_residual_11s[tr_idx]

                        group_values = defaultdict(list)

                        for code, res in zip(tr_codes, residual_tr):
                            group_values[code].append(res)

                        group_corr = {}

                        for code, vals in group_values.items():
                            vals = np.asarray(vals)

                            if len(vals) < min_group_count:
                                group_corr[code] = 0.0
                                continue

                            if stat_method == "mean":
                                raw_stat = float(np.mean(vals))
                            else:
                                raw_stat = trimmed_mean_11s_lite(vals)

                            raw_stat = max(raw_stat, 0.0)

                            positive_rate = np.mean(vals > 0)
                            count_reliability = len(vals) / (len(vals) + shrink_prior)
                            sign_reliability = max((positive_rate - 0.50) * 2.0, 0.0)

                            group_corr[code] = raw_stat * count_reliability * sign_reliability

                        val_corr = np.array([group_corr.get(code, 0.0) for code in val_codes])
                        test_corr = np.array([group_corr.get(code, 0.0) for code in test_codes])

                        raw_corr_oof[val_idx] = val_corr
                        raw_corr_test += test_corr / len(folds)

                        del group_values, group_corr, val_corr, test_corr

                    raw_corr_oof = np.clip(raw_corr_oof, 0, None)
                    raw_corr_test = np.clip(raw_corr_test, 0, None)

                    score_oof = risk_oof * raw_corr_oof
                    score_test = risk_test * raw_corr_test

                    for min_base_q in [0.80, 0.85]:
                        base_thr_oof = np.quantile(base_oof_11s, min_base_q)
                        base_thr_test = np.quantile(base_test_11s, min_base_q)

                        eligible_oof = base_oof_11s >= base_thr_oof
                        eligible_test = base_test_11s >= base_thr_test

                        if eligible_oof.sum() < 20:
                            continue

                        for top_frac in [0.005, 0.01, 0.02, 0.05, 0.10]:
                            score_eligible_oof = score_oof[eligible_oof]
                            score_eligible_test = score_test[eligible_test]

                            if len(score_eligible_oof) == 0 or np.max(score_eligible_oof) <= 0:
                                continue

                            if len(score_eligible_test) == 0 or np.max(score_eligible_test) <= 0:
                                continue

                            thr_oof = np.quantile(score_eligible_oof, 1.0 - top_frac)
                            thr_test = np.quantile(score_eligible_test, 1.0 - top_frac)

                            top_mask_oof = eligible_oof & (score_oof >= thr_oof) & (score_oof > 0)
                            top_mask_test = eligible_test & (score_test >= thr_test) & (score_test > 0)

                            for strength in [0.075, 0.10, 0.15, 0.20, 0.25]:
                                for cap in [0.50, 0.75, 1.00, 1.50, 2.00]:
                                    corr_oof = np.clip(strength * raw_corr_oof, 0, cap)
                                    corr_test = np.clip(strength * raw_corr_test, 0, cap)

                                    corr_oof = np.where(top_mask_oof, corr_oof, 0.0)
                                    corr_test = np.where(top_mask_test, corr_test, 0.0)

                                    pred_oof = np.clip(base_oof_11s + corr_oof, 0, None)
                                    pred_test = np.clip(base_test_11s + corr_test, 0, None)

                                    key = (
                                        f"B_lookup_{label_name}_{scheme_name}"
                                        f"_ming{min_group_count}_{stat_method}_prior{shrink_prior}"
                                        f"_minq{min_base_q}_top{top_frac}"
                                        f"_s{strength}_cap{cap}"
                                    )

                                    evaluate_candidate_11s_lite(
                                        key=key,
                                        method="B_empirical_lookup",
                                        pred_oof=pred_oof,
                                        pred_test=pred_test,
                                        corr_oof=corr_oof,
                                    )

                                    part_b_rows.append(candidate_rows_11s_lite[-1])

                                    del corr_oof, corr_test, pred_oof, pred_test

                    del raw_corr_oof, raw_corr_test, score_oof, score_test
                    gc.collect()

    del gate_oof, gate_test, resid_oof, resid_test, risk_oof, risk_test
    gc.collect()

part_b_df = pd.DataFrame(part_b_rows).sort_values("rmse")

print("=" * 100)
print("Top PART B results")
display(part_b_df.head(20))


# ==================================================
# PART C - Robust Final Selection
# ==================================================

print("=" * 100)
print("PART C - Robust Final Selection")

candidate_df_11s_lite = pd.DataFrame(candidate_rows_11s_lite)

candidate_df_11s_lite = candidate_df_11s_lite.sort_values("rmse").reset_index(drop=True)

print("Top candidates by raw RMSE")
display(candidate_df_11s_lite.head(30))

candidate_df_11s_lite_robust = candidate_df_11s_lite.sort_values("selection_score").reset_index(drop=True)

print("Top candidates by robust selection score")
display(candidate_df_11s_lite_robust.head(30))


print("=" * 100)
print("Selected 11S Lite candidates")
print("Raw best key     :", best_raw["key"])
print("Raw best method  :", best_raw["method"])
print("Raw best RMSE    :", best_raw["rmse"])
print("Raw delta vs 11O :", best_raw["rmse"] - BEST_11O_OOF)

print("-" * 100)

print("Robust best key     :", best_robust["key"])
print("Robust best method  :", best_robust["method"])
print("Robust best RMSE    :", best_robust["rmse"])
print("Robust selection    :", best_robust["selection_score"])
print("Robust delta vs 11O :", best_robust["rmse"] - BEST_11O_OOF)


raw_report_11s_lite = target_bin_report_11s_lite(
    best_raw["oof"],
    f"11S Lite raw best: {best_raw['key']}"
)

robust_report_11s_lite = target_bin_report_11s_lite(
    best_robust["oof"],
    f"11S Lite robust best: {best_robust['key']}"
)


# --------------------------------------------------
# Bin comparison
# --------------------------------------------------

print("=" * 100)
print("Bin comparison: 11O base vs 11S raw vs 11S robust")

compare_11s_lite = train_raw[[ID_COL, TARGET]].copy()
compare_11s_lite["base_pred"] = base_oof_11s
compare_11s_lite["raw_pred"] = best_raw["oof"]
compare_11s_lite["robust_pred"] = best_robust["oof"]

compare_11s_lite["base_error"] = compare_11s_lite["base_pred"] - compare_11s_lite[TARGET]
compare_11s_lite["raw_error"] = compare_11s_lite["raw_pred"] - compare_11s_lite[TARGET]
compare_11s_lite["robust_error"] = compare_11s_lite["robust_pred"] - compare_11s_lite[TARGET]

compare_11s_lite["base_se"] = compare_11s_lite["base_error"] ** 2
compare_11s_lite["raw_se"] = compare_11s_lite["raw_error"] ** 2
compare_11s_lite["robust_se"] = compare_11s_lite["robust_error"] ** 2

compare_11s_lite["target_bin_5"] = pd.qcut(
    compare_11s_lite[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11s_lite = (
    compare_11s_lite.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        raw_pred_mean=("raw_pred", "mean"),
        robust_pred_mean=("robust_pred", "mean"),

        base_bias=("base_error", "mean"),
        raw_bias=("raw_error", "mean"),
        robust_bias=("robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        raw_rmse=("raw_se", lambda x: np.sqrt(np.mean(x))),
        robust_rmse=("robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11s_lite["raw_rmse_delta"] = (
    bin_compare_11s_lite["raw_rmse"] - bin_compare_11s_lite["base_rmse"]
)

bin_compare_11s_lite["robust_rmse_delta"] = (
    bin_compare_11s_lite["robust_rmse"] - bin_compare_11s_lite["base_rmse"]
)

display(bin_compare_11s_lite.sort_values("base_rmse", ascending=False))


# --------------------------------------------------
# Save
# --------------------------------------------------

print("=" * 100)
print("Save 11S Lite submissions")

sub_11s_lite_raw = save_submission_11s_lite(
    best_raw["test"],
    "submission_11s_lite_raw_best.csv"
)

sub_11s_lite_robust = save_submission_11s_lite(
    best_robust["test"],
    "submission_11s_lite_robust_best.csv"
)

sub_11o_ref_from_11s_lite = save_submission_11s_lite(
    base_test_11s,
    "submission_11o_reference_from_11s_lite.csv"
)


# --------------------------------------------------
# Decision guide
# --------------------------------------------------

print("=" * 100)
print("11S Lite Decision Guide")

print(f"11O adaptive OOF        : {base_score_11s:.6f}")
print(f"11S Lite raw best OOF   : {best_raw['rmse']:.6f}")
print(f"11S Lite robust best OOF: {best_robust['rmse']:.6f}")
print(f"Raw delta vs 11O        : {best_raw['rmse'] - BEST_11O_OOF:.6f}")
print(f"Robust delta vs 11O     : {best_robust['rmse'] - BEST_11O_OOF:.6f}")

if best_raw["rmse"] < BEST_11O_OOF - 0.003:
    print("Raw best: STRONG candidate.")
elif best_raw["rmse"] < BEST_11O_OOF:
    print("Raw best: WEAK/MEDIUM candidate. Submit only if bin report is safe.")
else:
    print("Raw best: DO NOT FORCE.")

if best_robust["rmse"] < BEST_11O_OOF - 0.003:
    print("Robust best: STRONG candidate.")
elif best_robust["rmse"] < BEST_11O_OOF:
    print("Robust best: WEAK/MEDIUM candidate. Usually safer than raw if bins are stable.")
else:
    print("Robust best: DO NOT FORCE.")

print("=" * 100)
print("Please screenshot:")
print("1. part_a_df.head(20)")
print("2. part_b_df.head(20)")
print("3. candidate_df_11s_lite.head(30)")
print("4. candidate_df_11s_lite_robust.head(30)")
print("5. raw_report_11s_lite")
print("6. robust_report_11s_lite")
print("7. bin_compare_11s_lite")

In [ ]:
# ==================================================
# SECTION 11T - Tail Error Mining + Feature Discovery
# ==================================================
# Goal:
# 1. Analyze why very_high / high samples are still underpredicted.
# 2. Mine numeric and categorical features that separate missed-tail samples.
# 3. Create new 11T tail-risk features.
# 4. Retrain lightweight LGBM models with these new features.
# 5. Apply selective correction/blend against 11O adaptive.
#
# Key principle:
# Stop forcing correction blindly.
# Find signal that explains missed very_high.
#
# Paste after SECTION 11S Lite, or after SECTION 11O + 11R.
# Skip 11P and 11Q.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    print("LightGBM import issue:", e)


# ==================================================
# PART 0 - Setup
# ==================================================

print("=" * 100)
print("SECTION 11T - Tail Error Mining + Feature Discovery")

BEST_11O_OOF = 11.26696803195285
SEED_11T = globals().get("RANDOM_STATE", 42)
N_EST_11T = min(globals().get("MAIN_N_ESTIMATORS", 3000), 2500)
ES_11T = globals().get("EARLY_STOPPING_ROUNDS", 200)

required_vars_11t = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "X_fe",
    "X_test_fe",
    "cat_features",
    "num_features",
    "prepare_encoded_fold",
    "adaptive_oof",
    "final_adaptive_test",
]

missing_11t = [v for v in required_vars_11t if v not in globals()]
if len(missing_11t) > 0:
    raise NameError(f"Missing required variables for 11T: {missing_11t}")

base_oof_11t = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11t = np.clip(np.asarray(final_adaptive_test), 0, None)

base_score_11t = rmse(y, base_oof_11t)

print("Base 11O adaptive OOF:", base_score_11t)
print("Target to beat:", BEST_11O_OOF)

base_residual_11t = y.values - base_oof_11t
abs_residual_11t = np.abs(base_residual_11t)
positive_residual_11t = np.maximum(base_residual_11t, 0)

target_q80_11t = np.quantile(y.values, 0.80)
target_q85_11t = np.quantile(y.values, 0.85)
target_q90_11t = np.quantile(y.values, 0.90)

base_q75_11t = np.quantile(base_oof_11t, 0.75)
base_q80_11t = np.quantile(base_oof_11t, 0.80)
base_q85_11t = np.quantile(base_oof_11t, 0.85)
base_q90_11t = np.quantile(base_oof_11t, 0.90)


def rank_against_train_11t(train_values, values):
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)
    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


base_rank_oof_11t = rank_against_train_11t(base_oof_11t, base_oof_11t)
base_rank_test_11t = rank_against_train_11t(base_oof_11t, base_test_11t)

# Missed-tail definitions
missed_q85_res5_11t = (
    (y.values >= target_q85_11t) &
    (base_residual_11t >= 5)
)

missed_q90_res5_11t = (
    (y.values >= target_q90_11t) &
    (base_residual_11t >= 5)
)

good_tail_q85_11t = (
    (y.values >= target_q85_11t) &
    (abs_residual_11t <= 3)
)

tail_pool_q80_11t = y.values >= target_q80_11t

print("=" * 100)
print("Tail labels")
print("target q80:", target_q80_11t)
print("target q85:", target_q85_11t)
print("target q90:", target_q90_11t)
print("missed_q85_res5 count:", int(missed_q85_res5_11t.sum()))
print("missed_q90_res5 count:", int(missed_q90_res5_11t.sum()))
print("good_tail_q85 count:", int(good_tail_q85_11t.sum()))


# ==================================================
# Helpers
# ==================================================

def save_submission_11t(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11t(pred, label):
    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = np.clip(np.asarray(pred), 0, None)
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", rmse(y, np.clip(np.asarray(pred), 0, None)))
    display(report.sort_values("rmse", ascending=False))

    return report


def calc_candidate_metrics_11t(pred, correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = pd.DataFrame({
        "y": y.values,
        "pred": pred,
    })

    temp["err"] = temp["pred"] - temp["y"]
    temp["se"] = temp["err"] ** 2

    temp["bin"] = pd.qcut(
        temp["y"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    bin_rmse = temp.groupby("bin", observed=False)["se"].apply(lambda x: np.sqrt(np.mean(x)))
    bin_bias = temp.groupby("bin", observed=False)["err"].mean()

    if correction is None:
        correction = pred - base_oof_11t

    correction = np.asarray(correction)

    return {
        "rmse": rmse(y, pred),
        "very_high_rmse": float(bin_rmse.get("very_high", np.nan)),
        "high_rmse": float(bin_rmse.get("high", np.nan)),
        "medium_rmse": float(bin_rmse.get("medium", np.nan)),
        "low_rmse": float(bin_rmse.get("low", np.nan)),
        "very_low_rmse": float(bin_rmse.get("very_low", np.nan)),
        "very_high_bias": float(bin_bias.get("very_high", np.nan)),
        "high_bias": float(bin_bias.get("high", np.nan)),
        "medium_bias": float(bin_bias.get("medium", np.nan)),
        "corrected_count": int((correction > 1e-12).sum()),
        "corrected_rate": float(np.mean(correction > 1e-12)),
        "mean_correction": float(np.mean(correction)),
        "max_correction": float(np.max(correction)),
    }


def calc_benefit_metrics_11t(pred):
    pred = np.clip(np.asarray(pred), 0, None)

    base_se = (base_oof_11t - y.values) ** 2
    new_se = (pred - y.values) ** 2

    benefit = base_se - new_se
    changed = np.abs(pred - base_oof_11t) > 1e-12

    if changed.sum() == 0:
        return 0.0, 0.0, 0

    return (
        float(np.mean(benefit[changed] > 0)),
        float(np.mean(benefit[changed])),
        int(changed.sum()),
    )


base_report_11t = target_bin_report_11t(
    base_oof_11t,
    "11T base: 11O adaptive"
)


# ==================================================
# PART 1 - Numeric Tail Error Mining
# ==================================================

print("=" * 100)
print("PART 1 - Numeric Tail Error Mining")

numeric_cols_11t = [
    c for c in num_features
    if c in X_fe.columns and pd.api.types.is_numeric_dtype(X_fe[c])
]

print("Numeric columns checked:", len(numeric_cols_11t))

numeric_rows_11t = []

miss_mask = missed_q85_res5_11t
good_mask = good_tail_q85_11t

for col in numeric_cols_11t:
    s = pd.to_numeric(X_fe[col], errors="coerce")

    miss_vals = s[miss_mask].dropna().values
    good_vals = s[good_mask].dropna().values

    if len(miss_vals) < 20 or len(good_vals) < 20:
        continue

    miss_mean = float(np.mean(miss_vals))
    good_mean = float(np.mean(good_vals))
    miss_median = float(np.median(miss_vals))
    good_median = float(np.median(good_vals))

    miss_std = float(np.std(miss_vals))
    good_std = float(np.std(good_vals))
    pooled_std = np.sqrt((miss_std ** 2 + good_std ** 2) / 2.0)

    effect = 0.0 if pooled_std == 0 else (miss_mean - good_mean) / pooled_std

    missing_rate_miss = float(s[miss_mask].isna().mean())
    missing_rate_good = float(s[good_mask].isna().mean())
    missing_diff = missing_rate_miss - missing_rate_good

    s_fill = s.fillna(s.median())

    valid = s_fill.notna().values
    if valid.sum() < 100:
        continue

    try:
        q10 = np.quantile(s_fill[valid], 0.10)
        q90 = np.quantile(s_fill[valid], 0.90)

        high_value_mask = s_fill.values >= q90
        low_value_mask = s_fill.values <= q10

        miss_rate_high_value = float(miss_mask[high_value_mask].mean()) if high_value_mask.sum() > 0 else 0
        miss_rate_low_value = float(miss_mask[low_value_mask].mean()) if low_value_mask.sum() > 0 else 0

        residual_tail_corr = np.corrcoef(
            s_fill.values[tail_pool_q80_11t],
            positive_residual_11t[tail_pool_q80_11t]
        )[0, 1]

        if not np.isfinite(residual_tail_corr):
            residual_tail_corr = 0.0

    except Exception:
        miss_rate_high_value = 0
        miss_rate_low_value = 0
        residual_tail_corr = 0

    score = (
        abs(effect)
        + 3.0 * abs(missing_diff)
        + 2.0 * abs(miss_rate_high_value - miss_rate_low_value)
        + 0.5 * abs(residual_tail_corr)
    )

    numeric_rows_11t.append({
        "feature": col,
        "miss_count": len(miss_vals),
        "good_count": len(good_vals),
        "miss_mean": miss_mean,
        "good_mean": good_mean,
        "miss_median": miss_median,
        "good_median": good_median,
        "effect_size_miss_vs_good": effect,
        "missing_rate_miss": missing_rate_miss,
        "missing_rate_good": missing_rate_good,
        "missing_diff": missing_diff,
        "miss_rate_high_value": miss_rate_high_value,
        "miss_rate_low_value": miss_rate_low_value,
        "residual_tail_corr": residual_tail_corr,
        "score": score,
    })

numeric_mining_11t = pd.DataFrame(numeric_rows_11t)

if len(numeric_mining_11t) > 0:
    numeric_mining_11t = numeric_mining_11t.sort_values("score", ascending=False)
    print("Top numeric signals for missed-tail:")
    display(numeric_mining_11t.head(40))
else:
    print("No numeric mining rows produced.")


# ==================================================
# PART 2 - Categorical Tail Error Mining
# ==================================================

print("=" * 100)
print("PART 2 - Categorical Tail Error Mining")

cat_cols_11t = [c for c in cat_features if c in X_fe.columns]
cat_rows_11t = []

global_miss_rate_11t = float(missed_q85_res5_11t.mean())
global_resid_mean_11t = float(base_residual_11t.mean())

for col in cat_cols_11t:
    s = X_fe[col].astype(str).fillna("Missing")

    temp = pd.DataFrame({
        "cat": s.values,
        "miss": missed_q85_res5_11t.astype(int),
        "residual": base_residual_11t,
        "target": y.values,
        "base_pred": base_oof_11t,
    })

    g = (
        temp.groupby("cat")
        .agg(
            count=("miss", "count"),
            miss_count=("miss", "sum"),
            miss_rate=("miss", "mean"),
            residual_mean=("residual", "mean"),
            target_mean=("target", "mean"),
            base_pred_mean=("base_pred", "mean"),
        )
        .reset_index()
    )

    g = g[g["count"] >= 30].copy()

    if len(g) == 0:
        continue

    g["feature"] = col
    g["category"] = g["cat"].astype(str)
    g["miss_lift"] = g["miss_rate"] / max(global_miss_rate_11t, 1e-9)
    g["residual_lift"] = g["residual_mean"] - global_resid_mean_11t
    g["tail_group_score"] = (
        np.log1p(g["count"])
        * g["miss_lift"]
        * np.maximum(g["residual_mean"], 0)
    )

    cat_rows_11t.append(
        g[[
            "feature",
            "category",
            "count",
            "miss_count",
            "miss_rate",
            "miss_lift",
            "residual_mean",
            "residual_lift",
            "target_mean",
            "base_pred_mean",
            "tail_group_score",
        ]]
    )

if len(cat_rows_11t) > 0:
    categorical_mining_11t = pd.concat(cat_rows_11t, axis=0).sort_values(
        "tail_group_score",
        ascending=False
    )
    print("Top categorical groups for missed-tail:")
    display(categorical_mining_11t.head(50))
else:
    categorical_mining_11t = pd.DataFrame()
    print("No categorical mining rows produced.")


# ==================================================
# PART 3 - Build 11T Tail Features
# ==================================================

print("=" * 100)
print("PART 3 - Build 11T Tail Features")

selected_num_features_11t = []

if len(numeric_mining_11t) > 0:
    selected_num_features_11t = (
        numeric_mining_11t
        .sort_values("score", ascending=False)
        ["feature"]
        .drop_duplicates()
        .head(12)
        .tolist()
    )

selected_cat_groups_11t = []

if len(categorical_mining_11t) > 0:
    selected_cat_groups_11t = (
        categorical_mining_11t
        .sort_values("tail_group_score", ascending=False)
        .head(15)
        [["feature", "category"]]
        .to_dict("records")
    )

print("Selected numeric features:", selected_num_features_11t)
print("Selected categorical groups:", selected_cat_groups_11t[:10])

# Train statistics for feature creation
num_stats_11t = {}

for col in selected_num_features_11t:
    s = pd.to_numeric(X_fe[col], errors="coerce")
    med = float(s.median())
    q10 = float(s.quantile(0.10))
    q90 = float(s.quantile(0.90))

    num_stats_11t[col] = {
        "median": med,
        "q10": q10,
        "q90": q90,
        "values_sorted": np.sort(s.fillna(med).values),
    }


def safe_name_11t(x):
    x = str(x)
    keep = []
    for ch in x:
        if ch.isalnum():
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)[:60]


def add_11t_features(X, is_train=True):
    X_new = X.copy()

    n = X_new.shape[0]

    if is_train:
        base_pred = base_oof_11t
        base_rank = base_rank_oof_11t
    else:
        base_pred = base_test_11t
        base_rank = base_rank_test_11t

    # Prediction meta features
    X_new["11t_base_pred"] = base_pred
    X_new["11t_base_rank"] = base_rank
    X_new["11t_base_pred_log1p"] = np.log1p(base_pred)
    X_new["11t_base_pred_sq"] = base_pred ** 2

    # Base prediction tail flags
    X_new["11t_base_ge_q75"] = (base_pred >= base_q75_11t).astype(int)
    X_new["11t_base_ge_q80"] = (base_pred >= base_q80_11t).astype(int)
    X_new["11t_base_ge_q85"] = (base_pred >= base_q85_11t).astype(int)
    X_new["11t_base_ge_q90"] = (base_pred >= base_q90_11t).astype(int)

    # Model disagreement features if model_outputs exist
    meta_preds = []

    if "model_outputs" in globals():
        for m in ["catboost", "lightgbm", "xgboost"]:
            if m in model_outputs:
                if is_train and "oof" in model_outputs[m]:
                    meta_preds.append(np.asarray(model_outputs[m]["oof"]).reshape(-1, 1))
                elif (not is_train) and "test_pred" in model_outputs[m]:
                    meta_preds.append(np.asarray(model_outputs[m]["test_pred"]).reshape(-1, 1))

    optional_pairs = [
        ("global_oof", "global_test"),
        ("s6_oof", "s6_test"),
        ("ridge3_oof", "ridge3_test"),
    ]

    for oof_name, test_name in optional_pairs:
        if oof_name in globals() and test_name in globals():
            if is_train:
                meta_preds.append(np.asarray(globals()[oof_name]).reshape(-1, 1))
            else:
                meta_preds.append(np.asarray(globals()[test_name]).reshape(-1, 1))

    if len(meta_preds) > 0:
        M = np.hstack(meta_preds)
        X_new["11t_meta_mean"] = M.mean(axis=1)
        X_new["11t_meta_std"] = M.std(axis=1)
        X_new["11t_meta_min"] = M.min(axis=1)
        X_new["11t_meta_max"] = M.max(axis=1)
        X_new["11t_meta_range"] = M.max(axis=1) - M.min(axis=1)
        X_new["11t_base_minus_meta_mean"] = base_pred - M.mean(axis=1)
        X_new["11t_meta_max_minus_base"] = M.max(axis=1) - base_pred
    else:
        X_new["11t_meta_mean"] = base_pred
        X_new["11t_meta_std"] = 0
        X_new["11t_meta_min"] = base_pred
        X_new["11t_meta_max"] = base_pred
        X_new["11t_meta_range"] = 0
        X_new["11t_base_minus_meta_mean"] = 0
        X_new["11t_meta_max_minus_base"] = 0

    # Numeric mined features
    for col in selected_num_features_11t:
        if col not in X_new.columns:
            continue

        stats = num_stats_11t[col]
        med = stats["median"]
        q10 = stats["q10"]
        q90 = stats["q90"]
        sorted_train = stats["values_sorted"]

        s = pd.to_numeric(X_new[col], errors="coerce")
        s_fill = s.fillna(med).values

        fname = safe_name_11t(col)

        X_new[f"11t_{fname}_is_missing"] = s.isna().astype(int)
        X_new[f"11t_{fname}_rank"] = np.searchsorted(sorted_train, s_fill, side="right") / len(sorted_train)
        X_new[f"11t_{fname}_low_q10"] = (s_fill <= q10).astype(int)
        X_new[f"11t_{fname}_high_q90"] = (s_fill >= q90).astype(int)
        X_new[f"11t_{fname}_x_base_rank"] = X_new[f"11t_{fname}_rank"] * base_rank
        X_new[f"11t_{fname}_high_x_base_tail"] = (
            X_new[f"11t_{fname}_high_q90"] * X_new["11t_base_ge_q85"]
        )

    # Ratio and difference features from top numeric signals
    pair_nums = selected_num_features_11t[:6]

    for i in range(len(pair_nums)):
        for j in range(i + 1, len(pair_nums)):
            a = pair_nums[i]
            b = pair_nums[j]

            if a not in X_new.columns or b not in X_new.columns:
                continue

            a_stats = num_stats_11t[a]
            b_stats = num_stats_11t[b]

            av = pd.to_numeric(X_new[a], errors="coerce").fillna(a_stats["median"]).values
            bv = pd.to_numeric(X_new[b], errors="coerce").fillna(b_stats["median"]).values

            an = safe_name_11t(a)
            bn = safe_name_11t(b)

            ratio = av / (np.abs(bv) + 1e-6)
            ratio = np.clip(ratio, -50, 50)

            diff = av - bv

            X_new[f"11t_ratio_{an}_over_{bn}"] = ratio
            X_new[f"11t_diff_{an}_minus_{bn}"] = diff

    # Categorical suspicious group flags
    for idx, item in enumerate(selected_cat_groups_11t):
        col = item["feature"]
        cat = str(item["category"])

        if col not in X_new.columns:
            continue

        X_new[f"11t_catflag_{idx}_{safe_name_11t(col)}"] = (
            X_new[col].astype(str).fillna("Missing") == cat
        ).astype(int)

    return X_new


X_fe_11t = add_11t_features(X_fe, is_train=True)
X_test_fe_11t = add_11t_features(X_test_fe, is_train=False)

new_11t_cols = [c for c in X_fe_11t.columns if c.startswith("11t_")]

num_features_11t = list(num_features) + [
    c for c in new_11t_cols
    if c not in num_features
]

cat_features_11t = list(cat_features)

print("Original X_fe shape:", X_fe.shape)
print("11T X_fe shape:", X_fe_11t.shape)
print("New 11T feature count:", len(new_11t_cols))
print("Example new features:", new_11t_cols[:30])


# ==================================================
# PART 4 - Train Lightweight 11T LGBM Models
# ==================================================

print("=" * 100)
print("PART 4 - Train Lightweight 11T LGBM Models")


def fit_lgbm_11t(model, X_tr, y_tr, X_val, y_val, sample_weight=None):
    try:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(ES_11T, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
    except TypeError:
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sample_weight,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            early_stopping_rounds=ES_11T,
            verbose=False,
        )

    return model


def make_sample_weights_11t(mode):
    w = np.ones(len(y), dtype=float)

    if mode == "none":
        return None

    if mode == "tail_soft":
        w += 0.35 * (y.values >= target_q85_11t)
        w += 0.65 * missed_q85_res5_11t
        w += 0.20 * (base_rank_oof_11t >= 0.85)

    elif mode == "missed_soft":
        w += 0.80 * missed_q85_res5_11t
        w += 0.40 * missed_q90_res5_11t
        w += 0.20 * (base_rank_oof_11t >= 0.85)

    else:
        raise ValueError("Unknown weight mode")

    return w


model_configs_11t = [
    {
        "name": "lgb11t_plain31",
        "num_leaves": 31,
        "min_child_samples": 60,
        "reg_alpha": 0.5,
        "reg_lambda": 8.0,
        "weight_mode": "none",
    },
    {
        "name": "lgb11t_tailsoft31",
        "num_leaves": 31,
        "min_child_samples": 70,
        "reg_alpha": 0.7,
        "reg_lambda": 10.0,
        "weight_mode": "tail_soft",
    },
    {
        "name": "lgb11t_missedsoft31",
        "num_leaves": 31,
        "min_child_samples": 80,
        "reg_alpha": 1.0,
        "reg_lambda": 12.0,
        "weight_mode": "missed_soft",
    },
    {
        "name": "lgb11t_conservative63",
        "num_leaves": 63,
        "min_child_samples": 120,
        "reg_alpha": 1.0,
        "reg_lambda": 15.0,
        "weight_mode": "none",
    },
]

lgb_11t_outputs = {}
lgb_11t_rows = []

for cfg in model_configs_11t:
    name = cfg["name"]

    print("=" * 100)
    print("Training:", name)

    sample_weights_all = make_sample_weights_11t(cfg["weight_mode"])

    oof = np.zeros(len(y))
    test_pred = np.zeros(sample_submission.shape[0])
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw = X_fe_11t.iloc[tr_idx]
        X_val_raw = X_fe_11t.iloc[val_idx]
        X_te_raw = X_test_fe_11t

        y_tr = y.iloc[tr_idx]
        y_val = y.iloc[val_idx]

        X_tr, X_val, X_te, feature_names = prepare_encoded_fold(
            X_tr_raw,
            X_val_raw,
            X_te_raw,
            cat_features_11t,
            num_features_11t,
        )

        if sample_weights_all is None:
            w_tr = None
        else:
            w_tr = sample_weights_all[tr_idx]

        mdl = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=N_EST_11T,
            learning_rate=0.025,
            num_leaves=cfg["num_leaves"],
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=cfg["reg_alpha"],
            reg_lambda=cfg["reg_lambda"],
            min_child_samples=cfg["min_child_samples"],
            random_state=SEED_11T + 7000 + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        mdl = fit_lgbm_11t(
            mdl,
            X_tr,
            y_tr,
            X_val,
            y_val,
            sample_weight=w_tr,
        )

        val_pred = np.clip(mdl.predict(X_val), 0, None)
        te_pred = np.clip(mdl.predict(X_te), 0, None)

        oof[val_idx] = val_pred
        test_pred += te_pred / len(folds)

        fold_rmse = rmse(y_val, val_pred)
        fold_scores.append(fold_rmse)

        print(f"  Fold {fold}: RMSE={fold_rmse:.5f}")

        del X_tr, X_val, X_te, mdl
        gc.collect()

    oof = np.clip(oof, 0, None)
    test_pred = np.clip(test_pred, 0, None)

    score = rmse(y, oof)

    lgb_11t_outputs[name] = {
        "oof": oof,
        "test": test_pred,
    }

    lgb_11t_rows.append({
        "model": name,
        "oof_rmse": score,
        "delta_vs_11O": score - BEST_11O_OOF,
        "fold_std": np.std(fold_scores),
        "weight_mode": cfg["weight_mode"],
    })

    print(f"{name} OOF RMSE:", score)
    target_bin_report_11t(oof, f"11T model: {name}")

lgb_11t_df = pd.DataFrame(lgb_11t_rows).sort_values("oof_rmse")

print("=" * 100)
print("11T LGBM model results")
display(lgb_11t_df)


# ==================================================
# PART 5 - Blend and Selective Correction Against 11O
# ==================================================

print("=" * 100)
print("PART 5 - Blend and Selective Correction Against 11O")

candidate_rows_11t = []
best_11t = {
    "key": "base_11O_adaptive",
    "method": "base",
    "score": base_score_11t,
    "oof": base_oof_11t.copy(),
    "test": base_test_11t.copy(),
    "correction": np.zeros_like(base_oof_11t),
}


def evaluate_candidate_11t(key, method, pred_oof, pred_test, correction):
    global best_11t

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)
    correction = np.asarray(correction)

    metrics = calc_candidate_metrics_11t(pred_oof, correction)
    benefit_rate, avg_benefit, changed_count = calc_benefit_metrics_11t(pred_oof)

    row = {
        "key": key,
        "method": method,
        **metrics,
        "delta_vs_11O": metrics["rmse"] - BEST_11O_OOF,
        "benefit_positive_rate": benefit_rate,
        "avg_benefit_changed": avg_benefit,
        "changed_count": changed_count,
    }

    candidate_rows_11t.append(row)

    if metrics["rmse"] < best_11t["score"]:
        best_11t = {
            "key": key,
            "method": method,
            "score": metrics["rmse"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction": correction.copy(),
        }


# Base candidate
evaluate_candidate_11t(
    "base_11O_adaptive",
    "base",
    base_oof_11t,
    base_test_11t,
    np.zeros_like(base_oof_11t),
)

# Use 11R / 11S risk if available
if "missed_tail_outputs_11r" in globals() and "q85_res5" in missed_tail_outputs_11r:
    gate_oof_ref = missed_tail_outputs_11r["q85_res5"]["gate_oof"]
    gate_test_ref = missed_tail_outputs_11r["q85_res5"]["gate_test"]
    resid_oof_ref = missed_tail_outputs_11r["q85_res5"]["resid_oof"]
    resid_test_ref = missed_tail_outputs_11r["q85_res5"]["resid_test"]

    risk_oof_ref = np.asarray(gate_oof_ref) * np.asarray(resid_oof_ref)
    risk_test_ref = np.asarray(gate_test_ref) * np.asarray(resid_test_ref)
else:
    risk_oof_ref = base_rank_oof_11t.copy()
    risk_test_ref = base_rank_test_11t.copy()

for model_name, obj in lgb_11t_outputs.items():
    model_oof = obj["oof"]
    model_test = obj["test"]

    # Individual model candidate
    evaluate_candidate_11t(
        key=f"{model_name}_raw",
        method="raw_model",
        pred_oof=model_oof,
        pred_test=model_test,
        correction=model_oof - base_oof_11t,
    )

    # Global conservative blend
    for w in [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30]:
        blend_oof = np.clip((1 - w) * base_oof_11t + w * model_oof, 0, None)
        blend_test = np.clip((1 - w) * base_test_11t + w * model_test, 0, None)

        evaluate_candidate_11t(
            key=f"{model_name}_global_blend_w{w}",
            method="global_blend",
            pred_oof=blend_oof,
            pred_test=blend_test,
            correction=blend_oof - base_oof_11t,
        )

    # Selective positive correction
    delta_oof = np.maximum(model_oof - base_oof_11t, 0)
    delta_test = np.maximum(model_test - base_test_11t, 0)

    score_oof = risk_oof_ref * delta_oof
    score_test = risk_test_ref * delta_test

    for min_base_q in [0.70, 0.75, 0.80, 0.85]:
        base_thr_oof = np.quantile(base_oof_11t, min_base_q)
        base_thr_test = np.quantile(base_test_11t, min_base_q)

        eligible_oof = base_oof_11t >= base_thr_oof
        eligible_test = base_test_11t >= base_thr_test

        if eligible_oof.sum() < 20:
            continue

        for top_frac in [0.005, 0.01, 0.02, 0.05, 0.08, 0.10]:
            score_eligible_oof = score_oof[eligible_oof]
            score_eligible_test = score_test[eligible_test]

            if len(score_eligible_oof) == 0 or np.max(score_eligible_oof) <= 0:
                continue

            if len(score_eligible_test) == 0 or np.max(score_eligible_test) <= 0:
                continue

            thr_oof = np.quantile(score_eligible_oof, 1.0 - top_frac)
            thr_test = np.quantile(score_eligible_test, 1.0 - top_frac)

            top_mask_oof = eligible_oof & (score_oof >= thr_oof) & (score_oof > 0)
            top_mask_test = eligible_test & (score_test >= thr_test) & (score_test > 0)

            for strength in [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25]:
                for cap in [0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00]:
                    corr_oof = np.clip(strength * delta_oof, 0, cap)
                    corr_test = np.clip(strength * delta_test, 0, cap)

                    corr_oof = np.where(top_mask_oof, corr_oof, 0.0)
                    corr_test = np.where(top_mask_test, corr_test, 0.0)

                    final_oof = np.clip(base_oof_11t + corr_oof, 0, None)
                    final_test = np.clip(base_test_11t + corr_test, 0, None)

                    key = (
                        f"{model_name}_selective"
                        f"_minq{min_base_q}"
                        f"_top{top_frac}"
                        f"_s{strength}"
                        f"_cap{cap}"
                    )

                    evaluate_candidate_11t(
                        key=key,
                        method="selective_positive_correction",
                        pred_oof=final_oof,
                        pred_test=final_test,
                        correction=corr_oof,
                    )

candidate_df_11t = pd.DataFrame(candidate_rows_11t).sort_values("rmse")

print("=" * 100)
print("Top 11T candidates")
display(candidate_df_11t.head(40))

print("=" * 100)
print("Best 11T candidate")
print("Key:", best_11t["key"])
print("Method:", best_11t["method"])
print("RMSE:", best_11t["score"])
print("Delta vs 11O:", best_11t["score"] - BEST_11O_OOF)


# ==================================================
# PART 6 - Reports and Save
# ==================================================

best_11t_report = target_bin_report_11t(
    best_11t["oof"],
    f"Best 11T final: {best_11t['key']}"
)

print("=" * 100)
print("Bin comparison: 11O base vs best 11T")

compare_11t = train_raw[[ID_COL, TARGET]].copy()
compare_11t["base_pred"] = base_oof_11t
compare_11t["best_pred"] = best_11t["oof"]

compare_11t["base_error"] = compare_11t["base_pred"] - compare_11t[TARGET]
compare_11t["best_error"] = compare_11t["best_pred"] - compare_11t[TARGET]

compare_11t["base_se"] = compare_11t["base_error"] ** 2
compare_11t["best_se"] = compare_11t["best_error"] ** 2

compare_11t["target_bin_5"] = pd.qcut(
    compare_11t[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11t = (
    compare_11t.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        best_pred_mean=("best_pred", "mean"),

        base_bias=("base_error", "mean"),
        best_bias=("best_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        best_rmse=("best_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11t["rmse_delta_best_minus_base"] = (
    bin_compare_11t["best_rmse"] - bin_compare_11t["base_rmse"]
)

bin_compare_11t["bias_abs_delta_best_minus_base"] = (
    bin_compare_11t["best_bias"].abs() - bin_compare_11t["base_bias"].abs()
)

display(bin_compare_11t.sort_values("base_rmse", ascending=False))

print("=" * 100)
print("Save submissions")

sub_11t_best = save_submission_11t(
    best_11t["test"],
    "submission_11t_best.csv"
)

sub_11o_ref_11t = save_submission_11t(
    base_test_11t,
    "submission_11o_reference_from_11t.csv"
)


# ==================================================
# PART 7 - Decision Guide
# ==================================================

print("=" * 100)
print("11T Decision Guide")

print(f"11O adaptive OOF : {base_score_11t:.6f}")
print(f"Best 11T OOF     : {best_11t['score']:.6f}")
print(f"Delta vs 11O     : {best_11t['score'] - BEST_11O_OOF:.6f}")

if best_11t["score"] < BEST_11O_OOF - 0.01:
    print("Decision: STRONG. This finally found meaningful new signal.")
elif best_11t["score"] < BEST_11O_OOF - 0.003:
    print("Decision: GOOD. Submit if bin report is safe.")
elif best_11t["score"] < BEST_11O_OOF:
    print("Decision: WEAK/MEDIUM. Submit only as probe if slots are available.")
else:
    print("Decision: DO NOT FORCE. Keep 11O / 11S Lite as primary.")

print("=" * 100)
print("Please screenshot:")
print("1. numeric_mining_11t.head(40)")
print("2. categorical_mining_11t.head(50)")
print("3. lgb_11t_df")
print("4. candidate_df_11t.head(40)")
print("5. best_11t_report")
print("6. bin_compare_11t")

In [ ]:
# ==================================================
# SECTION 11U - Correction Optimizer After 11T
# ==================================================
# Goal:
# A. Fine-grid around best 11T selective correction
# B. Blend 11T correction with 11S-like stable correction
# C. Risk ensemble q85/q90 + guardrail filter
# D. Robust final selection
#
# Paste AFTER SECTION 11T.
# Skip 11P and 11Q.
#
# Main idea:
# 11T raw LGBM models are bad as standalone predictors,
# but their positive delta can be useful as selective correction.
#
# 11U = optimize correction dose, risk score, guardrail, and blend.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")


# ==================================================
# PART 0 - Setup
# ==================================================

print("=" * 100)
print("SECTION 11U - Correction Optimizer After 11T")

BEST_11O_OOF = 11.26696803195285

required_vars_11u = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "lgb_11t_outputs",
    "missed_tail_outputs_11r",
    "correction_df_11r",
    "correction_outputs_11r",
]

missing_11u = [v for v in required_vars_11u if v not in globals()]
if len(missing_11u) > 0:
    raise NameError(f"Missing required variables for 11U: {missing_11u}")

base_oof_11u = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11u = np.clip(np.asarray(final_adaptive_test), 0, None)

y_arr_11u = y.values
n_train_11u = len(y_arr_11u)
n_test_11u = sample_submission.shape[0]

base_score_11u = rmse(y, base_oof_11u)

print("Base 11O adaptive OOF:", base_score_11u)
print("Target to beat:", BEST_11O_OOF)

if "best_11t" in globals():
    print("Previous best 11T OOF:", best_11t.get("score", None))
else:
    print("Warning: best_11t not found. 11U still works using lgb_11t_outputs.")


# ==================================================
# Precompute target bins for fast metrics
# ==================================================

target_bin_11u = pd.qcut(
    y_arr_11u,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels_11u = ["very_low", "low", "medium", "high", "very_high"]
bin_masks_11u = {
    b: (target_bin_11u == b)
    for b in bin_labels_11u
}


def fast_rmse_11u(y_true, pred):
    pred = np.asarray(pred)
    return float(np.sqrt(np.mean((pred - y_true) ** 2)))


def rank_against_train_11u(train_values, values):
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)
    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def save_submission_11u(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11u(pred, label):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = pred
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", fast_rmse_11u(y_arr_11u, pred))
    display(report.sort_values("rmse", ascending=False))

    return report


def calc_metrics_11u(pred, correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    err = pred - y_arr_11u
    se = err ** 2

    if correction is None:
        correction = pred - base_oof_11u

    correction = np.asarray(correction)

    out = {
        "rmse": fast_rmse_11u(y_arr_11u, pred),
    }

    for b in bin_labels_11u:
        m = bin_masks_11u[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    changed = np.abs(pred - base_oof_11u) > 1e-12

    base_se = (base_oof_11u - y_arr_11u) ** 2
    new_se = (pred - y_arr_11u) ** 2
    benefit = base_se - new_se

    if changed.sum() > 0:
        benefit_positive_rate = float(np.mean(benefit[changed] > 0))
        avg_benefit_changed = float(np.mean(benefit[changed]))
    else:
        benefit_positive_rate = 0.0
        avg_benefit_changed = 0.0

    out.update({
        "corrected_count": int((correction > 1e-12).sum()),
        "corrected_rate": float(np.mean(correction > 1e-12)),
        "mean_correction": float(np.mean(correction)),
        "max_correction": float(np.max(correction)),
        "changed_count": int(changed.sum()),
        "benefit_positive_rate": benefit_positive_rate,
        "avg_benefit_changed": avg_benefit_changed,
        "delta_vs_11O": fast_rmse_11u(y_arr_11u, pred) - BEST_11O_OOF,
    })

    return out


base_metrics_11u = calc_metrics_11u(
    base_oof_11u,
    correction=np.zeros_like(base_oof_11u),
)

base_vh_rmse_11u = base_metrics_11u["very_high_rmse"]
base_high_rmse_11u = base_metrics_11u["high_rmse"]
base_medium_rmse_11u = base_metrics_11u["medium_rmse"]
base_low_rmse_11u = base_metrics_11u["low_rmse"]
base_vlow_rmse_11u = base_metrics_11u["very_low_rmse"]


def robust_penalty_11u(metrics):
    penalty = 0.0

    cc = metrics["corrected_count"]

    # Ideal correction count based on 11T/11S behavior:
    # not too tiny, not too broad.
    if cc > 0 and cc < 50:
        penalty += 0.00040
    if cc > 350:
        penalty += 0.00030
    if cc > 600:
        penalty += 0.00060

    # Very-high should not worsen.
    if metrics["very_high_rmse"] > base_vh_rmse_11u:
        penalty += 0.00060

    # Guard leakage to other bins.
    if metrics["high_rmse"] > base_high_rmse_11u + 0.055:
        penalty += 0.00040

    if metrics["medium_rmse"] > base_medium_rmse_11u + 0.060:
        penalty += 0.00050

    if metrics["low_rmse"] > base_low_rmse_11u + 0.035:
        penalty += 0.00030

    if metrics["very_low_rmse"] > base_vlow_rmse_11u + 0.035:
        penalty += 0.00030

    # Correction quality.
    if cc > 0:
        if metrics["benefit_positive_rate"] < 0.47:
            penalty += 0.00030
        if metrics["avg_benefit_changed"] < 0:
            penalty += 0.00080

    return penalty


candidate_rows_11u = []

best_raw_11u = {
    "key": "base_11O_adaptive",
    "method": "base",
    "rmse": base_score_11u,
    "selection_score": base_score_11u,
    "oof": base_oof_11u.copy(),
    "test": base_test_11u.copy(),
    "correction_oof": np.zeros_like(base_oof_11u),
    "correction_test": np.zeros_like(base_test_11u),
}

best_robust_11u = best_raw_11u.copy()

best_part_a_11u = {
    "key": None,
    "rmse": np.inf,
    "oof": None,
    "test": None,
    "correction_oof": None,
    "correction_test": None,
}


def evaluate_candidate_11u(
    key,
    method,
    pred_oof,
    pred_test,
    correction_oof,
    correction_test=None,
):
    global best_raw_11u, best_robust_11u, best_part_a_11u

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)
    correction_oof = np.clip(np.asarray(correction_oof), 0, None)

    if correction_test is None:
        correction_test = np.clip(pred_test - base_test_11u, 0, None)
    else:
        correction_test = np.clip(np.asarray(correction_test), 0, None)

    metrics = calc_metrics_11u(pred_oof, correction_oof)
    metrics["robust_penalty"] = robust_penalty_11u(metrics)
    metrics["selection_score"] = metrics["rmse"] + metrics["robust_penalty"]

    row = {
        "key": key,
        "method": method,
        **metrics,
    }

    candidate_rows_11u.append(row)

    if metrics["rmse"] < best_raw_11u["rmse"]:
        best_raw_11u = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }

    if metrics["selection_score"] < best_robust_11u["selection_score"]:
        best_robust_11u = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }

    if method == "A_fine_grid" and metrics["rmse"] < best_part_a_11u["rmse"]:
        best_part_a_11u = {
            "key": key,
            "rmse": metrics["rmse"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }


# Add base
evaluate_candidate_11u(
    key="base_11O_adaptive",
    method="base",
    pred_oof=base_oof_11u,
    pred_test=base_test_11u,
    correction_oof=np.zeros_like(base_oof_11u),
    correction_test=np.zeros_like(base_test_11u),
)


# Add previous best 11T if available
if "best_11t" in globals():
    corr_oof_prev_11t = np.clip(np.asarray(best_11t["oof"]) - base_oof_11u, 0, None)
    corr_test_prev_11t = np.clip(np.asarray(best_11t["test"]) - base_test_11u, 0, None)

    evaluate_candidate_11u(
        key="prev_best_11T",
        method="prev_11T",
        pred_oof=np.asarray(best_11t["oof"]),
        pred_test=np.asarray(best_11t["test"]),
        correction_oof=corr_oof_prev_11t,
        correction_test=corr_test_prev_11t,
    )


print("=" * 100)
print("Base bin report")
base_report_11u = target_bin_report_11u(
    base_oof_11u,
    "11U base: 11O adaptive",
)


# ==================================================
# PART 1 - Build Risk Scores
# ==================================================

print("=" * 100)
print("PART 1 - Build Risk Scores")


def get_risk_from_11r_label_11u(label_name):
    if label_name not in missed_tail_outputs_11r:
        return None

    obj = missed_tail_outputs_11r[label_name]

    gate_oof = np.clip(np.asarray(obj["gate_oof"]), 0, None)
    gate_test = np.clip(np.asarray(obj["gate_test"]), 0, None)

    resid_oof = np.clip(np.asarray(obj["resid_oof"]), 0, None)
    resid_test = np.clip(np.asarray(obj["resid_test"]), 0, None)

    risk_oof_raw = gate_oof * resid_oof
    risk_test_raw = gate_test * resid_test

    if np.max(risk_oof_raw) <= 0:
        risk_oof_rank = np.zeros_like(risk_oof_raw)
        risk_test_rank = np.zeros_like(risk_test_raw)
    else:
        risk_oof_rank = rank_against_train_11u(risk_oof_raw, risk_oof_raw)
        risk_test_rank = rank_against_train_11u(risk_oof_raw, risk_test_raw)

    return {
        "oof": risk_oof_rank,
        "test": risk_test_rank,
        "raw_oof": risk_oof_raw,
        "raw_test": risk_test_raw,
    }


risk_bank_11u = {}

for label in ["q85_res5", "q85_res8", "q90_res5", "q90_res8"]:
    risk_obj = get_risk_from_11r_label_11u(label)
    if risk_obj is not None:
        risk_bank_11u[label] = risk_obj

print("Available risk labels:", list(risk_bank_11u.keys()))

risk_variants_11u = {}

if "q85_res5" in risk_bank_11u:
    risk_variants_11u["risk_q85"] = (
        risk_bank_11u["q85_res5"]["oof"],
        risk_bank_11u["q85_res5"]["test"],
    )

if "q90_res5" in risk_bank_11u:
    risk_variants_11u["risk_q90"] = (
        risk_bank_11u["q90_res5"]["oof"],
        risk_bank_11u["q90_res5"]["test"],
    )

if "q85_res8" in risk_bank_11u:
    risk_variants_11u["risk_q85res8"] = (
        risk_bank_11u["q85_res8"]["oof"],
        risk_bank_11u["q85_res8"]["test"],
    )

if "q85_res5" in risk_bank_11u and "q90_res5" in risk_bank_11u:
    r85_oof = risk_bank_11u["q85_res5"]["oof"]
    r85_test = risk_bank_11u["q85_res5"]["test"]
    r90_oof = risk_bank_11u["q90_res5"]["oof"]
    r90_test = risk_bank_11u["q90_res5"]["test"]

    risk_variants_11u["risk_50q85_50q90"] = (
        0.50 * r85_oof + 0.50 * r90_oof,
        0.50 * r85_test + 0.50 * r90_test,
    )

    risk_variants_11u["risk_35q85_65q90"] = (
        0.35 * r85_oof + 0.65 * r90_oof,
        0.35 * r85_test + 0.65 * r90_test,
    )

    risk_variants_11u["risk_65q85_35q90"] = (
        0.65 * r85_oof + 0.35 * r90_oof,
        0.65 * r85_test + 0.35 * r90_test,
    )

if "q85_res5" in risk_bank_11u and "q85_res8" in risk_bank_11u and "q90_res5" in risk_bank_11u:
    r85_oof = risk_bank_11u["q85_res5"]["oof"]
    r85_test = risk_bank_11u["q85_res5"]["test"]
    r858_oof = risk_bank_11u["q85_res8"]["oof"]
    r858_test = risk_bank_11u["q85_res8"]["test"]
    r90_oof = risk_bank_11u["q90_res5"]["oof"]
    r90_test = risk_bank_11u["q90_res5"]["test"]

    risk_variants_11u["risk_mix_q85_q85res8_q90"] = (
        0.35 * r85_oof + 0.30 * r858_oof + 0.35 * r90_oof,
        0.35 * r85_test + 0.30 * r858_test + 0.35 * r90_test,
    )

print("Risk variants:", list(risk_variants_11u.keys()))

if len(risk_variants_11u) == 0:
    raise ValueError("No risk variants available. Run SECTION 11R first.")


# ==================================================
# PART 2 - Build Guardrail Features
# ==================================================

print("=" * 100)
print("PART 2 - Build Guardrail Features")

meta_oof_list_11u = []
meta_test_list_11u = []

# Base model outputs from earlier sections
if "model_outputs" in globals():
    for m in ["catboost", "lightgbm", "xgboost"]:
        if m in model_outputs:
            if "oof" in model_outputs[m] and "test_pred" in model_outputs[m]:
                meta_oof_list_11u.append(np.asarray(model_outputs[m]["oof"]).reshape(-1, 1))
                meta_test_list_11u.append(np.asarray(model_outputs[m]["test_pred"]).reshape(-1, 1))

# Optional ensemble variables
optional_pairs_11u = [
    ("global_oof", "global_test"),
    ("s6_oof", "s6_test"),
    ("ridge3_oof", "ridge3_test"),
    ("adaptive_oof", "final_adaptive_test"),
]

for oof_name, test_name in optional_pairs_11u:
    if oof_name in globals() and test_name in globals():
        meta_oof_list_11u.append(np.asarray(globals()[oof_name]).reshape(-1, 1))
        meta_test_list_11u.append(np.asarray(globals()[test_name]).reshape(-1, 1))

if len(meta_oof_list_11u) > 0:
    M_oof_guard_11u = np.hstack(meta_oof_list_11u)
    M_test_guard_11u = np.hstack(meta_test_list_11u)

    meta_max_minus_base_oof_11u = M_oof_guard_11u.max(axis=1) - base_oof_11u
    meta_max_minus_base_test_11u = M_test_guard_11u.max(axis=1) - base_test_11u

    meta_range_oof_11u = M_oof_guard_11u.max(axis=1) - M_oof_guard_11u.min(axis=1)
    meta_range_test_11u = M_test_guard_11u.max(axis=1) - M_test_guard_11u.min(axis=1)

    meta_std_oof_11u = M_oof_guard_11u.std(axis=1)
    meta_std_test_11u = M_test_guard_11u.std(axis=1)

else:
    print("No meta predictions found. Guardrails will be loose.")

    meta_max_minus_base_oof_11u = np.zeros(n_train_11u)
    meta_max_minus_base_test_11u = np.zeros(n_test_11u)

    meta_range_oof_11u = np.zeros(n_train_11u)
    meta_range_test_11u = np.zeros(n_test_11u)

    meta_std_oof_11u = np.zeros(n_train_11u)
    meta_std_test_11u = np.zeros(n_test_11u)


range_q50_11u = np.quantile(meta_range_oof_11u, 0.50)
range_q60_11u = np.quantile(meta_range_oof_11u, 0.60)
std_q50_11u = np.quantile(meta_std_oof_11u, 0.50)

guard_masks_11u = {
    "none": (
        np.ones(n_train_11u, dtype=bool),
        np.ones(n_test_11u, dtype=bool),
    ),
    "meta_positive": (
        meta_max_minus_base_oof_11u >= 0,
        meta_max_minus_base_test_11u >= 0,
    ),
    "range_q50": (
        meta_range_oof_11u >= range_q50_11u,
        meta_range_test_11u >= range_q50_11u,
    ),
    "range_q60": (
        meta_range_oof_11u >= range_q60_11u,
        meta_range_test_11u >= range_q60_11u,
    ),
    "meta_pos_or_range_q50": (
        (meta_max_minus_base_oof_11u >= 0) | (meta_range_oof_11u >= range_q50_11u),
        (meta_max_minus_base_test_11u >= 0) | (meta_range_test_11u >= range_q50_11u),
    ),
    "std_q50": (
        meta_std_oof_11u >= std_q50_11u,
        meta_std_test_11u >= std_q50_11u,
    ),
}

print("Guard modes:", list(guard_masks_11u.keys()))


# ==================================================
# PART 3 - PART A Fine Grid Around 11T
# ==================================================

print("=" * 100)
print("PART A - Fine Grid Around 11T Selective Correction")

source_model_order_11u = [
    "lgb11t_missedsoft31",
    "lgb11t_tailsoft31",
    "lgb11t_conservative63",
    "lgb11t_plain31",
]

source_model_names_11u = [
    m for m in source_model_order_11u
    if m in lgb_11t_outputs
]

if len(source_model_names_11u) == 0:
    source_model_names_11u = list(lgb_11t_outputs.keys())

print("Source models:", source_model_names_11u)

min_base_q_grid_11u = [0.78, 0.80, 0.82, 0.85]
top_frac_grid_11u = [0.025, 0.035, 0.05, 0.065, 0.08]
strength_grid_11u = [0.18, 0.20, 0.225, 0.25, 0.275, 0.30, 0.325]
cap_grid_11u = [1.25, 1.50, 1.75, 2.00, 2.25, 2.50, 3.00]
delta_min_grid_11u = [0.00, 0.25, 0.50, 1.00]

# Keep guard grid moderate so runtime stays sane.
guard_grid_11u = [
    "none",
    "meta_positive",
    "meta_pos_or_range_q50",
    "range_q50",
]

part_a_start_count = len(candidate_rows_11u)

for source_name in source_model_names_11u:
    source_oof = np.clip(np.asarray(lgb_11t_outputs[source_name]["oof"]), 0, None)
    source_test = np.clip(np.asarray(lgb_11t_outputs[source_name]["test"]), 0, None)

    delta_oof = np.maximum(source_oof - base_oof_11u, 0)
    delta_test = np.maximum(source_test - base_test_11u, 0)

    print("=" * 100)
    print("Fine grid source:", source_name)
    print("Source raw OOF:", fast_rmse_11u(y_arr_11u, source_oof))
    print("Mean positive delta OOF:", float(delta_oof.mean()))
    print("Max positive delta OOF:", float(delta_oof.max()))

    for risk_name, (risk_oof, risk_test) in risk_variants_11u.items():
        risk_oof = np.asarray(risk_oof)
        risk_test = np.asarray(risk_test)

        score_oof_base = risk_oof * delta_oof
        score_test_base = risk_test * delta_test

        for guard_name in guard_grid_11u:
            guard_oof, guard_test = guard_masks_11u[guard_name]

            for min_base_q in min_base_q_grid_11u:
                base_thr_oof = np.quantile(base_oof_11u, min_base_q)
                base_thr_test = np.quantile(base_test_11u, min_base_q)

                eligible_base_oof = base_oof_11u >= base_thr_oof
                eligible_base_test = base_test_11u >= base_thr_test

                for delta_min in delta_min_grid_11u:
                    eligible_oof = (
                        eligible_base_oof
                        & guard_oof
                        & (delta_oof >= delta_min)
                        & (score_oof_base > 0)
                    )

                    eligible_test = (
                        eligible_base_test
                        & guard_test
                        & (delta_test >= delta_min)
                        & (score_test_base > 0)
                    )

                    if eligible_oof.sum() < 20 or eligible_test.sum() < 5:
                        continue

                    score_eligible_oof = score_oof_base[eligible_oof]
                    score_eligible_test = score_test_base[eligible_test]

                    if np.max(score_eligible_oof) <= 0 or np.max(score_eligible_test) <= 0:
                        continue

                    for top_frac in top_frac_grid_11u:
                        thr_oof = np.quantile(score_eligible_oof, 1.0 - top_frac)
                        thr_test = np.quantile(score_eligible_test, 1.0 - top_frac)

                        top_mask_oof = eligible_oof & (score_oof_base >= thr_oof)
                        top_mask_test = eligible_test & (score_test_base >= thr_test)

                        if top_mask_oof.sum() < 5:
                            continue

                        for strength in strength_grid_11u:
                            raw_corr_oof = strength * delta_oof
                            raw_corr_test = strength * delta_test

                            for cap in cap_grid_11u:
                                corr_oof = np.clip(raw_corr_oof, 0, cap)
                                corr_test = np.clip(raw_corr_test, 0, cap)

                                corr_oof = np.where(top_mask_oof, corr_oof, 0.0)
                                corr_test = np.where(top_mask_test, corr_test, 0.0)

                                pred_oof = np.clip(base_oof_11u + corr_oof, 0, None)
                                pred_test = np.clip(base_test_11u + corr_test, 0, None)

                                key = (
                                    f"A_{source_name}"
                                    f"_{risk_name}"
                                    f"_guard{guard_name}"
                                    f"_minq{min_base_q}"
                                    f"_top{top_frac}"
                                    f"_dmin{delta_min}"
                                    f"_s{strength}"
                                    f"_cap{cap}"
                                )

                                evaluate_candidate_11u(
                                    key=key,
                                    method="A_fine_grid",
                                    pred_oof=pred_oof,
                                    pred_test=pred_test,
                                    correction_oof=corr_oof,
                                    correction_test=corr_test,
                                )

                                del corr_oof, corr_test, pred_oof, pred_test

    del source_oof, source_test, delta_oof, delta_test
    gc.collect()

part_a_df_11u = pd.DataFrame(candidate_rows_11u[part_a_start_count:]).sort_values("rmse")

print("=" * 100)
print("Top PART A fine-grid results")
display(part_a_df_11u.head(30))

print("=" * 100)
print("Best PART A candidate")
print("Key:", best_part_a_11u["key"])
print("RMSE:", best_part_a_11u["rmse"])


# ==================================================
# PART 4 - PART B Reconstruct 11S Correction and Blend
# ==================================================

print("=" * 100)
print("PART B - Blend 11T Correction With 11S Stable Correction")


def get_11r_correction_11u(key):
    obj = correction_outputs_11r[key]

    if "correction_oof" in obj and "correction_test" in obj:
        corr_oof = np.asarray(obj["correction_oof"])
        corr_test = np.asarray(obj["correction_test"])
    else:
        corr_oof = np.asarray(obj["oof"]) - base_oof_11u
        corr_test = np.asarray(obj["test"]) - base_test_11u

    corr_oof = np.clip(corr_oof, 0, None)
    corr_test = np.clip(corr_test, 0, None)

    return corr_oof, corr_test


corr_df_11u = correction_df_11r.copy().sort_values("rmse").reset_index(drop=True)

narrow_11s_df = corr_df_11u[
    (corr_df_11u["method"] == "missed_tail_residual") &
    (corr_df_11u["label_name"].astype(str).str.contains("q85_res5")) &
    (np.isclose(corr_df_11u["min_base_q"].astype(float), 0.85)) &
    (np.isclose(corr_df_11u["top_frac"].astype(float), 0.005))
].sort_values("rmse").head(1)

stable_11s_df = corr_df_11u[
    (corr_df_11u["method"] == "missed_tail_residual") &
    (corr_df_11u["label_name"].astype(str).str.contains("q85_res8")) &
    (corr_df_11u["corrected_count_oof"].astype(float) >= 100) &
    (corr_df_11u["corrected_count_oof"].astype(float) <= 400) &
    (corr_df_11u["cap"].astype(float) <= 1.0)
].sort_values("rmse").head(1)

if len(narrow_11s_df) == 0:
    print("11S narrow fallback.")
    narrow_11s_df = corr_df_11u.sort_values("rmse").head(1)

if len(stable_11s_df) == 0:
    print("11S stable fallback.")
    stable_11s_df = corr_df_11u[
        corr_df_11u["corrected_count_oof"].astype(float) >= 50
    ].sort_values("rmse").head(1)

narrow_key_11s = narrow_11s_df.iloc[0]["key"]
stable_key_11s = stable_11s_df.iloc[0]["key"]

print("11S narrow key:", narrow_key_11s)
print("11S stable key:", stable_key_11s)

n_corr_oof_11s, n_corr_test_11s = get_11r_correction_11u(narrow_key_11s)
s_corr_oof_11s, s_corr_test_11s = get_11r_correction_11u(stable_key_11s)

corr_11s_oof = np.clip(1.0 * n_corr_oof_11s + 0.75 * s_corr_oof_11s, 0, 3.0)
corr_11s_test = np.clip(1.0 * n_corr_test_11s + 0.75 * s_corr_test_11s, 0, 3.0)

pred_11s_recon_oof = np.clip(base_oof_11u + corr_11s_oof, 0, None)
pred_11s_recon_test = np.clip(base_test_11u + corr_11s_test, 0, None)

evaluate_candidate_11u(
    key="B_reconstructed_11S_blend_wn1_ws075_cap3",
    method="B_11S_reconstructed",
    pred_oof=pred_11s_recon_oof,
    pred_test=pred_11s_recon_test,
    correction_oof=corr_11s_oof,
    correction_test=corr_11s_test,
)

# Correction sources for blending
blend_sources_11u = []

if best_part_a_11u["correction_oof"] is not None:
    blend_sources_11u.append((
        "best_part_A",
        best_part_a_11u["correction_oof"],
        best_part_a_11u["correction_test"],
    ))

if "best_11t" in globals():
    corr_oof_best11t = np.clip(np.asarray(best_11t["oof"]) - base_oof_11u, 0, None)
    corr_test_best11t = np.clip(np.asarray(best_11t["test"]) - base_test_11u, 0, None)

    blend_sources_11u.append((
        "prev_best_11T",
        corr_oof_best11t,
        corr_test_best11t,
    ))

# Deduplicate roughly by name. Small list, no heavy memory.
w_t_grid_11u = [0.70, 0.85, 1.00, 1.10, 1.20, 1.35]
w_s_grid_11u = [0.00, 0.15, 0.25, 0.35, 0.50, 0.75]
total_cap_grid_11u = [1.50, 1.75, 2.00, 2.25, 2.50, 3.00, 3.50]

part_b_start_count = len(candidate_rows_11u)

for source_label, corr_t_oof, corr_t_test in blend_sources_11u:
    for wt in w_t_grid_11u:
        for ws in w_s_grid_11u:
            if wt == 0 and ws == 0:
                continue

            raw_corr_oof = wt * corr_t_oof + ws * corr_11s_oof
            raw_corr_test = wt * corr_t_test + ws * corr_11s_test

            for total_cap in total_cap_grid_11u:
                corr_oof = np.clip(raw_corr_oof, 0, total_cap)
                corr_test = np.clip(raw_corr_test, 0, total_cap)

                pred_oof = np.clip(base_oof_11u + corr_oof, 0, None)
                pred_test = np.clip(base_test_11u + corr_test, 0, None)

                key = (
                    f"B_blend_{source_label}"
                    f"_wt{wt}_ws{ws}_cap{total_cap}"
                )

                evaluate_candidate_11u(
                    key=key,
                    method="B_blend_11T_11S",
                    pred_oof=pred_oof,
                    pred_test=pred_test,
                    correction_oof=corr_oof,
                    correction_test=corr_test,
                )

                del corr_oof, corr_test, pred_oof, pred_test

part_b_df_11u = pd.DataFrame(candidate_rows_11u[part_b_start_count:]).sort_values("rmse")

print("=" * 100)
print("Top PART B blend results")
display(part_b_df_11u.head(30))


# ==================================================
# PART 5 - PART C Robust Final Selection
# ==================================================

print("=" * 100)
print("PART C - Robust Final Selection")

candidate_df_11u = pd.DataFrame(candidate_rows_11u)

candidate_df_11u = candidate_df_11u.sort_values("rmse").reset_index(drop=True)
candidate_df_11u_robust = candidate_df_11u.sort_values("selection_score").reset_index(drop=True)

print("Top 11U candidates by raw RMSE")
display(candidate_df_11u.head(40))

print("Top 11U candidates by robust selection score")
display(candidate_df_11u_robust.head(40))


print("=" * 100)
print("Selected 11U candidates")

print("Raw best key     :", best_raw_11u["key"])
print("Raw best method  :", best_raw_11u["method"])
print("Raw best RMSE    :", best_raw_11u["rmse"])
print("Raw delta vs 11O :", best_raw_11u["rmse"] - BEST_11O_OOF)

print("-" * 100)

print("Robust best key     :", best_robust_11u["key"])
print("Robust best method  :", best_robust_11u["method"])
print("Robust best RMSE    :", best_robust_11u["rmse"])
print("Robust selection    :", best_robust_11u["selection_score"])
print("Robust delta vs 11O :", best_robust_11u["rmse"] - BEST_11O_OOF)


raw_report_11u = target_bin_report_11u(
    best_raw_11u["oof"],
    f"11U raw best: {best_raw_11u['key']}",
)

robust_report_11u = target_bin_report_11u(
    best_robust_11u["oof"],
    f"11U robust best: {best_robust_11u['key']}",
)


# ==================================================
# PART 6 - Bin Comparison
# ==================================================

print("=" * 100)
print("Bin comparison: 11O base vs 11T previous vs 11U raw vs 11U robust")

compare_11u = train_raw[[ID_COL, TARGET]].copy()

compare_11u["base_pred"] = base_oof_11u
compare_11u["raw_pred"] = best_raw_11u["oof"]
compare_11u["robust_pred"] = best_robust_11u["oof"]

if "best_11t" in globals():
    compare_11u["prev11t_pred"] = np.asarray(best_11t["oof"])
else:
    compare_11u["prev11t_pred"] = base_oof_11u

for col in ["base_pred", "prev11t_pred", "raw_pred", "robust_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")

    compare_11u[err_col] = compare_11u[col] - compare_11u[TARGET]
    compare_11u[se_col] = compare_11u[err_col] ** 2

compare_11u["target_bin_5"] = pd.qcut(
    compare_11u[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11u = (
    compare_11u.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        prev11t_pred_mean=("prev11t_pred", "mean"),
        raw_pred_mean=("raw_pred", "mean"),
        robust_pred_mean=("robust_pred", "mean"),

        base_bias=("base_error", "mean"),
        prev11t_bias=("prev11t_error", "mean"),
        raw_bias=("raw_error", "mean"),
        robust_bias=("robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        prev11t_rmse=("prev11t_se", lambda x: np.sqrt(np.mean(x))),
        raw_rmse=("raw_se", lambda x: np.sqrt(np.mean(x))),
        robust_rmse=("robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11u["prev11t_rmse_delta"] = bin_compare_11u["prev11t_rmse"] - bin_compare_11u["base_rmse"]
bin_compare_11u["raw_rmse_delta"] = bin_compare_11u["raw_rmse"] - bin_compare_11u["base_rmse"]
bin_compare_11u["robust_rmse_delta"] = bin_compare_11u["robust_rmse"] - bin_compare_11u["base_rmse"]

display(bin_compare_11u.sort_values("base_rmse", ascending=False))


# ==================================================
# PART 7 - Save Submissions
# ==================================================

print("=" * 100)
print("Save 11U submissions")

sub_11u_raw = save_submission_11u(
    best_raw_11u["test"],
    "submission_11u_raw_best.csv",
)

sub_11u_robust = save_submission_11u(
    best_robust_11u["test"],
    "submission_11u_robust_best.csv",
)

if "best_11t" in globals():
    sub_11t_ref_from_11u = save_submission_11u(
        best_11t["test"],
        "submission_11t_reference_from_11u.csv",
    )

sub_11o_ref_from_11u = save_submission_11u(
    base_test_11u,
    "submission_11o_reference_from_11u.csv",
)


# ==================================================
# PART 8 - Decision Guide
# ==================================================

print("=" * 100)
print("11U Decision Guide")

prev_11t_score = best_11t["score"] if "best_11t" in globals() else np.nan

print(f"11O adaptive OOF      : {base_score_11u:.6f}")
if "best_11t" in globals():
    print(f"Previous 11T best OOF : {prev_11t_score:.6f}")

print(f"11U raw best OOF      : {best_raw_11u['rmse']:.6f}")
print(f"11U robust best OOF   : {best_robust_11u['rmse']:.6f}")

print(f"Raw delta vs 11O      : {best_raw_11u['rmse'] - BEST_11O_OOF:.6f}")
print(f"Robust delta vs 11O   : {best_robust_11u['rmse'] - BEST_11O_OOF:.6f}")

if "best_11t" in globals():
    print(f"Raw delta vs 11T      : {best_raw_11u['rmse'] - prev_11t_score:.6f}")
    print(f"Robust delta vs 11T   : {best_robust_11u['rmse'] - prev_11t_score:.6f}")

if best_raw_11u["rmse"] < BEST_11O_OOF - 0.010:
    print("Raw best decision: STRONG candidate.")
elif best_raw_11u["rmse"] < BEST_11O_OOF - 0.003:
    print("Raw best decision: GOOD candidate. Submit if bin report is safe.")
elif best_raw_11u["rmse"] < BEST_11O_OOF:
    print("Raw best decision: WEAK/MEDIUM candidate. Submit only as probe.")
else:
    print("Raw best decision: DO NOT FORCE.")

if best_robust_11u["rmse"] < BEST_11O_OOF - 0.010:
    print("Robust best decision: STRONG candidate.")
elif best_robust_11u["rmse"] < BEST_11O_OOF - 0.003:
    print("Robust best decision: GOOD candidate. Usually safer than raw if bins are stable.")
elif best_robust_11u["rmse"] < BEST_11O_OOF:
    print("Robust best decision: WEAK/MEDIUM candidate. Submit only as probe.")
else:
    print("Robust best decision: DO NOT FORCE.")

print("=" * 100)
print("Please screenshot:")
print("1. part_a_df_11u.head(30)")
print("2. part_b_df_11u.head(30)")
print("3. candidate_df_11u.head(40)")
print("4. candidate_df_11u_robust.head(40)")
print("5. raw_report_11u")
print("6. robust_report_11u")
print("7. bin_compare_11u")

In [ ]:
# ==================================================
# DOWNLOAD 11U ROBUST BEST
# ==================================================

from google.colab import files
import os

filename = "submission_11u_robust_best.csv"

if os.path.exists(filename):
    print(f"Downloading: {filename}")
    files.download(filename)
else:
    raise FileNotFoundError(
        f"{filename} belum ada. Pastikan SECTION 11U sudah selesai dan file sudah tersimpan."
    )

In [ ]:
# ==================================================
# SECTION 11V - Benefit-Gated Correction Pruning
# ==================================================
# Goal:
# 1. Use 11U raw/robust correction as source.
# 2. Learn which corrected OOF rows actually benefited.
# 3. Build fold-safe benefit gate classifier.
# 4. Shrink / prune 11U correction.
# 5. Save 11V raw and robust submissions.
#
# Paste AFTER SECTION 11U.
# Skip 11P and 11Q.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


# ==================================================
# PART 0 - Setup
# ==================================================

print("=" * 100)
print("SECTION 11V - Benefit-Gated Correction Pruning")

BEST_11O_OOF = 11.26696803195285

required_vars_11v = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "best_raw_11u",
    "best_robust_11u",
]

missing_11v = [v for v in required_vars_11v if v not in globals()]
if len(missing_11v) > 0:
    raise NameError(f"Missing required variables for 11V: {missing_11v}")

base_oof_11v = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11v = np.clip(np.asarray(final_adaptive_test), 0, None)

y_arr_11v = y.values
n_train_11v = len(y_arr_11v)
n_test_11v = sample_submission.shape[0]

base_score_11v = rmse(y, base_oof_11v)

print("Base 11O adaptive OOF:", base_score_11v)
print("Previous 11U raw OOF:", best_raw_11u["rmse"])
print("Previous 11U robust OOF:", best_robust_11u["rmse"])


# ==================================================
# Fast bin setup
# ==================================================

target_bin_11v = pd.qcut(
    y_arr_11v,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels_11v = ["very_low", "low", "medium", "high", "very_high"]
bin_masks_11v = {
    b: (target_bin_11v == b)
    for b in bin_labels_11v
}


def fast_rmse_11v(y_true, pred):
    pred = np.asarray(pred)
    return float(np.sqrt(np.mean((pred - y_true) ** 2)))


def rank_against_train_11v(train_values, values):
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)
    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def save_submission_11v(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11v(pred, label):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = pred
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", fast_rmse_11v(y_arr_11v, pred))
    display(report.sort_values("rmse", ascending=False))

    return report


def calc_metrics_11v(pred, correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    err = pred - y_arr_11v
    se = err ** 2

    if correction is None:
        correction = pred - base_oof_11v

    correction = np.asarray(correction)

    out = {
        "rmse": fast_rmse_11v(y_arr_11v, pred),
    }

    for b in bin_labels_11v:
        m = bin_masks_11v[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    changed = np.abs(pred - base_oof_11v) > 1e-12

    base_se = (base_oof_11v - y_arr_11v) ** 2
    new_se = (pred - y_arr_11v) ** 2
    benefit = base_se - new_se

    if changed.sum() > 0:
        benefit_positive_rate = float(np.mean(benefit[changed] > 0))
        avg_benefit_changed = float(np.mean(benefit[changed]))
    else:
        benefit_positive_rate = 0.0
        avg_benefit_changed = 0.0

    out.update({
        "corrected_count": int((correction > 1e-12).sum()),
        "corrected_rate": float(np.mean(correction > 1e-12)),
        "mean_correction": float(np.mean(correction)),
        "max_correction": float(np.max(correction)),
        "changed_count": int(changed.sum()),
        "benefit_positive_rate": benefit_positive_rate,
        "avg_benefit_changed": avg_benefit_changed,
        "delta_vs_11O": fast_rmse_11v(y_arr_11v, pred) - BEST_11O_OOF,
    })

    return out


base_metrics_11v = calc_metrics_11v(
    base_oof_11v,
    correction=np.zeros_like(base_oof_11v),
)

base_vh_rmse_11v = base_metrics_11v["very_high_rmse"]
base_high_rmse_11v = base_metrics_11v["high_rmse"]
base_medium_rmse_11v = base_metrics_11v["medium_rmse"]
base_low_rmse_11v = base_metrics_11v["low_rmse"]
base_vlow_rmse_11v = base_metrics_11v["very_low_rmse"]


def robust_penalty_11v(metrics):
    penalty = 0.0

    cc = metrics["corrected_count"]

    # Prefer pruning, but do not make correction too tiny.
    if cc > 0 and cc < 60:
        penalty += 0.00040

    # Too broad means too much leakage risk.
    if cc > 280:
        penalty += 0.00025
    if cc > 350:
        penalty += 0.00050

    # Very-high should remain improved.
    if metrics["very_high_rmse"] > base_vh_rmse_11v - 0.035:
        penalty += 0.00050

    # Leakage guards versus 11O base.
    if metrics["medium_rmse"] > base_medium_rmse_11v + 0.060:
        penalty += 0.00050

    if metrics["high_rmse"] > base_high_rmse_11v + 0.050:
        penalty += 0.00040

    if metrics["low_rmse"] > base_low_rmse_11v + 0.035:
        penalty += 0.00025

    if metrics["very_low_rmse"] > base_vlow_rmse_11v + 0.035:
        penalty += 0.00025

    # Benefit quality.
    if cc > 0:
        if metrics["benefit_positive_rate"] < 0.47:
            penalty += 0.00030
        if metrics["avg_benefit_changed"] < 0:
            penalty += 0.00100

    return penalty


candidate_rows_11v = []

best_raw_11v = {
    "key": "base_11O_adaptive",
    "method": "base",
    "rmse": base_score_11v,
    "selection_score": base_score_11v,
    "oof": base_oof_11v.copy(),
    "test": base_test_11v.copy(),
    "correction_oof": np.zeros_like(base_oof_11v),
    "correction_test": np.zeros_like(base_test_11v),
}

best_robust_11v = best_raw_11v.copy()


def evaluate_candidate_11v(
    key,
    method,
    pred_oof,
    pred_test,
    correction_oof,
    correction_test=None,
):
    global best_raw_11v, best_robust_11v

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)
    correction_oof = np.clip(np.asarray(correction_oof), 0, None)

    if correction_test is None:
        correction_test = np.clip(pred_test - base_test_11v, 0, None)
    else:
        correction_test = np.clip(np.asarray(correction_test), 0, None)

    metrics = calc_metrics_11v(pred_oof, correction_oof)
    metrics["robust_penalty"] = robust_penalty_11v(metrics)
    metrics["selection_score"] = metrics["rmse"] + metrics["robust_penalty"]

    row = {
        "key": key,
        "method": method,
        **metrics,
    }

    candidate_rows_11v.append(row)

    if metrics["rmse"] < best_raw_11v["rmse"]:
        best_raw_11v = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }

    if metrics["selection_score"] < best_robust_11v["selection_score"]:
        best_robust_11v = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }


# Base and previous 11U references
evaluate_candidate_11v(
    key="base_11O_adaptive",
    method="base",
    pred_oof=base_oof_11v,
    pred_test=base_test_11v,
    correction_oof=np.zeros_like(base_oof_11v),
    correction_test=np.zeros_like(base_test_11v),
)

evaluate_candidate_11v(
    key="prev_11u_raw",
    method="previous_11U",
    pred_oof=best_raw_11u["oof"],
    pred_test=best_raw_11u["test"],
    correction_oof=best_raw_11u["correction_oof"],
    correction_test=best_raw_11u["correction_test"],
)

evaluate_candidate_11v(
    key="prev_11u_robust",
    method="previous_11U",
    pred_oof=best_robust_11u["oof"],
    pred_test=best_robust_11u["test"],
    correction_oof=best_robust_11u["correction_oof"],
    correction_test=best_robust_11u["correction_test"],
)


# ==================================================
# PART 1 - Rebuild risk/meta features
# ==================================================

print("=" * 100)
print("PART 1 - Build 11V gate features")


def get_risk_from_11r_label_11v(label_name):
    if "missed_tail_outputs_11r" not in globals():
        return None

    if label_name not in missed_tail_outputs_11r:
        return None

    obj = missed_tail_outputs_11r[label_name]

    gate_oof = np.clip(np.asarray(obj["gate_oof"]), 0, None)
    gate_test = np.clip(np.asarray(obj["gate_test"]), 0, None)

    resid_oof = np.clip(np.asarray(obj["resid_oof"]), 0, None)
    resid_test = np.clip(np.asarray(obj["resid_test"]), 0, None)

    risk_oof_raw = gate_oof * resid_oof
    risk_test_raw = gate_test * resid_test

    if np.max(risk_oof_raw) <= 0:
        risk_oof_rank = np.zeros_like(risk_oof_raw)
        risk_test_rank = np.zeros_like(risk_test_raw)
    else:
        risk_oof_rank = rank_against_train_11v(risk_oof_raw, risk_oof_raw)
        risk_test_rank = rank_against_train_11v(risk_oof_raw, risk_test_raw)

    return risk_oof_rank, risk_test_rank


risk_features_train_11v = {}
risk_features_test_11v = {}

for label in ["q85_res5", "q85_res8", "q90_res5", "q90_res8"]:
    risk_pair = get_risk_from_11r_label_11v(label)
    if risk_pair is not None:
        risk_features_train_11v[f"risk_{label}"] = risk_pair[0]
        risk_features_test_11v[f"risk_{label}"] = risk_pair[1]

if "risk_q85_res5" in risk_features_train_11v and "risk_q90_res5" in risk_features_train_11v:
    risk_features_train_11v["risk_mix_50_50"] = (
        0.50 * risk_features_train_11v["risk_q85_res5"]
        + 0.50 * risk_features_train_11v["risk_q90_res5"]
    )
    risk_features_test_11v["risk_mix_50_50"] = (
        0.50 * risk_features_test_11v["risk_q85_res5"]
        + 0.50 * risk_features_test_11v["risk_q90_res5"]
    )

    risk_features_train_11v["risk_mix_35_65"] = (
        0.35 * risk_features_train_11v["risk_q85_res5"]
        + 0.65 * risk_features_train_11v["risk_q90_res5"]
    )
    risk_features_test_11v["risk_mix_35_65"] = (
        0.35 * risk_features_test_11v["risk_q85_res5"]
        + 0.65 * risk_features_test_11v["risk_q90_res5"]
    )

if len(risk_features_train_11v) == 0:
    print("Warning: no 11R risk features found. Using base rank only.")

base_rank_oof_11v = rank_against_train_11v(base_oof_11v, base_oof_11v)
base_rank_test_11v = rank_against_train_11v(base_oof_11v, base_test_11v)


# Meta features
meta_oof_list_11v = []
meta_test_list_11v = []

if "model_outputs" in globals():
    for m in ["catboost", "lightgbm", "xgboost"]:
        if m in model_outputs:
            if "oof" in model_outputs[m] and "test_pred" in model_outputs[m]:
                meta_oof_list_11v.append(np.asarray(model_outputs[m]["oof"]).reshape(-1, 1))
                meta_test_list_11v.append(np.asarray(model_outputs[m]["test_pred"]).reshape(-1, 1))

optional_pairs_11v = [
    ("global_oof", "global_test"),
    ("s6_oof", "s6_test"),
    ("ridge3_oof", "ridge3_test"),
    ("adaptive_oof", "final_adaptive_test"),
]

for oof_name, test_name in optional_pairs_11v:
    if oof_name in globals() and test_name in globals():
        meta_oof_list_11v.append(np.asarray(globals()[oof_name]).reshape(-1, 1))
        meta_test_list_11v.append(np.asarray(globals()[test_name]).reshape(-1, 1))

if len(meta_oof_list_11v) > 0:
    M_oof_11v = np.hstack(meta_oof_list_11v)
    M_test_11v = np.hstack(meta_test_list_11v)

    meta_mean_oof_11v = M_oof_11v.mean(axis=1)
    meta_mean_test_11v = M_test_11v.mean(axis=1)

    meta_std_oof_11v = M_oof_11v.std(axis=1)
    meta_std_test_11v = M_test_11v.std(axis=1)

    meta_range_oof_11v = M_oof_11v.max(axis=1) - M_oof_11v.min(axis=1)
    meta_range_test_11v = M_test_11v.max(axis=1) - M_test_11v.min(axis=1)

    meta_max_minus_base_oof_11v = M_oof_11v.max(axis=1) - base_oof_11v
    meta_max_minus_base_test_11v = M_test_11v.max(axis=1) - base_test_11v
else:
    meta_mean_oof_11v = base_oof_11v.copy()
    meta_mean_test_11v = base_test_11v.copy()

    meta_std_oof_11v = np.zeros(n_train_11v)
    meta_std_test_11v = np.zeros(n_test_11v)

    meta_range_oof_11v = np.zeros(n_train_11v)
    meta_range_test_11v = np.zeros(n_test_11v)

    meta_max_minus_base_oof_11v = np.zeros(n_train_11v)
    meta_max_minus_base_test_11v = np.zeros(n_test_11v)


# Optional source model delta from 11T
delta_11t_oof_11v = np.zeros(n_train_11v)
delta_11t_test_11v = np.zeros(n_test_11v)

if "lgb_11t_outputs" in globals() and "lgb11t_missedsoft31" in lgb_11t_outputs:
    delta_11t_oof_11v = np.maximum(
        np.asarray(lgb_11t_outputs["lgb11t_missedsoft31"]["oof"]) - base_oof_11v,
        0,
    )
    delta_11t_test_11v = np.maximum(
        np.asarray(lgb_11t_outputs["lgb11t_missedsoft31"]["test"]) - base_test_11v,
        0,
    )


# ==================================================
# PART 2 - Build benefit gate features
# ==================================================

def make_gate_features_11v(source_corr_oof, source_corr_test):
    source_corr_oof = np.clip(np.asarray(source_corr_oof), 0, None)
    source_corr_test = np.clip(np.asarray(source_corr_test), 0, None)

    train_dict = {
        "base_pred": base_oof_11v,
        "base_rank": base_rank_oof_11v,
        "corr_source": source_corr_oof,
        "corr_rank": rank_against_train_11v(source_corr_oof, source_corr_oof),
        "corr_x_base_rank": source_corr_oof * base_rank_oof_11v,
        "meta_mean": meta_mean_oof_11v,
        "meta_std": meta_std_oof_11v,
        "meta_range": meta_range_oof_11v,
        "meta_max_minus_base": meta_max_minus_base_oof_11v,
        "delta_11t": delta_11t_oof_11v,
        "delta_11t_x_corr": delta_11t_oof_11v * source_corr_oof,
    }

    test_dict = {
        "base_pred": base_test_11v,
        "base_rank": base_rank_test_11v,
        "corr_source": source_corr_test,
        "corr_rank": rank_against_train_11v(source_corr_oof, source_corr_test),
        "corr_x_base_rank": source_corr_test * base_rank_test_11v,
        "meta_mean": meta_mean_test_11v,
        "meta_std": meta_std_test_11v,
        "meta_range": meta_range_test_11v,
        "meta_max_minus_base": meta_max_minus_base_test_11v,
        "delta_11t": delta_11t_test_11v,
        "delta_11t_x_corr": delta_11t_test_11v * source_corr_test,
    }

    for k in risk_features_train_11v:
        train_dict[k] = risk_features_train_11v[k]
        test_dict[k] = risk_features_test_11v[k]
        train_dict[f"{k}_x_corr"] = risk_features_train_11v[k] * source_corr_oof
        test_dict[f"{k}_x_corr"] = risk_features_test_11v[k] * source_corr_test

    X_gate_train = pd.DataFrame(train_dict)
    X_gate_test = pd.DataFrame(test_dict)

    return X_gate_train, X_gate_test


def get_positive_class_prob_11v(model, X):
    proba = model.predict_proba(X)

    if hasattr(model, "classes_"):
        classes = model.classes_
    elif hasattr(model, "steps") and hasattr(model.steps[-1][1], "classes_"):
        classes = model.steps[-1][1].classes_
    else:
        return proba[:, -1]

    if 1 in classes:
        idx = list(classes).index(1)
    else:
        idx = -1

    return proba[:, idx]


def build_benefit_gate_11v(source_name, source_corr_oof, source_corr_test):
    print("=" * 100)
    print("Building benefit gate for:", source_name)

    source_corr_oof = np.clip(np.asarray(source_corr_oof), 0, None)
    source_corr_test = np.clip(np.asarray(source_corr_test), 0, None)

    source_pred_oof = np.clip(base_oof_11v + source_corr_oof, 0, None)

    base_se = (base_oof_11v - y_arr_11v) ** 2
    source_se = (source_pred_oof - y_arr_11v) ** 2

    benefit = base_se - source_se
    corrected_mask = source_corr_oof > 1e-12

    benefit_label = ((benefit > 0) & corrected_mask).astype(int)

    X_gate_train, X_gate_test = make_gate_features_11v(
        source_corr_oof,
        source_corr_test,
    )

    prob_logit_oof = np.zeros(n_train_11v)
    prob_hgb_oof = np.zeros(n_train_11v)

    prob_logit_test_sum = np.zeros(n_test_11v)
    prob_hgb_test_sum = np.zeros(n_test_11v)

    valid_fold_count = 0

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        tr_mask = corrected_mask[tr_idx]
        val_mask = corrected_mask[val_idx]

        tr_rows = np.asarray(tr_idx)[tr_mask]
        val_rows = np.asarray(val_idx)[val_mask]

        if len(tr_rows) < 40 or len(np.unique(benefit_label[tr_rows])) < 2:
            print(f"  Fold {fold}: skipped, not enough class diversity.")
            continue

        X_tr = X_gate_train.iloc[tr_rows]
        y_tr = benefit_label[tr_rows]

        X_val = X_gate_train.iloc[val_rows] if len(val_rows) > 0 else X_gate_train.iloc[val_idx]
        val_target_rows = val_rows if len(val_rows) > 0 else val_idx

        # Model 1: conservative logistic gate
        logit = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(
                C=0.7,
                class_weight="balanced",
                max_iter=1000,
                random_state=4200 + fold,
            ),
        )

        logit.fit(X_tr, y_tr)

        prob_logit_oof[val_target_rows] = get_positive_class_prob_11v(
            logit,
            X_val,
        )

        prob_logit_test_sum += get_positive_class_prob_11v(
            logit,
            X_gate_test,
        )

        # Model 2: small nonlinear gate
        hgb = make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingClassifier(
                max_iter=150,
                learning_rate=0.035,
                max_leaf_nodes=7,
                min_samples_leaf=18,
                l2_regularization=2.0,
                random_state=5200 + fold,
            ),
        )

        hgb.fit(X_tr, y_tr)

        prob_hgb_oof[val_target_rows] = get_positive_class_prob_11v(
            hgb,
            X_val,
        )

        prob_hgb_test_sum += get_positive_class_prob_11v(
            hgb,
            X_gate_test,
        )

        valid_fold_count += 1

        print(
            f"  Fold {fold}: train corrected={len(tr_rows)}, "
            f"val corrected={len(val_rows)}, positive rate={y_tr.mean():.4f}"
        )

    if valid_fold_count == 0:
        print("No valid benefit-gate folds. Using simple correction rank fallback.")

        prob_rank_oof = rank_against_train_11v(source_corr_oof, source_corr_oof)
        prob_rank_test = rank_against_train_11v(source_corr_oof, source_corr_test)

        prob_logit_oof = prob_rank_oof
        prob_hgb_oof = prob_rank_oof

        prob_logit_test = prob_rank_test
        prob_hgb_test = prob_rank_test
    else:
        prob_logit_test = prob_logit_test_sum / valid_fold_count
        prob_hgb_test = prob_hgb_test_sum / valid_fold_count

    prob_avg_oof = 0.50 * prob_logit_oof + 0.50 * prob_hgb_oof
    prob_avg_test = 0.50 * prob_logit_test + 0.50 * prob_hgb_test

    prob_max_oof = np.maximum(prob_logit_oof, prob_hgb_oof)
    prob_max_test = np.maximum(prob_logit_test, prob_hgb_test)

    # Zero out rows without source correction
    prob_logit_oof = np.where(corrected_mask, prob_logit_oof, 0.0)
    prob_hgb_oof = np.where(corrected_mask, prob_hgb_oof, 0.0)
    prob_avg_oof = np.where(corrected_mask, prob_avg_oof, 0.0)
    prob_max_oof = np.where(corrected_mask, prob_max_oof, 0.0)

    test_corrected_mask = source_corr_test > 1e-12
    prob_logit_test = np.where(test_corrected_mask, prob_logit_test, 0.0)
    prob_hgb_test = np.where(test_corrected_mask, prob_hgb_test, 0.0)
    prob_avg_test = np.where(test_corrected_mask, prob_avg_test, 0.0)
    prob_max_test = np.where(test_corrected_mask, prob_max_test, 0.0)

    gate_df = pd.DataFrame({
        "source": source_name,
        "corrected_count": [int(corrected_mask.sum())],
        "actual_benefit_positive_rate": [
            float(np.mean(benefit[corrected_mask] > 0)) if corrected_mask.sum() > 0 else 0
        ],
        "actual_avg_benefit": [
            float(np.mean(benefit[corrected_mask])) if corrected_mask.sum() > 0 else 0
        ],
        "valid_fold_count": [valid_fold_count],
        "logit_prob_mean": [float(prob_logit_oof[corrected_mask].mean()) if corrected_mask.sum() > 0 else 0],
        "hgb_prob_mean": [float(prob_hgb_oof[corrected_mask].mean()) if corrected_mask.sum() > 0 else 0],
        "avg_prob_mean": [float(prob_avg_oof[corrected_mask].mean()) if corrected_mask.sum() > 0 else 0],
    })

    display(gate_df)

    return {
        "logit_oof": prob_logit_oof,
        "logit_test": prob_logit_test,
        "hgb_oof": prob_hgb_oof,
        "hgb_test": prob_hgb_test,
        "avg_oof": prob_avg_oof,
        "avg_test": prob_avg_test,
        "max_oof": prob_max_oof,
        "max_test": prob_max_test,
        "benefit": benefit,
        "benefit_label": benefit_label,
        "corrected_mask": corrected_mask,
    }


# ==================================================
# PART 3 - Build source corrections
# ==================================================

print("=" * 100)
print("PART 3 - Source corrections")

source_bank_11v = {
    "11u_raw": {
        "corr_oof": np.clip(np.asarray(best_raw_11u["correction_oof"]), 0, None),
        "corr_test": np.clip(np.asarray(best_raw_11u["correction_test"]), 0, None),
    },
    "11u_robust": {
        "corr_oof": np.clip(np.asarray(best_robust_11u["correction_oof"]), 0, None),
        "corr_test": np.clip(np.asarray(best_robust_11u["correction_test"]), 0, None),
    },
}

if "best_11t" in globals():
    source_bank_11v["11t_best"] = {
        "corr_oof": np.clip(np.asarray(best_11t["oof"]) - base_oof_11v, 0, None),
        "corr_test": np.clip(np.asarray(best_11t["test"]) - base_test_11v, 0, None),
    }

for name, obj in source_bank_11v.items():
    corr_oof = obj["corr_oof"]
    pred_oof = np.clip(base_oof_11v + corr_oof, 0, None)

    m = calc_metrics_11v(pred_oof, corr_oof)

    print(
        name,
        "| rmse:", round(m["rmse"], 6),
        "| corrected:", m["corrected_count"],
        "| benefit positive:", round(m["benefit_positive_rate"], 4),
        "| avg benefit:", round(m["avg_benefit_changed"], 4),
    )


# ==================================================
# PART 4 - Benefit gates
# ==================================================

print("=" * 100)
print("PART 4 - Build benefit gates")

gate_outputs_11v = {}

for source_name, obj in source_bank_11v.items():
    gate_outputs_11v[source_name] = build_benefit_gate_11v(
        source_name,
        obj["corr_oof"],
        obj["corr_test"],
    )

gc.collect()


# ==================================================
# PART 5 - Prune and shrink correction
# ==================================================

print("=" * 100)
print("PART 5 - Prune and shrink grid")

prob_types_11v = ["avg", "logit", "hgb", "max"]

source_names_grid_11v = ["11u_robust", "11u_raw"]

if "11t_best" in source_bank_11v:
    source_names_grid_11v.append("11t_best")

# Main grid around 11U.
source_weight_grid_11v = [0.85, 0.95, 1.00, 1.05, 1.10, 1.20]

high_keep_frac_grid_11v = [0.25, 0.35, 0.45, 0.55, 0.65, 0.80]
mid_keep_frac_grid_11v = [0.55, 0.70, 0.85, 1.00]

high_weight_grid_11v = [0.90, 1.00, 1.10]
mid_weight_grid_11v = [0.30, 0.50, 0.70]
low_weight_grid_11v = [0.00, 0.10, 0.20]

cap_grid_11v = [2.50, 3.00, 3.50]

for source_name in source_names_grid_11v:
    corr_oof_base = source_bank_11v[source_name]["corr_oof"]
    corr_test_base = source_bank_11v[source_name]["corr_test"]

    source_mask_oof = corr_oof_base > 1e-12
    source_mask_test = corr_test_base > 1e-12

    if source_mask_oof.sum() < 20:
        continue

    for prob_type in prob_types_11v:
        prob_oof = gate_outputs_11v[source_name][f"{prob_type}_oof"]
        prob_test = gate_outputs_11v[source_name][f"{prob_type}_test"]

        valid_prob_oof = prob_oof[source_mask_oof]
        valid_prob_test = prob_test[source_mask_test]

        if len(valid_prob_oof) == 0 or np.max(valid_prob_oof) <= 0:
            continue

        if len(valid_prob_test) == 0 or np.max(valid_prob_test) <= 0:
            continue

        for high_keep_frac in high_keep_frac_grid_11v:
            high_thr_oof = np.quantile(valid_prob_oof, 1.0 - high_keep_frac)
            high_thr_test = np.quantile(valid_prob_test, 1.0 - high_keep_frac)

            high_mask_oof = source_mask_oof & (prob_oof >= high_thr_oof)
            high_mask_test = source_mask_test & (prob_test >= high_thr_test)

            for mid_keep_frac in mid_keep_frac_grid_11v:
                if mid_keep_frac < high_keep_frac:
                    continue

                mid_thr_oof = np.quantile(valid_prob_oof, 1.0 - mid_keep_frac)
                mid_thr_test = np.quantile(valid_prob_test, 1.0 - mid_keep_frac)

                mid_mask_oof = source_mask_oof & (prob_oof >= mid_thr_oof)
                mid_mask_test = source_mask_test & (prob_test >= mid_thr_test)

                for source_weight in source_weight_grid_11v:
                    weighted_corr_oof = source_weight * corr_oof_base
                    weighted_corr_test = source_weight * corr_test_base

                    for high_weight in high_weight_grid_11v:
                        for mid_weight in mid_weight_grid_11v:
                            for low_weight in low_weight_grid_11v:

                                shrink_oof = np.full(n_train_11v, low_weight)
                                shrink_test = np.full(n_test_11v, low_weight)

                                shrink_oof[mid_mask_oof] = mid_weight
                                shrink_test[mid_mask_test] = mid_weight

                                shrink_oof[high_mask_oof] = high_weight
                                shrink_test[high_mask_test] = high_weight

                                for cap in cap_grid_11v:
                                    corr_oof = np.clip(weighted_corr_oof * shrink_oof, 0, cap)
                                    corr_test = np.clip(weighted_corr_test * shrink_test, 0, cap)

                                    pred_oof = np.clip(base_oof_11v + corr_oof, 0, None)
                                    pred_test = np.clip(base_test_11v + corr_test, 0, None)

                                    key = (
                                        f"V_{source_name}_{prob_type}"
                                        f"_sw{source_weight}"
                                        f"_hk{high_keep_frac}"
                                        f"_mk{mid_keep_frac}"
                                        f"_hw{high_weight}"
                                        f"_mw{mid_weight}"
                                        f"_lw{low_weight}"
                                        f"_cap{cap}"
                                    )

                                    evaluate_candidate_11v(
                                        key=key,
                                        method="benefit_gated_shrink",
                                        pred_oof=pred_oof,
                                        pred_test=pred_test,
                                        correction_oof=corr_oof,
                                        correction_test=corr_test,
                                    )

                                    del corr_oof, corr_test, pred_oof, pred_test

                    del weighted_corr_oof, weighted_corr_test

    gc.collect()


# ==================================================
# PART 6 - Optional local blend raw + robust after pruning
# ==================================================

print("=" * 100)
print("PART 6 - Optional blend top gated candidates")

candidate_df_temp_11v = pd.DataFrame(candidate_rows_11v).sort_values("rmse")

top_gated_keys_11v = (
    candidate_df_temp_11v[
        candidate_df_temp_11v["method"] == "benefit_gated_shrink"
    ]
    .head(8)["key"]
    .tolist()
)

# Store actual top predictions by rerun would be annoying, so use current best raw/robust only for lightweight blend.
# This part is intentionally small and safe.

for wt_raw in [0.25, 0.40, 0.50, 0.60, 0.75]:
    corr_oof = (
        wt_raw * best_raw_11v["correction_oof"]
        + (1.0 - wt_raw) * best_robust_11v["correction_oof"]
    )

    corr_test = (
        wt_raw * best_raw_11v["correction_test"]
        + (1.0 - wt_raw) * best_robust_11v["correction_test"]
    )

    for cap in [2.50, 3.00, 3.50]:
        corr_oof_cap = np.clip(corr_oof, 0, cap)
        corr_test_cap = np.clip(corr_test, 0, cap)

        pred_oof = np.clip(base_oof_11v + corr_oof_cap, 0, None)
        pred_test = np.clip(base_test_11v + corr_test_cap, 0, None)

        key = f"V_blend_raw_robust_wraw{wt_raw}_cap{cap}"

        evaluate_candidate_11v(
            key=key,
            method="blend_11v_raw_robust",
            pred_oof=pred_oof,
            pred_test=pred_test,
            correction_oof=corr_oof_cap,
            correction_test=corr_test_cap,
        )


# ==================================================
# PART 7 - Final selection
# ==================================================

print("=" * 100)
print("PART 7 - Final selection")

candidate_df_11v = pd.DataFrame(candidate_rows_11v)
candidate_df_11v = candidate_df_11v.sort_values("rmse").reset_index(drop=True)

candidate_df_11v_robust = candidate_df_11v.sort_values("selection_score").reset_index(drop=True)

print("Top 11V candidates by raw RMSE")
display(candidate_df_11v.head(40))

print("Top 11V candidates by robust selection score")
display(candidate_df_11v_robust.head(40))

print("=" * 100)
print("Selected 11V candidates")

print("Raw best key     :", best_raw_11v["key"])
print("Raw best method  :", best_raw_11v["method"])
print("Raw best RMSE    :", best_raw_11v["rmse"])
print("Raw delta vs 11O :", best_raw_11v["rmse"] - BEST_11O_OOF)
print("Raw delta vs 11U robust:", best_raw_11v["rmse"] - best_robust_11u["rmse"])

print("-" * 100)

print("Robust best key     :", best_robust_11v["key"])
print("Robust best method  :", best_robust_11v["method"])
print("Robust best RMSE    :", best_robust_11v["rmse"])
print("Robust selection    :", best_robust_11v["selection_score"])
print("Robust delta vs 11O :", best_robust_11v["rmse"] - BEST_11O_OOF)
print("Robust delta vs 11U robust:", best_robust_11v["rmse"] - best_robust_11u["rmse"])


raw_report_11v = target_bin_report_11v(
    best_raw_11v["oof"],
    f"11V raw best: {best_raw_11v['key']}",
)

robust_report_11v = target_bin_report_11v(
    best_robust_11v["oof"],
    f"11V robust best: {best_robust_11v['key']}",
)


# ==================================================
# PART 8 - Bin comparison
# ==================================================

print("=" * 100)
print("Bin comparison: 11O vs 11U robust vs 11V raw vs 11V robust")

compare_11v = train_raw[[ID_COL, TARGET]].copy()

compare_11v["base_pred"] = base_oof_11v
compare_11v["u_robust_pred"] = np.asarray(best_robust_11u["oof"])
compare_11v["v_raw_pred"] = best_raw_11v["oof"]
compare_11v["v_robust_pred"] = best_robust_11v["oof"]

for col in ["base_pred", "u_robust_pred", "v_raw_pred", "v_robust_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")

    compare_11v[err_col] = compare_11v[col] - compare_11v[TARGET]
    compare_11v[se_col] = compare_11v[err_col] ** 2

compare_11v["target_bin_5"] = pd.qcut(
    compare_11v[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11v = (
    compare_11v.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        u_robust_pred_mean=("u_robust_pred", "mean"),
        v_raw_pred_mean=("v_raw_pred", "mean"),
        v_robust_pred_mean=("v_robust_pred", "mean"),

        base_bias=("base_error", "mean"),
        u_robust_bias=("u_robust_error", "mean"),
        v_raw_bias=("v_raw_error", "mean"),
        v_robust_bias=("v_robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        u_robust_rmse=("u_robust_se", lambda x: np.sqrt(np.mean(x))),
        v_raw_rmse=("v_raw_se", lambda x: np.sqrt(np.mean(x))),
        v_robust_rmse=("v_robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11v["u_robust_rmse_delta"] = (
    bin_compare_11v["u_robust_rmse"] - bin_compare_11v["base_rmse"]
)

bin_compare_11v["v_raw_rmse_delta"] = (
    bin_compare_11v["v_raw_rmse"] - bin_compare_11v["base_rmse"]
)

bin_compare_11v["v_robust_rmse_delta"] = (
    bin_compare_11v["v_robust_rmse"] - bin_compare_11v["base_rmse"]
)

display(bin_compare_11v.sort_values("base_rmse", ascending=False))


# ==================================================
# PART 9 - Save
# ==================================================

print("=" * 100)
print("Save 11V submissions")

sub_11v_raw = save_submission_11v(
    best_raw_11v["test"],
    "submission_11v_raw_best.csv",
)

sub_11v_robust = save_submission_11v(
    best_robust_11v["test"],
    "submission_11v_robust_best.csv",
)

sub_11u_robust_ref_from_11v = save_submission_11v(
    best_robust_11u["test"],
    "submission_11u_robust_reference_from_11v.csv",
)

sub_11o_ref_from_11v = save_submission_11v(
    base_test_11v,
    "submission_11o_reference_from_11v.csv",
)


# ==================================================
# PART 10 - Decision guide
# ==================================================

print("=" * 100)
print("11V Decision Guide")

print(f"11O adaptive OOF       : {base_score_11v:.6f}")
print(f"11U raw best OOF       : {best_raw_11u['rmse']:.6f}")
print(f"11U robust best OOF    : {best_robust_11u['rmse']:.6f}")
print(f"11V raw best OOF       : {best_raw_11v['rmse']:.6f}")
print(f"11V robust best OOF    : {best_robust_11v['rmse']:.6f}")

print(f"11V raw delta vs 11O   : {best_raw_11v['rmse'] - BEST_11O_OOF:.6f}")
print(f"11V robust delta vs 11O: {best_robust_11v['rmse'] - BEST_11O_OOF:.6f}")

print(f"11V raw delta vs 11U robust   : {best_raw_11v['rmse'] - best_robust_11u['rmse']:.6f}")
print(f"11V robust delta vs 11U robust: {best_robust_11v['rmse'] - best_robust_11u['rmse']:.6f}")

if best_raw_11v["rmse"] < best_robust_11u["rmse"] - 0.001:
    print("Raw best decision: GOOD improvement over 11U. Submit if bin report is safe.")
elif best_raw_11v["rmse"] < best_robust_11u["rmse"]:
    print("Raw best decision: SMALL improvement over 11U. Submit only as probe.")
else:
    print("Raw best decision: did not beat 11U robust. Do not force.")

if best_robust_11v["rmse"] < best_robust_11u["rmse"] - 0.001:
    print("Robust best decision: GOOD improvement over 11U. Prefer this if bins are safer.")
elif best_robust_11v["rmse"] < best_robust_11u["rmse"]:
    print("Robust best decision: SMALL improvement over 11U. Submit only as probe.")
else:
    print("Robust best decision: did not beat 11U robust. Keep 11U robust as primary.")

print("=" * 100)
print("Please screenshot:")
print("1. candidate_df_11v.head(40)")
print("2. candidate_df_11v_robust.head(40)")
print("3. raw_report_11v")
print("4. robust_report_11v")
print("5. bin_compare_11v")

In [ ]:
# ==================================================
# DOWNLOAD 11V ROBUST BEST
# ==================================================

from google.colab import files
import os

filename = "submission_11v_robust_best.csv"

if os.path.exists(filename):
    print(f"Downloading: {filename}")
    files.download(filename)
else:
    raise FileNotFoundError(
        f"{filename} belum ada. Pastikan SECTION 11V sudah selesai dan file sudah tersimpan."
    )

In [ ]:
# ==================================================
# SECTION 11W - Micro Surgery After 11V
# ==================================================
# Goal:
# 1. Local refine around 11V best benefit-gated correction.
# 2. Add tiny extreme-tail q90 overlay only on very selective rows.
# 3. Keep medium/high leakage controlled.
# 4. Save 11W raw and robust submissions.
#
# Paste AFTER SECTION 11V.
# Skip 11P and 11Q.
#
# Main rule:
# 11W only replaces 11V if it beats 11V with safe bins.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")


# ==================================================
# PART 0 - Setup
# ==================================================

print("=" * 100)
print("SECTION 11W - Micro Surgery After 11V")

BEST_11O_OOF = 11.26696803195285

required_vars_11w = [
    "train_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "best_raw_11v",
    "best_robust_11v",
]

missing_11w = [v for v in required_vars_11w if v not in globals()]
if len(missing_11w) > 0:
    raise NameError(f"Missing required variables for 11W: {missing_11w}")

base_oof_11w = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11w = np.clip(np.asarray(final_adaptive_test), 0, None)

y_arr_11w = y.values
n_train_11w = len(y_arr_11w)
n_test_11w = sample_submission.shape[0]

base_score_11w = rmse(y, base_oof_11w)

prev_11v_raw_score = best_raw_11v["rmse"]
prev_11v_robust_score = best_robust_11v["rmse"]
prev_11v_best_score = min(prev_11v_raw_score, prev_11v_robust_score)

print("Base 11O adaptive OOF:", base_score_11w)
print("Previous 11V raw OOF:", prev_11v_raw_score)
print("Previous 11V robust OOF:", prev_11v_robust_score)
print("Previous 11V best OOF:", prev_11v_best_score)


# ==================================================
# Fast bin setup
# ==================================================

target_bin_11w = pd.qcut(
    y_arr_11w,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels_11w = ["very_low", "low", "medium", "high", "very_high"]

bin_masks_11w = {
    b: (target_bin_11w == b)
    for b in bin_labels_11w
}


def fast_rmse_11w(y_true, pred):
    pred = np.asarray(pred)
    return float(np.sqrt(np.mean((pred - y_true) ** 2)))


def rank_against_train_11w(train_values, values):
    train_sorted = np.sort(np.asarray(train_values))
    values = np.asarray(values)
    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def save_submission_11w(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11w(pred, label):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = pred
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", fast_rmse_11w(y_arr_11w, pred))
    display(report.sort_values("rmse", ascending=False))

    return report


def calc_metrics_11w(pred, correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    err = pred - y_arr_11w
    se = err ** 2

    if correction is None:
        correction = pred - base_oof_11w

    correction = np.clip(np.asarray(correction), 0, None)

    out = {
        "rmse": fast_rmse_11w(y_arr_11w, pred),
    }

    for b in bin_labels_11w:
        m = bin_masks_11w[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    changed = np.abs(pred - base_oof_11w) > 1e-12

    base_se = (base_oof_11w - y_arr_11w) ** 2
    new_se = (pred - y_arr_11w) ** 2
    benefit = base_se - new_se

    if changed.sum() > 0:
        benefit_positive_rate = float(np.mean(benefit[changed] > 0))
        avg_benefit_changed = float(np.mean(benefit[changed]))
    else:
        benefit_positive_rate = 0.0
        avg_benefit_changed = 0.0

    out.update({
        "corrected_count": int((correction > 1e-12).sum()),
        "corrected_rate": float(np.mean(correction > 1e-12)),
        "mean_correction": float(np.mean(correction)),
        "max_correction": float(np.max(correction)),
        "changed_count": int(changed.sum()),
        "benefit_positive_rate": benefit_positive_rate,
        "avg_benefit_changed": avg_benefit_changed,
        "delta_vs_11O": fast_rmse_11w(y_arr_11w, pred) - BEST_11O_OOF,
        "delta_vs_11V_best": fast_rmse_11w(y_arr_11w, pred) - prev_11v_best_score,
    })

    return out


base_metrics_11w = calc_metrics_11w(
    base_oof_11w,
    correction=np.zeros_like(base_oof_11w),
)

prev_11v_raw_metrics_11w = calc_metrics_11w(
    best_raw_11v["oof"],
    best_raw_11v["correction_oof"],
)

prev_11v_robust_metrics_11w = calc_metrics_11w(
    best_robust_11v["oof"],
    best_robust_11v["correction_oof"],
)

prev_best_metrics_11w = (
    prev_11v_raw_metrics_11w
    if prev_11v_raw_metrics_11w["rmse"] <= prev_11v_robust_metrics_11w["rmse"]
    else prev_11v_robust_metrics_11w
)

base_vh_rmse_11w = base_metrics_11w["very_high_rmse"]
base_high_rmse_11w = base_metrics_11w["high_rmse"]
base_medium_rmse_11w = base_metrics_11w["medium_rmse"]
base_low_rmse_11w = base_metrics_11w["low_rmse"]
base_vlow_rmse_11w = base_metrics_11w["very_low_rmse"]

prev_vh_rmse_11w = prev_best_metrics_11w["very_high_rmse"]
prev_high_rmse_11w = prev_best_metrics_11w["high_rmse"]
prev_medium_rmse_11w = prev_best_metrics_11w["medium_rmse"]
prev_low_rmse_11w = prev_best_metrics_11w["low_rmse"]
prev_vlow_rmse_11w = prev_best_metrics_11w["very_low_rmse"]

print("=" * 100)
print("Previous 11V best metrics snapshot")
display(pd.DataFrame([prev_best_metrics_11w]))


def robust_penalty_11w(metrics):
    penalty = 0.0

    cc = metrics["corrected_count"]

    # 11W target: keep correction moderately selective.
    if cc > 0 and cc < 120:
        penalty += 0.00035

    if cc > 240:
        penalty += 0.00025

    if cc > 300:
        penalty += 0.00055

    # Very-high should not collapse.
    if metrics["very_high_rmse"] > 19.445:
        penalty += 0.00045

    if metrics["very_high_rmse"] > prev_vh_rmse_11w + 0.015:
        penalty += 0.00035

    # Medium/high leakage guard.
    if metrics["medium_rmse"] > 9.230:
        penalty += 0.00035

    if metrics["medium_rmse"] > base_medium_rmse_11w + 0.035:
        penalty += 0.00025

    if metrics["high_rmse"] > 8.945:
        penalty += 0.00035

    if metrics["high_rmse"] > base_high_rmse_11w + 0.045:
        penalty += 0.00025

    if metrics["low_rmse"] > base_low_rmse_11w + 0.025:
        penalty += 0.00020

    if metrics["very_low_rmse"] > base_vlow_rmse_11w + 0.025:
        penalty += 0.00020

    # Benefit quality.
    if cc > 0:
        if metrics["avg_benefit_changed"] < 12:
            penalty += 0.00030

        if metrics["avg_benefit_changed"] < 8:
            penalty += 0.00050

        if metrics["benefit_positive_rate"] < 0.42:
            penalty += 0.00030

    return penalty


candidate_rows_11w = []

best_raw_11w = {
    "key": "prev_11v_raw",
    "method": "previous_11V",
    "rmse": best_raw_11v["rmse"],
    "selection_score": best_raw_11v["rmse"],
    "oof": np.asarray(best_raw_11v["oof"]).copy(),
    "test": np.asarray(best_raw_11v["test"]).copy(),
    "correction_oof": np.asarray(best_raw_11v["correction_oof"]).copy(),
    "correction_test": np.asarray(best_raw_11v["correction_test"]).copy(),
}

best_robust_11w = {
    "key": "prev_11v_robust",
    "method": "previous_11V",
    "rmse": best_robust_11v["rmse"],
    "selection_score": best_robust_11v["rmse"],
    "oof": np.asarray(best_robust_11v["oof"]).copy(),
    "test": np.asarray(best_robust_11v["test"]).copy(),
    "correction_oof": np.asarray(best_robust_11v["correction_oof"]).copy(),
    "correction_test": np.asarray(best_robust_11v["correction_test"]).copy(),
}


def evaluate_candidate_11w(
    key,
    method,
    pred_oof,
    pred_test,
    correction_oof,
    correction_test=None,
):
    global best_raw_11w, best_robust_11w

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)
    correction_oof = np.clip(np.asarray(correction_oof), 0, None)

    if correction_test is None:
        correction_test = np.clip(pred_test - base_test_11w, 0, None)
    else:
        correction_test = np.clip(np.asarray(correction_test), 0, None)

    metrics = calc_metrics_11w(pred_oof, correction_oof)
    metrics["robust_penalty"] = robust_penalty_11w(metrics)
    metrics["selection_score"] = metrics["rmse"] + metrics["robust_penalty"]

    row = {
        "key": key,
        "method": method,
        **metrics,
    }

    candidate_rows_11w.append(row)

    if metrics["rmse"] < best_raw_11w["rmse"]:
        best_raw_11w = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }

    if metrics["selection_score"] < best_robust_11w["selection_score"]:
        best_robust_11w = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "correction_oof": correction_oof.copy(),
            "correction_test": correction_test.copy(),
        }


# Add references
evaluate_candidate_11w(
    key="base_11O_adaptive",
    method="base",
    pred_oof=base_oof_11w,
    pred_test=base_test_11w,
    correction_oof=np.zeros_like(base_oof_11w),
    correction_test=np.zeros_like(base_test_11w),
)

evaluate_candidate_11w(
    key="prev_11v_raw",
    method="previous_11V",
    pred_oof=best_raw_11v["oof"],
    pred_test=best_raw_11v["test"],
    correction_oof=best_raw_11v["correction_oof"],
    correction_test=best_raw_11v["correction_test"],
)

evaluate_candidate_11w(
    key="prev_11v_robust",
    method="previous_11V",
    pred_oof=best_robust_11v["oof"],
    pred_test=best_robust_11v["test"],
    correction_oof=best_robust_11v["correction_oof"],
    correction_test=best_robust_11v["correction_test"],
)


# ==================================================
# PART 1 - Validate required 11V internals or rebuild fallback
# ==================================================

print("=" * 100)
print("PART 1 - Prepare 11W inputs")

if "source_bank_11v" not in globals():
    print("source_bank_11v not found. Rebuilding minimal source bank from 11V best objects.")

    source_bank_11v = {
        "11u_raw": {
            "corr_oof": np.clip(np.asarray(best_raw_11v["correction_oof"]), 0, None),
            "corr_test": np.clip(np.asarray(best_raw_11v["correction_test"]), 0, None),
        },
        "11u_robust": {
            "corr_oof": np.clip(np.asarray(best_robust_11v["correction_oof"]), 0, None),
            "corr_test": np.clip(np.asarray(best_robust_11v["correction_test"]), 0, None),
        },
    }

if "gate_outputs_11v" not in globals():
    raise NameError(
        "gate_outputs_11v not found. Run SECTION 11V first because 11W needs its gate probabilities."
    )

print("Available 11V source bank:", list(source_bank_11v.keys()))
print("Available 11V gate outputs:", list(gate_outputs_11v.keys()))


# ==================================================
# PART 2 - Local refinement around 11V best
# ==================================================

print("=" * 100)
print("PART 2 - Local refinement around 11V best")

part_a_start_11w = len(candidate_rows_11w)

source_names_11w = []

for s in ["11u_raw", "11u_robust"]:
    if s in source_bank_11v and s in gate_outputs_11v:
        source_names_11w.append(s)

if len(source_names_11w) == 0:
    raise ValueError("No valid 11V sources found for local refinement.")

prob_types_11w = ["max", "avg", "hgb", "logit"]

source_weight_grid_11w = [0.98, 1.00, 1.03, 1.05, 1.08, 1.10, 1.15]
high_keep_frac_grid_11w = [0.38, 0.42, 0.45, 0.48, 0.52]
mid_keep_frac_grid_11w = [0.60, 0.65, 0.70, 0.75, 0.80]
high_weight_grid_11w = [0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
mid_weight_grid_11w = [0.20, 0.25, 0.30, 0.35, 0.40]
low_weight_grid_11w = [0.00, 0.05]
cap_grid_11w = [3.25, 3.50, 3.75, 4.00]

for source_name in source_names_11w:
    corr_oof_base = np.clip(np.asarray(source_bank_11v[source_name]["corr_oof"]), 0, None)
    corr_test_base = np.clip(np.asarray(source_bank_11v[source_name]["corr_test"]), 0, None)

    source_mask_oof = corr_oof_base > 1e-12
    source_mask_test = corr_test_base > 1e-12

    if source_mask_oof.sum() < 20:
        continue

    print("=" * 100)
    print("Local source:", source_name)
    print("Source corrected OOF:", int(source_mask_oof.sum()))
    print("Source corrected TEST:", int(source_mask_test.sum()))

    for prob_type in prob_types_11w:
        prob_key_oof = f"{prob_type}_oof"
        prob_key_test = f"{prob_type}_test"

        if prob_key_oof not in gate_outputs_11v[source_name]:
            continue

        prob_oof = np.asarray(gate_outputs_11v[source_name][prob_key_oof])
        prob_test = np.asarray(gate_outputs_11v[source_name][prob_key_test])

        valid_prob_oof = prob_oof[source_mask_oof]
        valid_prob_test = prob_test[source_mask_test]

        if len(valid_prob_oof) == 0 or np.max(valid_prob_oof) <= 0:
            continue

        if len(valid_prob_test) == 0 or np.max(valid_prob_test) <= 0:
            continue

        for high_keep_frac in high_keep_frac_grid_11w:
            high_thr_oof = np.quantile(valid_prob_oof, 1.0 - high_keep_frac)
            high_thr_test = np.quantile(valid_prob_test, 1.0 - high_keep_frac)

            high_mask_oof = source_mask_oof & (prob_oof >= high_thr_oof)
            high_mask_test = source_mask_test & (prob_test >= high_thr_test)

            for mid_keep_frac in mid_keep_frac_grid_11w:
                if mid_keep_frac < high_keep_frac:
                    continue

                mid_thr_oof = np.quantile(valid_prob_oof, 1.0 - mid_keep_frac)
                mid_thr_test = np.quantile(valid_prob_test, 1.0 - mid_keep_frac)

                mid_mask_oof = source_mask_oof & (prob_oof >= mid_thr_oof)
                mid_mask_test = source_mask_test & (prob_test >= mid_thr_test)

                for source_weight in source_weight_grid_11w:
                    weighted_corr_oof = source_weight * corr_oof_base
                    weighted_corr_test = source_weight * corr_test_base

                    for high_weight in high_weight_grid_11w:
                        for mid_weight in mid_weight_grid_11w:
                            for low_weight in low_weight_grid_11w:

                                shrink_oof = np.full(n_train_11w, low_weight)
                                shrink_test = np.full(n_test_11w, low_weight)

                                shrink_oof[mid_mask_oof] = mid_weight
                                shrink_test[mid_mask_test] = mid_weight

                                shrink_oof[high_mask_oof] = high_weight
                                shrink_test[high_mask_test] = high_weight

                                raw_corr_oof = weighted_corr_oof * shrink_oof
                                raw_corr_test = weighted_corr_test * shrink_test

                                for cap in cap_grid_11w:
                                    corr_oof = np.clip(raw_corr_oof, 0, cap)
                                    corr_test = np.clip(raw_corr_test, 0, cap)

                                    pred_oof = np.clip(base_oof_11w + corr_oof, 0, None)
                                    pred_test = np.clip(base_test_11w + corr_test, 0, None)

                                    key = (
                                        f"W_local_{source_name}_{prob_type}"
                                        f"_sw{source_weight}"
                                        f"_hk{high_keep_frac}"
                                        f"_mk{mid_keep_frac}"
                                        f"_hw{high_weight}"
                                        f"_mw{mid_weight}"
                                        f"_lw{low_weight}"
                                        f"_cap{cap}"
                                    )

                                    evaluate_candidate_11w(
                                        key=key,
                                        method="W_local_refine",
                                        pred_oof=pred_oof,
                                        pred_test=pred_test,
                                        correction_oof=corr_oof,
                                        correction_test=corr_test,
                                    )

                                    del corr_oof, corr_test, pred_oof, pred_test

                    del weighted_corr_oof, weighted_corr_test

    gc.collect()

part_a_df_11w = pd.DataFrame(candidate_rows_11w[part_a_start_11w:]).sort_values("rmse")

print("=" * 100)
print("Top PART A local refinement results")
display(part_a_df_11w.head(40))

print("Best after PART A raw:", best_raw_11w["key"], best_raw_11w["rmse"])
print("Best after PART A robust:", best_robust_11w["key"], best_robust_11w["rmse"], best_robust_11w["selection_score"])


# ==================================================
# PART 3 - Build q90 extreme-tail overlay signal
# ==================================================

print("=" * 100)
print("PART 3 - Build tiny q90 extreme-tail overlay")

base_rank_oof_11w = rank_against_train_11w(base_oof_11w, base_oof_11w)
base_rank_test_11w = rank_against_train_11w(base_oof_11w, base_test_11w)

# Main delta source for overlay
delta_11t_oof_11w = np.zeros(n_train_11w)
delta_11t_test_11w = np.zeros(n_test_11w)

if "lgb_11t_outputs" in globals() and "lgb11t_missedsoft31" in lgb_11t_outputs:
    delta_11t_oof_11w = np.maximum(
        np.asarray(lgb_11t_outputs["lgb11t_missedsoft31"]["oof"]) - base_oof_11w,
        0,
    )
    delta_11t_test_11w = np.maximum(
        np.asarray(lgb_11t_outputs["lgb11t_missedsoft31"]["test"]) - base_test_11w,
        0,
    )
elif "best_11t" in globals():
    delta_11t_oof_11w = np.maximum(np.asarray(best_11t["oof"]) - base_oof_11w, 0)
    delta_11t_test_11w = np.maximum(np.asarray(best_11t["test"]) - base_test_11w, 0)
else:
    delta_11t_oof_11w = np.maximum(np.asarray(best_raw_11v["correction_oof"]), 0)
    delta_11t_test_11w = np.maximum(np.asarray(best_raw_11v["correction_test"]), 0)


def get_overlay_risk_11w():
    # Prefer q90_res5 because it was the precision-oriented missed-tail detector.
    if "missed_tail_outputs_11r" in globals() and "q90_res5" in missed_tail_outputs_11r:
        obj = missed_tail_outputs_11r["q90_res5"]

        gate_oof = np.clip(np.asarray(obj["gate_oof"]), 0, None)
        gate_test = np.clip(np.asarray(obj["gate_test"]), 0, None)

        resid_oof = np.clip(np.asarray(obj["resid_oof"]), 0, None)
        resid_test = np.clip(np.asarray(obj["resid_test"]), 0, None)

        raw_oof = gate_oof * resid_oof
        raw_test = gate_test * resid_test

        if np.max(raw_oof) > 0:
            rank_oof = rank_against_train_11w(raw_oof, raw_oof)
            rank_test = rank_against_train_11w(raw_oof, raw_test)
        else:
            rank_oof = np.zeros_like(raw_oof)
            rank_test = np.zeros_like(raw_test)

        return rank_oof, rank_test, raw_oof, raw_test

    if "risk_features_train_11v" in globals() and "risk_q90_res5" in risk_features_train_11v:
        rank_oof = np.asarray(risk_features_train_11v["risk_q90_res5"])
        rank_test = np.asarray(risk_features_test_11v["risk_q90_res5"])

        return rank_oof, rank_test, rank_oof, rank_test

    print("q90 risk unavailable. Fallback to base rank.")
    return base_rank_oof_11w, base_rank_test_11w, base_rank_oof_11w, base_rank_test_11w


risk_q90_oof_11w, risk_q90_test_11w, raw_q90_oof_11w, raw_q90_test_11w = get_overlay_risk_11w()

# Overlay amount is conservative: combine LGB delta and q90 residual signal, then clip.
overlay_amount_oof_11w = np.clip(
    np.maximum(delta_11t_oof_11w, 0.50 * raw_q90_oof_11w),
    0,
    8.0,
)

overlay_amount_test_11w = np.clip(
    np.maximum(delta_11t_test_11w, 0.50 * raw_q90_test_11w),
    0,
    8.0,
)

overlay_score_oof_11w = (
    risk_q90_oof_11w
    * (overlay_amount_oof_11w + 1e-9)
    * (0.50 + base_rank_oof_11w)
)

overlay_score_test_11w = (
    risk_q90_test_11w
    * (overlay_amount_test_11w + 1e-9)
    * (0.50 + base_rank_test_11w)
)

print("Overlay score OOF max:", float(np.max(overlay_score_oof_11w)))
print("Overlay amount OOF mean/max:", float(np.mean(overlay_amount_oof_11w)), float(np.max(overlay_amount_oof_11w)))


# ==================================================
# PART 4 - Tiny extreme-tail overlay
# ==================================================

print("=" * 100)
print("PART 4 - Tiny extreme-tail overlay")

part_b_start_11w = len(candidate_rows_11w)

# Snapshot seeds before overlay so loop does not chase itself.
overlay_seed_bank_11w = {
    "seed_prev_11v_raw": {
        "corr_oof": np.clip(np.asarray(best_raw_11v["correction_oof"]), 0, None),
        "corr_test": np.clip(np.asarray(best_raw_11v["correction_test"]), 0, None),
    },
    "seed_prev_11v_robust": {
        "corr_oof": np.clip(np.asarray(best_robust_11v["correction_oof"]), 0, None),
        "corr_test": np.clip(np.asarray(best_robust_11v["correction_test"]), 0, None),
    },
    "seed_best_after_local_raw": {
        "corr_oof": np.clip(np.asarray(best_raw_11w["correction_oof"]), 0, None),
        "corr_test": np.clip(np.asarray(best_raw_11w["correction_test"]), 0, None),
    },
    "seed_best_after_local_robust": {
        "corr_oof": np.clip(np.asarray(best_robust_11w["correction_oof"]), 0, None),
        "corr_test": np.clip(np.asarray(best_robust_11w["correction_test"]), 0, None),
    },
}

min_base_rank_grid_11w = [0.78, 0.82, 0.85, 0.88, 0.90]
top_frac_grid_overlay_11w = [0.003, 0.005, 0.0075, 0.010, 0.015, 0.020, 0.030]
overlay_strength_grid_11w = [0.025, 0.050, 0.075, 0.100, 0.125, 0.150]
overlay_cap_grid_11w = [0.25, 0.50, 0.75, 1.00]
existing_corr_max_grid_11w = [1.50, 2.00, 2.50, 3.50, 99.0]
total_cap_grid_overlay_11w = [3.50, 3.75, 4.00]

for seed_name, seed_obj in overlay_seed_bank_11w.items():
    seed_corr_oof = np.clip(np.asarray(seed_obj["corr_oof"]), 0, None)
    seed_corr_test = np.clip(np.asarray(seed_obj["corr_test"]), 0, None)

    print("=" * 100)
    print("Overlay seed:", seed_name)
    print("Seed RMSE:", fast_rmse_11w(y_arr_11w, base_oof_11w + seed_corr_oof))
    print("Seed corrected:", int((seed_corr_oof > 1e-12).sum()))

    for min_base_rank in min_base_rank_grid_11w:
        base_eligible_oof = base_rank_oof_11w >= min_base_rank
        base_eligible_test = base_rank_test_11w >= min_base_rank

        for existing_corr_max in existing_corr_max_grid_11w:
            eligible_oof = (
                base_eligible_oof
                & (seed_corr_oof <= existing_corr_max)
                & (overlay_score_oof_11w > 0)
                & (overlay_amount_oof_11w > 0)
            )

            eligible_test = (
                base_eligible_test
                & (seed_corr_test <= existing_corr_max)
                & (overlay_score_test_11w > 0)
                & (overlay_amount_test_11w > 0)
            )

            if eligible_oof.sum() < 10 or eligible_test.sum() < 1:
                continue

            score_eligible_oof = overlay_score_oof_11w[eligible_oof]
            score_eligible_test = overlay_score_test_11w[eligible_test]

            if np.max(score_eligible_oof) <= 0 or np.max(score_eligible_test) <= 0:
                continue

            for top_frac in top_frac_grid_overlay_11w:
                thr_oof = np.quantile(score_eligible_oof, 1.0 - top_frac)
                thr_test = np.quantile(score_eligible_test, 1.0 - top_frac)

                top_mask_oof = eligible_oof & (overlay_score_oof_11w >= thr_oof)
                top_mask_test = eligible_test & (overlay_score_test_11w >= thr_test)

                if top_mask_oof.sum() < 3:
                    continue

                for strength in overlay_strength_grid_11w:
                    raw_overlay_oof = strength * overlay_amount_oof_11w
                    raw_overlay_test = strength * overlay_amount_test_11w

                    for overlay_cap in overlay_cap_grid_11w:
                        overlay_oof = np.clip(raw_overlay_oof, 0, overlay_cap)
                        overlay_test = np.clip(raw_overlay_test, 0, overlay_cap)

                        overlay_oof = np.where(top_mask_oof, overlay_oof, 0.0)
                        overlay_test = np.where(top_mask_test, overlay_test, 0.0)

                        for total_cap in total_cap_grid_overlay_11w:
                            final_corr_oof = np.clip(seed_corr_oof + overlay_oof, 0, total_cap)
                            final_corr_test = np.clip(seed_corr_test + overlay_test, 0, total_cap)

                            pred_oof = np.clip(base_oof_11w + final_corr_oof, 0, None)
                            pred_test = np.clip(base_test_11w + final_corr_test, 0, None)

                            key = (
                                f"W_overlay_{seed_name}"
                                f"_minrank{min_base_rank}"
                                f"_existmax{existing_corr_max}"
                                f"_top{top_frac}"
                                f"_s{strength}"
                                f"_ocap{overlay_cap}"
                                f"_tcap{total_cap}"
                            )

                            evaluate_candidate_11w(
                                key=key,
                                method="W_tiny_q90_overlay",
                                pred_oof=pred_oof,
                                pred_test=pred_test,
                                correction_oof=final_corr_oof,
                                correction_test=final_corr_test,
                            )

                            del final_corr_oof, final_corr_test, pred_oof, pred_test

                    del overlay_oof, overlay_test

    gc.collect()

part_b_df_11w = pd.DataFrame(candidate_rows_11w[part_b_start_11w:]).sort_values("rmse")

print("=" * 100)
print("Top PART B overlay results")
display(part_b_df_11w.head(40))

print("Best after PART B raw:", best_raw_11w["key"], best_raw_11w["rmse"])
print("Best after PART B robust:", best_robust_11w["key"], best_robust_11w["rmse"], best_robust_11w["selection_score"])


# ==================================================
# PART 5 - Final selection
# ==================================================

print("=" * 100)
print("PART 5 - Final selection")

candidate_df_11w = pd.DataFrame(candidate_rows_11w)
candidate_df_11w = candidate_df_11w.sort_values("rmse").reset_index(drop=True)

candidate_df_11w_robust = candidate_df_11w.sort_values("selection_score").reset_index(drop=True)

print("Top 11W candidates by raw RMSE")
display(candidate_df_11w.head(50))

print("Top 11W candidates by robust selection score")
display(candidate_df_11w_robust.head(50))

print("=" * 100)
print("Selected 11W candidates")

print("Raw best key     :", best_raw_11w["key"])
print("Raw best method  :", best_raw_11w["method"])
print("Raw best RMSE    :", best_raw_11w["rmse"])
print("Raw selection    :", best_raw_11w["selection_score"])
print("Raw delta vs 11O :", best_raw_11w["rmse"] - BEST_11O_OOF)
print("Raw delta vs 11V :", best_raw_11w["rmse"] - prev_11v_best_score)

print("-" * 100)

print("Robust best key     :", best_robust_11w["key"])
print("Robust best method  :", best_robust_11w["method"])
print("Robust best RMSE    :", best_robust_11w["rmse"])
print("Robust selection    :", best_robust_11w["selection_score"])
print("Robust delta vs 11O :", best_robust_11w["rmse"] - BEST_11O_OOF)
print("Robust delta vs 11V :", best_robust_11w["rmse"] - prev_11v_best_score)

raw_report_11w = target_bin_report_11w(
    best_raw_11w["oof"],
    f"11W raw best: {best_raw_11w['key']}",
)

robust_report_11w = target_bin_report_11w(
    best_robust_11w["oof"],
    f"11W robust best: {best_robust_11w['key']}",
)


# ==================================================
# PART 6 - Bin comparison
# ==================================================

print("=" * 100)
print("Bin comparison: 11O vs 11V robust vs 11W raw vs 11W robust")

compare_11w = train_raw[[ID_COL, TARGET]].copy()

compare_11w["base_pred"] = base_oof_11w
compare_11w["v_robust_pred"] = np.asarray(best_robust_11v["oof"])
compare_11w["w_raw_pred"] = best_raw_11w["oof"]
compare_11w["w_robust_pred"] = best_robust_11w["oof"]

for col in ["base_pred", "v_robust_pred", "w_raw_pred", "w_robust_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")

    compare_11w[err_col] = compare_11w[col] - compare_11w[TARGET]
    compare_11w[se_col] = compare_11w[err_col] ** 2

compare_11w["target_bin_5"] = pd.qcut(
    compare_11w[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11w = (
    compare_11w.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        v_robust_pred_mean=("v_robust_pred", "mean"),
        w_raw_pred_mean=("w_raw_pred", "mean"),
        w_robust_pred_mean=("w_robust_pred", "mean"),

        base_bias=("base_error", "mean"),
        v_robust_bias=("v_robust_error", "mean"),
        w_raw_bias=("w_raw_error", "mean"),
        w_robust_bias=("w_robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        v_robust_rmse=("v_robust_se", lambda x: np.sqrt(np.mean(x))),
        w_raw_rmse=("w_raw_se", lambda x: np.sqrt(np.mean(x))),
        w_robust_rmse=("w_robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11w["v_robust_rmse_delta"] = (
    bin_compare_11w["v_robust_rmse"] - bin_compare_11w["base_rmse"]
)

bin_compare_11w["w_raw_rmse_delta"] = (
    bin_compare_11w["w_raw_rmse"] - bin_compare_11w["base_rmse"]
)

bin_compare_11w["w_robust_rmse_delta"] = (
    bin_compare_11w["w_robust_rmse"] - bin_compare_11w["base_rmse"]
)

bin_compare_11w["w_raw_minus_v_robust_rmse"] = (
    bin_compare_11w["w_raw_rmse"] - bin_compare_11w["v_robust_rmse"]
)

bin_compare_11w["w_robust_minus_v_robust_rmse"] = (
    bin_compare_11w["w_robust_rmse"] - bin_compare_11w["v_robust_rmse"]
)

display(bin_compare_11w.sort_values("base_rmse", ascending=False))


# ==================================================
# PART 7 - Save submissions
# ==================================================

print("=" * 100)
print("Save 11W submissions")

sub_11w_raw = save_submission_11w(
    best_raw_11w["test"],
    "submission_11w_raw_best.csv",
)

sub_11w_robust = save_submission_11w(
    best_robust_11w["test"],
    "submission_11w_robust_best.csv",
)

sub_11v_robust_ref_from_11w = save_submission_11w(
    best_robust_11v["test"],
    "submission_11v_robust_reference_from_11w.csv",
)

sub_11o_ref_from_11w = save_submission_11w(
    base_test_11w,
    "submission_11o_reference_from_11w.csv",
)


# ==================================================
# PART 8 - Decision guide
# ==================================================

print("=" * 100)
print("11W Decision Guide")

print(f"11O adaptive OOF       : {base_score_11w:.6f}")
print(f"11V raw best OOF       : {prev_11v_raw_score:.6f}")
print(f"11V robust best OOF    : {prev_11v_robust_score:.6f}")
print(f"11W raw best OOF       : {best_raw_11w['rmse']:.6f}")
print(f"11W robust best OOF    : {best_robust_11w['rmse']:.6f}")

print(f"11W raw delta vs 11O   : {best_raw_11w['rmse'] - BEST_11O_OOF:.6f}")
print(f"11W robust delta vs 11O: {best_robust_11w['rmse'] - BEST_11O_OOF:.6f}")

print(f"11W raw delta vs 11V   : {best_raw_11w['rmse'] - prev_11v_best_score:.6f}")
print(f"11W robust delta vs 11V: {best_robust_11w['rmse'] - prev_11v_best_score:.6f}")

if best_raw_11w["rmse"] < prev_11v_best_score - 0.001:
    print("Raw best decision: GOOD improvement over 11V. Submit if bin report is safe.")
elif best_raw_11w["rmse"] < prev_11v_best_score:
    print("Raw best decision: SMALL improvement over 11V. Submit only as probe.")
else:
    print("Raw best decision: did not beat 11V. Do not force.")

if best_robust_11w["rmse"] < prev_11v_best_score - 0.001:
    print("Robust best decision: GOOD improvement over 11V. Prefer this if bins are safer.")
elif best_robust_11w["rmse"] < prev_11v_best_score:
    print("Robust best decision: SMALL improvement over 11V. Submit only as probe.")
else:
    print("Robust best decision: did not beat 11V. Keep 11V robust as primary.")

print("=" * 100)
print("Please screenshot:")
print("1. part_a_df_11w.head(40)")
print("2. part_b_df_11w.head(40)")
print("3. candidate_df_11w.head(50)")
print("4. candidate_df_11w_robust.head(50)")
print("5. raw_report_11w")
print("6. robust_report_11w")
print("7. bin_compare_11w")

In [ ]:
# ==================================================
# DOWNLOAD 11W RAW + ROBUST BEST
# ==================================================

from google.colab import files
import os

filenames = [
    "submission_11w_robust_best.csv",
    "submission_11w_raw_best.csv",
]

for filename in filenames:
    if os.path.exists(filename):
        print(f"Downloading: {filename}")
        files.download(filename)
    else:
        print(f"Not found: {filename}")

In [ ]:
# ==================================================
# SECTION 11X - Signed Dual-Tail & Local Residual Memory
# ==================================================
# Goal:
# 1. Keep 11W as positive high-tail anchor.
# 2. Add negative low-tail correction for overpredicted very_low / low samples.
# 3. Add signed hierarchical group residual correction.
# 4. Optional KNN local residual memory.
# 5. Blend components and save 11X raw / robust.
#
# Paste AFTER SECTION 11W.
# Skip 11P and 11Q.
#
# Key difference from 11T-11W:
# 11X is not just "more positive tail correction".
# It can move predictions up OR down.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


# ==================================================
# PART 0 - Setup
# ==================================================

print("=" * 100)
print("SECTION 11X - Signed Dual-Tail & Local Residual Memory")

BEST_11O_OOF = 11.26696803195285

# Set False if runtime is too long.
RUN_KNN_11X = True

required_vars_11x = [
    "train_raw",
    "test_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "best_raw_11w",
    "best_robust_11w",
]

missing_11x = [v for v in required_vars_11x if v not in globals()]
if len(missing_11x) > 0:
    raise NameError(f"Missing required variables for 11X: {missing_11x}")

base_oof_11x = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11x = np.clip(np.asarray(final_adaptive_test), 0, None)

y_arr_11x = y.values
n_train_11x = len(y_arr_11x)
n_test_11x = sample_submission.shape[0]

base_score_11x = rmse(y, base_oof_11x)

# Use the better 11W object as seed.
if best_raw_11w["rmse"] <= best_robust_11w["rmse"]:
    seed_11w_obj = best_raw_11w
    seed_11w_name = "11w_raw"
else:
    seed_11w_obj = best_robust_11w
    seed_11w_name = "11w_robust"

prev_11w_score = seed_11w_obj["rmse"]

# 11W correction is positive high-tail anchor.
pos_corr_oof_11x = np.clip(np.asarray(seed_11w_obj["correction_oof"]), 0, None)
pos_corr_test_11x = np.clip(np.asarray(seed_11w_obj["correction_test"]), 0, None)

seed_pred_oof_11x = np.clip(base_oof_11x + pos_corr_oof_11x, 0, None)
seed_pred_test_11x = np.clip(base_test_11x + pos_corr_test_11x, 0, None)

print("Base 11O adaptive OOF:", base_score_11x)
print("Seed 11W name:", seed_11w_name)
print("Seed 11W OOF:", prev_11w_score)
print("Seed 11W positive corrected count:", int((pos_corr_oof_11x > 1e-12).sum()))


# ==================================================
# Fast bin setup
# ==================================================

target_bin_11x = pd.qcut(
    y_arr_11x,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels_11x = ["very_low", "low", "medium", "high", "very_high"]

bin_masks_11x = {
    b: (target_bin_11x == b)
    for b in bin_labels_11x
}


def fast_rmse_11x(y_true, pred):
    pred = np.asarray(pred)
    return float(np.sqrt(np.mean((pred - y_true) ** 2)))


def rank_against_train_11x(train_values, values):
    train_values = np.asarray(train_values)
    values = np.asarray(values)

    train_sorted = np.sort(train_values[np.isfinite(train_values)])
    if len(train_sorted) == 0:
        return np.zeros_like(values, dtype=float)

    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def save_submission_11x(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11x(pred, label):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = pred
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", fast_rmse_11x(y_arr_11x, pred))
    display(report.sort_values("rmse", ascending=False))

    return report


def calc_metrics_11x(pred, signed_correction=None):
    pred = np.clip(np.asarray(pred), 0, None)

    if signed_correction is None:
        signed_correction = pred - base_oof_11x

    signed_correction = np.asarray(signed_correction)

    err = pred - y_arr_11x
    se = err ** 2

    out = {
        "rmse": fast_rmse_11x(y_arr_11x, pred),
    }

    for b in bin_labels_11x:
        m = bin_masks_11x[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    changed = np.abs(pred - base_oof_11x) > 1e-12

    base_se = (base_oof_11x - y_arr_11x) ** 2
    new_se = (pred - y_arr_11x) ** 2
    benefit = base_se - new_se

    if changed.sum() > 0:
        benefit_positive_rate = float(np.mean(benefit[changed] > 0))
        avg_benefit_changed = float(np.mean(benefit[changed]))
    else:
        benefit_positive_rate = 0.0
        avg_benefit_changed = 0.0

    out.update({
        "pos_corr_count": int((signed_correction > 1e-12).sum()),
        "neg_corr_count": int((signed_correction < -1e-12).sum()),
        "changed_count": int(changed.sum()),
        "signed_corr_mean": float(np.mean(signed_correction)),
        "signed_corr_min": float(np.min(signed_correction)),
        "signed_corr_max": float(np.max(signed_correction)),
        "max_abs_corr": float(np.max(np.abs(signed_correction))),
        "benefit_positive_rate": benefit_positive_rate,
        "avg_benefit_changed": avg_benefit_changed,
        "delta_vs_11O": fast_rmse_11x(y_arr_11x, pred) - BEST_11O_OOF,
        "delta_vs_11W": fast_rmse_11x(y_arr_11x, pred) - prev_11w_score,
    })

    return out


base_metrics_11x = calc_metrics_11x(
    base_oof_11x,
    signed_correction=np.zeros_like(base_oof_11x),
)

seed_metrics_11x = calc_metrics_11x(
    seed_pred_oof_11x,
    signed_correction=pos_corr_oof_11x,
)

base_vlow_rmse_11x = base_metrics_11x["very_low_rmse"]
base_low_rmse_11x = base_metrics_11x["low_rmse"]
base_medium_rmse_11x = base_metrics_11x["medium_rmse"]
base_high_rmse_11x = base_metrics_11x["high_rmse"]
base_vh_rmse_11x = base_metrics_11x["very_high_rmse"]

seed_vlow_rmse_11x = seed_metrics_11x["very_low_rmse"]
seed_low_rmse_11x = seed_metrics_11x["low_rmse"]
seed_medium_rmse_11x = seed_metrics_11x["medium_rmse"]
seed_high_rmse_11x = seed_metrics_11x["high_rmse"]
seed_vh_rmse_11x = seed_metrics_11x["very_high_rmse"]

print("=" * 100)
print("Seed 11W metrics")
display(pd.DataFrame([seed_metrics_11x]))


def robust_penalty_11x(metrics):
    penalty = 0.0

    # 11X must not lose the hard-earned high-tail performance too much.
    if metrics["very_high_rmse"] > seed_vh_rmse_11x + 0.020:
        penalty += 0.00050

    if metrics["very_high_rmse"] > 19.455:
        penalty += 0.00050

    # Main target: improve very_low / low, or at least don't harm them.
    if metrics["very_low_rmse"] > seed_vlow_rmse_11x + 0.005:
        penalty += 0.00045

    if metrics["low_rmse"] > seed_low_rmse_11x + 0.008:
        penalty += 0.00035

    # Keep mid/high leakage safe.
    if metrics["medium_rmse"] > seed_medium_rmse_11x + 0.020:
        penalty += 0.00035

    if metrics["medium_rmse"] > 9.240:
        penalty += 0.00040

    if metrics["high_rmse"] > seed_high_rmse_11x + 0.020:
        penalty += 0.00035

    if metrics["high_rmse"] > 8.960:
        penalty += 0.00040

    # Signed correction should not become too broad or violent.
    if metrics["neg_corr_count"] > 900:
        penalty += 0.00035

    if metrics["pos_corr_count"] > 500:
        penalty += 0.00035

    if metrics["max_abs_corr"] > 6.0:
        penalty += 0.00040

    # Benefit sanity.
    if metrics["changed_count"] > 0:
        if metrics["avg_benefit_changed"] < 6:
            penalty += 0.00040
        if metrics["benefit_positive_rate"] < 0.38:
            penalty += 0.00030

    return penalty


candidate_rows_11x = []

best_raw_11x = {
    "key": "seed_11W",
    "method": "previous_11W",
    "rmse": prev_11w_score,
    "selection_score": prev_11w_score,
    "oof": seed_pred_oof_11x.copy(),
    "test": seed_pred_test_11x.copy(),
    "signed_correction_oof": pos_corr_oof_11x.copy(),
    "signed_correction_test": pos_corr_test_11x.copy(),
}

best_robust_11x = best_raw_11x.copy()

method_best_11x = {}


def update_method_best_11x(method, key, pred_oof, pred_test, corr_oof, corr_test, metrics):
    if method not in method_best_11x or metrics["rmse"] < method_best_11x[method]["rmse"]:
        method_best_11x[method] = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "oof": np.asarray(pred_oof).copy(),
            "test": np.asarray(pred_test).copy(),
            "corr_oof": np.asarray(corr_oof).copy(),
            "corr_test": np.asarray(corr_test).copy(),
        }


def evaluate_candidate_11x(
    key,
    method,
    pred_oof,
    pred_test,
    signed_correction_oof,
    signed_correction_test,
):
    global best_raw_11x, best_robust_11x

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)

    signed_correction_oof = np.asarray(signed_correction_oof)
    signed_correction_test = np.asarray(signed_correction_test)

    metrics = calc_metrics_11x(pred_oof, signed_correction_oof)
    metrics["robust_penalty"] = robust_penalty_11x(metrics)
    metrics["selection_score"] = metrics["rmse"] + metrics["robust_penalty"]

    row = {
        "key": key,
        "method": method,
        **metrics,
    }

    candidate_rows_11x.append(row)

    update_method_best_11x(
        method=method,
        key=key,
        pred_oof=pred_oof,
        pred_test=pred_test,
        corr_oof=signed_correction_oof,
        corr_test=signed_correction_test,
        metrics=metrics,
    )

    if metrics["rmse"] < best_raw_11x["rmse"]:
        best_raw_11x = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "signed_correction_oof": signed_correction_oof.copy(),
            "signed_correction_test": signed_correction_test.copy(),
        }

    if metrics["selection_score"] < best_robust_11x["selection_score"]:
        best_robust_11x = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
            "signed_correction_oof": signed_correction_oof.copy(),
            "signed_correction_test": signed_correction_test.copy(),
        }


# Add references
evaluate_candidate_11x(
    key="base_11O_adaptive",
    method="base",
    pred_oof=base_oof_11x,
    pred_test=base_test_11x,
    signed_correction_oof=np.zeros_like(base_oof_11x),
    signed_correction_test=np.zeros_like(base_test_11x),
)

evaluate_candidate_11x(
    key="seed_11W",
    method="previous_11W",
    pred_oof=seed_pred_oof_11x,
    pred_test=seed_pred_test_11x,
    signed_correction_oof=pos_corr_oof_11x,
    signed_correction_test=pos_corr_test_11x,
)


# ==================================================
# PART 1 - Feature matrix for low-tail model
# ==================================================

print("=" * 100)
print("PART 1 - Build low-tail model features")

def clean_numeric_frame_11x(df):
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def make_numeric_source_11x():
    if "X_fe" in globals() and "X_test_fe" in globals():
        X_train_src = X_fe.copy()
        X_test_src = X_test_fe.copy()
        print("Using X_fe / X_test_fe for 11X features.")
    else:
        X_train_src = train_raw.drop(columns=[TARGET], errors="ignore").copy()
        X_test_src = test_raw.copy()
        print("Using train_raw / test_raw numeric fallback for 11X features.")

    numeric_cols = [
        c for c in X_train_src.columns
        if c in X_test_src.columns
        and pd.api.types.is_numeric_dtype(X_train_src[c])
        and pd.api.types.is_numeric_dtype(X_test_src[c])
        and c not in [TARGET, ID_COL]
    ]

    priority_keywords = [
        "particle",
        "fine",
        "coarse",
        "acidity",
        "cation",
        "CEC",
        "Ca_",
        "Mg_",
        "spectral",
        "band",
        "ratio",
        "index",
        "organic",
        "clay",
        "sand",
        "silt",
    ]

    priority_cols = [
        c for c in numeric_cols
        if any(k.lower() in c.lower() for k in priority_keywords)
    ]

    remaining_cols = [c for c in numeric_cols if c not in priority_cols]

    # Keep feature count controlled.
    selected_cols = priority_cols[:80] + remaining_cols[:60]

    X_train_num = X_train_src[selected_cols].copy()
    X_test_num = X_test_src[selected_cols].copy()

    return clean_numeric_frame_11x(X_train_num), clean_numeric_frame_11x(X_test_num), selected_cols


X_low_train_base_11x, X_low_test_base_11x, selected_num_cols_11x = make_numeric_source_11x()

base_rank_oof_11x = rank_against_train_11x(base_oof_11x, base_oof_11x)
base_rank_test_11x = rank_against_train_11x(base_oof_11x, base_test_11x)

seed_rank_oof_11x = rank_against_train_11x(seed_pred_oof_11x, seed_pred_oof_11x)
seed_rank_test_11x = rank_against_train_11x(seed_pred_oof_11x, seed_pred_test_11x)

resid_base_oof_11x = y_arr_11x - base_oof_11x
over_amount_oof_11x = np.clip(base_oof_11x - y_arr_11x, 0, None)

X_low_train_11x = X_low_train_base_11x.copy()
X_low_test_11x = X_low_test_base_11x.copy()

extra_train_11x = pd.DataFrame({
    "11x_base_pred": base_oof_11x,
    "11x_base_rank": base_rank_oof_11x,
    "11x_seed_pred": seed_pred_oof_11x,
    "11x_seed_rank": seed_rank_oof_11x,
    "11x_pos_corr": pos_corr_oof_11x,
    "11x_pos_corr_rank": rank_against_train_11x(pos_corr_oof_11x, pos_corr_oof_11x),
    "11x_base_minus_seed": base_oof_11x - seed_pred_oof_11x,
})

extra_test_11x = pd.DataFrame({
    "11x_base_pred": base_test_11x,
    "11x_base_rank": base_rank_test_11x,
    "11x_seed_pred": seed_pred_test_11x,
    "11x_seed_rank": seed_rank_test_11x,
    "11x_pos_corr": pos_corr_test_11x,
    "11x_pos_corr_rank": rank_against_train_11x(pos_corr_oof_11x, pos_corr_test_11x),
    "11x_base_minus_seed": base_test_11x - seed_pred_test_11x,
})

X_low_train_11x = pd.concat(
    [X_low_train_11x.reset_index(drop=True), extra_train_11x.reset_index(drop=True)],
    axis=1,
)

X_low_test_11x = pd.concat(
    [X_low_test_11x.reset_index(drop=True), extra_test_11x.reset_index(drop=True)],
    axis=1,
)

X_low_train_11x = clean_numeric_frame_11x(X_low_train_11x)
X_low_test_11x = clean_numeric_frame_11x(X_low_test_11x)

print("Low-tail feature shape train:", X_low_train_11x.shape)
print("Low-tail feature shape test :", X_low_test_11x.shape)


# ==================================================
# PART 2 - Fold-safe low-tail overprediction detector
# ==================================================

print("=" * 100)
print("PART 2 - Low-tail overprediction detector")

q20_11x = np.quantile(y_arr_11x, 0.20)
q25_11x = np.quantile(y_arr_11x, 0.25)
q30_11x = np.quantile(y_arr_11x, 0.30)

low_label_configs_11x = {
    "low_q20_res4": {
        "target_thr": q20_11x,
        "res_thr": 4.0,
    },
    "low_q20_res6": {
        "target_thr": q20_11x,
        "res_thr": 6.0,
    },
    "low_q25_res5": {
        "target_thr": q25_11x,
        "res_thr": 5.0,
    },
    "low_q30_res6": {
        "target_thr": q30_11x,
        "res_thr": 6.0,
    },
}

low_outputs_11x = {}

for label_name, cfg in low_label_configs_11x.items():
    target_thr = cfg["target_thr"]
    res_thr = cfg["res_thr"]

    low_label = ((y_arr_11x <= target_thr) & ((base_oof_11x - y_arr_11x) >= res_thr)).astype(int)

    print("=" * 100)
    print("Low label:", label_name)
    print("target_thr:", target_thr, "res_thr:", res_thr)
    print("positive count:", int(low_label.sum()), "positive rate:", float(low_label.mean()))

    proba_oof = np.zeros(n_train_11x)
    amount_oof = np.zeros(n_train_11x)

    proba_test_sum = np.zeros(n_test_11x)
    amount_test_sum = np.zeros(n_test_11x)

    valid_fold_count = 0

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr = X_low_train_11x.iloc[tr_idx]
        X_val = X_low_train_11x.iloc[val_idx]

        y_cls_tr = low_label[tr_idx]
        y_amt_tr = over_amount_oof_11x[tr_idx]

        # Emphasize low target and actual overprediction rows.
        sample_weight = np.ones(len(tr_idx))
        sample_weight += 1.5 * (y_arr_11x[tr_idx] <= q30_11x)
        sample_weight += 2.0 * (y_cls_tr == 1)
        sample_weight += np.clip(y_amt_tr / 8.0, 0, 2.0)

        if len(np.unique(y_cls_tr)) < 2 or y_cls_tr.sum() < 10:
            print(f"  Fold {fold}: skipped classifier, fallback mean.")
            fold_prior = float(y_cls_tr.mean()) if len(y_cls_tr) > 0 else 0.0
            proba_oof[val_idx] = fold_prior
            proba_test_sum += fold_prior
        else:
            clf = make_pipeline(
                SimpleImputer(strategy="median"),
                HistGradientBoostingClassifier(
                    max_iter=160,
                    learning_rate=0.035,
                    max_leaf_nodes=9,
                    min_samples_leaf=18,
                    l2_regularization=2.5,
                    random_state=6100 + fold,
                ),
            )

            clf.fit(
                X_tr,
                y_cls_tr,
                histgradientboostingclassifier__sample_weight=sample_weight,
            )

            p_val = clf.predict_proba(X_val)[:, 1]
            p_test = clf.predict_proba(X_low_test_11x)[:, 1]

            proba_oof[val_idx] = p_val
            proba_test_sum += p_test

        # Regress overprediction amount.
        reg_weight = np.ones(len(tr_idx))
        reg_weight += 1.5 * (y_arr_11x[tr_idx] <= q30_11x)
        reg_weight += np.clip(y_amt_tr / 6.0, 0, 3.0)

        reg = make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                max_iter=180,
                learning_rate=0.035,
                max_leaf_nodes=9,
                min_samples_leaf=18,
                l2_regularization=2.5,
                random_state=7100 + fold,
            ),
        )

        reg.fit(
            X_tr,
            y_amt_tr,
            histgradientboostingregressor__sample_weight=reg_weight,
        )

        amt_val = np.clip(reg.predict(X_val), 0, 20)
        amt_test = np.clip(reg.predict(X_low_test_11x), 0, 20)

        amount_oof[val_idx] = amt_val
        amount_test_sum += amt_test

        valid_fold_count += 1

        print(
            f"  Fold {fold}: cls_pos={int(y_cls_tr.sum())}, "
            f"mean_proba_val={float(np.mean(proba_oof[val_idx])):.4f}, "
            f"mean_amt_val={float(np.mean(amt_val)):.4f}"
        )

    if valid_fold_count == 0:
        raise RuntimeError(f"No valid folds for low label {label_name}")

    proba_test = proba_test_sum / valid_fold_count
    amount_test = amount_test_sum / valid_fold_count

    low_outputs_11x[label_name] = {
        "proba_oof": np.clip(proba_oof, 0, 1),
        "proba_test": np.clip(proba_test, 0, 1),
        "amount_oof": np.clip(amount_oof, 0, 20),
        "amount_test": np.clip(amount_test, 0, 20),
        "label": low_label,
    }

    print(
        "Finished",
        label_name,
        "| proba mean:",
        float(np.mean(proba_oof)),
        "| amount mean:",
        float(np.mean(amount_oof)),
    )

gc.collect()


# ==================================================
# PART 3 - Evaluate low-tail negative correction
# ==================================================

print("=" * 100)
print("PART 3 - Low-tail negative correction grid")

part_low_start_11x = len(candidate_rows_11x)

best_low_neg_11x = {
    "rmse": np.inf,
    "key": None,
    "neg_oof": np.zeros(n_train_11x),
    "neg_test": np.zeros(n_test_11x),
}

low_source_weight_grid_11x = [0.95, 1.00, 1.05]
max_base_rank_grid_11x = [0.25, 0.30, 0.35, 0.40, 0.50]
top_frac_grid_low_11x = [0.02, 0.04, 0.06, 0.08, 0.10, 0.15]
min_prob_grid_11x = [0.10, 0.20, 0.30, 0.40]
strength_grid_low_11x = [0.10, 0.15, 0.20, 0.30, 0.40, 0.55]
cap_grid_low_11x = [0.50, 0.75, 1.00, 1.50, 2.00, 2.50]

for label_name, obj in low_outputs_11x.items():
    p_oof = obj["proba_oof"]
    p_test = obj["proba_test"]

    a_oof = obj["amount_oof"]
    a_test = obj["amount_test"]

    # Stronger low score for low base rank and high overprediction amount.
    low_score_oof = p_oof * a_oof * (1.05 - np.clip(base_rank_oof_11x, 0, 1))
    low_score_test = p_test * a_test * (1.05 - np.clip(base_rank_test_11x, 0, 1))

    for w_pos in low_source_weight_grid_11x:
        pos_oof = np.clip(w_pos * pos_corr_oof_11x, 0, 5.5)
        pos_test = np.clip(w_pos * pos_corr_test_11x, 0, 5.5)

        for max_base_rank in max_base_rank_grid_11x:
            eligible_oof_base = base_rank_oof_11x <= max_base_rank
            eligible_test_base = base_rank_test_11x <= max_base_rank

            for min_prob in min_prob_grid_11x:
                eligible_oof = eligible_oof_base & (p_oof >= min_prob) & (low_score_oof > 0)
                eligible_test = eligible_test_base & (p_test >= min_prob) & (low_score_test > 0)

                if eligible_oof.sum() < 10 or eligible_test.sum() < 1:
                    continue

                score_eligible_oof = low_score_oof[eligible_oof]
                score_eligible_test = low_score_test[eligible_test]

                if np.max(score_eligible_oof) <= 0 or np.max(score_eligible_test) <= 0:
                    continue

                for top_frac in top_frac_grid_low_11x:
                    thr_oof = np.quantile(score_eligible_oof, 1.0 - top_frac)
                    thr_test = np.quantile(score_eligible_test, 1.0 - top_frac)

                    mask_oof = eligible_oof & (low_score_oof >= thr_oof)
                    mask_test = eligible_test & (low_score_test >= thr_test)

                    if mask_oof.sum() < 5:
                        continue

                    for strength in strength_grid_low_11x:
                        raw_neg_oof = strength * p_oof * a_oof
                        raw_neg_test = strength * p_test * a_test

                        for cap in cap_grid_low_11x:
                            neg_oof = np.clip(raw_neg_oof, 0, cap)
                            neg_test = np.clip(raw_neg_test, 0, cap)

                            neg_oof = np.where(mask_oof, neg_oof, 0.0)
                            neg_test = np.where(mask_test, neg_test, 0.0)

                            signed_corr_oof = np.clip(pos_oof - neg_oof, -3.5, 5.5)
                            signed_corr_test = np.clip(pos_test - neg_test, -3.5, 5.5)

                            pred_oof = np.clip(base_oof_11x + signed_corr_oof, 0, None)
                            pred_test = np.clip(base_test_11x + signed_corr_test, 0, None)

                            key = (
                                f"X_low_{label_name}"
                                f"_wpos{w_pos}"
                                f"_maxrank{max_base_rank}"
                                f"_p{min_prob}"
                                f"_top{top_frac}"
                                f"_s{strength}"
                                f"_cap{cap}"
                            )

                            evaluate_candidate_11x(
                                key=key,
                                method="X_low_negative",
                                pred_oof=pred_oof,
                                pred_test=pred_test,
                                signed_correction_oof=signed_corr_oof,
                                signed_correction_test=signed_corr_test,
                            )

                            score_now = fast_rmse_11x(y_arr_11x, pred_oof)
                            if score_now < best_low_neg_11x["rmse"]:
                                best_low_neg_11x = {
                                    "rmse": score_now,
                                    "key": key,
                                    "neg_oof": neg_oof.copy(),
                                    "neg_test": neg_test.copy(),
                                }

                            del neg_oof, neg_test, signed_corr_oof, signed_corr_test, pred_oof, pred_test

        del pos_oof, pos_test

    gc.collect()

low_df_11x = pd.DataFrame(candidate_rows_11x[part_low_start_11x:]).sort_values("rmse")

print("=" * 100)
print("Top low-tail negative results")
display(low_df_11x.head(40))

print("Best low negative key:", best_low_neg_11x["key"])
print("Best low negative RMSE:", best_low_neg_11x["rmse"])


# ==================================================
# PART 4 - Hierarchical signed group residual
# ==================================================

print("=" * 100)
print("PART 4 - Hierarchical signed group residual")

def get_col_for_group_11x(col, is_train=True):
    if is_train:
        if "X_fe" in globals() and col in X_fe.columns:
            return X_fe[col].astype(str).reset_index(drop=True)
        if col in train_raw.columns:
            return train_raw[col].astype(str).reset_index(drop=True)
        return None
    else:
        if "X_test_fe" in globals() and col in X_test_fe.columns:
            return X_test_fe[col].astype(str).reset_index(drop=True)
        if col in test_raw.columns:
            return test_raw[col].astype(str).reset_index(drop=True)
        return None


group_frame_train_11x = pd.DataFrame(index=np.arange(n_train_11x))
group_frame_test_11x = pd.DataFrame(index=np.arange(n_test_11x))

single_group_candidates_11x = [
    "source_id",
    "geo_zone_micro",
    "geo_zone_meso",
    "geo_zone_macro",
    "geo_meso_micro",
    "geo_macro_meso",
    "biome_land_cover",
    "land_cover_geo_macro",
    "soil_type",
    "land_cover",
    "biome",
]

for col in single_group_candidates_11x:
    s_tr = get_col_for_group_11x(col, True)
    s_te = get_col_for_group_11x(col, False)

    if s_tr is not None and s_te is not None:
        group_frame_train_11x[col] = s_tr.fillna("Unknown").astype(str)
        group_frame_test_11x[col] = s_te.fillna("Unknown").astype(str)

# Add base prediction bins as categorical residual segments.
base_edges_5_11x = np.unique(np.quantile(base_oof_11x, np.linspace(0, 1, 6)))
base_edges_10_11x = np.unique(np.quantile(base_oof_11x, np.linspace(0, 1, 11)))

if len(base_edges_5_11x) > 2:
    group_frame_train_11x["base_bin5"] = pd.cut(base_oof_11x, bins=base_edges_5_11x, include_lowest=True, labels=False).astype(str)
    group_frame_test_11x["base_bin5"] = pd.cut(base_test_11x, bins=base_edges_5_11x, include_lowest=True, labels=False).astype(str)

if len(base_edges_10_11x) > 2:
    group_frame_train_11x["base_bin10"] = pd.cut(base_oof_11x, bins=base_edges_10_11x, include_lowest=True, labels=False).astype(str)
    group_frame_test_11x["base_bin10"] = pd.cut(base_test_11x, bins=base_edges_10_11x, include_lowest=True, labels=False).astype(str)

# Composite groups.
def add_composite_group_11x(name, cols):
    if all(c in group_frame_train_11x.columns for c in cols) and all(c in group_frame_test_11x.columns for c in cols):
        group_frame_train_11x[name] = group_frame_train_11x[cols].astype(str).agg("__".join, axis=1)
        group_frame_test_11x[name] = group_frame_test_11x[cols].astype(str).agg("__".join, axis=1)

add_composite_group_11x("source_base5", ["source_id", "base_bin5"])
add_composite_group_11x("source_base10", ["source_id", "base_bin10"])
add_composite_group_11x("geo_micro_base5", ["geo_zone_micro", "base_bin5"])
add_composite_group_11x("geo_micro_base10", ["geo_zone_micro", "base_bin10"])
add_composite_group_11x("geo_meso_base5", ["geo_zone_meso", "base_bin5"])
add_composite_group_11x("geo_meso_micro_base5", ["geo_meso_micro", "base_bin5"])
add_composite_group_11x("biome_land_base5", ["biome_land_cover", "base_bin5"])
add_composite_group_11x("land_geo_base5", ["land_cover_geo_macro", "base_bin5"])

available_group_cols_11x = [
    c for c in group_frame_train_11x.columns
    if c in group_frame_test_11x.columns
]

print("Available group columns:", available_group_cols_11x)


def make_group_key_11x(df, cols):
    if isinstance(cols, str):
        return df[cols].astype(str)
    return df[list(cols)].astype(str).agg("||".join, axis=1)


def fold_safe_group_residual_11x(group_col, min_count=50, shrink_k=100, stat="mean"):
    # Residual after 11W seed, so group correction attacks remaining signed bias.
    residual_seed = y_arr_11x - seed_pred_oof_11x

    corr_oof = np.zeros(n_train_11x)
    corr_test_sum = np.zeros(n_test_11x)

    valid_fold_count = 0

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        tr_key = make_group_key_11x(group_frame_train_11x.iloc[tr_idx], group_col)
        val_key = make_group_key_11x(group_frame_train_11x.iloc[val_idx], group_col)
        test_key = make_group_key_11x(group_frame_test_11x, group_col)

        tmp = pd.DataFrame({
            "key": tr_key.values,
            "residual": residual_seed[tr_idx],
        })

        if stat == "median":
            agg = tmp.groupby("key")["residual"].agg(["count", "median"]).reset_index()
            agg["raw_resid"] = agg["median"]
        else:
            agg = tmp.groupby("key")["residual"].agg(["count", "mean"]).reset_index()
            agg["raw_resid"] = agg["mean"]

        agg = agg[agg["count"] >= min_count].copy()

        if len(agg) == 0:
            valid_fold_count += 1
            continue

        agg["shrunk_resid"] = agg["raw_resid"] * (agg["count"] / (agg["count"] + shrink_k))

        mapping = dict(zip(agg["key"], agg["shrunk_resid"]))

        corr_oof[val_idx] = val_key.map(mapping).fillna(0.0).values
        corr_test_sum += test_key.map(mapping).fillna(0.0).values

        valid_fold_count += 1

    if valid_fold_count == 0:
        return corr_oof, np.zeros(n_test_11x)

    corr_test = corr_test_sum / valid_fold_count

    return corr_oof, corr_test


print("=" * 100)
print("PART 4B - Group residual grid")

part_group_start_11x = len(candidate_rows_11x)

best_group_signed_11x = {
    "rmse": np.inf,
    "key": None,
    "signed_oof": np.zeros(n_train_11x),
    "signed_test": np.zeros(n_test_11x),
}

group_min_count_grid_11x = [25, 50, 80, 120]
group_shrink_k_grid_11x = [50, 100, 200, 400]
group_stat_grid_11x = ["mean", "median"]
group_strength_grid_11x = [0.10, 0.15, 0.25, 0.35, 0.50]
group_cap_grid_11x = [0.35, 0.50, 0.75, 1.00, 1.50]
w_pos_grid_group_11x = [0.95, 1.00, 1.05]

group_cache_11x = {}

for group_col in available_group_cols_11x:
    for min_count in group_min_count_grid_11x:
        for shrink_k in group_shrink_k_grid_11x:
            for stat in group_stat_grid_11x:
                cache_key = (group_col, min_count, shrink_k, stat)

                corr_group_oof, corr_group_test = fold_safe_group_residual_11x(
                    group_col=group_col,
                    min_count=min_count,
                    shrink_k=shrink_k,
                    stat=stat,
                )

                group_cache_11x[cache_key] = (corr_group_oof, corr_group_test)

                if np.max(np.abs(corr_group_oof)) <= 1e-12:
                    continue

                for w_pos in w_pos_grid_group_11x:
                    pos_oof = np.clip(w_pos * pos_corr_oof_11x, 0, 5.5)
                    pos_test = np.clip(w_pos * pos_corr_test_11x, 0, 5.5)

                    for strength in group_strength_grid_11x:
                        raw_signed_oof = strength * corr_group_oof
                        raw_signed_test = strength * corr_group_test

                        for cap in group_cap_grid_11x:
                            group_signed_oof = np.clip(raw_signed_oof, -cap, cap)
                            group_signed_test = np.clip(raw_signed_test, -cap, cap)

                            signed_corr_oof = np.clip(pos_oof + group_signed_oof, -3.5, 5.5)
                            signed_corr_test = np.clip(pos_test + group_signed_test, -3.5, 5.5)

                            pred_oof = np.clip(base_oof_11x + signed_corr_oof, 0, None)
                            pred_test = np.clip(base_test_11x + signed_corr_test, 0, None)

                            key = (
                                f"X_group_{group_col}"
                                f"_min{min_count}"
                                f"_k{shrink_k}"
                                f"_{stat}"
                                f"_wpos{w_pos}"
                                f"_s{strength}"
                                f"_cap{cap}"
                            )

                            evaluate_candidate_11x(
                                key=key,
                                method="X_group_signed",
                                pred_oof=pred_oof,
                                pred_test=pred_test,
                                signed_correction_oof=signed_corr_oof,
                                signed_correction_test=signed_corr_test,
                            )

                            score_now = fast_rmse_11x(y_arr_11x, pred_oof)
                            if score_now < best_group_signed_11x["rmse"]:
                                best_group_signed_11x = {
                                    "rmse": score_now,
                                    "key": key,
                                    "signed_oof": group_signed_oof.copy(),
                                    "signed_test": group_signed_test.copy(),
                                }

                            del group_signed_oof, group_signed_test, signed_corr_oof, signed_corr_test, pred_oof, pred_test

                    del pos_oof, pos_test

                gc.collect()

group_df_11x = pd.DataFrame(candidate_rows_11x[part_group_start_11x:]).sort_values("rmse")

print("=" * 100)
print("Top group residual results")
display(group_df_11x.head(40))

print("Best group key:", best_group_signed_11x["key"])
print("Best group RMSE:", best_group_signed_11x["rmse"])


# ==================================================
# PART 5 - Optional KNN local residual memory
# ==================================================

print("=" * 100)
print("PART 5 - Optional KNN local residual memory")

best_knn_signed_11x = {
    "rmse": np.inf,
    "key": None,
    "signed_oof": np.zeros(n_train_11x),
    "signed_test": np.zeros(n_test_11x),
}

knn_df_11x = pd.DataFrame()

if RUN_KNN_11X:
    part_knn_start_11x = len(candidate_rows_11x)

    # Use a controlled feature set for KNN.
    knn_priority_cols = [
        c for c in selected_num_cols_11x
        if any(k.lower() in c.lower() for k in [
            "particle",
            "fine",
            "coarse",
            "acidity",
            "cec",
            "cation",
            "ratio",
            "spectral",
            "band",
            "ca_",
            "mg_",
        ])
    ]

    if len(knn_priority_cols) < 8:
        knn_priority_cols = selected_num_cols_11x[:40]
    else:
        knn_priority_cols = knn_priority_cols[:50]

    X_knn_train_11x = X_low_train_base_11x[knn_priority_cols].copy()
    X_knn_test_11x = X_low_test_base_11x[knn_priority_cols].copy()

    # Add prediction-space coordinates.
    X_knn_train_11x["11x_base_pred"] = base_oof_11x
    X_knn_train_11x["11x_base_rank"] = base_rank_oof_11x
    X_knn_train_11x["11x_seed_pred"] = seed_pred_oof_11x

    X_knn_test_11x["11x_base_pred"] = base_test_11x
    X_knn_test_11x["11x_base_rank"] = base_rank_test_11x
    X_knn_test_11x["11x_seed_pred"] = seed_pred_test_11x

    X_knn_train_11x = clean_numeric_frame_11x(X_knn_train_11x)
    X_knn_test_11x = clean_numeric_frame_11x(X_knn_test_11x)

    residual_seed_11x = y_arr_11x - seed_pred_oof_11x

    def compute_knn_residual_11x(n_neighbors=40):
        corr_oof = np.zeros(n_train_11x)
        corr_test_sum = np.zeros(n_test_11x)
        valid_fold_count = 0

        for fold, (tr_idx, val_idx) in enumerate(folds, 1):
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()

            X_tr_imp = imputer.fit_transform(X_knn_train_11x.iloc[tr_idx])
            X_val_imp = imputer.transform(X_knn_train_11x.iloc[val_idx])
            X_test_imp = imputer.transform(X_knn_test_11x)

            X_tr_scaled = scaler.fit_transform(X_tr_imp)
            X_val_scaled = scaler.transform(X_val_imp)
            X_test_scaled = scaler.transform(X_test_imp)

            nn = NearestNeighbors(
                n_neighbors=min(n_neighbors, len(tr_idx)),
                metric="euclidean",
                n_jobs=-1,
            )

            nn.fit(X_tr_scaled)

            dist_val, ind_val = nn.kneighbors(X_val_scaled)
            dist_test, ind_test = nn.kneighbors(X_test_scaled)

            resid_tr = residual_seed_11x[tr_idx]

            weights_val = 1.0 / (dist_val + 1e-4)
            weights_test = 1.0 / (dist_test + 1e-4)

            corr_val = np.sum(weights_val * resid_tr[ind_val], axis=1) / np.sum(weights_val, axis=1)
            corr_test = np.sum(weights_test * resid_tr[ind_test], axis=1) / np.sum(weights_test, axis=1)

            corr_oof[val_idx] = corr_val
            corr_test_sum += corr_test

            valid_fold_count += 1

            print(
                f"  KNN fold {fold}, k={n_neighbors}: "
                f"val corr mean={float(np.mean(corr_val)):.4f}, "
                f"abs={float(np.mean(np.abs(corr_val))):.4f}"
            )

        corr_test_avg = corr_test_sum / max(valid_fold_count, 1)

        return corr_oof, corr_test_avg

    knn_cache_11x = {}

    for k in [20, 40, 80]:
        print("=" * 100)
        print("Computing KNN residual k =", k)

        raw_knn_oof, raw_knn_test = compute_knn_residual_11x(n_neighbors=k)
        knn_cache_11x[k] = (raw_knn_oof, raw_knn_test)

        abs_oof = np.abs(raw_knn_oof)
        abs_test = np.abs(raw_knn_test)

        for top_frac in [0.20, 0.35, 0.50, 0.75, 1.00]:
            if top_frac < 1.0:
                thr_oof = np.quantile(abs_oof, 1.0 - top_frac)
                thr_test = np.quantile(abs_test, 1.0 - top_frac)

                mask_oof = abs_oof >= thr_oof
                mask_test = abs_test >= thr_test
            else:
                mask_oof = np.ones(n_train_11x, dtype=bool)
                mask_test = np.ones(n_test_11x, dtype=bool)

            for w_pos in [0.95, 1.00, 1.05]:
                pos_oof = np.clip(w_pos * pos_corr_oof_11x, 0, 5.5)
                pos_test = np.clip(w_pos * pos_corr_test_11x, 0, 5.5)

                for strength in [0.05, 0.10, 0.15, 0.20, 0.30]:
                    for cap in [0.35, 0.50, 0.75, 1.00, 1.50]:
                        signed_knn_oof = np.where(mask_oof, np.clip(strength * raw_knn_oof, -cap, cap), 0.0)
                        signed_knn_test = np.where(mask_test, np.clip(strength * raw_knn_test, -cap, cap), 0.0)

                        signed_corr_oof = np.clip(pos_oof + signed_knn_oof, -3.5, 5.5)
                        signed_corr_test = np.clip(pos_test + signed_knn_test, -3.5, 5.5)

                        pred_oof = np.clip(base_oof_11x + signed_corr_oof, 0, None)
                        pred_test = np.clip(base_test_11x + signed_corr_test, 0, None)

                        key = (
                            f"X_knn_k{k}"
                            f"_top{top_frac}"
                            f"_wpos{w_pos}"
                            f"_s{strength}"
                            f"_cap{cap}"
                        )

                        evaluate_candidate_11x(
                            key=key,
                            method="X_knn_signed",
                            pred_oof=pred_oof,
                            pred_test=pred_test,
                            signed_correction_oof=signed_corr_oof,
                            signed_correction_test=signed_corr_test,
                        )

                        score_now = fast_rmse_11x(y_arr_11x, pred_oof)
                        if score_now < best_knn_signed_11x["rmse"]:
                            best_knn_signed_11x = {
                                "rmse": score_now,
                                "key": key,
                                "signed_oof": signed_knn_oof.copy(),
                                "signed_test": signed_knn_test.copy(),
                            }

                        del signed_knn_oof, signed_knn_test, signed_corr_oof, signed_corr_test, pred_oof, pred_test

                del pos_oof, pos_test

        gc.collect()

    knn_df_11x = pd.DataFrame(candidate_rows_11x[part_knn_start_11x:]).sort_values("rmse")

    print("=" * 100)
    print("Top KNN residual results")
    display(knn_df_11x.head(40))

    print("Best KNN key:", best_knn_signed_11x["key"])
    print("Best KNN RMSE:", best_knn_signed_11x["rmse"])
else:
    print("RUN_KNN_11X = False, skipping KNN.")


# ==================================================
# PART 6 - Component blend
# ==================================================

print("=" * 100)
print("PART 6 - Component blend")

part_blend_start_11x = len(candidate_rows_11x)

# Components:
# pos_corr: positive 11W anchor
# low_neg: positive amount to subtract
# group_signed: can be positive or negative
# knn_signed: can be positive or negative

low_neg_oof = best_low_neg_11x["neg_oof"]
low_neg_test = best_low_neg_11x["neg_test"]

group_signed_oof = best_group_signed_11x["signed_oof"]
group_signed_test = best_group_signed_11x["signed_test"]

knn_signed_oof = best_knn_signed_11x["signed_oof"]
knn_signed_test = best_knn_signed_11x["signed_test"]

w_pos_grid_11x = [0.95, 1.00, 1.03, 1.05, 1.08]
w_low_grid_11x = [0.00, 0.50, 0.75, 1.00, 1.25]
w_group_grid_11x = [0.00, 0.50, 0.75, 1.00, 1.25]
w_knn_grid_11x = [0.00, 0.50, 0.75, 1.00] if RUN_KNN_11X else [0.00]

total_pos_cap_grid_11x = [4.00, 4.50, 5.00]
total_neg_cap_grid_11x = [1.50, 2.00, 2.50, 3.00]

for w_pos in w_pos_grid_11x:
    for w_low in w_low_grid_11x:
        for w_group in w_group_grid_11x:
            for w_knn in w_knn_grid_11x:
                if w_low == 0 and w_group == 0 and w_knn == 0 and w_pos == 1.00:
                    continue

                raw_extra_oof = (
                    w_pos * pos_corr_oof_11x
                    - w_low * low_neg_oof
                    + w_group * group_signed_oof
                    + w_knn * knn_signed_oof
                )

                raw_extra_test = (
                    w_pos * pos_corr_test_11x
                    - w_low * low_neg_test
                    + w_group * group_signed_test
                    + w_knn * knn_signed_test
                )

                for pos_cap in total_pos_cap_grid_11x:
                    for neg_cap in total_neg_cap_grid_11x:
                        signed_corr_oof = np.clip(raw_extra_oof, -neg_cap, pos_cap)
                        signed_corr_test = np.clip(raw_extra_test, -neg_cap, pos_cap)

                        pred_oof = np.clip(base_oof_11x + signed_corr_oof, 0, None)
                        pred_test = np.clip(base_test_11x + signed_corr_test, 0, None)

                        key = (
                            f"X_blend"
                            f"_wpos{w_pos}"
                            f"_wlow{w_low}"
                            f"_wgrp{w_group}"
                            f"_wknn{w_knn}"
                            f"_pcap{pos_cap}"
                            f"_ncap{neg_cap}"
                        )

                        evaluate_candidate_11x(
                            key=key,
                            method="X_component_blend",
                            pred_oof=pred_oof,
                            pred_test=pred_test,
                            signed_correction_oof=signed_corr_oof,
                            signed_correction_test=signed_corr_test,
                        )

                        del signed_corr_oof, signed_corr_test, pred_oof, pred_test

                del raw_extra_oof, raw_extra_test

blend_df_11x = pd.DataFrame(candidate_rows_11x[part_blend_start_11x:]).sort_values("rmse")

print("=" * 100)
print("Top component blend results")
display(blend_df_11x.head(50))


# ==================================================
# PART 7 - Final selection
# ==================================================

print("=" * 100)
print("PART 7 - Final selection")

candidate_df_11x = pd.DataFrame(candidate_rows_11x)
candidate_df_11x = candidate_df_11x.sort_values("rmse").reset_index(drop=True)

candidate_df_11x_robust = candidate_df_11x.sort_values("selection_score").reset_index(drop=True)

print("Top 11X candidates by raw RMSE")
display(candidate_df_11x.head(60))

print("Top 11X candidates by robust selection score")
display(candidate_df_11x_robust.head(60))

print("=" * 100)
print("Method best summary")

method_summary_11x = []
for method, obj in method_best_11x.items():
    method_summary_11x.append({
        "method": method,
        "key": obj["key"],
        "rmse": obj["rmse"],
    })

method_summary_11x = pd.DataFrame(method_summary_11x).sort_values("rmse")
display(method_summary_11x)


print("=" * 100)
print("Selected 11X candidates")

print("Raw best key     :", best_raw_11x["key"])
print("Raw best method  :", best_raw_11x["method"])
print("Raw best RMSE    :", best_raw_11x["rmse"])
print("Raw selection    :", best_raw_11x["selection_score"])
print("Raw delta vs 11O :", best_raw_11x["rmse"] - BEST_11O_OOF)
print("Raw delta vs 11W :", best_raw_11x["rmse"] - prev_11w_score)

print("-" * 100)

print("Robust best key     :", best_robust_11x["key"])
print("Robust best method  :", best_robust_11x["method"])
print("Robust best RMSE    :", best_robust_11x["rmse"])
print("Robust selection    :", best_robust_11x["selection_score"])
print("Robust delta vs 11O :", best_robust_11x["rmse"] - BEST_11O_OOF)
print("Robust delta vs 11W :", best_robust_11x["rmse"] - prev_11w_score)

raw_report_11x = target_bin_report_11x(
    best_raw_11x["oof"],
    f"11X raw best: {best_raw_11x['key']}",
)

robust_report_11x = target_bin_report_11x(
    best_robust_11x["oof"],
    f"11X robust best: {best_robust_11x['key']}",
)


# ==================================================
# PART 8 - Bin comparison
# ==================================================

print("=" * 100)
print("Bin comparison: 11O vs 11W seed vs 11X raw vs 11X robust")

compare_11x = train_raw[[ID_COL, TARGET]].copy()

compare_11x["base_pred"] = base_oof_11x
compare_11x["w_seed_pred"] = seed_pred_oof_11x
compare_11x["x_raw_pred"] = best_raw_11x["oof"]
compare_11x["x_robust_pred"] = best_robust_11x["oof"]

for col in ["base_pred", "w_seed_pred", "x_raw_pred", "x_robust_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")

    compare_11x[err_col] = compare_11x[col] - compare_11x[TARGET]
    compare_11x[se_col] = compare_11x[err_col] ** 2

compare_11x["target_bin_5"] = pd.qcut(
    compare_11x[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11x = (
    compare_11x.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        w_seed_pred_mean=("w_seed_pred", "mean"),
        x_raw_pred_mean=("x_raw_pred", "mean"),
        x_robust_pred_mean=("x_robust_pred", "mean"),

        base_bias=("base_error", "mean"),
        w_seed_bias=("w_seed_error", "mean"),
        x_raw_bias=("x_raw_error", "mean"),
        x_robust_bias=("x_robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        w_seed_rmse=("w_seed_se", lambda x: np.sqrt(np.mean(x))),
        x_raw_rmse=("x_raw_se", lambda x: np.sqrt(np.mean(x))),
        x_robust_rmse=("x_robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11x["w_seed_rmse_delta_vs_base"] = (
    bin_compare_11x["w_seed_rmse"] - bin_compare_11x["base_rmse"]
)

bin_compare_11x["x_raw_rmse_delta_vs_base"] = (
    bin_compare_11x["x_raw_rmse"] - bin_compare_11x["base_rmse"]
)

bin_compare_11x["x_robust_rmse_delta_vs_base"] = (
    bin_compare_11x["x_robust_rmse"] - bin_compare_11x["base_rmse"]
)

bin_compare_11x["x_raw_minus_w_seed_rmse"] = (
    bin_compare_11x["x_raw_rmse"] - bin_compare_11x["w_seed_rmse"]
)

bin_compare_11x["x_robust_minus_w_seed_rmse"] = (
    bin_compare_11x["x_robust_rmse"] - bin_compare_11x["w_seed_rmse"]
)

display(bin_compare_11x.sort_values("base_rmse", ascending=False))


# ==================================================
# PART 9 - Save submissions
# ==================================================

print("=" * 100)
print("Save 11X submissions")

sub_11x_raw = save_submission_11x(
    best_raw_11x["test"],
    "submission_11x_raw_best.csv",
)

sub_11x_robust = save_submission_11x(
    best_robust_11x["test"],
    "submission_11x_robust_best.csv",
)

sub_11w_ref_from_11x = save_submission_11x(
    seed_pred_test_11x,
    "submission_11w_reference_from_11x.csv",
)

sub_11o_ref_from_11x = save_submission_11x(
    base_test_11x,
    "submission_11o_reference_from_11x.csv",
)


# ==================================================
# PART 10 - Decision guide
# ==================================================

print("=" * 100)
print("11X Decision Guide")

print(f"11O adaptive OOF        : {base_score_11x:.6f}")
print(f"11W seed OOF            : {prev_11w_score:.6f}")
print(f"11X raw best OOF        : {best_raw_11x['rmse']:.6f}")
print(f"11X robust best OOF     : {best_robust_11x['rmse']:.6f}")

print(f"11X raw delta vs 11O    : {best_raw_11x['rmse'] - BEST_11O_OOF:.6f}")
print(f"11X robust delta vs 11O : {best_robust_11x['rmse'] - BEST_11O_OOF:.6f}")

print(f"11X raw delta vs 11W    : {best_raw_11x['rmse'] - prev_11w_score:.6f}")
print(f"11X robust delta vs 11W : {best_robust_11x['rmse'] - prev_11w_score:.6f}")

if best_raw_11x["rmse"] < prev_11w_score - 0.003:
    print("Raw best decision: STRONG new approach. Submit if bin report is safe.")
elif best_raw_11x["rmse"] < prev_11w_score - 0.001:
    print("Raw best decision: GOOD improvement over 11W. Submit if bin report is safe.")
elif best_raw_11x["rmse"] < prev_11w_score:
    print("Raw best decision: SMALL improvement over 11W. Submit only as probe.")
else:
    print("Raw best decision: did not beat 11W. Keep 11W.")

if best_robust_11x["rmse"] < prev_11w_score - 0.003:
    print("Robust best decision: STRONG new approach. Prefer robust if bins are safer.")
elif best_robust_11x["rmse"] < prev_11w_score - 0.001:
    print("Robust best decision: GOOD improvement over 11W. Prefer this if bin report is safer.")
elif best_robust_11x["rmse"] < prev_11w_score:
    print("Robust best decision: SMALL improvement over 11W. Submit only as probe.")
else:
    print("Robust best decision: did not beat 11W. Keep 11W.")

print("=" * 100)
print("Please screenshot:")
print("1. low_df_11x.head(40)")
print("2. group_df_11x.head(40)")
print("3. knn_df_11x.head(40) if RUN_KNN_11X=True")
print("4. blend_df_11x.head(50)")
print("5. candidate_df_11x.head(60)")
print("6. candidate_df_11x_robust.head(60)")
print("7. method_summary_11x")
print("8. raw_report_11x")
print("9. robust_report_11x")
print("10. bin_compare_11x")

In [ ]:
# ==================================================
# DOWNLOAD 11X RAW + ROBUST BEST
# ==================================================

from google.colab import files
import os

filenames = [
    "submission_11x_robust_best.csv",
    "submission_11x_raw_best.csv",
    "submission_11w_reference_from_11x.csv",
]

for filename in filenames:
    if os.path.exists(filename):
        print(f"Downloading: {filename}")
        files.download(filename)
    else:
        print(f"Not found: {filename}")

In [ ]:
# ==================================================
# SECTION 11Y - Test-Weighted Target Encoding Base Rebuild
# ==================================================
# Goal:
# 1. Build adversarial validation score: train-like vs test-like.
# 2. Build fold-safe target encoding and residual target encoding.
# 3. Train new TE-weighted base models.
# 4. Stack / blend new models with 11X.
# 5. Save 11Y raw and robust submissions.
#
# Paste AFTER SECTION 11X.
# Skip 11P and 11Q.
#
# This is NOT another correction-only section.
# This section rebuilds feature space and trains new models.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import roc_auc_score

# Optional model libraries
HAS_LGBM_11Y = False
HAS_XGB_11Y = False
HAS_CAT_11Y = False

try:
    import lightgbm as lgb
    HAS_LGBM_11Y = True
except Exception as e:
    print("LightGBM unavailable:", e)

try:
    import xgboost as xgb
    HAS_XGB_11Y = True
except Exception as e:
    print("XGBoost unavailable:", e)

try:
    from catboost import CatBoostRegressor, Pool
    HAS_CAT_11Y = True
except Exception as e:
    print("CatBoost unavailable:", e)


# ==================================================
# PART 0 - Config
# ==================================================

print("=" * 100)
print("SECTION 11Y - Test-Weighted Target Encoding Base Rebuild")

BEST_11O_OOF = 11.26696803195285

# Runtime switches
RUN_LGBM_11Y = HAS_LGBM_11Y
RUN_XGB_11Y = HAS_XGB_11Y
RUN_CAT_11Y = HAS_CAT_11Y

# If runtime is too long, turn XGB/CatBoost False.
# RUN_XGB_11Y = False
# RUN_CAT_11Y = False

required_vars_11y = [
    "train_raw",
    "test_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "best_raw_11x",
    "best_robust_11x",
]

missing_11y = [v for v in required_vars_11y if v not in globals()]
if len(missing_11y) > 0:
    raise NameError(f"Missing required variables for 11Y: {missing_11y}")

y_arr_11y = y.values
n_train_11y = len(y_arr_11y)
n_test_11y = sample_submission.shape[0]

base_oof_11y = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_11y = np.clip(np.asarray(final_adaptive_test), 0, None)

base_score_11y = rmse(y, base_oof_11y)

# Use best 11X as seed.
if best_raw_11x["rmse"] <= best_robust_11x["rmse"]:
    seed_11x_obj = best_raw_11x
    seed_11x_name = "11x_raw"
else:
    seed_11x_obj = best_robust_11x
    seed_11x_name = "11x_robust"

seed_11x_oof_11y = np.clip(np.asarray(seed_11x_obj["oof"]), 0, None)
seed_11x_test_11y = np.clip(np.asarray(seed_11x_obj["test"]), 0, None)
seed_11x_score_11y = float(seed_11x_obj["rmse"])

# 11W reference if available.
if "best_raw_11w" in globals() and "best_robust_11w" in globals():
    if best_raw_11w["rmse"] <= best_robust_11w["rmse"]:
        seed_11w_obj_11y = best_raw_11w
    else:
        seed_11w_obj_11y = best_robust_11w

    seed_11w_oof_11y = np.clip(np.asarray(seed_11w_obj_11y["oof"]), 0, None)
    seed_11w_test_11y = np.clip(np.asarray(seed_11w_obj_11y["test"]), 0, None)
    seed_11w_score_11y = float(seed_11w_obj_11y["rmse"])
else:
    seed_11w_oof_11y = seed_11x_oof_11y.copy()
    seed_11w_test_11y = seed_11x_test_11y.copy()
    seed_11w_score_11y = seed_11x_score_11y

print("Base 11O OOF:", base_score_11y)
print("Seed 11X name:", seed_11x_name)
print("Seed 11X OOF:", seed_11x_score_11y)
print("Seed 11W OOF:", seed_11w_score_11y)
print("RUN_LGBM_11Y:", RUN_LGBM_11Y)
print("RUN_XGB_11Y :", RUN_XGB_11Y)
print("RUN_CAT_11Y :", RUN_CAT_11Y)


# ==================================================
# Fast utilities
# ==================================================

target_bin_11y = pd.qcut(
    y_arr_11y,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels_11y = ["very_low", "low", "medium", "high", "very_high"]

bin_masks_11y = {
    b: (target_bin_11y == b)
    for b in bin_labels_11y
}


def fast_rmse_11y(y_true, pred):
    pred = np.asarray(pred)
    return float(np.sqrt(np.mean((pred - y_true) ** 2)))


def weighted_rmse_11y(y_true, pred, weight):
    y_true = np.asarray(y_true)
    pred = np.asarray(pred)
    weight = np.asarray(weight)

    weight = weight / np.mean(weight)
    return float(np.sqrt(np.average((pred - y_true) ** 2, weights=weight)))


def rank_against_train_11y(train_values, values):
    train_values = np.asarray(train_values)
    values = np.asarray(values)

    train_sorted = np.sort(train_values[np.isfinite(train_values)])
    if len(train_sorted) == 0:
        return np.zeros_like(values, dtype=float)

    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def save_submission_11y(pred, filename):
    pred = np.clip(np.asarray(pred), 0, None)

    sub = sample_submission.copy()
    sub[TARGET] = pred

    assert sub.shape[0] == sample_submission.shape[0]
    assert list(sub.columns) == [ID_COL, TARGET]
    assert sub[TARGET].isna().sum() == 0
    assert np.isfinite(sub[TARGET]).all()
    assert (sub[TARGET] < 0).sum() == 0
    assert sub[ID_COL].equals(sample_submission[ID_COL])

    sub.to_csv(filename, index=False)

    print("=" * 100)
    print(f"Saved: {filename}")
    display(sub.head())
    display(sub[TARGET].describe().to_frame().T)

    return sub


def target_bin_report_11y(pred, label):
    pred = np.clip(np.asarray(pred), 0, None)

    temp = train_raw[[ID_COL, TARGET]].copy()
    temp["pred"] = pred
    temp["error"] = temp["pred"] - temp[TARGET]
    temp["abs_error"] = temp["error"].abs()
    temp["squared_error"] = temp["error"] ** 2

    temp["target_bin_5"] = pd.qcut(
        temp[TARGET],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )

    report = (
        temp.groupby("target_bin_5", observed=False)
        .agg(
            count=(TARGET, "count"),
            target_mean=(TARGET, "mean"),
            pred_mean=("pred", "mean"),
            bias=("error", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda x: np.sqrt(np.mean(x))),
            target_min=(TARGET, "min"),
            target_max=(TARGET, "max"),
        )
        .reset_index()
    )

    report["abs_bias"] = report["bias"].abs()

    print("=" * 100)
    print(label)
    print("Overall RMSE:", fast_rmse_11y(y_arr_11y, pred))
    display(report.sort_values("rmse", ascending=False))

    return report


def clean_numeric_frame_11y(df):
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)

    for c in out.columns:
        if not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out


# Placeholder, overwritten after adversarial weighting.
adv_weight_train_11y = np.ones(n_train_11y)


def calc_metrics_11y(pred):
    pred = np.clip(np.asarray(pred), 0, None)

    err = pred - y_arr_11y
    se = err ** 2

    out = {
        "rmse": fast_rmse_11y(y_arr_11y, pred),
        "adv_weighted_rmse": weighted_rmse_11y(y_arr_11y, pred, adv_weight_train_11y),
    }

    for b in bin_labels_11y:
        m = bin_masks_11y[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    out["delta_vs_11O"] = out["rmse"] - BEST_11O_OOF
    out["delta_vs_11X"] = out["rmse"] - seed_11x_score_11y

    return out


candidate_rows_11y = []

best_raw_11y = {
    "key": "seed_11X",
    "method": "previous_11X",
    "rmse": seed_11x_score_11y,
    "selection_score": seed_11x_score_11y,
    "oof": seed_11x_oof_11y.copy(),
    "test": seed_11x_test_11y.copy(),
}

best_robust_11y = best_raw_11y.copy()


def robust_penalty_11y(metrics):
    penalty = 0.0

    # Very-high/high are the risky part from 11X raw.
    seed_metrics = calc_metrics_11y(seed_11x_oof_11y)

    if metrics["very_high_rmse"] > seed_metrics["very_high_rmse"] + 0.025:
        penalty += 0.00060

    if metrics["high_rmse"] > seed_metrics["high_rmse"] + 0.020:
        penalty += 0.00040

    # Don't lose low-tail improvement too badly.
    if metrics["very_low_rmse"] > seed_metrics["very_low_rmse"] + 0.020:
        penalty += 0.00040

    if metrics["low_rmse"] > seed_metrics["low_rmse"] + 0.020:
        penalty += 0.00040

    # Public proxy: adversarial weighted RMSE should not be worse.
    if metrics["adv_weighted_rmse"] > seed_metrics["adv_weighted_rmse"] + 0.020:
        penalty += 0.00060

    if metrics["adv_weighted_rmse"] > metrics["rmse"] + 0.25:
        penalty += 0.00030

    return penalty


def evaluate_candidate_11y(key, method, pred_oof, pred_test):
    global best_raw_11y, best_robust_11y

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)

    metrics = calc_metrics_11y(pred_oof)
    metrics["robust_penalty"] = robust_penalty_11y(metrics)
    metrics["selection_score"] = metrics["rmse"] + metrics["robust_penalty"]

    row = {
        "key": key,
        "method": method,
        **metrics,
    }

    candidate_rows_11y.append(row)

    if metrics["rmse"] < best_raw_11y["rmse"]:
        best_raw_11y = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
        }

    if metrics["selection_score"] < best_robust_11y["selection_score"]:
        best_robust_11y = {
            "key": key,
            "method": method,
            "rmse": metrics["rmse"],
            "selection_score": metrics["selection_score"],
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
        }


# References
evaluate_candidate_11y(
    key="base_11O_adaptive",
    method="base",
    pred_oof=base_oof_11y,
    pred_test=base_test_11y,
)

evaluate_candidate_11y(
    key="seed_11W",
    method="previous_11W",
    pred_oof=seed_11w_oof_11y,
    pred_test=seed_11w_test_11y,
)

evaluate_candidate_11y(
    key="seed_11X",
    method="previous_11X",
    pred_oof=seed_11x_oof_11y,
    pred_test=seed_11x_test_11y,
)


# ==================================================
# PART 1 - Source frames
# ==================================================

print("=" * 100)
print("PART 1 - Source frames")

if "X_fe" in globals() and "X_test_fe" in globals():
    X_source_train_11y = X_fe.copy().reset_index(drop=True)
    X_source_test_11y = X_test_fe.copy().reset_index(drop=True)
    print("Using X_fe / X_test_fe.")
else:
    X_source_train_11y = train_raw.drop(columns=[TARGET], errors="ignore").copy().reset_index(drop=True)
    X_source_test_11y = test_raw.copy().reset_index(drop=True)
    print("Using train_raw / test_raw fallback.")

# Numeric base columns
numeric_cols_all_11y = [
    c for c in X_source_train_11y.columns
    if c in X_source_test_11y.columns
    and pd.api.types.is_numeric_dtype(X_source_train_11y[c])
    and pd.api.types.is_numeric_dtype(X_source_test_11y[c])
    and c not in [ID_COL, TARGET]
]

# Keep most columns, but cap to avoid silly width.
priority_keywords_11y = [
    "particle",
    "fine",
    "coarse",
    "acidity",
    "cation",
    "cec",
    "Ca_",
    "Mg_",
    "spectral",
    "band",
    "ratio",
    "index",
    "organic",
    "clay",
    "sand",
    "silt",
    "geo",
    "source",
]

priority_num_cols_11y = [
    c for c in numeric_cols_all_11y
    if any(k.lower() in c.lower() for k in priority_keywords_11y)
]

remaining_num_cols_11y = [
    c for c in numeric_cols_all_11y
    if c not in priority_num_cols_11y
]

selected_num_cols_11y = priority_num_cols_11y[:120] + remaining_num_cols_11y[:100]

X_num_train_11y = clean_numeric_frame_11y(X_source_train_11y[selected_num_cols_11y])
X_num_test_11y = clean_numeric_frame_11y(X_source_test_11y[selected_num_cols_11y])

print("Numeric cols selected:", len(selected_num_cols_11y))


# ==================================================
# PART 2 - Adversarial validation
# ==================================================

print("=" * 100)
print("PART 2 - Adversarial validation train vs test")

def make_adv_features_11y(Xtr, Xte):
    train_df = Xtr.copy().reset_index(drop=True)
    test_df = Xte.copy().reset_index(drop=True)

    numeric_cols = [
        c for c in train_df.columns
        if c in test_df.columns
        and pd.api.types.is_numeric_dtype(train_df[c])
        and pd.api.types.is_numeric_dtype(test_df[c])
        and c not in [ID_COL, TARGET]
    ]

    numeric_cols = numeric_cols[:150]

    adv_train = clean_numeric_frame_11y(train_df[numeric_cols])
    adv_test = clean_numeric_frame_11y(test_df[numeric_cols])

    # Add categorical frequency features for train-test shift.
    cat_cols = [
        c for c in train_df.columns
        if c in test_df.columns
        and c not in numeric_cols
        and c not in [ID_COL, TARGET]
    ]

    cat_cols = cat_cols[:40]

    for c in cat_cols:
        all_vals = pd.concat([
            train_df[c].astype(str).fillna("Unknown"),
            test_df[c].astype(str).fillna("Unknown")
        ], axis=0)

        freq = all_vals.value_counts(normalize=True)

        adv_train[f"adv_freq_{c}"] = train_df[c].astype(str).fillna("Unknown").map(freq).fillna(0).values
        adv_test[f"adv_freq_{c}"] = test_df[c].astype(str).fillna("Unknown").map(freq).fillna(0).values

        train_seen = set(train_df[c].astype(str).fillna("Unknown").unique())
        adv_test[f"adv_unseen_in_train_{c}"] = (~test_df[c].astype(str).fillna("Unknown").isin(train_seen)).astype(int).values
        adv_train[f"adv_unseen_in_train_{c}"] = 0

    return clean_numeric_frame_11y(adv_train), clean_numeric_frame_11y(adv_test)


X_adv_train_11y, X_adv_test_11y = make_adv_features_11y(
    X_source_train_11y,
    X_source_test_11y,
)

X_adv_all_11y = pd.concat([X_adv_train_11y, X_adv_test_11y], axis=0).reset_index(drop=True)
adv_label_11y = np.r_[np.zeros(n_train_11y), np.ones(n_test_11y)].astype(int)

adv_score_all_11y = np.zeros(n_train_11y + n_test_11y)

adv_cv_11y = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=8127,
)

adv_auc_scores_11y = []

for fold, (tr_idx, val_idx) in enumerate(adv_cv_11y.split(X_adv_all_11y, adv_label_11y), 1):
    clf = make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(
            max_iter=180,
            learning_rate=0.045,
            max_leaf_nodes=15,
            min_samples_leaf=25,
            l2_regularization=2.0,
            random_state=8100 + fold,
        ),
    )

    clf.fit(X_adv_all_11y.iloc[tr_idx], adv_label_11y[tr_idx])

    p_val = clf.predict_proba(X_adv_all_11y.iloc[val_idx])[:, 1]
    adv_score_all_11y[val_idx] = p_val

    auc = roc_auc_score(adv_label_11y[val_idx], p_val)
    adv_auc_scores_11y.append(auc)

    print(f"ADV fold {fold} AUC: {auc:.5f}")

adv_score_train_11y = adv_score_all_11y[:n_train_11y]
adv_score_test_11y = adv_score_all_11y[n_train_11y:]

adv_rank_train_11y = rank_against_train_11y(adv_score_train_11y, adv_score_train_11y)
adv_rank_test_11y = rank_against_train_11y(adv_score_train_11y, adv_score_test_11y)

# Public proxy weights
adv_weight_train_11y = 1.0 + 1.50 * adv_rank_train_11y
adv_weight_train_11y = adv_weight_train_11y / np.mean(adv_weight_train_11y)

print("=" * 100)
print("Adversarial validation summary")
print("ADV AUC mean:", np.mean(adv_auc_scores_11y), "std:", np.std(adv_auc_scores_11y))
print("Train adv score mean:", float(np.mean(adv_score_train_11y)))
print("Test adv score mean :", float(np.mean(adv_score_test_11y)))
print("Adv weight min/max  :", float(np.min(adv_weight_train_11y)), float(np.max(adv_weight_train_11y)))


# ==================================================
# PART 3 - Group keys and fold-safe target encoding
# ==================================================

print("=" * 100)
print("PART 3 - Fold-safe target encoding")

def get_group_col_11y(col, is_train=True):
    if is_train:
        if col in X_source_train_11y.columns:
            return X_source_train_11y[col].astype(str).fillna("Unknown").reset_index(drop=True)
        if col in train_raw.columns:
            return train_raw[col].astype(str).fillna("Unknown").reset_index(drop=True)
        return None
    else:
        if col in X_source_test_11y.columns:
            return X_source_test_11y[col].astype(str).fillna("Unknown").reset_index(drop=True)
        if col in test_raw.columns:
            return test_raw[col].astype(str).fillna("Unknown").reset_index(drop=True)
        return None


group_train_11y = pd.DataFrame(index=np.arange(n_train_11y))
group_test_11y = pd.DataFrame(index=np.arange(n_test_11y))

single_group_candidates_11y = [
    "source_id",
    "geo_zone_micro",
    "geo_zone_meso",
    "geo_zone_macro",
    "geo_meso_micro",
    "geo_macro_meso",
    "biome_land_cover",
    "land_cover_geo_macro",
    "biome",
    "land_cover",
    "soil_type",
]

for col in single_group_candidates_11y:
    tr_col = get_group_col_11y(col, True)
    te_col = get_group_col_11y(col, False)

    if tr_col is not None and te_col is not None:
        group_train_11y[col] = tr_col
        group_test_11y[col] = te_col

# Prediction bins
base_edges5_11y = np.unique(np.quantile(base_oof_11y, np.linspace(0, 1, 6)))
base_edges10_11y = np.unique(np.quantile(base_oof_11y, np.linspace(0, 1, 11)))
x_edges5_11y = np.unique(np.quantile(seed_11x_oof_11y, np.linspace(0, 1, 6)))

if len(base_edges5_11y) > 2:
    group_train_11y["base_bin5"] = pd.cut(
        base_oof_11y,
        bins=base_edges5_11y,
        include_lowest=True,
        labels=False,
    ).astype(str)

    group_test_11y["base_bin5"] = pd.cut(
        base_test_11y,
        bins=base_edges5_11y,
        include_lowest=True,
        labels=False,
    ).astype(str)

if len(base_edges10_11y) > 2:
    group_train_11y["base_bin10"] = pd.cut(
        base_oof_11y,
        bins=base_edges10_11y,
        include_lowest=True,
        labels=False,
    ).astype(str)

    group_test_11y["base_bin10"] = pd.cut(
        base_test_11y,
        bins=base_edges10_11y,
        include_lowest=True,
        labels=False,
    ).astype(str)

if len(x_edges5_11y) > 2:
    group_train_11y["x_bin5"] = pd.cut(
        seed_11x_oof_11y,
        bins=x_edges5_11y,
        include_lowest=True,
        labels=False,
    ).astype(str)

    group_test_11y["x_bin5"] = pd.cut(
        seed_11x_test_11y,
        bins=x_edges5_11y,
        include_lowest=True,
        labels=False,
    ).astype(str)


def add_composite_11y(name, cols):
    if all(c in group_train_11y.columns for c in cols) and all(c in group_test_11y.columns for c in cols):
        group_train_11y[name] = group_train_11y[cols].astype(str).agg("__".join, axis=1)
        group_test_11y[name] = group_test_11y[cols].astype(str).agg("__".join, axis=1)


# Interactions. These are the main new signal.
add_composite_11y("source_geo_meso", ["source_id", "geo_zone_meso"])
add_composite_11y("source_geo_micro", ["source_id", "geo_zone_micro"])
add_composite_11y("source_base5", ["source_id", "base_bin5"])
add_composite_11y("source_base10", ["source_id", "base_bin10"])
add_composite_11y("source_xbin5", ["source_id", "x_bin5"])
add_composite_11y("geo_micro_base5", ["geo_zone_micro", "base_bin5"])
add_composite_11y("geo_micro_base10", ["geo_zone_micro", "base_bin10"])
add_composite_11y("geo_meso_base5", ["geo_zone_meso", "base_bin5"])
add_composite_11y("geo_meso_micro_base5", ["geo_meso_micro", "base_bin5"])
add_composite_11y("biome_land_base5", ["biome_land_cover", "base_bin5"])
add_composite_11y("land_geo_base5", ["land_cover_geo_macro", "base_bin5"])
add_composite_11y("biome_xbin5", ["biome", "x_bin5"])

group_cols_11y = list(group_train_11y.columns)

print("Group columns for TE:", group_cols_11y)


def fold_safe_te_one_col_11y(col, shrink_k=80):
    global_y = float(np.mean(y_arr_11y))
    global_res_base = float(np.mean(y_arr_11y - base_oof_11y))
    global_res_x = float(np.mean(y_arr_11y - seed_11x_oof_11y))

    te_train = pd.DataFrame(index=np.arange(n_train_11y))
    te_test_sum = pd.DataFrame({
        f"{col}_te_mean": np.zeros(n_test_11y),
        f"{col}_te_res_base": np.zeros(n_test_11y),
        f"{col}_te_res_x": np.zeros(n_test_11y),
        f"{col}_te_log_count": np.zeros(n_test_11y),
        f"{col}_te_std": np.zeros(n_test_11y),
    })

    for new_col in te_test_sum.columns:
        te_train[new_col] = np.nan

    key_train_all = group_train_11y[col].astype(str)
    key_test_all = group_test_11y[col].astype(str)

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        key_tr = key_train_all.iloc[tr_idx]
        key_val = key_train_all.iloc[val_idx]

        tmp = pd.DataFrame({
            "key": key_tr.values,
            "y": y_arr_11y[tr_idx],
            "res_base": y_arr_11y[tr_idx] - base_oof_11y[tr_idx],
            "res_x": y_arr_11y[tr_idx] - seed_11x_oof_11y[tr_idx],
        })

        agg = (
            tmp.groupby("key")
            .agg(
                count=("y", "count"),
                y_mean=("y", "mean"),
                y_std=("y", "std"),
                res_base_mean=("res_base", "mean"),
                res_x_mean=("res_x", "mean"),
            )
            .reset_index()
        )

        agg["shrink"] = agg["count"] / (agg["count"] + shrink_k)

        agg[f"{col}_te_mean"] = global_y + (agg["y_mean"] - global_y) * agg["shrink"]
        agg[f"{col}_te_res_base"] = global_res_base + (agg["res_base_mean"] - global_res_base) * agg["shrink"]
        agg[f"{col}_te_res_x"] = global_res_x + (agg["res_x_mean"] - global_res_x) * agg["shrink"]
        agg[f"{col}_te_log_count"] = np.log1p(agg["count"])
        agg[f"{col}_te_std"] = agg["y_std"].fillna(0)

        use_cols = [
            "key",
            f"{col}_te_mean",
            f"{col}_te_res_base",
            f"{col}_te_res_x",
            f"{col}_te_log_count",
            f"{col}_te_std",
        ]

        mapping_df = agg[use_cols].set_index("key")

        mapped_val = key_val.map(mapping_df.to_dict(orient="index"))

        # Faster explicit joins
        val_df = pd.DataFrame({"key": key_val.values})
        val_df = val_df.merge(mapping_df.reset_index(), on="key", how="left")

        val_df[f"{col}_te_mean"] = val_df[f"{col}_te_mean"].fillna(global_y)
        val_df[f"{col}_te_res_base"] = val_df[f"{col}_te_res_base"].fillna(global_res_base)
        val_df[f"{col}_te_res_x"] = val_df[f"{col}_te_res_x"].fillna(global_res_x)
        val_df[f"{col}_te_log_count"] = val_df[f"{col}_te_log_count"].fillna(0)
        val_df[f"{col}_te_std"] = val_df[f"{col}_te_std"].fillna(0)

        for c in te_test_sum.columns:
            te_train.loc[val_idx, c] = val_df[c].values

        test_df = pd.DataFrame({"key": key_test_all.values})
        test_df = test_df.merge(mapping_df.reset_index(), on="key", how="left")

        test_df[f"{col}_te_mean"] = test_df[f"{col}_te_mean"].fillna(global_y)
        test_df[f"{col}_te_res_base"] = test_df[f"{col}_te_res_base"].fillna(global_res_base)
        test_df[f"{col}_te_res_x"] = test_df[f"{col}_te_res_x"].fillna(global_res_x)
        test_df[f"{col}_te_log_count"] = test_df[f"{col}_te_log_count"].fillna(0)
        test_df[f"{col}_te_std"] = test_df[f"{col}_te_std"].fillna(0)

        for c in te_test_sum.columns:
            te_test_sum[c] += test_df[c].values / len(folds)

    te_train = te_train.fillna({
        f"{col}_te_mean": global_y,
        f"{col}_te_res_base": global_res_base,
        f"{col}_te_res_x": global_res_x,
        f"{col}_te_log_count": 0,
        f"{col}_te_std": 0,
    })

    return te_train, te_test_sum


te_train_parts_11y = []
te_test_parts_11y = []

for col in group_cols_11y:
    print("Building TE:", col)

    # Larger k for high-cardinality interaction groups.
    nunique = group_train_11y[col].nunique()
    if nunique > 500:
        k = 160
    elif nunique > 200:
        k = 120
    else:
        k = 80

    te_tr, te_te = fold_safe_te_one_col_11y(col, shrink_k=k)

    te_train_parts_11y.append(te_tr)
    te_test_parts_11y.append(te_te)

    gc.collect()

TE_train_11y = pd.concat(te_train_parts_11y, axis=1)
TE_test_11y = pd.concat(te_test_parts_11y, axis=1)

TE_train_11y = clean_numeric_frame_11y(TE_train_11y)
TE_test_11y = clean_numeric_frame_11y(TE_test_11y)

print("TE train shape:", TE_train_11y.shape)
print("TE test shape :", TE_test_11y.shape)


# ==================================================
# PART 4 - Final 11Y model matrix
# ==================================================

print("=" * 100)
print("PART 4 - Final 11Y feature matrix")

base_rank_oof_11y = rank_against_train_11y(base_oof_11y, base_oof_11y)
base_rank_test_11y = rank_against_train_11y(base_oof_11y, base_test_11y)

x_rank_oof_11y = rank_against_train_11y(seed_11x_oof_11y, seed_11x_oof_11y)
x_rank_test_11y = rank_against_train_11y(seed_11x_oof_11y, seed_11x_test_11y)

extra_train_11y = pd.DataFrame({
    "11y_base_pred": base_oof_11y,
    "11y_base_rank": base_rank_oof_11y,
    "11y_11w_pred": seed_11w_oof_11y,
    "11y_11x_pred": seed_11x_oof_11y,
    "11y_11x_rank": x_rank_oof_11y,
    "11y_11x_minus_base": seed_11x_oof_11y - base_oof_11y,
    "11y_11x_minus_11w": seed_11x_oof_11y - seed_11w_oof_11y,
    "11y_adv_score": adv_score_train_11y,
    "11y_adv_rank": adv_rank_train_11y,
})

extra_test_11y = pd.DataFrame({
    "11y_base_pred": base_test_11y,
    "11y_base_rank": base_rank_test_11y,
    "11y_11w_pred": seed_11w_test_11y,
    "11y_11x_pred": seed_11x_test_11y,
    "11y_11x_rank": x_rank_test_11y,
    "11y_11x_minus_base": seed_11x_test_11y - base_test_11y,
    "11y_11x_minus_11w": seed_11x_test_11y - seed_11w_test_11y,
    "11y_adv_score": adv_score_test_11y,
    "11y_adv_rank": adv_rank_test_11y,
})

# Add previous model predictions if available.
if "model_outputs" in globals():
    for m in ["catboost", "lightgbm", "xgboost"]:
        if m in model_outputs:
            if "oof" in model_outputs[m] and "test_pred" in model_outputs[m]:
                extra_train_11y[f"11y_prev_{m}_pred"] = np.asarray(model_outputs[m]["oof"])
                extra_test_11y[f"11y_prev_{m}_pred"] = np.asarray(model_outputs[m]["test_pred"])

optional_pred_pairs_11y = [
    ("global_oof", "global_test"),
    ("s6_oof", "s6_test"),
    ("ridge3_oof", "ridge3_test"),
]

for oof_name, test_name in optional_pred_pairs_11y:
    if oof_name in globals() and test_name in globals():
        extra_train_11y[f"11y_prev_{oof_name}"] = np.asarray(globals()[oof_name])
        extra_test_11y[f"11y_prev_{oof_name}"] = np.asarray(globals()[test_name])

X_11y = pd.concat(
    [
        X_num_train_11y.reset_index(drop=True),
        TE_train_11y.reset_index(drop=True),
        extra_train_11y.reset_index(drop=True),
    ],
    axis=1,
)

X_test_11y = pd.concat(
    [
        X_num_test_11y.reset_index(drop=True),
        TE_test_11y.reset_index(drop=True),
        extra_test_11y.reset_index(drop=True),
    ],
    axis=1,
)

X_11y = clean_numeric_frame_11y(X_11y)
X_test_11y = clean_numeric_frame_11y(X_test_11y)

# Remove duplicate columns if any
X_11y = X_11y.loc[:, ~X_11y.columns.duplicated()].copy()
X_test_11y = X_test_11y[X_11y.columns].copy()

print("X_11y shape:", X_11y.shape)
print("X_test_11y shape:", X_test_11y.shape)


# ==================================================
# PART 5 - Sample weights
# ==================================================

print("=" * 100)
print("PART 5 - Sample weights")

q20_11y = np.quantile(y_arr_11y, 0.20)
q80_11y = np.quantile(y_arr_11y, 0.80)
q90_11y = np.quantile(y_arr_11y, 0.90)

resid_seed_11y = y_arr_11y - seed_11x_oof_11y
abs_resid_seed_11y = np.abs(resid_seed_11y)

weight_bank_11y = {}

weight_bank_11y["normal"] = np.ones(n_train_11y)

weight_bank_11y["adv1"] = 1.0 + 1.00 * adv_rank_train_11y
weight_bank_11y["adv2"] = 1.0 + 2.00 * adv_rank_train_11y

weight_bank_11y["adv_tail"] = (
    1.0
    + 1.25 * adv_rank_train_11y
    + 0.35 * (y_arr_11y <= q20_11y)
    + 0.35 * (y_arr_11y >= q80_11y)
)

weight_bank_11y["adv_resid"] = (
    1.0
    + 1.25 * adv_rank_train_11y
    + 0.40 * rank_against_train_11y(abs_resid_seed_11y, abs_resid_seed_11y)
)

weight_bank_11y["signed_tail"] = (
    1.0
    + 0.50 * (y_arr_11y <= q20_11y)
    + 0.50 * (y_arr_11y >= q90_11y)
    + 0.75 * adv_rank_train_11y
)

for k in list(weight_bank_11y.keys()):
    w = np.asarray(weight_bank_11y[k], dtype=float)
    w = np.clip(w, 0.25, 5.0)
    w = w / np.mean(w)
    weight_bank_11y[k] = w

    print(k, "min/mean/max:", float(w.min()), float(w.mean()), float(w.max()))


# ==================================================
# PART 6 - Train 11Y base models
# ==================================================

print("=" * 100)
print("PART 6 - Train 11Y base models")

model_outputs_11y = {}


def add_model_output_11y(name, oof, test, fold_scores, model_type):
    oof = np.clip(np.asarray(oof), 0, None)
    test = np.clip(np.asarray(test), 0, None)

    score = fast_rmse_11y(y_arr_11y, oof)
    adv_score = weighted_rmse_11y(y_arr_11y, oof, adv_weight_train_11y)

    model_outputs_11y[name] = {
        "name": name,
        "type": model_type,
        "oof": oof,
        "test": test,
        "rmse": score,
        "adv_weighted_rmse": adv_score,
        "fold_scores": fold_scores,
    }

    print("=" * 100)
    print(f"Model {name} finished")
    print("OOF RMSE:", score)
    print("ADV weighted RMSE:", adv_score)
    print("Fold scores:", fold_scores)

    evaluate_candidate_11y(
        key=name,
        method=f"raw_{model_type}",
        pred_oof=oof,
        pred_test=test,
    )


def train_lgb_11y(name, params, weight_name):
    if not RUN_LGBM_11Y:
        print("Skipping LGBM:", name)
        return

    oof = np.zeros(n_train_11y)
    test_sum = np.zeros(n_test_11y)
    fold_scores = []

    w_all = weight_bank_11y[weight_name]

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr = X_11y.iloc[tr_idx]
        X_val = X_11y.iloc[val_idx]

        y_tr = y_arr_11y[tr_idx]
        y_val = y_arr_11y[val_idx]

        w_tr = w_all[tr_idx]
        w_val = w_all[val_idx]

        model = lgb.LGBMRegressor(**params)

        try:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=w_tr,
                eval_set=[(X_val, y_val)],
                eval_sample_weight=[w_val],
                eval_metric="rmse",
                callbacks=[
                    lgb.early_stopping(250, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )
        except TypeError:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=w_tr,
                eval_set=[(X_val, y_val)],
                eval_metric="rmse",
            )

        val_pred = np.clip(model.predict(X_val), 0, None)
        test_pred = np.clip(model.predict(X_test_11y), 0, None)

        oof[val_idx] = val_pred
        test_sum += test_pred / len(folds)

        fold_score = fast_rmse_11y(y_val, val_pred)
        fold_scores.append(fold_score)

        print(f"{name} fold {fold}: {fold_score:.6f}")

        del model, X_tr, X_val
        gc.collect()

    add_model_output_11y(name, oof, test_sum, fold_scores, "lgbm")


def train_xgb_11y(name, params, weight_name):
    if not RUN_XGB_11Y:
        print("Skipping XGB:", name)
        return

    oof = np.zeros(n_train_11y)
    test_sum = np.zeros(n_test_11y)
    fold_scores = []

    w_all = weight_bank_11y[weight_name]

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr = X_11y.iloc[tr_idx]
        X_val = X_11y.iloc[val_idx]

        y_tr = y_arr_11y[tr_idx]
        y_val = y_arr_11y[val_idx]

        w_tr = w_all[tr_idx]
        w_val = w_all[val_idx]

        model = xgb.XGBRegressor(**params)

        try:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=w_tr,
                eval_set=[(X_val, y_val)],
                sample_weight_eval_set=[w_val],
                verbose=False,
            )
        except TypeError:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=w_tr,
                eval_set=[(X_val, y_val)],
                verbose=False,
            )

        val_pred = np.clip(model.predict(X_val), 0, None)
        test_pred = np.clip(model.predict(X_test_11y), 0, None)

        oof[val_idx] = val_pred
        test_sum += test_pred / len(folds)

        fold_score = fast_rmse_11y(y_val, val_pred)
        fold_scores.append(fold_score)

        print(f"{name} fold {fold}: {fold_score:.6f}")

        del model, X_tr, X_val
        gc.collect()

    add_model_output_11y(name, oof, test_sum, fold_scores, "xgb")


def train_cat_11y(name, params, weight_name):
    if not RUN_CAT_11Y:
        print("Skipping CatBoost:", name)
        return

    oof = np.zeros(n_train_11y)
    test_sum = np.zeros(n_test_11y)
    fold_scores = []

    w_all = weight_bank_11y[weight_name]

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr = X_11y.iloc[tr_idx]
        X_val = X_11y.iloc[val_idx]

        y_tr = y_arr_11y[tr_idx]
        y_val = y_arr_11y[val_idx]

        w_tr = w_all[tr_idx]
        w_val = w_all[val_idx]

        train_pool = Pool(X_tr, y_tr, weight=w_tr)
        val_pool = Pool(X_val, y_val, weight=w_val)
        test_pool = Pool(X_test_11y)

        model = CatBoostRegressor(**params)

        model.fit(
            train_pool,
            eval_set=val_pool,
            use_best_model=True,
            verbose=False,
        )

        val_pred = np.clip(model.predict(val_pool), 0, None)
        test_pred = np.clip(model.predict(test_pool), 0, None)

        oof[val_idx] = val_pred
        test_sum += test_pred / len(folds)

        fold_score = fast_rmse_11y(y_val, val_pred)
        fold_scores.append(fold_score)

        print(f"{name} fold {fold}: {fold_score:.6f}")

        del model, train_pool, val_pool, test_pool, X_tr, X_val
        gc.collect()

    add_model_output_11y(name, oof, test_sum, fold_scores, "cat")


# Model params
lgb_params_base_11y = dict(
    n_estimators=6000,
    learning_rate=0.018,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=28,
    subsample=0.85,
    colsample_bytree=0.78,
    reg_alpha=0.15,
    reg_lambda=2.0,
    objective="regression",
    random_state=9111,
    n_jobs=-1,
    verbosity=-1,
)

lgb_params_deep_11y = dict(
    n_estimators=5000,
    learning_rate=0.020,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=35,
    subsample=0.82,
    colsample_bytree=0.72,
    reg_alpha=0.30,
    reg_lambda=3.0,
    objective="regression",
    random_state=9112,
    n_jobs=-1,
    verbosity=-1,
)

xgb_params_11y = dict(
    n_estimators=2600,
    learning_rate=0.018,
    max_depth=5,
    min_child_weight=8,
    subsample=0.82,
    colsample_bytree=0.76,
    reg_alpha=0.15,
    reg_lambda=4.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=9211,
    n_jobs=-1,
)

cat_params_11y = dict(
    iterations=4500,
    learning_rate=0.018,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=9311,
    l2_leaf_reg=7.0,
    random_strength=0.7,
    bagging_temperature=0.4,
    od_type="Iter",
    od_wait=250,
    allow_writing_files=False,
)


# Train models
train_lgb_11y(
    name="lgb11y_te_adv1",
    params={**lgb_params_base_11y, "random_state": 9111},
    weight_name="adv1",
)

train_lgb_11y(
    name="lgb11y_te_adv_resid",
    params={**lgb_params_base_11y, "random_state": 9112},
    weight_name="adv_resid",
)

train_lgb_11y(
    name="lgb11y_te_deep_adv_tail",
    params={**lgb_params_deep_11y, "random_state": 9113},
    weight_name="adv_tail",
)

train_xgb_11y(
    name="xgb11y_te_adv_tail",
    params={**xgb_params_11y, "random_state": 9211},
    weight_name="adv_tail",
)

train_cat_11y(
    name="cat11y_te_signed_tail",
    params={**cat_params_11y, "random_seed": 9311},
    weight_name="signed_tail",
)

gc.collect()


# ==================================================
# PART 7 - Blend and stack with 11X
# ==================================================

print("=" * 100)
print("PART 7 - Blend and stack with 11X")

model_summary_11y = []

for name, obj in model_outputs_11y.items():
    model_summary_11y.append({
        "name": name,
        "type": obj["type"],
        "rmse": obj["rmse"],
        "adv_weighted_rmse": obj["adv_weighted_rmse"],
        "fold_std": float(np.std(obj["fold_scores"])),
    })

model_summary_11y = pd.DataFrame(model_summary_11y).sort_values("rmse")

print("11Y model summary")
display(model_summary_11y)

# Simple blends with seed 11X
for name, obj in model_outputs_11y.items():
    m_oof = obj["oof"]
    m_test = obj["test"]

    for w in [-0.15, -0.10, -0.05, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
        pred_oof = np.clip((1.0 - w) * seed_11x_oof_11y + w * m_oof, 0, None)
        pred_test = np.clip((1.0 - w) * seed_11x_test_11y + w * m_test, 0, None)

        evaluate_candidate_11y(
            key=f"blend_11x_{name}_w{w}",
            method="blend_11x_model",
            pred_oof=pred_oof,
            pred_test=pred_test,
        )

# Pairwise model ensemble then blend with 11X
model_names_11y = list(model_outputs_11y.keys())

for i in range(len(model_names_11y)):
    for j in range(i + 1, len(model_names_11y)):
        n1 = model_names_11y[i]
        n2 = model_names_11y[j]

        oof_pair = 0.5 * model_outputs_11y[n1]["oof"] + 0.5 * model_outputs_11y[n2]["oof"]
        test_pair = 0.5 * model_outputs_11y[n1]["test"] + 0.5 * model_outputs_11y[n2]["test"]

        for w in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
            pred_oof = np.clip((1.0 - w) * seed_11x_oof_11y + w * oof_pair, 0, None)
            pred_test = np.clip((1.0 - w) * seed_11x_test_11y + w * test_pair, 0, None)

            evaluate_candidate_11y(
                key=f"blend_11x_pair_{n1}_{n2}_w{w}",
                method="blend_11x_pair",
                pred_oof=pred_oof,
                pred_test=pred_test,
            )


# ==================================================
# PART 8 - Fold-safe meta stacking
# ==================================================

print("=" * 100)
print("PART 8 - Fold-safe meta stacking")

stack_feature_train_11y = pd.DataFrame({
    "base_11O": base_oof_11y,
    "seed_11W": seed_11w_oof_11y,
    "seed_11X": seed_11x_oof_11y,
    "adv_rank": adv_rank_train_11y,
})

stack_feature_test_11y = pd.DataFrame({
    "base_11O": base_test_11y,
    "seed_11W": seed_11w_test_11y,
    "seed_11X": seed_11x_test_11y,
    "adv_rank": adv_rank_test_11y,
})

for name, obj in model_outputs_11y.items():
    stack_feature_train_11y[name] = obj["oof"]
    stack_feature_test_11y[name] = obj["test"]

# Add useful deltas
for name in model_outputs_11y.keys():
    stack_feature_train_11y[f"{name}_minus_x"] = stack_feature_train_11y[name] - seed_11x_oof_11y
    stack_feature_test_11y[f"{name}_minus_x"] = stack_feature_test_11y[name] - seed_11x_test_11y

stack_feature_train_11y = clean_numeric_frame_11y(stack_feature_train_11y)
stack_feature_test_11y = clean_numeric_frame_11y(stack_feature_test_11y)

stack_alphas_11y = [0.01, 0.1, 1.0, 10.0, 100.0]
stack_weight_names_11y = ["normal", "adv1", "adv_resid", "signed_tail"]

for alpha in stack_alphas_11y:
    for weight_name in stack_weight_names_11y:
        w_all = weight_bank_11y[weight_name]

        stack_oof = np.zeros(n_train_11y)
        stack_test_sum = np.zeros(n_test_11y)
        coef_rows = []

        for fold, (tr_idx, val_idx) in enumerate(folds, 1):
            model = make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler(),
                Ridge(alpha=alpha, random_state=9400 + fold),
            )

            model.fit(
                stack_feature_train_11y.iloc[tr_idx],
                y_arr_11y[tr_idx],
                ridge__sample_weight=w_all[tr_idx],
            )

            val_pred = np.clip(model.predict(stack_feature_train_11y.iloc[val_idx]), 0, None)
            test_pred = np.clip(model.predict(stack_feature_test_11y), 0, None)

            stack_oof[val_idx] = val_pred
            stack_test_sum += test_pred / len(folds)

        evaluate_candidate_11y(
            key=f"ridge_stack_alpha{alpha}_{weight_name}",
            method="ridge_stack",
            pred_oof=stack_oof,
            pred_test=stack_test_sum,
        )

# Positive linear stack on prediction columns only
pred_cols_for_pos_stack_11y = ["base_11O", "seed_11W", "seed_11X"] + model_names_11y

if len(model_names_11y) > 0:
    for weight_name in ["normal", "adv1", "adv_resid"]:
        w_all = weight_bank_11y[weight_name]

        pos_oof = np.zeros(n_train_11y)
        pos_test_sum = np.zeros(n_test_11y)

        for fold, (tr_idx, val_idx) in enumerate(folds, 1):
            model = LinearRegression(positive=True)

            model.fit(
                stack_feature_train_11y.iloc[tr_idx][pred_cols_for_pos_stack_11y],
                y_arr_11y[tr_idx],
                sample_weight=w_all[tr_idx],
            )

            val_pred = np.clip(
                model.predict(stack_feature_train_11y.iloc[val_idx][pred_cols_for_pos_stack_11y]),
                0,
                None,
            )

            test_pred = np.clip(
                model.predict(stack_feature_test_11y[pred_cols_for_pos_stack_11y]),
                0,
                None,
            )

            pos_oof[val_idx] = val_pred
            pos_test_sum += test_pred / len(folds)

        evaluate_candidate_11y(
            key=f"positive_linear_stack_{weight_name}",
            method="positive_linear_stack",
            pred_oof=pos_oof,
            pred_test=pos_test_sum,
        )


# ==================================================
# PART 9 - Final selection
# ==================================================

print("=" * 100)
print("PART 9 - Final selection")

candidate_df_11y = pd.DataFrame(candidate_rows_11y)
candidate_df_11y = candidate_df_11y.sort_values("rmse").reset_index(drop=True)

candidate_df_11y_robust = candidate_df_11y.sort_values("selection_score").reset_index(drop=True)

print("Top 11Y candidates by raw RMSE")
display(candidate_df_11y.head(80))

print("Top 11Y candidates by robust selection score")
display(candidate_df_11y_robust.head(80))

print("=" * 100)
print("Selected 11Y candidates")

print("Raw best key     :", best_raw_11y["key"])
print("Raw best method  :", best_raw_11y["method"])
print("Raw best RMSE    :", best_raw_11y["rmse"])
print("Raw selection    :", best_raw_11y["selection_score"])
print("Raw delta vs 11O :", best_raw_11y["rmse"] - BEST_11O_OOF)
print("Raw delta vs 11X :", best_raw_11y["rmse"] - seed_11x_score_11y)

print("-" * 100)

print("Robust best key     :", best_robust_11y["key"])
print("Robust best method  :", best_robust_11y["method"])
print("Robust best RMSE    :", best_robust_11y["rmse"])
print("Robust selection    :", best_robust_11y["selection_score"])
print("Robust delta vs 11O :", best_robust_11y["rmse"] - BEST_11O_OOF)
print("Robust delta vs 11X :", best_robust_11y["rmse"] - seed_11x_score_11y)

raw_report_11y = target_bin_report_11y(
    best_raw_11y["oof"],
    f"11Y raw best: {best_raw_11y['key']}",
)

robust_report_11y = target_bin_report_11y(
    best_robust_11y["oof"],
    f"11Y robust best: {best_robust_11y['key']}",
)


# ==================================================
# PART 10 - Bin comparison
# ==================================================

print("=" * 100)
print("Bin comparison: 11O vs 11X seed vs 11Y raw vs 11Y robust")

compare_11y = train_raw[[ID_COL, TARGET]].copy()

compare_11y["base_pred"] = base_oof_11y
compare_11y["x_seed_pred"] = seed_11x_oof_11y
compare_11y["y_raw_pred"] = best_raw_11y["oof"]
compare_11y["y_robust_pred"] = best_robust_11y["oof"]

for col in ["base_pred", "x_seed_pred", "y_raw_pred", "y_robust_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")

    compare_11y[err_col] = compare_11y[col] - compare_11y[TARGET]
    compare_11y[se_col] = compare_11y[err_col] ** 2

compare_11y["target_bin_5"] = pd.qcut(
    compare_11y[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11y = (
    compare_11y.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_pred_mean=("base_pred", "mean"),
        x_seed_pred_mean=("x_seed_pred", "mean"),
        y_raw_pred_mean=("y_raw_pred", "mean"),
        y_robust_pred_mean=("y_robust_pred", "mean"),

        base_bias=("base_error", "mean"),
        x_seed_bias=("x_seed_error", "mean"),
        y_raw_bias=("y_raw_error", "mean"),
        y_robust_bias=("y_robust_error", "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        x_seed_rmse=("x_seed_se", lambda x: np.sqrt(np.mean(x))),
        y_raw_rmse=("y_raw_se", lambda x: np.sqrt(np.mean(x))),
        y_robust_rmse=("y_robust_se", lambda x: np.sqrt(np.mean(x))),
    )
    .reset_index()
)

bin_compare_11y["x_seed_rmse_delta_vs_base"] = (
    bin_compare_11y["x_seed_rmse"] - bin_compare_11y["base_rmse"]
)

bin_compare_11y["y_raw_rmse_delta_vs_base"] = (
    bin_compare_11y["y_raw_rmse"] - bin_compare_11y["base_rmse"]
)

bin_compare_11y["y_robust_rmse_delta_vs_base"] = (
    bin_compare_11y["y_robust_rmse"] - bin_compare_11y["base_rmse"]
)

bin_compare_11y["y_raw_minus_x_seed_rmse"] = (
    bin_compare_11y["y_raw_rmse"] - bin_compare_11y["x_seed_rmse"]
)

bin_compare_11y["y_robust_minus_x_seed_rmse"] = (
    bin_compare_11y["y_robust_rmse"] - bin_compare_11y["x_seed_rmse"]
)

display(bin_compare_11y.sort_values("base_rmse", ascending=False))


# ==================================================
# PART 11 - Save submissions
# ==================================================

print("=" * 100)
print("Save 11Y submissions")

sub_11y_raw = save_submission_11y(
    best_raw_11y["test"],
    "submission_11y_raw_best.csv",
)

sub_11y_robust = save_submission_11y(
    best_robust_11y["test"],
    "submission_11y_robust_best.csv",
)

sub_11x_ref_from_11y = save_submission_11y(
    seed_11x_test_11y,
    "submission_11x_reference_from_11y.csv",
)

sub_11o_ref_from_11y = save_submission_11y(
    base_test_11y,
    "submission_11o_reference_from_11y.csv",
)


# ==================================================
# PART 12 - Decision guide
# ==================================================

print("=" * 100)
print("11Y Decision Guide")

print(f"11O adaptive OOF        : {base_score_11y:.6f}")
print(f"11X seed OOF            : {seed_11x_score_11y:.6f}")
print(f"11Y raw best OOF        : {best_raw_11y['rmse']:.6f}")
print(f"11Y robust best OOF     : {best_robust_11y['rmse']:.6f}")

print(f"11Y raw delta vs 11O    : {best_raw_11y['rmse'] - BEST_11O_OOF:.6f}")
print(f"11Y robust delta vs 11O : {best_robust_11y['rmse'] - BEST_11O_OOF:.6f}")

print(f"11Y raw delta vs 11X    : {best_raw_11y['rmse'] - seed_11x_score_11y:.6f}")
print(f"11Y robust delta vs 11X : {best_robust_11y['rmse'] - seed_11x_score_11y:.6f}")

if best_raw_11y["rmse"] < seed_11x_score_11y - 0.050:
    print("Raw best decision: STRONG. This is the kind of jump needed for top-10 chasing.")
elif best_raw_11y["rmse"] < seed_11x_score_11y - 0.015:
    print("Raw best decision: GOOD new-model improvement. Submit if bin and public proxy are safe.")
elif best_raw_11y["rmse"] < seed_11x_score_11y - 0.003:
    print("Raw best decision: modest improvement. Submit as probe.")
elif best_raw_11y["rmse"] < seed_11x_score_11y:
    print("Raw best decision: tiny improvement. Not enough for top-10, submit only if slots spare.")
else:
    print("Raw best decision: did not beat 11X. Keep 11X.")

if best_robust_11y["rmse"] < seed_11x_score_11y - 0.050:
    print("Robust best decision: STRONG and safer. Prefer this.")
elif best_robust_11y["rmse"] < seed_11x_score_11y - 0.015:
    print("Robust best decision: GOOD. Prefer this if raw bins are risky.")
elif best_robust_11y["rmse"] < seed_11x_score_11y - 0.003:
    print("Robust best decision: modest improvement. Submit as probe.")
elif best_robust_11y["rmse"] < seed_11x_score_11y:
    print("Robust best decision: tiny improvement. Not enough for top-10, but safer probe.")
else:
    print("Robust best decision: did not beat 11X. Keep 11X.")

print("=" * 100)
print("Please screenshot:")
print("1. ADV AUC summary")
print("2. model_summary_11y")
print("3. candidate_df_11y.head(80)")
print("4. candidate_df_11y_robust.head(80)")
print("5. raw_report_11y")
print("6. robust_report_11y")
print("7. bin_compare_11y")

In [ ]:
# ==================================================
# DOWNLOAD 11Y RAW + ROBUST BEST
# ==================================================

from google.colab import files
import os

filenames = [
    "submission_11y_raw_best.csv",
    "submission_11y_robust_best.csv",
    "submission_11x_reference_from_11y.csv",
]

for filename in filenames:
    if os.path.exists(filename):
        print(f"Downloading: {filename}")
        files.download(filename)
    else:
        print(f"Not found: {filename}")

In [ ]:
# ==================================================
# SECTION 11Z - SINGLE PUBLIC-SAFE SUBMISSION
# ==================================================
# Goal:
# 1. Internally test 11Z public-safe 11X family.
# 2. Internally test 12A cluster-gated residual.
# 3. Internally test 12B residual model over 11X.
# 4. Select ONE candidate using public_safe_score.
# 5. Save ONLY ONE file:
#       submission_12abc_single_public_safe.csv
#
# Paste AFTER SECTION 11X.
#
# Important:
# - Does NOT use 11Y as anchor.
# - Does NOT use XGB negative anti-signal.
# - Does NOT save many submissions.
# - Output is only one final selected public-safe CSV.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor


# ==================================================
# PART 0 - Setup
# ==================================================

print("=" * 100)
print("SECTION 12ABC - SINGLE PUBLIC-SAFE SUBMISSION")

BEST_11O_OOF = 11.26696803195285

required_vars_12abc = [
    "train_raw",
    "test_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "folds",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "best_raw_11x",
    "best_robust_11x",
]

missing_12abc = [v for v in required_vars_12abc if v not in globals()]
if len(missing_12abc) > 0:
    raise NameError(f"Missing required variables for 12ABC: {missing_12abc}")

y_arr_12abc = y.values
n_train_12abc = len(y_arr_12abc)
n_test_12abc = sample_submission.shape[0]

base_oof_12abc = np.clip(np.asarray(adaptive_oof), 0, None)
base_test_12abc = np.clip(np.asarray(final_adaptive_test), 0, None)

base_score_12abc = rmse(y, base_oof_12abc)

# 11X anchors
x_raw_oof_12abc = np.clip(np.asarray(best_raw_11x["oof"]), 0, None)
x_raw_test_12abc = np.clip(np.asarray(best_raw_11x["test"]), 0, None)
x_raw_score_12abc = float(best_raw_11x["rmse"])

x_rob_oof_12abc = np.clip(np.asarray(best_robust_11x["oof"]), 0, None)
x_rob_test_12abc = np.clip(np.asarray(best_robust_11x["test"]), 0, None)
x_rob_score_12abc = float(best_robust_11x["rmse"])

# 11W anchor if available
if "best_raw_11w" in globals() and "best_robust_11w" in globals():
    if best_raw_11w["rmse"] <= best_robust_11w["rmse"]:
        w_obj_12abc = best_raw_11w
        w_name_12abc = "11w_raw"
    else:
        w_obj_12abc = best_robust_11w
        w_name_12abc = "11w_robust"

    w_oof_12abc = np.clip(np.asarray(w_obj_12abc["oof"]), 0, None)
    w_test_12abc = np.clip(np.asarray(w_obj_12abc["test"]), 0, None)
    w_score_12abc = float(w_obj_12abc["rmse"])
else:
    w_name_12abc = "fallback_11x_robust"
    w_oof_12abc = x_rob_oof_12abc.copy()
    w_test_12abc = x_rob_test_12abc.copy()
    w_score_12abc = x_rob_score_12abc

print("Base 11O OOF:", base_score_12abc)
print("11W anchor:", w_name_12abc, "| OOF:", w_score_12abc)
print("11X raw OOF:", x_raw_score_12abc)
print("11X robust OOF:", x_rob_score_12abc)


# ==================================================
# PART 1 - Utilities
# ==================================================

target_bin_12abc = pd.qcut(
    y_arr_12abc,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels_12abc = ["very_low", "low", "medium", "high", "very_high"]

bin_masks_12abc = {
    b: (target_bin_12abc == b)
    for b in bin_labels_12abc
}


def fast_rmse_12abc(y_true, pred):
    pred = np.asarray(pred)
    return float(np.sqrt(np.mean((pred - y_true) ** 2)))


def rank_against_train_12abc(train_values, values):
    train_values = np.asarray(train_values)
    values = np.asarray(values)

    train_sorted = np.sort(train_values[np.isfinite(train_values)])
    if len(train_sorted) == 0:
        return np.zeros_like(values, dtype=float)

    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def clean_numeric_frame_12abc(df):
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)

    for c in out.columns:
        if not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out


def safe_corr_from_obj_12abc(obj, oof_pred, test_pred):
    if "signed_correction_oof" in obj and "signed_correction_test" in obj:
        corr_oof = np.asarray(obj["signed_correction_oof"])
        corr_test = np.asarray(obj["signed_correction_test"])
    elif "correction_oof" in obj and "correction_test" in obj:
        corr_oof = np.asarray(obj["correction_oof"])
        corr_test = np.asarray(obj["correction_test"])
    else:
        corr_oof = np.asarray(oof_pred) - base_oof_12abc
        corr_test = np.asarray(test_pred) - base_test_12abc

    return corr_oof, corr_test


def describe_test_dist_12abc(pred, prefix):
    pred = np.asarray(pred)

    return {
        f"{prefix}_test_mean": float(np.mean(pred)),
        f"{prefix}_test_std": float(np.std(pred)),
        f"{prefix}_test_q01": float(np.quantile(pred, 0.01)),
        f"{prefix}_test_q05": float(np.quantile(pred, 0.05)),
        f"{prefix}_test_q50": float(np.quantile(pred, 0.50)),
        f"{prefix}_test_q95": float(np.quantile(pred, 0.95)),
        f"{prefix}_test_q99": float(np.quantile(pred, 0.99)),
        f"{prefix}_test_max": float(np.max(pred)),
    }


def calc_metrics_12abc(pred_oof, pred_test):
    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)

    err = pred_oof - y_arr_12abc
    se = err ** 2

    out = {
        "rmse": fast_rmse_12abc(y_arr_12abc, pred_oof),
    }

    for b in bin_labels_12abc:
        m = bin_masks_12abc[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    signed_corr_oof = pred_oof - base_oof_12abc

    out["pos_corr_count"] = int((signed_corr_oof > 1e-12).sum())
    out["neg_corr_count"] = int((signed_corr_oof < -1e-12).sum())
    out["changed_count"] = int((np.abs(signed_corr_oof) > 1e-12).sum())
    out["signed_corr_mean"] = float(np.mean(signed_corr_oof))
    out["signed_corr_min"] = float(np.min(signed_corr_oof))
    out["signed_corr_max"] = float(np.max(signed_corr_oof))
    out["max_abs_corr"] = float(np.max(np.abs(signed_corr_oof)))

    base_se = (base_oof_12abc - y_arr_12abc) ** 2
    new_se = (pred_oof - y_arr_12abc) ** 2
    benefit = base_se - new_se

    changed = np.abs(signed_corr_oof) > 1e-12

    if changed.sum() > 0:
        out["benefit_positive_rate"] = float(np.mean(benefit[changed] > 0))
        out["avg_benefit_changed"] = float(np.mean(benefit[changed]))
    else:
        out["benefit_positive_rate"] = 0.0
        out["avg_benefit_changed"] = 0.0

    out["delta_vs_11O"] = out["rmse"] - BEST_11O_OOF
    out["delta_vs_11X_raw"] = out["rmse"] - x_raw_score_12abc
    out["delta_vs_11X_robust"] = out["rmse"] - x_rob_score_12abc

    # Test distribution distance from known public-good 11X raw.
    out["test_mae_vs_11x_raw"] = float(np.mean(np.abs(pred_test - x_raw_test_12abc)))
    out["test_rmse_vs_11x_raw"] = float(np.sqrt(np.mean((pred_test - x_raw_test_12abc) ** 2)))

    out["test_mean_diff_vs_11x_raw"] = float(np.mean(pred_test) - np.mean(x_raw_test_12abc))
    out["test_std_diff_vs_11x_raw"] = float(np.std(pred_test) - np.std(x_raw_test_12abc))
    out["test_q95_diff_vs_11x_raw"] = float(np.quantile(pred_test, 0.95) - np.quantile(x_raw_test_12abc, 0.95))
    out["test_q99_diff_vs_11x_raw"] = float(np.quantile(pred_test, 0.99) - np.quantile(x_raw_test_12abc, 0.99))

    out.update(describe_test_dist_12abc(pred_test, "candidate"))

    return out


# Reference metrics for penalties
x_raw_metrics_12abc = calc_metrics_12abc(x_raw_oof_12abc, x_raw_test_12abc)
x_rob_metrics_12abc = calc_metrics_12abc(x_rob_oof_12abc, x_rob_test_12abc)
w_metrics_12abc = calc_metrics_12abc(w_oof_12abc, w_test_12abc)


def public_safe_penalty_12abc(metrics, method):
    penalty = 0.0

    # --------------------------------------------------
    # 1. Avoid OOF-only trap.
    # Candidate may be better than 11X, but must stay close to 11X test distribution.
    # --------------------------------------------------

    if metrics["test_mae_vs_11x_raw"] > 0.35:
        penalty += 0.0010
    if metrics["test_mae_vs_11x_raw"] > 0.60:
        penalty += 0.0020
    if abs(metrics["test_mean_diff_vs_11x_raw"]) > 0.25:
        penalty += 0.0008
    if abs(metrics["test_q95_diff_vs_11x_raw"]) > 0.75:
        penalty += 0.0010
    if abs(metrics["test_q99_diff_vs_11x_raw"]) > 1.25:
        penalty += 0.0012

    # --------------------------------------------------
    # 2. High and very-high guard.
    # 11X public worked, but high/very_high were risky.
    # Don't let candidate over-damage them.
    # --------------------------------------------------

    if metrics["very_high_rmse"] > x_raw_metrics_12abc["very_high_rmse"] + 0.010:
        penalty += 0.0012
    if metrics["very_high_rmse"] > x_raw_metrics_12abc["very_high_rmse"] + 0.020:
        penalty += 0.0020

    if metrics["high_rmse"] > x_raw_metrics_12abc["high_rmse"] + 0.010:
        penalty += 0.0010
    if metrics["high_rmse"] > x_raw_metrics_12abc["high_rmse"] + 0.020:
        penalty += 0.0018

    # --------------------------------------------------
    # 3. Don't lose low/medium improvement too much.
    # --------------------------------------------------

    if metrics["very_low_rmse"] > x_raw_metrics_12abc["very_low_rmse"] + 0.025:
        penalty += 0.0008

    if metrics["low_rmse"] > x_raw_metrics_12abc["low_rmse"] + 0.025:
        penalty += 0.0008

    if metrics["medium_rmse"] > x_raw_metrics_12abc["medium_rmse"] + 0.025:
        penalty += 0.0008

    # --------------------------------------------------
    # 4. Correction aggressiveness guard.
    # --------------------------------------------------

    if metrics["max_abs_corr"] > 5.5:
        penalty += 0.0010

    if metrics["neg_corr_count"] > 9500:
        penalty += 0.0010

    if metrics["changed_count"] > 10000:
        penalty += 0.0008

    # --------------------------------------------------
    # 5. Method-specific guard.
    # 12A/12B can help, but should not win purely by aggressive broad correction.
    # --------------------------------------------------

    if method.startswith("12a") and metrics["test_mae_vs_11x_raw"] > 0.50:
        penalty += 0.0010

    if method.startswith("12b") and metrics["test_mae_vs_11x_raw"] > 0.45:
        penalty += 0.0010

    # --------------------------------------------------
    # 6. Reward meaningful OOF improvement without letting it dominate too much.
    # --------------------------------------------------

    if metrics["rmse"] < x_raw_score_12abc - 0.005:
        penalty -= 0.0004

    if metrics["rmse"] < x_raw_score_12abc - 0.010:
        penalty -= 0.0006

    return penalty


candidate_rows_12abc = []
candidate_store_12abc = {}


def add_candidate_12abc(key, method, pred_oof, pred_test, store_arrays=True):
    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)

    metrics = calc_metrics_12abc(pred_oof, pred_test)

    penalty = public_safe_penalty_12abc(metrics, method)

    # Main score:
    # Mostly OOF RMSE, but with enough penalty to avoid 11Y-style OOF trap.
    public_safe_score = metrics["rmse"] + penalty

    row = {
        "key": key,
        "method": method,
        **metrics,
        "public_safe_penalty": penalty,
        "public_safe_score": public_safe_score,
    }

    candidate_rows_12abc.append(row)

    if store_arrays:
        candidate_store_12abc[key] = {
            "key": key,
            "method": method,
            "oof": pred_oof.copy(),
            "test": pred_test.copy(),
        }


# Add references
add_candidate_12abc(
    key="ref_11o_adaptive",
    method="reference",
    pred_oof=base_oof_12abc,
    pred_test=base_test_12abc,
)

add_candidate_12abc(
    key="ref_11w_best",
    method="reference",
    pred_oof=w_oof_12abc,
    pred_test=w_test_12abc,
)

add_candidate_12abc(
    key="ref_11x_raw",
    method="reference",
    pred_oof=x_raw_oof_12abc,
    pred_test=x_raw_test_12abc,
)

add_candidate_12abc(
    key="ref_11x_robust",
    method="reference",
    pred_oof=x_rob_oof_12abc,
    pred_test=x_rob_test_12abc,
)


# ==================================================
# PART 2 - Component reconstruction from 11X
# ==================================================

print("=" * 100)
print("PART 2 - Reconstruct 11X components")

# Main 11W positive anchor
w_corr_oof_12abc = w_oof_12abc - base_oof_12abc
w_corr_test_12abc = w_test_12abc - base_test_12abc

# 11X raw/robust signed corrections
x_raw_corr_oof_12abc = x_raw_oof_12abc - base_oof_12abc
x_raw_corr_test_12abc = x_raw_test_12abc - base_test_12abc

x_rob_corr_oof_12abc = x_rob_oof_12abc - base_oof_12abc
x_rob_corr_test_12abc = x_rob_test_12abc - base_test_12abc

# Prefer exact components from 11X if still in memory.
if "pos_corr_oof_11x" in globals() and "pos_corr_test_11x" in globals():
    pos_corr_oof_12abc = np.asarray(pos_corr_oof_11x)
    pos_corr_test_12abc = np.asarray(pos_corr_test_11x)
else:
    # Fallback: 11W positive correction.
    pos_corr_oof_12abc = np.clip(w_corr_oof_12abc, 0, None)
    pos_corr_test_12abc = np.clip(w_corr_test_12abc, 0, None)

if "best_low_neg_11x" in globals() and "neg_oof" in best_low_neg_11x:
    low_neg_oof_12abc = np.asarray(best_low_neg_11x["neg_oof"])
    low_neg_test_12abc = np.asarray(best_low_neg_11x["neg_test"])
else:
    # Fallback: estimate low negative component from robust correction minus positive.
    estimated_low_oof = np.clip(pos_corr_oof_12abc - x_rob_corr_oof_12abc, 0, None)
    estimated_low_test = np.clip(pos_corr_test_12abc - x_rob_corr_test_12abc, 0, None)

    low_neg_oof_12abc = estimated_low_oof
    low_neg_test_12abc = estimated_low_test

if "best_group_signed_11x" in globals() and "signed_oof" in best_group_signed_11x:
    group_signed_oof_12abc = np.asarray(best_group_signed_11x["signed_oof"])
    group_signed_test_12abc = np.asarray(best_group_signed_11x["signed_test"])
else:
    # Fallback: raw minus robust approximates group signed residual contribution.
    group_signed_oof_12abc = x_raw_corr_oof_12abc - x_rob_corr_oof_12abc
    group_signed_test_12abc = x_raw_corr_test_12abc - x_rob_corr_test_12abc

print("pos corr OOF mean/min/max:", np.mean(pos_corr_oof_12abc), np.min(pos_corr_oof_12abc), np.max(pos_corr_oof_12abc))
print("low neg OOF mean/min/max :", np.mean(low_neg_oof_12abc), np.min(low_neg_oof_12abc), np.max(low_neg_oof_12abc))
print("group OOF mean/min/max   :", np.mean(group_signed_oof_12abc), np.min(group_signed_oof_12abc), np.max(group_signed_oof_12abc))


# ==================================================
# PART 3 - 11Z public-safe 11X family
# ==================================================

print("=" * 100)
print("PART 3 - 11Z public-safe 11X family")

# A. Interpolation raw / robust
for w_raw in [0.25, 0.40, 0.50, 0.60, 0.75]:
    pred_oof = w_raw * x_raw_oof_12abc + (1.0 - w_raw) * x_rob_oof_12abc
    pred_test = w_raw * x_raw_test_12abc + (1.0 - w_raw) * x_rob_test_12abc

    add_candidate_12abc(
        key=f"11z_interp_raw{w_raw}_rob{1.0 - w_raw}",
        method="11z_interpolation",
        pred_oof=pred_oof,
        pred_test=pred_test,
    )

# B. Rebuild around components.
w_pos_grid_11z = [1.00, 1.03, 1.05, 1.08, 1.10]
w_low_grid_11z = [0.00, 0.50, 0.75, 1.00]
w_group_grid_11z = [0.00, 0.25, 0.40, 0.50, 0.65, 0.75, 0.85, 1.00]
pos_cap_grid_11z = [4.00, 4.25, 4.50, 4.75, 5.00]
neg_cap_grid_11z = [1.00, 1.25, 1.50]

for w_pos in w_pos_grid_11z:
    for w_low in w_low_grid_11z:
        for w_group in w_group_grid_11z:
            raw_corr_oof = (
                w_pos * pos_corr_oof_12abc
                - w_low * low_neg_oof_12abc
                + w_group * group_signed_oof_12abc
            )

            raw_corr_test = (
                w_pos * pos_corr_test_12abc
                - w_low * low_neg_test_12abc
                + w_group * group_signed_test_12abc
            )

            for pcap in pos_cap_grid_11z:
                for ncap in neg_cap_grid_11z:
                    signed_corr_oof = np.clip(raw_corr_oof, -ncap, pcap)
                    signed_corr_test = np.clip(raw_corr_test, -ncap, pcap)

                    pred_oof = np.clip(base_oof_12abc + signed_corr_oof, 0, None)
                    pred_test = np.clip(base_test_12abc + signed_corr_test, 0, None)

                    key = (
                        f"11z_family"
                        f"_wpos{w_pos}"
                        f"_wlow{w_low}"
                        f"_wgrp{w_group}"
                        f"_pcap{pcap}"
                        f"_ncap{ncap}"
                    )

                    add_candidate_12abc(
                        key=key,
                        method="11z_family",
                        pred_oof=pred_oof,
                        pred_test=pred_test,
                        store_arrays=True,
                    )

                    del signed_corr_oof, signed_corr_test, pred_oof, pred_test

            del raw_corr_oof, raw_corr_test

gc.collect()


# ==================================================
# PART 4 - 12A cluster-gated residual
# ==================================================

print("=" * 100)
print("PART 4 - 12A cluster-gated residual")

# Use 11X raw as residual anchor because it is public-proven.
seed_oof_12a = x_raw_oof_12abc
seed_test_12a = x_raw_test_12abc
residual_seed_12a = y_arr_12abc - seed_oof_12a

def get_col_12abc(col, is_train=True):
    if is_train:
        if "X_fe" in globals() and col in X_fe.columns:
            return X_fe[col].astype(str).fillna("Unknown").reset_index(drop=True)
        if col in train_raw.columns:
            return train_raw[col].astype(str).fillna("Unknown").reset_index(drop=True)
        return None
    else:
        if "X_test_fe" in globals() and col in X_test_fe.columns:
            return X_test_fe[col].astype(str).fillna("Unknown").reset_index(drop=True)
        if col in test_raw.columns:
            return test_raw[col].astype(str).fillna("Unknown").reset_index(drop=True)
        return None


cluster_train_12a = pd.DataFrame(index=np.arange(n_train_12abc))
cluster_test_12a = pd.DataFrame(index=np.arange(n_test_12abc))

base_edges5_12a = np.unique(np.quantile(base_oof_12abc, np.linspace(0, 1, 6)))
x_edges5_12a = np.unique(np.quantile(x_raw_oof_12abc, np.linspace(0, 1, 6)))

if len(base_edges5_12a) > 2:
    cluster_train_12a["base_bin5"] = pd.cut(
        base_oof_12abc,
        bins=base_edges5_12a,
        include_lowest=True,
        labels=False,
    ).astype(str)

    cluster_test_12a["base_bin5"] = pd.cut(
        base_test_12abc,
        bins=base_edges5_12a,
        include_lowest=True,
        labels=False,
    ).astype(str)

if len(x_edges5_12a) > 2:
    cluster_train_12a["x_bin5"] = pd.cut(
        x_raw_oof_12abc,
        bins=x_edges5_12a,
        include_lowest=True,
        labels=False,
    ).astype(str)

    cluster_test_12a["x_bin5"] = pd.cut(
        x_raw_test_12abc,
        bins=x_edges5_12a,
        include_lowest=True,
        labels=False,
    ).astype(str)

single_cluster_cols_12a = [
    "source_id",
    "geo_zone_micro",
    "geo_zone_meso",
    "geo_meso_micro",
    "geo_macro_meso",
    "biome_land_cover",
    "land_cover_geo_macro",
    "biome",
]

for col in single_cluster_cols_12a:
    tr_col = get_col_12abc(col, True)
    te_col = get_col_12abc(col, False)

    if tr_col is not None and te_col is not None:
        cluster_train_12a[col] = tr_col
        cluster_test_12a[col] = te_col


def add_cluster_combo_12a(name, cols):
    if all(c in cluster_train_12a.columns for c in cols) and all(c in cluster_test_12a.columns for c in cols):
        cluster_train_12a[name] = cluster_train_12a[cols].astype(str).agg("__".join, axis=1)
        cluster_test_12a[name] = cluster_test_12a[cols].astype(str).agg("__".join, axis=1)


add_cluster_combo_12a("source_base5", ["source_id", "base_bin5"])
add_cluster_combo_12a("source_xbin5", ["source_id", "x_bin5"])
add_cluster_combo_12a("geo_micro_base5", ["geo_zone_micro", "base_bin5"])
add_cluster_combo_12a("geo_meso_base5", ["geo_zone_meso", "base_bin5"])
add_cluster_combo_12a("geo_meso_micro_base5", ["geo_meso_micro", "base_bin5"])
add_cluster_combo_12a("biome_land_base5", ["biome_land_cover", "base_bin5"])
add_cluster_combo_12a("land_geo_base5", ["land_cover_geo_macro", "base_bin5"])
add_cluster_combo_12a("biome_xbin5", ["biome", "x_bin5"])

cluster_cols_12a = [
    c for c in [
        "source_base5",
        "source_xbin5",
        "geo_micro_base5",
        "geo_meso_base5",
        "geo_meso_micro_base5",
        "biome_land_base5",
        "land_geo_base5",
        "biome_xbin5",
        "source_id",
        "geo_zone_micro",
        "geo_zone_meso",
        "biome_land_cover",
    ]
    if c in cluster_train_12a.columns and c in cluster_test_12a.columns
]

print("12A cluster columns:", cluster_cols_12a)


def fold_safe_cluster_residual_12a(group_col, min_count=80, shrink_k=150, consistency_min=0.58):
    corr_oof = np.zeros(n_train_12abc)
    corr_test_sum = np.zeros(n_test_12abc)

    valid_fold_count = 0

    key_train_all = cluster_train_12a[group_col].astype(str)
    key_test_all = cluster_test_12a[group_col].astype(str)

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        tmp = pd.DataFrame({
            "key": key_train_all.iloc[tr_idx].values,
            "resid": residual_seed_12a[tr_idx],
        })

        agg = (
            tmp.groupby("key")
            .agg(
                count=("resid", "count"),
                mean_resid=("resid", "mean"),
                median_resid=("resid", "median"),
                pos_rate=("resid", lambda x: np.mean(x > 0)),
                abs_mean=("resid", lambda x: np.mean(np.abs(x))),
            )
            .reset_index()
        )

        # Direction consistency:
        # either mostly positive residual or mostly negative residual.
        agg["direction_consistency"] = np.maximum(agg["pos_rate"], 1.0 - agg["pos_rate"])

        agg = agg[
            (agg["count"] >= min_count)
            & (agg["direction_consistency"] >= consistency_min)
        ].copy()

        if len(agg) == 0:
            valid_fold_count += 1
            continue

        # Robust signed residual:
        # median is safer, mean adds magnitude.
        agg["raw_corr"] = 0.65 * agg["median_resid"] + 0.35 * agg["mean_resid"]

        agg["shrunk_corr"] = agg["raw_corr"] * (
            agg["count"] / (agg["count"] + shrink_k)
        )

        mapping = dict(zip(agg["key"], agg["shrunk_corr"]))

        val_key = key_train_all.iloc[val_idx]
        test_key = key_test_all

        corr_oof[val_idx] = val_key.map(mapping).fillna(0.0).values
        corr_test_sum += test_key.map(mapping).fillna(0.0).values

        valid_fold_count += 1

    corr_test = corr_test_sum / max(valid_fold_count, 1)

    return corr_oof, corr_test


cluster_cache_12a = {}

for group_col in cluster_cols_12a:
    for min_count in [50, 80, 120, 160]:
        for shrink_k in [100, 200, 400]:
            for consistency_min in [0.56, 0.60, 0.65]:
                cache_key = (group_col, min_count, shrink_k, consistency_min)

                corr_oof, corr_test = fold_safe_cluster_residual_12a(
                    group_col=group_col,
                    min_count=min_count,
                    shrink_k=shrink_k,
                    consistency_min=consistency_min,
                )

                if np.max(np.abs(corr_oof)) <= 1e-12:
                    continue

                cluster_cache_12a[cache_key] = (corr_oof, corr_test)

                for strength in [0.10, 0.15, 0.20, 0.30, 0.40, 0.55]:
                    for cap in [0.35, 0.50, 0.75, 1.00, 1.25]:
                        signed_extra_oof = np.clip(strength * corr_oof, -cap, cap)
                        signed_extra_test = np.clip(strength * corr_test, -cap, cap)

                        pred_oof = np.clip(seed_oof_12a + signed_extra_oof, 0, None)
                        pred_test = np.clip(seed_test_12a + signed_extra_test, 0, None)

                        key = (
                            f"12a_cluster_{group_col}"
                            f"_min{min_count}"
                            f"_k{shrink_k}"
                            f"_cons{consistency_min}"
                            f"_s{strength}"
                            f"_cap{cap}"
                        )

                        add_candidate_12abc(
                            key=key,
                            method="12a_cluster_gated",
                            pred_oof=pred_oof,
                            pred_test=pred_test,
                            store_arrays=True,
                        )

                        del signed_extra_oof, signed_extra_test, pred_oof, pred_test

                gc.collect()

print("12A cluster cache count:", len(cluster_cache_12a))


# ==================================================
# PART 5 - 12B residual model over 11X
# ==================================================

print("=" * 100)
print("PART 5 - 12B residual model over 11X")

# Residual target from public-proven 11X raw.
residual_target_12b = y_arr_12abc - x_raw_oof_12abc

if "X_fe" in globals() and "X_test_fe" in globals():
    X_source_train_12b = X_fe.copy().reset_index(drop=True)
    X_source_test_12b = X_test_fe.copy().reset_index(drop=True)
else:
    X_source_train_12b = train_raw.drop(columns=[TARGET], errors="ignore").copy().reset_index(drop=True)
    X_source_test_12b = test_raw.copy().reset_index(drop=True)

num_cols_12b = [
    c for c in X_source_train_12b.columns
    if c in X_source_test_12b.columns
    and pd.api.types.is_numeric_dtype(X_source_train_12b[c])
    and pd.api.types.is_numeric_dtype(X_source_test_12b[c])
    and c not in [TARGET, ID_COL]
]

priority_keywords_12b = [
    "particle",
    "fine",
    "coarse",
    "acidity",
    "cec",
    "cation",
    "ratio",
    "spectral",
    "band",
    "ca_",
    "mg_",
    "geo",
    "source",
    "organic",
]

priority_cols_12b = [
    c for c in num_cols_12b
    if any(k.lower() in c.lower() for k in priority_keywords_12b)
]

remaining_cols_12b = [c for c in num_cols_12b if c not in priority_cols_12b]

selected_cols_12b = priority_cols_12b[:100] + remaining_cols_12b[:70]

X_res_train_12b = clean_numeric_frame_12abc(X_source_train_12b[selected_cols_12b])
X_res_test_12b = clean_numeric_frame_12abc(X_source_test_12b[selected_cols_12b])

# Add prediction-space and correction-space features.
base_rank_train_12b = rank_against_train_12abc(base_oof_12abc, base_oof_12abc)
base_rank_test_12b = rank_against_train_12abc(base_oof_12abc, base_test_12abc)

x_rank_train_12b = rank_against_train_12abc(x_raw_oof_12abc, x_raw_oof_12abc)
x_rank_test_12b = rank_against_train_12abc(x_raw_oof_12abc, x_raw_test_12abc)

extra_train_12b = pd.DataFrame({
    "12b_base_pred": base_oof_12abc,
    "12b_base_rank": base_rank_train_12b,
    "12b_11w_pred": w_oof_12abc,
    "12b_11x_raw_pred": x_raw_oof_12abc,
    "12b_11x_robust_pred": x_rob_oof_12abc,
    "12b_11x_rank": x_rank_train_12b,
    "12b_xraw_minus_base": x_raw_oof_12abc - base_oof_12abc,
    "12b_xrob_minus_base": x_rob_oof_12abc - base_oof_12abc,
    "12b_xraw_minus_xrob": x_raw_oof_12abc - x_rob_oof_12abc,
})

extra_test_12b = pd.DataFrame({
    "12b_base_pred": base_test_12abc,
    "12b_base_rank": base_rank_test_12b,
    "12b_11w_pred": w_test_12abc,
    "12b_11x_raw_pred": x_raw_test_12abc,
    "12b_11x_robust_pred": x_rob_test_12abc,
    "12b_11x_rank": x_rank_test_12b,
    "12b_xraw_minus_base": x_raw_test_12abc - base_test_12abc,
    "12b_xrob_minus_base": x_rob_test_12abc - base_test_12abc,
    "12b_xraw_minus_xrob": x_raw_test_12abc - x_rob_test_12abc,
})

# Add cluster residual features from top cached 12A signals, but only a few.
if len(cluster_cache_12a) > 0:
    # Use absolute train signal as quick proxy.
    cache_strength = []
    for k, (coof, ctest) in cluster_cache_12a.items():
        cache_strength.append((k, float(np.mean(np.abs(coof))), coof, ctest))

    cache_strength = sorted(cache_strength, key=lambda x: x[1], reverse=True)[:8]

    for idx, (k, _, coof, ctest) in enumerate(cache_strength):
        extra_train_12b[f"12b_cluster_feat_{idx}"] = coof
        extra_test_12b[f"12b_cluster_feat_{idx}"] = ctest

X_res_train_12b = pd.concat(
    [X_res_train_12b.reset_index(drop=True), extra_train_12b.reset_index(drop=True)],
    axis=1,
)

X_res_test_12b = pd.concat(
    [X_res_test_12b.reset_index(drop=True), extra_test_12b.reset_index(drop=True)],
    axis=1,
)

X_res_train_12b = clean_numeric_frame_12abc(X_res_train_12b)
X_res_test_12b = clean_numeric_frame_12abc(X_res_test_12b)

X_res_train_12b = X_res_train_12b.loc[:, ~X_res_train_12b.columns.duplicated()].copy()
X_res_test_12b = X_res_test_12b[X_res_train_12b.columns].copy()

print("12B residual feature shape:", X_res_train_12b.shape, X_res_test_12b.shape)


def train_residual_model_12b(name, params, sample_weight_mode):
    resid_oof = np.zeros(n_train_12abc)
    resid_test_sum = np.zeros(n_test_12abc)

    abs_resid = np.abs(residual_target_12b)
    abs_rank = rank_against_train_12abc(abs_resid, abs_resid)

    if sample_weight_mode == "normal":
        weight_all = np.ones(n_train_12abc)
    elif sample_weight_mode == "abs_resid":
        weight_all = 1.0 + 1.25 * abs_rank
    elif sample_weight_mode == "tail_resid":
        q20 = np.quantile(y_arr_12abc, 0.20)
        q85 = np.quantile(y_arr_12abc, 0.85)
        weight_all = (
            1.0
            + 1.00 * abs_rank
            + 0.35 * (y_arr_12abc <= q20)
            + 0.35 * (y_arr_12abc >= q85)
        )
    else:
        weight_all = np.ones(n_train_12abc)

    weight_all = np.clip(weight_all, 0.5, 4.0)
    weight_all = weight_all / np.mean(weight_all)

    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        model = make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(**params),
        )

        model.fit(
            X_res_train_12b.iloc[tr_idx],
            residual_target_12b[tr_idx],
            histgradientboostingregressor__sample_weight=weight_all[tr_idx],
        )

        val_resid = model.predict(X_res_train_12b.iloc[val_idx])
        test_resid = model.predict(X_res_test_12b)

        resid_oof[val_idx] = val_resid
        resid_test_sum += test_resid / len(folds)

        pred_val = np.clip(x_raw_oof_12abc[val_idx] + 0.10 * np.clip(val_resid, -1, 1), 0, None)
        fold_score = fast_rmse_12abc(y_arr_12abc[val_idx], pred_val)
        fold_scores.append(fold_score)

        print(f"{name} fold {fold}: proxy score={fold_score:.6f}")

        del model
        gc.collect()

    print(name, "residual mean/std:", float(np.mean(resid_oof)), float(np.std(resid_oof)))
    return resid_oof, resid_test_sum, fold_scores


residual_model_configs_12b = {
    "hgb_small": dict(
        max_iter=180,
        learning_rate=0.035,
        max_leaf_nodes=9,
        min_samples_leaf=25,
        l2_regularization=3.0,
        random_state=12101,
    ),
    "hgb_medium": dict(
        max_iter=240,
        learning_rate=0.030,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=4.0,
        random_state=12102,
    ),
    "hgb_smooth": dict(
        max_iter=220,
        learning_rate=0.025,
        max_leaf_nodes=7,
        min_samples_leaf=40,
        l2_regularization=6.0,
        random_state=12103,
    ),
}

residual_outputs_12b = {}

for cfg_name, params in residual_model_configs_12b.items():
    for sw_mode in ["normal", "abs_resid", "tail_resid"]:
        name = f"12b_{cfg_name}_{sw_mode}"

        resid_oof, resid_test, fold_scores = train_residual_model_12b(
            name=name,
            params=params,
            sample_weight_mode=sw_mode,
        )

        residual_outputs_12b[name] = {
            "resid_oof": resid_oof,
            "resid_test": resid_test,
            "fold_scores": fold_scores,
        }

        for shrink in [0.025, 0.05, 0.075, 0.10, 0.15, 0.20]:
            for cap in [0.25, 0.35, 0.50, 0.75, 1.00]:
                signed_resid_oof = np.clip(shrink * resid_oof, -cap, cap)
                signed_resid_test = np.clip(shrink * resid_test, -cap, cap)

                pred_oof = np.clip(x_raw_oof_12abc + signed_resid_oof, 0, None)
                pred_test = np.clip(x_raw_test_12abc + signed_resid_test, 0, None)

                key = (
                    f"{name}"
                    f"_shrink{shrink}"
                    f"_cap{cap}"
                )

                add_candidate_12abc(
                    key=key,
                    method="12b_residual_model",
                    pred_oof=pred_oof,
                    pred_test=pred_test,
                    store_arrays=True,
                )

                del signed_resid_oof, signed_resid_test, pred_oof, pred_test

        gc.collect()


# ==================================================
# PART 6 - Final ranking
# ==================================================

print("=" * 100)
print("PART 6 - Final ranking")

candidate_df_12abc = pd.DataFrame(candidate_rows_12abc)

candidate_df_12abc = candidate_df_12abc.sort_values("rmse").reset_index(drop=True)
candidate_df_12abc_safe = candidate_df_12abc.sort_values("public_safe_score").reset_index(drop=True)

print("Top candidates by OOF RMSE")
display(candidate_df_12abc.head(60))

print("Top candidates by public-safe score")
display(candidate_df_12abc_safe.head(60))

print("=" * 100)
print("Method best summary")

method_best_12abc = (
    candidate_df_12abc
    .sort_values("rmse")
    .groupby("method", as_index=False)
    .head(3)
    .sort_values(["method", "rmse"])
)

display(method_best_12abc)


# ==================================================
# PART 7 - Select ONE final candidate
# ==================================================

print("=" * 100)
print("PART 7 - Select one final candidate")

# Hard guard:
# We don't want an OOF-only monster that moves too far from 11X raw.
# But we also require a meaningful attempt to beat 11X.
selection_pool_12abc = candidate_df_12abc_safe.copy()

# Basic filters.
filtered_12abc = selection_pool_12abc[
    (selection_pool_12abc["rmse"] <= x_raw_score_12abc + 0.003)
    & (selection_pool_12abc["test_mae_vs_11x_raw"] <= 0.60)
    & (selection_pool_12abc["very_high_rmse"] <= x_raw_metrics_12abc["very_high_rmse"] + 0.020)
    & (selection_pool_12abc["high_rmse"] <= x_raw_metrics_12abc["high_rmse"] + 0.020)
].copy()

if len(filtered_12abc) == 0:
    print("No candidate passed strict filters. Relaxing filters slightly.")
    filtered_12abc = selection_pool_12abc[
        (selection_pool_12abc["rmse"] <= x_raw_score_12abc + 0.006)
        & (selection_pool_12abc["test_mae_vs_11x_raw"] <= 0.80)
        & (selection_pool_12abc["very_high_rmse"] <= x_raw_metrics_12abc["very_high_rmse"] + 0.035)
    ].copy()

if len(filtered_12abc) == 0:
    print("Still no candidate passed. Fallback to 11X raw.")
    selected_row_12abc = candidate_df_12abc[
        candidate_df_12abc["key"] == "ref_11x_raw"
    ].iloc[0]
else:
    # Bias toward actual improvement, but still sorted by public-safe score.
    # This avoids choosing 11X raw if a similar safe candidate improves OOF enough.
    filtered_12abc["final_selector_score"] = (
        filtered_12abc["public_safe_score"]
        + 0.20 * np.maximum(filtered_12abc["rmse"] - x_raw_score_12abc, 0)
        - 0.10 * np.maximum(x_raw_score_12abc - filtered_12abc["rmse"], 0)
    )

    filtered_12abc = filtered_12abc.sort_values("final_selector_score").reset_index(drop=True)
    selected_row_12abc = filtered_12abc.iloc[0]

print("=" * 100)
print("Selected candidate")
display(pd.DataFrame([selected_row_12abc]))

selected_key_12abc = selected_row_12abc["key"]
selected_method_12abc = selected_row_12abc["method"]

if selected_key_12abc not in candidate_store_12abc:
    raise KeyError(f"Selected key not found in candidate store: {selected_key_12abc}")

selected_oof_12abc = candidate_store_12abc[selected_key_12abc]["oof"]
selected_test_12abc = candidate_store_12abc[selected_key_12abc]["test"]

print("Selected key   :", selected_key_12abc)
print("Selected method:", selected_method_12abc)
print("Selected RMSE  :", selected_row_12abc["rmse"])
print("Delta vs 11X   :", selected_row_12abc["rmse"] - x_raw_score_12abc)
print("Public-safe score:", selected_row_12abc["public_safe_score"])


# ==================================================
# PART 8 - Reports
# ==================================================

print("=" * 100)
print("PART 8 - Bin comparison")

compare_12abc = train_raw[[ID_COL, TARGET]].copy()

compare_12abc["base_pred"] = base_oof_12abc
compare_12abc["w_pred"] = w_oof_12abc
compare_12abc["x_raw_pred"] = x_raw_oof_12abc
compare_12abc["x_robust_pred"] = x_rob_oof_12abc
compare_12abc["selected_pred"] = selected_oof_12abc

for col in ["base_pred", "w_pred", "x_raw_pred", "x_robust_pred", "selected_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")

    compare_12abc[err_col] = compare_12abc[col] - compare_12abc[TARGET]
    compare_12abc[se_col] = compare_12abc[err_col] ** 2

compare_12abc["target_bin_5"] = pd.qcut(
    compare_12abc[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_12abc = (
    compare_12abc.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        w_rmse=("w_se", lambda x: np.sqrt(np.mean(x))),
        x_raw_rmse=("x_raw_se", lambda x: np.sqrt(np.mean(x))),
        x_robust_rmse=("x_robust_se", lambda x: np.sqrt(np.mean(x))),
        selected_rmse=("selected_se", lambda x: np.sqrt(np.mean(x))),

        base_bias=("base_error", "mean"),
        w_bias=("w_error", "mean"),
        x_raw_bias=("x_raw_error", "mean"),
        x_robust_bias=("x_robust_error", "mean"),
        selected_bias=("selected_error", "mean"),
    )
    .reset_index()
)

bin_compare_12abc["selected_minus_x_raw_rmse"] = (
    bin_compare_12abc["selected_rmse"] - bin_compare_12abc["x_raw_rmse"]
)

bin_compare_12abc["selected_minus_w_rmse"] = (
    bin_compare_12abc["selected_rmse"] - bin_compare_12abc["w_rmse"]
)

display(bin_compare_12abc.sort_values("base_rmse", ascending=False))


print("=" * 100)
print("PART 9 - Test distribution comparison")

test_dist_compare_12abc = pd.DataFrame([
    {
        "name": "11O_base",
        **describe_test_dist_12abc(base_test_12abc, "pred"),
    },
    {
        "name": "11W",
        **describe_test_dist_12abc(w_test_12abc, "pred"),
    },
    {
        "name": "11X_raw",
        **describe_test_dist_12abc(x_raw_test_12abc, "pred"),
    },
    {
        "name": "11X_robust",
        **describe_test_dist_12abc(x_rob_test_12abc, "pred"),
    },
    {
        "name": "12ABC_selected",
        **describe_test_dist_12abc(selected_test_12abc, "pred"),
    },
])

display(test_dist_compare_12abc)


# ==================================================
# PART 10 - Save ONLY ONE submission
# ==================================================

print("=" * 100)
print("PART 10 - Save ONLY ONE final submission")

final_filename_12abc = "submission_12abc_single_public_safe.csv"

final_sub_12abc = sample_submission.copy()
final_sub_12abc[TARGET] = np.clip(selected_test_12abc, 0, None)

assert final_sub_12abc.shape[0] == sample_submission.shape[0]
assert list(final_sub_12abc.columns) == [ID_COL, TARGET]
assert final_sub_12abc[TARGET].isna().sum() == 0
assert np.isfinite(final_sub_12abc[TARGET]).all()
assert (final_sub_12abc[TARGET] < 0).sum() == 0
assert final_sub_12abc[ID_COL].equals(sample_submission[ID_COL])

final_sub_12abc.to_csv(final_filename_12abc, index=False)

print("Saved:", final_filename_12abc)
display(final_sub_12abc.head())
display(final_sub_12abc[TARGET].describe().to_frame().T)


# ==================================================
# PART 11 - Decision guide
# ==================================================

print("=" * 100)
print("12ABC Decision Guide")

print(f"11O adaptive OOF       : {base_score_12abc:.6f}")
print(f"11W OOF                : {w_score_12abc:.6f}")
print(f"11X raw OOF            : {x_raw_score_12abc:.6f}")
print(f"11X robust OOF         : {x_rob_score_12abc:.6f}")
print(f"12ABC selected OOF     : {selected_row_12abc['rmse']:.6f}")

print(f"Selected delta vs 11O  : {selected_row_12abc['rmse'] - BEST_11O_OOF:.6f}")
print(f"Selected delta vs 11X  : {selected_row_12abc['rmse'] - x_raw_score_12abc:.6f}")
print(f"Selected public-safe   : {selected_row_12abc['public_safe_score']:.6f}")
print(f"Selected test MAE vs 11X raw: {selected_row_12abc['test_mae_vs_11x_raw']:.6f}")

if selected_row_12abc["rmse"] < x_raw_score_12abc - 0.010:
    print("Decision: GOOD OOF improvement with public-safe selector. Submit this single file.")
elif selected_row_12abc["rmse"] < x_raw_score_12abc - 0.003:
    print("Decision: modest OOF improvement. Submit as the next public-safe probe.")
elif selected_key_12abc == "ref_11x_raw":
    print("Decision: selector chose 11X raw. This means new candidates were too risky.")
else:
    print("Decision: tiny OOF improvement. Submit only if you accept public-probe risk.")

print("=" * 100)
print("Only CSV saved:")
print(final_filename_12abc)
print("=" * 100)
print("Please screenshot:")
print("1. candidate_df_12abc.head(60)")
print("2. candidate_df_12abc_safe.head(60)")
print("3. selected candidate row")
print("4. bin_compare_12abc")
print("5. test_dist_compare_12abc")

In [ ]:
# ==================================================
# DOWNLOAD 11Z SINGLE PUBLIC-SAFE SUBMISSION
# ==================================================

from google.colab import files
import os

filename = "submission_12abc_single_public_safe.csv"

if os.path.exists(filename):
    print("Downloading:", filename)
    files.download(filename)
else:
    raise FileNotFoundError(f"{filename} not found. Run SECTION 12ABC first.")

In [ ]:
# ==================================================
# SECTION 11Z2 LITE - RAM SAFE PUBLIC-ACTIVE 11X REFINEMENT
# ==================================================
# Goal:
# - Fix 11Z issue: selected test prediction was identical to 11X raw.
# - Avoid RAM crash by NOT storing arrays for all candidates.
# - Search smaller but useful 11X-family grid.
# - Save ONLY ONE file:
#       submission_11z2_lite_single_public_active.csv
#
# Paste AFTER SECTION 11X.
# Do NOT use 11Y.

import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

print("=" * 100)
print("SECTION 11Z2 LITE - RAM SAFE PUBLIC-ACTIVE 11X REFINEMENT")

BEST_11O_OOF = 11.26696803195285

required_vars_11z2_lite = [
    "train_raw",
    "test_raw",
    "sample_submission",
    "TARGET",
    "ID_COL",
    "y",
    "rmse",
    "adaptive_oof",
    "final_adaptive_test",
    "best_raw_11x",
    "best_robust_11x",
]

missing_11z2_lite = [v for v in required_vars_11z2_lite if v not in globals()]
if len(missing_11z2_lite) > 0:
    raise NameError(f"Missing required variables for 11Z2 Lite: {missing_11z2_lite}")

y_arr = y.values

base_oof = np.clip(np.asarray(adaptive_oof), 0, None)
base_test = np.clip(np.asarray(final_adaptive_test), 0, None)

x_raw_oof = np.clip(np.asarray(best_raw_11x["oof"]), 0, None)
x_raw_test = np.clip(np.asarray(best_raw_11x["test"]), 0, None)
x_raw_score = float(best_raw_11x["rmse"])

x_rob_oof = np.clip(np.asarray(best_robust_11x["oof"]), 0, None)
x_rob_test = np.clip(np.asarray(best_robust_11x["test"]), 0, None)
x_rob_score = float(best_robust_11x["rmse"])

if "best_raw_11w" in globals() and "best_robust_11w" in globals():
    if best_raw_11w["rmse"] <= best_robust_11w["rmse"]:
        w_obj = best_raw_11w
        w_name = "11w_raw"
    else:
        w_obj = best_robust_11w
        w_name = "11w_robust"

    w_oof = np.clip(np.asarray(w_obj["oof"]), 0, None)
    w_test = np.clip(np.asarray(w_obj["test"]), 0, None)
    w_score = float(w_obj["rmse"])
else:
    w_name = "fallback_11x_robust"
    w_oof = x_rob_oof.copy()
    w_test = x_rob_test.copy()
    w_score = x_rob_score

print("11O base OOF   :", rmse(y, base_oof))
print("11W anchor     :", w_name, "|", w_score)
print("11X raw OOF    :", x_raw_score)
print("11X robust OOF :", x_rob_score)

# ==================================================
# Utilities
# ==================================================

target_bin = pd.qcut(
    y_arr,
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
).astype(str)

bin_labels = ["very_low", "low", "medium", "high", "very_high"]
bin_masks = {b: (target_bin == b) for b in bin_labels}


def fast_rmse(y_true, pred):
    return float(np.sqrt(np.mean((np.asarray(pred) - y_true) ** 2)))


def rank_against_train(train_values, values):
    train_values = np.asarray(train_values)
    values = np.asarray(values)
    train_sorted = np.sort(train_values[np.isfinite(train_values)])

    if len(train_sorted) == 0:
        return np.zeros_like(values, dtype=float)

    return np.searchsorted(train_sorted, values, side="right") / len(train_sorted)


def describe_pred_dist(pred, prefix):
    pred = np.asarray(pred)

    return {
        f"{prefix}_mean": float(np.mean(pred)),
        f"{prefix}_std": float(np.std(pred)),
        f"{prefix}_q01": float(np.quantile(pred, 0.01)),
        f"{prefix}_q05": float(np.quantile(pred, 0.05)),
        f"{prefix}_q50": float(np.quantile(pred, 0.50)),
        f"{prefix}_q95": float(np.quantile(pred, 0.95)),
        f"{prefix}_q99": float(np.quantile(pred, 0.99)),
        f"{prefix}_max": float(np.max(pred)),
    }


def calc_metrics(pred_oof, pred_test):
    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)

    err = pred_oof - y_arr
    se = err ** 2

    out = {
        "rmse": fast_rmse(y_arr, pred_oof),
    }

    for b in bin_labels:
        m = bin_masks[b]
        out[f"{b}_rmse"] = float(np.sqrt(np.mean(se[m])))
        out[f"{b}_bias"] = float(np.mean(err[m]))

    signed_corr = pred_oof - base_oof

    out["pos_corr_count"] = int((signed_corr > 1e-12).sum())
    out["neg_corr_count"] = int((signed_corr < -1e-12).sum())
    out["changed_count"] = int((np.abs(signed_corr) > 1e-12).sum())
    out["signed_corr_mean"] = float(np.mean(signed_corr))
    out["signed_corr_min"] = float(np.min(signed_corr))
    out["signed_corr_max"] = float(np.max(signed_corr))
    out["max_abs_corr"] = float(np.max(np.abs(signed_corr)))

    base_se = (base_oof - y_arr) ** 2
    new_se = (pred_oof - y_arr) ** 2
    benefit = base_se - new_se
    changed = np.abs(signed_corr) > 1e-12

    if changed.sum() > 0:
        out["benefit_positive_rate"] = float(np.mean(benefit[changed] > 0))
        out["avg_benefit_changed"] = float(np.mean(benefit[changed]))
    else:
        out["benefit_positive_rate"] = 0.0
        out["avg_benefit_changed"] = 0.0

    out["delta_vs_11O"] = out["rmse"] - BEST_11O_OOF
    out["delta_vs_11X_raw"] = out["rmse"] - x_raw_score
    out["delta_vs_11X_robust"] = out["rmse"] - x_rob_score

    out["test_mae_vs_11x_raw"] = float(np.mean(np.abs(pred_test - x_raw_test)))
    out["test_rmse_vs_11x_raw"] = float(np.sqrt(np.mean((pred_test - x_raw_test) ** 2)))
    out["test_max_abs_vs_11x_raw"] = float(np.max(np.abs(pred_test - x_raw_test)))

    out["test_mean_diff_vs_11x_raw"] = float(np.mean(pred_test) - np.mean(x_raw_test))
    out["test_std_diff_vs_11x_raw"] = float(np.std(pred_test) - np.std(x_raw_test))
    out["test_q95_diff_vs_11x_raw"] = float(np.quantile(pred_test, 0.95) - np.quantile(x_raw_test, 0.95))
    out["test_q99_diff_vs_11x_raw"] = float(np.quantile(pred_test, 0.99) - np.quantile(x_raw_test, 0.99))

    out.update(describe_pred_dist(pred_test, "test_pred"))

    return out


x_raw_metrics = calc_metrics(x_raw_oof, x_raw_test)
x_rob_metrics = calc_metrics(x_rob_oof, x_rob_test)


def public_active_score(metrics):
    penalty = 0.0

    # Force active test, unlike failed 11Z.
    if metrics["test_mae_vs_11x_raw"] < 0.010:
        penalty += 0.0500
    elif metrics["test_mae_vs_11x_raw"] < 0.020:
        penalty += 0.0100
    elif metrics["test_mae_vs_11x_raw"] < 0.030:
        penalty += 0.0040

    # Too different from public-proven 11X is risky.
    if metrics["test_mae_vs_11x_raw"] > 0.35:
        penalty += 0.0010
    if metrics["test_mae_vs_11x_raw"] > 0.55:
        penalty += 0.0030
    if metrics["test_mae_vs_11x_raw"] > 0.80:
        penalty += 0.0080

    # Protect high / very_high.
    if metrics["very_high_rmse"] > x_raw_metrics["very_high_rmse"] + 0.015:
        penalty += 0.0015
    if metrics["very_high_rmse"] > x_raw_metrics["very_high_rmse"] + 0.030:
        penalty += 0.0030

    if metrics["high_rmse"] > x_raw_metrics["high_rmse"] + 0.015:
        penalty += 0.0012
    if metrics["high_rmse"] > x_raw_metrics["high_rmse"] + 0.030:
        penalty += 0.0025

    # Test distribution guard.
    if abs(metrics["test_mean_diff_vs_11x_raw"]) > 0.30:
        penalty += 0.0010
    if abs(metrics["test_q95_diff_vs_11x_raw"]) > 0.75:
        penalty += 0.0010
    if abs(metrics["test_q99_diff_vs_11x_raw"]) > 1.25:
        penalty += 0.0012

    # Reward meaningful OOF gain a little.
    if metrics["rmse"] < x_raw_score - 0.001:
        penalty -= 0.0002
    if metrics["rmse"] < x_raw_score - 0.003:
        penalty -= 0.0004

    return metrics["rmse"] + penalty, penalty


# ==================================================
# Component reconstruction
# ==================================================

print("=" * 100)
print("Reconstruct 11X components")

w_corr_oof = w_oof - base_oof
w_corr_test = w_test - base_test

x_raw_corr_oof = x_raw_oof - base_oof
x_raw_corr_test = x_raw_test - base_test

x_rob_corr_oof = x_rob_oof - base_oof
x_rob_corr_test = x_rob_test - base_test

if "pos_corr_oof_11x" in globals() and "pos_corr_test_11x" in globals():
    pos_corr_oof = np.asarray(pos_corr_oof_11x)
    pos_corr_test = np.asarray(pos_corr_test_11x)
else:
    pos_corr_oof = np.clip(w_corr_oof, 0, None)
    pos_corr_test = np.clip(w_corr_test, 0, None)

if "best_low_neg_11x" in globals() and "neg_oof" in best_low_neg_11x:
    low_neg_oof = np.asarray(best_low_neg_11x["neg_oof"])
    low_neg_test = np.asarray(best_low_neg_11x["neg_test"])
else:
    low_neg_oof = np.clip(pos_corr_oof - x_rob_corr_oof, 0, None)
    low_neg_test = np.clip(pos_corr_test - x_rob_corr_test, 0, None)

if "best_group_signed_11x" in globals() and "signed_oof" in best_group_signed_11x:
    group_signed_oof = np.asarray(best_group_signed_11x["signed_oof"])
    group_signed_test = np.asarray(best_group_signed_11x["signed_test"])
else:
    group_signed_oof = x_raw_corr_oof - x_rob_corr_oof
    group_signed_test = x_raw_corr_test - x_rob_corr_test

print("pos corr OOF mean/min/max  :", np.mean(pos_corr_oof), np.min(pos_corr_oof), np.max(pos_corr_oof))
print("low neg OOF mean/min/max   :", np.mean(low_neg_oof), np.min(low_neg_oof), np.max(low_neg_oof))
print("group OOF mean/min/max     :", np.mean(group_signed_oof), np.min(group_signed_oof), np.max(group_signed_oof))
print("group TEST mean/min/max    :", np.mean(group_signed_test), np.min(group_signed_test), np.max(group_signed_test))
print("raw vs robust test MAE     :", np.mean(np.abs(x_raw_test - x_rob_test)))


# ==================================================
# Guarded group variants
# ==================================================

base_rank_oof = rank_against_train(base_oof, base_oof)
base_rank_test = rank_against_train(base_oof, base_test)


def guard_group(group_oof, group_test, mode):
    group_oof = np.asarray(group_oof).copy()
    group_test = np.asarray(group_test).copy()

    if mode == "none":
        return group_oof, group_test

    if mode == "tail_shrink_soft":
        factor_oof = np.ones_like(group_oof)
        factor_test = np.ones_like(group_test)

        factor_oof[base_rank_oof > 0.80] = 0.65
        factor_oof[base_rank_oof > 0.90] = 0.40

        factor_test[base_rank_test > 0.80] = 0.65
        factor_test[base_rank_test > 0.90] = 0.40

        return group_oof * factor_oof, group_test * factor_test

    if mode == "no_negative_tail":
        out_oof = group_oof.copy()
        out_test = group_test.copy()

        mask_oof = (base_rank_oof > 0.80) & (out_oof < 0)
        mask_test = (base_rank_test > 0.80) & (out_test < 0)

        out_oof[mask_oof] *= 0.20
        out_test[mask_test] *= 0.20

        return out_oof, out_test

    raise ValueError(f"Unknown guard mode: {mode}")


guard_modes = ["none", "tail_shrink_soft", "no_negative_tail"]

guarded_group_bank = {}

for mode in guard_modes:
    goof, gtest = guard_group(group_signed_oof, group_signed_test, mode)
    guarded_group_bank[mode] = (goof, gtest)

    print(
        mode,
        "| OOF mean/min/max:",
        float(np.mean(goof)),
        float(np.min(goof)),
        float(np.max(goof)),
        "| TEST mean/min/max:",
        float(np.mean(gtest)),
        float(np.min(gtest)),
        float(np.max(gtest)),
    )


# ==================================================
# Candidate search, RAM safe
# ==================================================

print("=" * 100)
print("Candidate search, RAM safe")

candidate_rows = []

best_selected = None
best_selected_oof = None
best_selected_test = None

best_active = None
best_active_oof = None
best_active_test = None

MIN_ACTIVE = 0.030
MAX_ACTIVE = 0.550


def consider_candidate(key, method, pred_oof, pred_test):
    global best_selected, best_selected_oof, best_selected_test
    global best_active, best_active_oof, best_active_test

    pred_oof = np.clip(np.asarray(pred_oof), 0, None)
    pred_test = np.clip(np.asarray(pred_test), 0, None)

    metrics = calc_metrics(pred_oof, pred_test)
    score, penalty = public_active_score(metrics)

    row = {
        "key": key,
        "method": method,
        **metrics,
        "public_active_penalty": penalty,
        "public_active_score": score,
    }

    candidate_rows.append(row)

    active_ok = (
        (metrics["test_mae_vs_11x_raw"] >= MIN_ACTIVE)
        and (metrics["test_mae_vs_11x_raw"] <= MAX_ACTIVE)
        and (metrics["rmse"] <= x_raw_score + 0.005)
        and (metrics["very_high_rmse"] <= x_raw_metrics["very_high_rmse"] + 0.030)
        and (metrics["high_rmse"] <= x_raw_metrics["high_rmse"] + 0.030)
    )

    # Track best active candidate by public_active_score
    if active_ok:
        selector_score = (
            score
            + 0.25 * max(metrics["rmse"] - x_raw_score, 0)
            - 0.10 * max(x_raw_score - metrics["rmse"], 0)
        )

        row["final_selector_score"] = selector_score

        if best_active is None or selector_score < best_active["final_selector_score"]:
            best_active = row.copy()
            best_active_oof = pred_oof.copy()
            best_active_test = pred_test.copy()

    # Fallback best non-reference, still avoid exact clone.
    clone_bad = metrics["test_mae_vs_11x_raw"] < 0.010
    if not clone_bad:
        fallback_score = score

        row["fallback_score"] = fallback_score

        if best_selected is None or fallback_score < best_selected["fallback_score"]:
            best_selected = row.copy()
            best_selected_oof = pred_oof.copy()
            best_selected_test = pred_test.copy()


# References only for reporting
consider_candidate("ref_11X_raw", "reference", x_raw_oof, x_raw_test)
consider_candidate("ref_11X_robust", "reference", x_rob_oof, x_rob_test)
consider_candidate("ref_11W", "reference", w_oof, w_test)


# A. Raw / robust interpolation
for w_raw in [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]:
    pred_oof = w_raw * x_raw_oof + (1.0 - w_raw) * x_rob_oof
    pred_test = w_raw * x_raw_test + (1.0 - w_raw) * x_rob_test

    consider_candidate(
        key=f"11z2lite_interp_raw{w_raw:.2f}_rob{1.0 - w_raw:.2f}",
        method="11z2lite_interpolation",
        pred_oof=pred_oof,
        pred_test=pred_test,
    )


# B. Smaller component grid
w_pos_grid = [1.00, 1.05, 1.08]
w_low_grid = [0.50, 0.75, 1.00]
w_group_grid = [0.25, 0.50, 0.75, 0.95]
pcap_grid = [4.50, 5.00]
ncap_grid = [1.00, 1.50]

for guard_mode, (group_oof_guarded, group_test_guarded) in guarded_group_bank.items():
    for w_pos in w_pos_grid:
        for w_low in w_low_grid:
            for w_group in w_group_grid:
                raw_corr_oof = (
                    w_pos * pos_corr_oof
                    - w_low * low_neg_oof
                    + w_group * group_oof_guarded
                )

                raw_corr_test = (
                    w_pos * pos_corr_test
                    - w_low * low_neg_test
                    + w_group * group_test_guarded
                )

                for pcap in pcap_grid:
                    for ncap in ncap_grid:
                        signed_corr_oof = np.clip(raw_corr_oof, -ncap, pcap)
                        signed_corr_test = np.clip(raw_corr_test, -ncap, pcap)

                        pred_oof = base_oof + signed_corr_oof
                        pred_test = base_test + signed_corr_test

                        key = (
                            f"11z2lite_family"
                            f"_guard{guard_mode}"
                            f"_wpos{w_pos}"
                            f"_wlow{w_low}"
                            f"_wgrp{w_group}"
                            f"_pcap{pcap}"
                            f"_ncap{ncap}"
                        )

                        consider_candidate(
                            key=key,
                            method="11z2lite_family",
                            pred_oof=pred_oof,
                            pred_test=pred_test,
                        )

                del raw_corr_oof, raw_corr_test
                gc.collect()


# C. Low-only variants
for w_pos in [1.00, 1.05, 1.08]:
    for w_low in [0.50, 0.75, 1.00]:
        for pcap in [4.50, 5.00]:
            for ncap in [1.00, 1.50]:
                signed_corr_oof = np.clip(
                    w_pos * pos_corr_oof - w_low * low_neg_oof,
                    -ncap,
                    pcap,
                )

                signed_corr_test = np.clip(
                    w_pos * pos_corr_test - w_low * low_neg_test,
                    -ncap,
                    pcap,
                )

                pred_oof = base_oof + signed_corr_oof
                pred_test = base_test + signed_corr_test

                consider_candidate(
                    key=f"11z2lite_lowonly_wpos{w_pos}_wlow{w_low}_pcap{pcap}_ncap{ncap}",
                    method="11z2lite_low_only",
                    pred_oof=pred_oof,
                    pred_test=pred_test,
                )

                del signed_corr_oof, signed_corr_test, pred_oof, pred_test
                gc.collect()


# ==================================================
# Final selection
# ==================================================

print("=" * 100)
print("Final selection")

candidate_df_11z2_lite = pd.DataFrame(candidate_rows)

print("Top by RMSE")
display(candidate_df_11z2_lite.sort_values("rmse").head(40))

print("Top by public_active_score")
display(candidate_df_11z2_lite.sort_values("public_active_score").head(40))

if best_active is not None:
    selected_row = best_active
    selected_oof = best_active_oof
    selected_test = best_active_test
    print("Selected from ACTIVE pool.")
else:
    selected_row = best_selected
    selected_oof = best_selected_oof
    selected_test = best_selected_test
    print("No strict active candidate. Selected fallback non-clone.")

if selected_row is None:
    raise RuntimeError("No valid non-clone candidate found. Stop and keep 11X raw.")

print("=" * 100)
print("Selected candidate")
display(pd.DataFrame([selected_row]))

print("Selected key        :", selected_row["key"])
print("Selected method     :", selected_row["method"])
print("Selected OOF        :", selected_row["rmse"])
print("Delta vs 11X raw    :", selected_row["delta_vs_11X_raw"])
print("Test MAE vs 11X raw :", selected_row["test_mae_vs_11x_raw"])
print("Public active score :", selected_row["public_active_score"])


# ==================================================
# Reports
# ==================================================

compare = train_raw[[ID_COL, TARGET]].copy()

compare["base_pred"] = base_oof
compare["w_pred"] = w_oof
compare["x_raw_pred"] = x_raw_oof
compare["x_robust_pred"] = x_rob_oof
compare["selected_pred"] = selected_oof

for col in ["base_pred", "w_pred", "x_raw_pred", "x_robust_pred", "selected_pred"]:
    err_col = col.replace("_pred", "_error")
    se_col = col.replace("_pred", "_se")
    compare[err_col] = compare[col] - compare[TARGET]
    compare[se_col] = compare[err_col] ** 2

compare["target_bin_5"] = pd.qcut(
    compare[TARGET],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop",
)

bin_compare_11z2_lite = (
    compare.groupby("target_bin_5", observed=False)
    .agg(
        count=(TARGET, "count"),
        target_mean=(TARGET, "mean"),

        base_rmse=("base_se", lambda x: np.sqrt(np.mean(x))),
        w_rmse=("w_se", lambda x: np.sqrt(np.mean(x))),
        x_raw_rmse=("x_raw_se", lambda x: np.sqrt(np.mean(x))),
        x_robust_rmse=("x_robust_se", lambda x: np.sqrt(np.mean(x))),
        selected_rmse=("selected_se", lambda x: np.sqrt(np.mean(x))),

        base_bias=("base_error", "mean"),
        w_bias=("w_error", "mean"),
        x_raw_bias=("x_raw_error", "mean"),
        x_robust_bias=("x_robust_error", "mean"),
        selected_bias=("selected_error", "mean"),
    )
    .reset_index()
)

bin_compare_11z2_lite["selected_minus_x_raw_rmse"] = (
    bin_compare_11z2_lite["selected_rmse"] - bin_compare_11z2_lite["x_raw_rmse"]
)

display(bin_compare_11z2_lite.sort_values("base_rmse", ascending=False))

test_dist_compare_11z2_lite = pd.DataFrame([
    {"name": "11O_base", **describe_pred_dist(base_test, "pred")},
    {"name": "11W", **describe_pred_dist(w_test, "pred")},
    {"name": "11X_raw", **describe_pred_dist(x_raw_test, "pred")},
    {"name": "11X_robust", **describe_pred_dist(x_rob_test, "pred")},
    {"name": "11Z2_lite_selected", **describe_pred_dist(selected_test, "pred")},
])

display(test_dist_compare_11z2_lite)


# ==================================================
# Save one file
# ==================================================

final_filename = "submission_11z2_lite_single_public_active.csv"

final_sub = sample_submission.copy()
final_sub[TARGET] = np.clip(selected_test, 0, None)

assert final_sub.shape[0] == sample_submission.shape[0]
assert list(final_sub.columns) == [ID_COL, TARGET]
assert final_sub[TARGET].isna().sum() == 0
assert np.isfinite(final_sub[TARGET]).all()
assert (final_sub[TARGET] < 0).sum() == 0
assert final_sub[ID_COL].equals(sample_submission[ID_COL])

final_sub.to_csv(final_filename, index=False)

print("=" * 100)
print("Saved:", final_filename)
display(final_sub.head())
display(final_sub[TARGET].describe().to_frame().T)

print("=" * 100)
print("11Z2 Lite Decision Guide")
print(f"11X raw OOF          : {x_raw_score:.6f}")
print(f"Selected OOF         : {selected_row['rmse']:.6f}")
print(f"Delta vs 11X raw     : {selected_row['rmse'] - x_raw_score:.6f}")
print(f"Test MAE vs 11X raw  : {selected_row['test_mae_vs_11x_raw']:.6f}")
print(f"File                 : {final_filename}")

if selected_row["test_mae_vs_11x_raw"] < 0.010:
    print("WARNING: still too similar to 11X raw. Do NOT submit.")
elif selected_row["rmse"] < x_raw_score:
    print("Decision: OK. Active on test and improves OOF.")
elif selected_row["rmse"] <= x_raw_score + 0.004:
    print("Decision: public probe only. Active on test, OOF near 11X.")
else:
    print("Decision: risky. OOF worse than 11X.")

In [ ]:
from google.colab import files
import os

filename = "submission_11z2_lite_single_public_active.csv"

if os.path.exists(filename):
    print("Downloading:", filename)
    files.download(filename)
else:
    raise FileNotFoundError(filename)

## SECTION 12 - Final Reporting Assets for Research Questions

This section builds the evidence assets needed for final written answers. The final answers should be written after inspecting these outputs, not guessed.

Assets included:

- Final experiment log
- OOF prediction vs actual
- Residual distribution
- Residual by target range
- Feature importance tables for CatBoost, LightGBM, and XGBoost
- Fair feature-engineering impact comparison
- Geographic/ecosystem summaries
- Research question answer template

In [ ]:
# Final experiment table including ensemble
ensemble_log_row = {
    "model_name": f"Final Ensemble - {best_ensemble['ensemble_name']}",
    "features": "all engineered features",
    "mean_cv_rmse": float(best_ensemble["oof_rmse"]),
    "std_cv_rmse": np.nan,
    "fold_scores": None,
    "notes": f"Weights: {best_weights}",
    "mean_train_rmse": np.nan,
    "train_valid_gap": np.nan,
}
final_experiment_log = pd.concat([experiment_log_df, pd.DataFrame([ensemble_log_row])], ignore_index=True).sort_values("mean_cv_rmse")
print("Final experiment log:")
display(final_experiment_log)

# Residual assets
residuals = y.values - final_oof_pred
residual_df = pd.DataFrame({
    TARGET: y.values,
    "oof_prediction": final_oof_pred,
    "oof_prediction_clipped": final_oof_pred_clipped,
    "residual": residuals,
    "abs_residual": np.abs(residuals),
})
residual_df["target_bin"] = pd.qcut(residual_df[TARGET], q=5, duplicates="drop")
print("Residual summary:")
display(residual_df.describe())
print("Residual by target bin:")
display(residual_df.groupby("target_bin", observed=False)["abs_residual"].agg(["count", "mean", "median", "std", "max"]))

plt.figure(figsize=(6, 6))
plt.scatter(residual_df[TARGET], residual_df["oof_prediction"], alpha=0.4)
lims = [min(residual_df[TARGET].min(), residual_df["oof_prediction"].min()), max(residual_df[TARGET].max(), residual_df["oof_prediction"].max())]
plt.plot(lims, lims)
plt.xlabel("Actual")
plt.ylabel("OOF Prediction")
plt.title("OOF Prediction vs Actual")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(residual_df["residual"], bins=50)
plt.xlabel("Residual = actual - prediction")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.scatter(residual_df[TARGET], residual_df["residual"], alpha=0.4)
plt.axhline(0)
plt.xlabel("Actual Target")
plt.ylabel("Residual")
plt.title("Residual vs Actual Target")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance summaries
def summarize_importance(fi_df: pd.DataFrame, top_n: int = 30) -> pd.DataFrame:
    if fi_df is None or fi_df.empty:
        return pd.DataFrame()
    return (
        fi_df.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
        .head(top_n)
    )

importance_assets = {}
for name in ["catboost", "lightgbm", "xgboost"]:
    fi = summarize_importance(model_outputs[name].get("feature_importance"), top_n=30)
    importance_assets[name] = fi
    print(f"Top feature importance - {name}:")
    display(fi)
    if not fi.empty:
        plt.figure(figsize=(8, 6))
        plt.barh(fi["feature"][::-1], fi["importance"][::-1])
        plt.xlabel("Mean Importance")
        plt.title(f"Top Feature Importance - {name}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Fair feature engineering impact comparison: same model family, raw vs engineered features.
# Uses a lightweight LightGBM setting so it does not dominate runtime.

def quick_lgbm_cv_score(X: pd.DataFrame, X_test_unused: pd.DataFrame, y: pd.Series, folds, label: str):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    oof = np.zeros(len(y))
    scores = []
    dummy_test = X_test_unused.copy()
    for fold, (tr_idx, val_idx) in enumerate(folds, 1):
        X_tr_raw, X_val_raw = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_tr, X_val, _, _ = prepare_encoded_fold(X_tr_raw, X_val_raw, dummy_test, cat_cols, num_cols)
        model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=300 if FAST_MODE else 800,
            learning_rate=0.04,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            verbosity=-1,
        )
        model = fit_lgbm_compatible(model, X_tr, y_tr, X_val, y_val)
        val_pred = model.predict(X_val)
        oof[val_idx] = val_pred
        scores.append(rmse(y_val, val_pred))
    return {"label": label, "oof_rmse": rmse(y, oof), "mean_fold_rmse": float(np.mean(scores)), "std_fold_rmse": float(np.std(scores))}

raw_lgb_score = quick_lgbm_cv_score(X_base.reset_index(drop=True), X_test_base.reset_index(drop=True), y.reset_index(drop=True), folds, "LightGBM raw features")
fe_lgb_score = quick_lgbm_cv_score(X_fe.reset_index(drop=True), X_test_fe.reset_index(drop=True), y.reset_index(drop=True), folds, "LightGBM engineered features")
fe_impact_df = pd.DataFrame([raw_lgb_score, fe_lgb_score])
fe_impact_df["improvement_vs_raw"] = raw_lgb_score["oof_rmse"] - fe_impact_df["oof_rmse"]
print("Fair feature engineering impact comparison:")
display(fe_impact_df)

### Research Question Answer Template

Use the following template after reviewing the generated outputs:

1. **Urgency of predicting soil organic content**  
   Evidence to use: business/scientific context, target distribution, and the potential benefit of rapid estimation using only the official competition dataset.

2. **Overfit or underfit analysis**  
   Evidence to use: mean train RMSE, mean validation RMSE, train-validation gap, fold stability, and residual plots.

3. **Organic content distribution across geographic zones**  
   Evidence to use: `target_by_geo_zone_macro`, `target_by_geo_zone_meso`, and `target_by_geo_zone_micro`.

4. **Correlation/pattern across geographic hierarchy**  
   Evidence to use: geographic grouped means, medians, counts, and extreme-zone comparisons.

5. **Ecosystem averages under low acidity and low CEC**  
   Evidence to use: `low_acidity_low_cec` filtered summaries by biome, land cover, and macro zone.

6. **Outliers from land cover and geography combinations**  
   Evidence to use: `land_cover_geo_outliers`, especially groups with enough sample count.

7. **Multicollinearity and high-correlation features**  
   Evidence to use: `high_corr_pairs_abs_gt_0_8`, correlation heatmap, and model type discussion.

8. **Feature engineering explanation**  
   Evidence to use: engineered feature list, fair raw-vs-engineered feature comparison, and feature importance.

9. **Model choice and performance**  
   Evidence to use: final experiment log, CatBoost/LightGBM/XGBoost CV RMSE, and final ensemble RMSE.

10. **Competition-rule compliance: no external data**  
   The modeling pipeline and final submission use only `train.csv`, `test.csv`, and `sample_submission.csv`. No external dataset, web data, geospatial source, climate source, remote-sensing source, or additional lookup table is downloaded, merged, or used for training/submission. If the written answer discusses external data, clearly state that it is not used because the competition rules prohibit it.


In [ ]:
# Save useful outputs for reporting
Path("outputs").mkdir(exist_ok=True)
final_experiment_log.to_csv("outputs/final_experiment_log.csv", index=False)
ensemble_results.to_csv("outputs/ensemble_results.csv", index=False)
residual_df.to_csv("outputs/oof_residuals.csv", index=False)
fe_impact_df.to_csv("outputs/feature_engineering_impact.csv", index=False)

for key, value in research_assets.items():
    if isinstance(value, pd.DataFrame):
        safe_key = key.replace("/", "_").replace(" ", "_")
        value.to_csv(f"outputs/{safe_key}.csv", index=False)

for name, fi in importance_assets.items():
    if isinstance(fi, pd.DataFrame) and not fi.empty:
        fi.to_csv(f"outputs/feature_importance_{name}.csv", index=False)

print("Saved reporting assets to outputs/ folder")
print("Final notebook workflow complete. Ready to submit submission.csv to Kaggle.")